In [1]:
"""CELL 1 — UC-RCF-NBM configuration and frozen experiment contract.

Paste this complete file into the first cell of a new notebook.  The cell creates
an isolated output tree for Experiment 2 and freezes every choice that may affect
the subsequent nested CARE evaluation.  It does not read event labels or sensor
measurements.
"""

from __future__ import annotations

import hashlib
import importlib.metadata
import json
import os
import platform
import random
import sys
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any


# Keep numerical libraries reproducible and avoid thread oversubscription.  These
# values are set before importing NumPy; setdefault preserves an explicit user
# choice made before this cell runs.
for _variable in (
    "OMP_NUM_THREADS",
    "MKL_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
):
    os.environ.setdefault(_variable, "1")

import numpy as np
import pandas as pd


# =============================================================================
# 1. Paths and experiment identity
# =============================================================================

METHOD_NAME = "UC-RCF-NBM"
METHOD_LONG_NAME = (
    "Uncertainty-Calibrated Robust Cross-Fitted Normal-Behaviour Model"
)
METHOD_VERSION = "1.0.0-pre-outer-evaluation"
EXPERIMENT_ID = "uc_rcf_nbm_care_v6_exp01"

PROJECT_ROOT = Path.cwd().resolve()

# Environment overrides make the cell portable without changing the scientific
# contract.  On Umar's workstation the defaults point to the existing CARE data
# and the Implementation output tree.
CARE_DATA_ROOT = Path(
    os.environ.get(
        "UC_RCF_NBM_DATA_ROOT",
        r"F:\Umar-Wisal-Work\Wisal-Bearings-Work\CARE_To_Compare\CARE_To_Compare",
    )
).resolve()

OUTPUT_ROOT = Path(
    os.environ.get(
        "UC_RCF_NBM_OUTPUT_ROOT",
        str(PROJECT_ROOT / "outputs" / EXPERIMENT_ID),
    )
).resolve()

CONTRACT_DIR = OUTPUT_ROOT / "00_contract"
INVENTORY_DIR = OUTPUT_ROOT / "01_inventory"
QUALITY_DIR = OUTPUT_ROOT / "02_quality"
CACHE_DIR = OUTPUT_ROOT / "03_cache"
MODEL_DIR = OUTPUT_ROOT / "04_models"
PREDICTION_DIR = OUTPUT_ROOT / "05_outer_predictions"
TABLE_DIR = OUTPUT_ROOT / "06_tables"
FIGURE_DIR = OUTPUT_ROOT / "07_figures"
LOG_DIR = OUTPUT_ROOT / "08_logs"

OUTPUT_DIRECTORIES = (
    OUTPUT_ROOT,
    CONTRACT_DIR,
    INVENTORY_DIR,
    QUALITY_DIR,
    CACHE_DIR,
    MODEL_DIR,
    PREDICTION_DIR,
    TABLE_DIR,
    FIGURE_DIR,
    LOG_DIR,
)


# =============================================================================
# 2. Frozen scientific configuration
# =============================================================================

@dataclass(frozen=True)
class DatasetContract:
    """Expected properties of the current CARE-to-Compare release."""

    name: str = "Wind Turbine SCADA Data For Early Fault Detection"
    release: str = "v6"
    zenodo_doi: str = "10.5281/zenodo.15846963"
    sampling_minutes: int = 10
    farms: tuple[str, ...] = ("Wind Farm A", "Wind Farm B", "Wind Farm C")
    expected_cases_by_farm: tuple[int, ...] = (22, 15, 58)
    expected_anomaly_by_farm: tuple[int, ...] = (12, 6, 27)
    expected_normal_by_farm: tuple[int, ...] = (10, 9, 31)
    expected_total_cases: int = 95
    expected_anomaly_cases: int = 45
    expected_normal_cases: int = 50
    expected_assets: int = 36
    metadata_columns: tuple[str, ...] = (
        "id",
        "time_stamp",
        "asset_id",
        "train_test",
        "status_type_id",
    )
    source_train_labels: tuple[str, ...] = ("train", "training", "0")
    source_prediction_labels: tuple[str, ...] = (
        "test",
        "prediction",
        "predict",
        "1",
    )
    normal_status_ids: tuple[int, ...] = (0, 2)


@dataclass(frozen=True)
class QualityContract:
    """Primary signal-quality and temporal-continuity rules."""

    primary_statistics: tuple[str, ...] = ("avg",)
    sensitivity_statistics: tuple[str, ...] = ("avg", "std")
    standard_deviation_primary: bool = False
    maximum_gap_minutes: int = 10
    minimum_training_rows: int = 2_000
    minimum_sensor_availability: float = 0.70
    minimum_segment_steps: int = 144
    zero_coded_missingness_policy: str = (
        "metadata-guided plus training-only constant-zero-run audit"
    )
    constant_zero_run_min_steps: int = 144
    counter_policy: str = "within-segment first difference; negative resets missing"
    angle_policy: str = "sine-cosine encoding"
    driver_imputation: str = "training-normal median"
    target_imputation: str = "masked from fitting and scoring"
    temporal_operations_restart_at_gaps: bool = True


@dataclass(frozen=True)
class MeanModelContract:
    """Operating-conditioned mean response model."""

    model: str = "multi-output ridge normal-behaviour model"
    basis: str = "linear + quadratic + wind cubic + restricted physical interactions"
    driver_description_patterns: tuple[str, ...] = (
        "wind speed",
        "active power",
        "reactive power",
        "rotor speed",
        "generator speed",
        "generator acceleration",
        "torque",
        "ambient temperature",
    )
    maximum_core_interaction_drivers: int = 5
    ridge_grid: tuple[float, ...] = (
        1.0e-4,
        1.0e-2,
        1.0,
        1.0e2,
        1.0e4,
    )
    crossfit_folds: int = 5
    temporal_block_steps: int = 432
    embargo_steps: int = 144
    penalty_objective: str = "masked out-of-fold mean squared error"
    modelability_r2_grid: tuple[float, ...] = (0.0, 0.10, 0.30)
    fallback_minimum_r2: float = 0.0
    residual_cap: float = 10.0


@dataclass(frozen=True)
class UncertaintyContract:
    """Aleatoric, support, and blocked conformal uncertainty rules."""

    aleatoric_model: str = "multi-output ridge on log squared cross-fitted residual"
    scale_ridge_grid: tuple[float, ...] = (
        1.0e-4,
        1.0e-2,
        1.0,
        1.0e2,
        1.0e4,
    )
    scale_target_epsilon: float = 1.0e-6
    scale_crossfit_folds: int = 5
    support_uncertainty: str = "regularized nonlinear-design leverage"
    support_reference_scaling: str = "training cross-fitted median and IQR"
    support_nonnegative_for_total_scale: bool = True
    total_scale_rule: str = "aleatoric_scale * sqrt(1 + positive_support_uncertainty)"
    conformal_method: str = "embargoed blocked cross-conformal calibration"
    conformal_coverage_grid: tuple[float, ...] = (0.95, 0.98, 0.99)
    conformal_block_steps: int = 144
    minimum_conformal_blocks: int = 30
    finite_sample_quantile_correction: bool = True
    exact_exchangeability_claimed: bool = False
    primary_uncertainty_outputs: tuple[str, ...] = (
        "aleatoric_scale",
        "support_uncertainty",
        "total_predictive_scale",
        "conformal_interval",
        "interval_exceedance",
    )


@dataclass(frozen=True)
class AlarmContract:
    """Multichannel health evidence and sequential decision rule."""

    sensor_weighting: str = "cross-fitted modelability and calibration quality"
    health_indicator: str = "weighted mean of positive conformal interval excess"
    evidence_head_grid: tuple[str, ...] = (
        "all_modellable",
        "temperature_modellable",
    )
    smoothing_steps_grid: tuple[int, ...] = (36, 72, 144)
    smoothing: str = "causal rolling median within continuous segments"
    health_threshold_quantile_grid: tuple[float, ...] = (
        0.95,
        0.98,
        0.99,
        0.995,
    )
    timestamp_rule: str = "smoothed health indicator strictly exceeds frozen threshold"
    post_alarm_latching: bool = False
    criticality_feedback_into_predictions: bool = False


@dataclass(frozen=True)
class EvaluationContract:
    """Nested asset-grouped validation and official CARE scoring."""

    outer_strategy: str = "leave-one-asset-out"
    outer_group_column: str = "asset_id"
    inner_strategy: str = "grouped folds by asset on outer-development assets only"
    inner_group_column: str = "asset_id"
    inner_splits: int = 5
    selection_objective: str = "official CARE"
    deterministic_tie_break: tuple[str, ...] = (
        "higher normal accuracy",
        "higher event F0.5",
        "narrower mean normal interval",
        "lexicographic configuration",
    )
    outer_labels_available_during_selection: bool = False
    outer_predictions_generated_once: bool = True
    legacy_transformer_test_is_confirmation: bool = False
    primary_endpoint: str = "pooled outer-fold official CARE"
    secondary_endpoints: tuple[str, ...] = (
        "coverage F0.5",
        "normal accuracy",
        "event reliability F0.5",
        "earliness weighted score",
        "event sensitivity",
        "event false-alarm count",
        "median lead time",
        "normal prediction-interval coverage",
        "normal interval width",
        "coverage error",
        "interval score",
        "selective risk",
    )
    bootstrap_unit: str = "asset"
    bootstrap_replicates: int = 10_000
    bootstrap_seed: int = 42
    multiple_comparison_adjustment: str = "Holm"


@dataclass(frozen=True)
class CareContract:
    """Published CARE settings retained for benchmark comparability."""

    beta: float = 0.5
    component_weights: tuple[float, ...] = (1.0, 1.0, 1.0, 2.0)
    criticality_threshold: int = 72
    criticality_update_state: str = "normal/actionable timestamps"
    raw_timestamp_predictions_only: bool = True
    farm_a_anomaly_prediction_status_policy: str = (
        "ignore status mask as prescribed by CARE v6 dataset notes"
    )
    farm_a_normal_prediction_status_policy: str = "normal-status timestamps"
    farms_b_c_prediction_status_policy: str = "normal-status timestamps"
    official_reference_unit_tests_required: bool = True


@dataclass(frozen=True)
class ReproducibilityContract:
    seed: int = 42
    deterministic_algorithms: bool = True
    float_dtype: str = "float64 for fitting; float32 permitted for caches"
    source_files_read_only: bool = True
    contract_change_requires_new_experiment_id: bool = True
    save_outer_predictions_before_label_scoring: bool = True


DATASET = DatasetContract()
QUALITY = QualityContract()
MEAN_MODEL = MeanModelContract()
UNCERTAINTY = UncertaintyContract()
ALARM = AlarmContract()
EVALUATION = EvaluationContract()
CARE = CareContract()
REPRODUCIBILITY = ReproducibilityContract()


# Predictor and calibration code may access sensor values, timestamps, partition
# identity, and asset identity.  The following fields are forbidden until an
# outer prediction has been frozen.  status_type_id is used only to select normal
# fitting rows and by the final CARE mask; it is never a predictor input.
FORBIDDEN_PREDICTOR_FIELDS = (
    "event_label",
    "event_label_raw",
    "is_anomaly",
    "event_start",
    "event_start_id",
    "event_end",
    "event_end_id",
    "event_description",
    "fault_type",
    "care_ground_truth",
)

PERMITTED_STATUS_USES = (
    "normal-training row eligibility",
    "final official CARE actionable mask",
)

LABEL_ACCESS_RULES = {
    "case_preprocessing": "event/status outcome fields forbidden as predictors",
    "mean_model_fit": "normal training partition only; no prediction event labels",
    "uncertainty_model_fit": "cross-fitted normal training residuals only",
    "conformal_calibration": "cross-fitted normal training nonconformity only",
    "inner_selection": "labels from inner-development assets only",
    "outer_inference": "outer event labels and boundaries unopened",
    "outer_scoring": "labels opened only after outer predictions are saved and hashed",
}


# =============================================================================
# 3. Validation, serialization, and reproducibility helpers
# =============================================================================

def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def json_default(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        numeric = float(value)
        return numeric if np.isfinite(numeric) else None
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (pd.Timestamp, datetime)):
        return value.isoformat()
    if value is pd.NA or value is pd.NaT:
        return None
    raise TypeError(f"{type(value).__name__} is not JSON serializable")


def canonical_json(data: Any) -> str:
    return json.dumps(
        data,
        default=json_default,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )


def sha256_json(data: Any) -> str:
    return hashlib.sha256(canonical_json(data).encode("utf-8")).hexdigest()


def save_json(data: Any, output_path: Path) -> None:
    """Atomically save human-readable JSON."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = output_path.with_suffix(output_path.suffix + ".tmp")
    temporary_path.write_text(
        json.dumps(
            data,
            default=json_default,
            ensure_ascii=False,
            sort_keys=True,
            indent=2,
        )
        + "\n",
        encoding="utf-8",
    )
    temporary_path.replace(output_path)


def package_version(distribution_name: str) -> str | None:
    try:
        return importlib.metadata.version(distribution_name)
    except importlib.metadata.PackageNotFoundError:
        return None


def set_global_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch

        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        if REPRODUCIBILITY.deterministic_algorithms:
            torch.use_deterministic_algorithms(True, warn_only=True)
            if torch.backends.cudnn.is_available():
                torch.backends.cudnn.deterministic = True
                torch.backends.cudnn.benchmark = False
    except ImportError:
        pass


def validate_contract() -> None:
    if sum(DATASET.expected_cases_by_farm) != DATASET.expected_total_cases:
        raise ValueError("Farm case counts do not equal expected_total_cases.")
    if sum(DATASET.expected_anomaly_by_farm) != DATASET.expected_anomaly_cases:
        raise ValueError("Farm anomaly counts do not equal expected_anomaly_cases.")
    if sum(DATASET.expected_normal_by_farm) != DATASET.expected_normal_cases:
        raise ValueError("Farm normal counts do not equal expected_normal_cases.")
    if DATASET.expected_anomaly_cases + DATASET.expected_normal_cases != DATASET.expected_total_cases:
        raise ValueError("Anomaly and normal totals do not equal total cases.")
    if len(DATASET.farms) != len(DATASET.expected_cases_by_farm):
        raise ValueError("Farm names and expected counts have different lengths.")
    if QUALITY.primary_statistics != ("avg",):
        raise ValueError("Primary analysis must remain average-signal only.")
    if QUALITY.maximum_gap_minutes != DATASET.sampling_minutes:
        raise ValueError("A continuity gap must begin after one missing sample.")
    if not 0.0 < QUALITY.minimum_sensor_availability <= 1.0:
        raise ValueError("minimum_sensor_availability must lie in (0, 1].")
    if MEAN_MODEL.crossfit_folds < 3 or UNCERTAINTY.scale_crossfit_folds < 3:
        raise ValueError("At least three blocked cross-fitting folds are required.")
    if MEAN_MODEL.embargo_steps <= 0:
        raise ValueError("Temporal cross-fitting requires a positive embargo.")
    if any(not 0.0 < value < 1.0 for value in UNCERTAINTY.conformal_coverage_grid):
        raise ValueError("Conformal coverage levels must lie in (0, 1).")
    if any(not 0.0 < value < 1.0 for value in ALARM.health_threshold_quantile_grid):
        raise ValueError("Health threshold quantiles must lie in (0, 1).")
    if ALARM.post_alarm_latching or ALARM.criticality_feedback_into_predictions:
        raise ValueError("Post-alarm latching and CARE feedback are prohibited.")
    if not CARE.raw_timestamp_predictions_only:
        raise ValueError("Official CARE must receive raw timestamp predictions.")
    if EVALUATION.outer_group_column != "asset_id":
        raise ValueError("Outer evaluation must remain grouped by asset_id.")
    if EVALUATION.outer_labels_available_during_selection:
        raise ValueError("Outer labels cannot be available during selection.")
    if EVALUATION.legacy_transformer_test_is_confirmation:
        raise ValueError("The opened legacy Transformer test cannot be confirmatory.")


validate_contract()
set_global_seed(REPRODUCIBILITY.seed)

if not CARE_DATA_ROOT.is_dir():
    raise FileNotFoundError(
        "CARE v6 dataset root not found. Set UC_RCF_NBM_DATA_ROOT before running "
        f"Cell 1. Current path:\n{CARE_DATA_ROOT}"
    )

for _directory in OUTPUT_DIRECTORIES:
    _directory.mkdir(parents=True, exist_ok=True)


# =============================================================================
# 4. Freeze the scientific contract
# =============================================================================

SCIENTIFIC_CONTRACT = {
    "method": {
        "name": METHOD_NAME,
        "long_name": METHOD_LONG_NAME,
        "version": METHOD_VERSION,
        "experiment_id": EXPERIMENT_ID,
    },
    "dataset": asdict(DATASET),
    "quality": asdict(QUALITY),
    "mean_model": asdict(MEAN_MODEL),
    "uncertainty": asdict(UNCERTAINTY),
    "alarm": asdict(ALARM),
    "evaluation": asdict(EVALUATION),
    "care": asdict(CARE),
    "reproducibility": asdict(REPRODUCIBILITY),
    "forbidden_predictor_fields": FORBIDDEN_PREDICTOR_FIELDS,
    "permitted_status_uses": PERMITTED_STATUS_USES,
    "label_access_rules": LABEL_ACCESS_RULES,
}

CONTRACT_SHA256 = sha256_json(SCIENTIFIC_CONTRACT)
CONTRACT_PATH = CONTRACT_DIR / "scientific_experiment_contract.json"

if CONTRACT_PATH.exists():
    existing_contract = json.loads(CONTRACT_PATH.read_text(encoding="utf-8"))
    existing_hash = str(existing_contract.get("contract_sha256", ""))
    if existing_hash != CONTRACT_SHA256:
        raise RuntimeError(
            "A different scientific contract already exists for this experiment ID. "
            "Do not overwrite it. Change EXPERIMENT_ID to start a genuinely new "
            f"experiment. Existing={existing_hash or '<missing>'}, "
            f"current={CONTRACT_SHA256}."
        )
    CONTRACT_STATE = "existing identical contract verified"
else:
    save_json(
        {
            "contract_sha256": CONTRACT_SHA256,
            "created_at_utc": utc_now(),
            "scientific_contract": SCIENTIFIC_CONTRACT,
        },
        CONTRACT_PATH,
    )
    CONTRACT_STATE = "new contract frozen"


ENVIRONMENT_SNAPSHOT = {
    "recorded_at_utc": utc_now(),
    "experiment_id": EXPERIMENT_ID,
    "contract_sha256": CONTRACT_SHA256,
    "project_root": str(PROJECT_ROOT),
    "dataset_root": str(CARE_DATA_ROOT),
    "output_root": str(OUTPUT_ROOT),
    "python": sys.version,
    "python_executable": sys.executable,
    "platform": platform.platform(),
    "packages": {
        name: package_version(name)
        for name in (
            "numpy",
            "pandas",
            "scipy",
            "scikit-learn",
            "matplotlib",
            "seaborn",
            "torch",
        )
    },
}
save_json(ENVIRONMENT_SNAPSHOT, CONTRACT_DIR / "environment_snapshot.json")


# =============================================================================
# 5. Cell summary
# =============================================================================

print("=" * 92)
print("UC-RCF-NBM EXPERIMENT INITIALIZED — SCIENTIFIC CONTRACT FROZEN")
print("=" * 92)
print(f"Method                         : {METHOD_LONG_NAME}")
print(f"Experiment ID                  : {EXPERIMENT_ID}")
print(f"CARE dataset release           : {DATASET.release} ({DATASET.zenodo_doi})")
print(
    "Expected cases                 : "
    f"{DATASET.expected_total_cases} "
    f"({DATASET.expected_anomaly_cases} anomaly / "
    f"{DATASET.expected_normal_cases} normal)"
)
print(f"Expected assets                : {DATASET.expected_assets}")
print(f"Primary feature statistics     : {QUALITY.primary_statistics}")
print(f"Mean model                     : {MEAN_MODEL.model}")
print(f"Temporal cross-fitting         : {MEAN_MODEL.crossfit_folds} folds, "
      f"{MEAN_MODEL.temporal_block_steps} steps/block, "
      f"{MEAN_MODEL.embargo_steps} steps embargo")
print(f"Aleatoric uncertainty          : {UNCERTAINTY.aleatoric_model}")
print(f"Support uncertainty            : {UNCERTAINTY.support_uncertainty}")
print(f"Conformal calibration          : {UNCERTAINTY.conformal_method}")
print(f"Outer evaluation               : {EVALUATION.outer_strategy}")
print(f"Primary endpoint               : {EVALUATION.primary_endpoint}")
print(f"Post-alarm latching            : {ALARM.post_alarm_latching}")
print(f"Legacy Transformer confirmation: {EVALUATION.legacy_transformer_test_is_confirmation}")
print(f"Contract state                 : {CONTRACT_STATE}")
print(f"Contract SHA-256               : {CONTRACT_SHA256}")
print(f"Dataset root                   : {CARE_DATA_ROOT}")
print(f"Output root                    : {OUTPUT_ROOT}")
print("Source data modified           : No — read-only contract")
print("=" * 92)
print("CELL 1 COMPLETED SUCCESSFULLY — UC-RCF-NBM EXPERIMENT CONTRACT LOCKED")



UC-RCF-NBM EXPERIMENT INITIALIZED — SCIENTIFIC CONTRACT FROZEN
Method                         : Uncertainty-Calibrated Robust Cross-Fitted Normal-Behaviour Model
Experiment ID                  : uc_rcf_nbm_care_v6_exp01
CARE dataset release           : v6 (10.5281/zenodo.15846963)
Expected cases                 : 95 (45 anomaly / 50 normal)
Expected assets                : 36
Primary feature statistics     : ('avg',)
Mean model                     : multi-output ridge normal-behaviour model
Temporal cross-fitting         : 5 folds, 432 steps/block, 144 steps embargo
Aleatoric uncertainty          : multi-output ridge on log squared cross-fitted residual
Support uncertainty            : regularized nonlinear-design leverage
Conformal calibration          : embargoed blocked cross-conformal calibration
Outer evaluation               : leave-one-asset-out
Primary endpoint               : pooled outer-fold official CARE
Post-alarm latching            : False
Legacy Transformer confirmation

In [2]:
"""CELL 2 — CARE v6 inventory, schema, taxonomy, and leakage-safe quality audit.

Paste this complete file into the second cell of the UC-RCF-NBM notebook and
run it only after Cell 1 has printed its successful contract-lock banner.

This cell deliberately fits no model and chooses no hyperparameter.  It builds
a safe case registry, places outcome-only metadata in a separate lockbox,
constructs a metadata-grounded signal taxonomy, and audits signal quality using
only normal-status rows in each source training partition.  The audit is
diagnostic: channel eligibility is recomputed inside every outer fold later.
"""

from __future__ import annotations

import hashlib
import json
import re
from collections import defaultdict
from functools import lru_cache
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print


# =============================================================================
# 0. Cell-1 compatibility and leakage guards
# =============================================================================

EXPECTED_CELL1_CONTRACT_SHA256 = (
    "827641aecd8e807193ad193d64c274319756092faa53bdb5084f310f62041f49"
)

_required_cell1_names = (
    "CARE_DATA_ROOT",
    "OUTPUT_ROOT",
    "INVENTORY_DIR",
    "QUALITY_DIR",
    "CONTRACT_SHA256",
    "DATASET",
    "QUALITY",
    "MEAN_MODEL",
    "FORBIDDEN_PREDICTOR_FIELDS",
    "save_json",
    "sha256_json",
    "utc_now",
)
_missing_cell1_names = [name for name in _required_cell1_names if name not in globals()]
if _missing_cell1_names:
    raise RuntimeError(
        "Run UC-RCF-NBM Cell 1 before Cell 2. Missing notebook objects: "
        + ", ".join(_missing_cell1_names)
    )

if CONTRACT_SHA256 != EXPECTED_CELL1_CONTRACT_SHA256:
    raise RuntimeError(
        "Cell 1 does not match the frozen UC-RCF-NBM experiment contract. "
        f"Expected {EXPECTED_CELL1_CONTRACT_SHA256}, received {CONTRACT_SHA256}. "
        "Do not mix cells from different experiments."
    )

if not Path(CARE_DATA_ROOT).is_dir():
    raise FileNotFoundError(f"CARE v6 dataset root not found:\n{CARE_DATA_ROOT}")

for _directory in (Path(INVENTORY_DIR), Path(QUALITY_DIR)):
    _directory.mkdir(parents=True, exist_ok=True)


# Labels and event boundaries are allowed here only to validate the benchmark
# inventory and construct the outcome lockbox.  They never enter CASE_REGISTRY,
# CARE_FEATURE_REGISTRY, FARM_SCHEMAS, or the quality-audit functions.
SAFE_CASE_COLUMNS = (
    "case_key",
    "farm",
    "asset_id",          # Canonical farm-qualified asset identifier.
    "source_asset_id",   # Identifier written in the CARE source file.
    "event_id",          # Case/file identifier only; never a predictor.
    "file_path",
    "relative_file_path",
    "size_bytes",
)

OUTCOME_ONLY_COLUMNS = (
    "event_label_raw",
    "is_anomaly",
    "event_start",
    "event_start_id",
    "event_end",
    "event_end_id",
    "event_description",
)

if set(FORBIDDEN_PREDICTOR_FIELDS) - set(OUTCOME_ONLY_COLUMNS) - {
    "event_label",
    "care_ground_truth",
    "fault_type",
}:
    raise RuntimeError("Cell 1 and Cell 2 outcome-field definitions are inconsistent.")


# =============================================================================
# 1. Robust CARE readers and canonical names
# =============================================================================

CSV_ENCODINGS = ("utf-8-sig", "utf-8", "cp1252", "latin-1")
CSV_SEPARATORS = (",", ";", "\t", "|")
EVENT_FILE_PATTERN = re.compile(r"^\d+$")
AUDIT_CHUNK_ROWS = 20_000

COLUMN_ALIASES = {
    "timestamp": "time_stamp",
    "time": "time_stamp",
    "datetime": "time_stamp",
    "asset": "asset_id",
    "assetid": "asset_id",
    "turbine_id": "asset_id",
    "wt_id": "asset_id",
    "status_id": "status_type_id",
    "status_type": "status_type_id",
    "train_or_test": "train_test",
    "split": "train_test",
    "eventid": "event_id",
    "event_start_index": "event_start_id",
    "event_end_index": "event_end_id",
    "start_id": "event_start_id",
    "end_id": "event_end_id",
    "event_start_time": "event_start",
    "event_end_time": "event_end",
    "start_time": "event_start",
    "end_time": "event_end",
    "sensor_name": "feature_name",
    "signal_name": "feature_name",
    "column_name": "feature_name",
    "featurename": "feature_name",
    "featuredescription": "feature_description",
    "description": "feature_description",
}


def standardize_column_name(raw: Any) -> str:
    """Convert CARE column aliases to stable snake_case names."""
    text = str(raw).strip().lstrip("\ufeff")
    text = re.sub(r"(?<=[a-z0-9])(?=[A-Z])", "_", text)
    text = re.sub(r"[^A-Za-z0-9]+", "_", text).strip("_").lower()
    return COLUMN_ALIASES.get(text, text)


def find_column(frame: pd.DataFrame, candidates: Iterable[str]) -> str | None:
    for candidate in candidates:
        canonical = standardize_column_name(candidate)
        if canonical in frame.columns:
            return canonical
    return None


def normalize_identifier(value: Any) -> str:
    """Represent numeric and textual asset identifiers without float artifacts."""
    if pd.isna(value):
        raise ValueError("Missing asset identifier")
    text = str(value).strip()
    try:
        numeric = float(text)
        if np.isfinite(numeric) and numeric.is_integer():
            return str(int(numeric))
    except ValueError:
        pass
    if not text:
        raise ValueError("Empty asset identifier")
    return text


def parse_care_timestamps(values: pd.Series) -> pd.Series:
    cleaned = values.astype("string").str.strip().replace(
        {"": pd.NA, "None": pd.NA, "NULL": pd.NA, "nan": pd.NA, "NaT": pd.NA}
    )
    try:
        parsed = pd.to_datetime(
            cleaned,
            format="mixed",
            dayfirst=True,
            errors="coerce",
            utc=True,
        )
    except (TypeError, ValueError):
        parsed = pd.to_datetime(cleaned, dayfirst=True, errors="coerce", utc=True)
    return parsed.dt.tz_convert(None)


@lru_cache(maxsize=None)
def read_care_header(file_path: Path) -> tuple[tuple[str, ...], tuple[str, ...], str, str]:
    """Detect an event CSV's encoding and separator using its header only."""
    file_path = Path(file_path)
    best: tuple[int, int, list[str], list[str], str, str] | None = None
    metadata = set(DATASET.metadata_columns)
    for encoding in CSV_ENCODINGS:
        for separator in CSV_SEPARATORS:
            try:
                header = pd.read_csv(
                    file_path,
                    encoding=encoding,
                    sep=separator,
                    nrows=0,
                    engine="python",
                )
            except Exception:
                continue
            raw = [str(column).strip().lstrip("\ufeff") for column in header.columns]
            canonical = [standardize_column_name(column) for column in raw]
            if len(canonical) < 2 or len(canonical) != len(set(canonical)):
                continue
            candidate = (
                len(metadata & set(canonical)),
                len(canonical),
                raw,
                canonical,
                encoding,
                separator,
            )
            if best is None or candidate[:2] > best[:2]:
                best = candidate
    if best is None or best[0] < len(metadata):
        raise RuntimeError(
            "Could not identify all required CARE event columns in:\n"
            f"{file_path}\nRequired: {sorted(metadata)}"
        )
    return tuple(best[2]), tuple(best[3]), best[4], best[5]


def read_care_chunks(
    file_path: Path,
    usecols: Iterable[str],
    chunksize: int = AUDIT_CHUNK_ROWS,
) -> Iterable[pd.DataFrame]:
    """Stream selected CARE columns and return canonical column names."""
    raw, canonical, encoding, separator = read_care_header(Path(file_path))
    canonical_to_raw = dict(zip(canonical, raw))
    requested = [standardize_column_name(column) for column in usecols]
    missing = [column for column in requested if column not in canonical_to_raw]
    if missing:
        raise KeyError(f"{Path(file_path).name} lacks columns: {', '.join(missing)}")

    reader = pd.read_csv(
        file_path,
        encoding=encoding,
        sep=separator,
        usecols=[canonical_to_raw[column] for column in requested],
        chunksize=chunksize,
        low_memory=False,
    )
    for chunk in reader:
        chunk.columns = [standardize_column_name(column) for column in chunk.columns]
        if "time_stamp" in chunk.columns:
            chunk["time_stamp"] = parse_care_timestamps(chunk["time_stamp"])
        yield chunk


def read_metadata_csv(file_path: Path, required: set[str]) -> pd.DataFrame:
    """Parse a small CARE metadata CSV despite encoding or delimiter variation."""
    for encoding in CSV_ENCODINGS:
        for skiprows in range(0, 20):
            for separator in CSV_SEPARATORS:
                try:
                    frame = pd.read_csv(
                        file_path,
                        encoding=encoding,
                        sep=separator,
                        skiprows=skiprows,
                        engine="python",
                    )
                except Exception:
                    continue
                frame = frame.dropna(axis=0, how="all").dropna(axis=1, how="all")
                if frame.empty:
                    continue
                frame.columns = [standardize_column_name(c) for c in frame.columns]
                if len(frame.columns) != len(set(frame.columns)):
                    continue
                if required <= set(frame.columns):
                    return frame.reset_index(drop=True)
    raise RuntimeError(f"No table containing {sorted(required)} found in:\n{file_path}")


def select_unique_metadata_file(
    candidates: Iterable[Path],
    required: set[str],
    description: str,
) -> tuple[Path, pd.DataFrame]:
    parsed: list[tuple[Path, pd.DataFrame]] = []
    for path in candidates:
        try:
            parsed.append((Path(path), read_metadata_csv(Path(path), required)))
        except RuntimeError:
            continue
    if len(parsed) != 1:
        names = [str(path) for path, _ in parsed]
        raise RuntimeError(
            f"Expected exactly one parseable {description}; found {len(parsed)}: {names}"
        )
    return parsed[0]


def sha256_file(file_path: Path, maximum_bytes: int | None = None) -> str:
    digest = hashlib.sha256()
    remaining = maximum_bytes
    with Path(file_path).open("rb") as handle:
        while True:
            size = 1024 * 1024 if remaining is None else min(1024 * 1024, remaining)
            if size <= 0:
                break
            block = handle.read(size)
            if not block:
                break
            digest.update(block)
            if remaining is not None:
                remaining -= len(block)
    return digest.hexdigest()


def save_csv_atomic(frame: pd.DataFrame, output_path: Path) -> None:
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = output_path.with_suffix(output_path.suffix + ".tmp")
    frame.to_csv(temporary, index=False, lineterminator="\n")
    temporary.replace(output_path)


def dataframe_sha256(frame: pd.DataFrame, sort_columns: Iterable[str]) -> str:
    ordered = frame.sort_values(list(sort_columns), kind="stable").reset_index(drop=True)
    payload = ordered.to_csv(
        index=False,
        lineterminator="\n",
        na_rep="<NA>",
        float_format="%.12g",
    )
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


# =============================================================================
# 2. File discovery and lightweight dataset manifest
# =============================================================================

def infer_farm(path: Path) -> str | None:
    compact = "".join(character.lower() for character in str(path.parent) if character.isalnum())
    for code in ("a", "b", "c"):
        if f"windfarm{code}" in compact or f"farm{code}" in compact:
            return f"Wind Farm {code.upper()}"
    return None


_all_csv_files = sorted(
    path for path in Path(CARE_DATA_ROOT).rglob("*")
    if path.is_file() and path.suffix.lower() == ".csv"
)

_event_records: list[dict[str, Any]] = []
_event_info_files: dict[str, list[Path]] = defaultdict(list)
_feature_description_files: dict[str, list[Path]] = defaultdict(list)

for _path in _all_csv_files:
    _farm = infer_farm(_path)
    _stem = _path.stem.lower()
    if EVENT_FILE_PATTERN.fullmatch(_path.stem):
        if _farm is None:
            raise RuntimeError(f"Numeric event CSV is outside a recognized farm: {_path}")
        _event_records.append(
            {
                "farm": _farm,
                "event_id": int(_path.stem),
                "file_path": _path,
                "relative_file_path": _path.relative_to(CARE_DATA_ROOT).as_posix(),
                "size_bytes": int(_path.stat().st_size),
            }
        )
    elif "feature" in _stem:
        _feature_description_files[_farm].append(_path)
    elif "event" in _stem:
        _event_info_files[_farm].append(_path)

EVENT_FILE_INVENTORY = (
    pd.DataFrame(_event_records)
    .sort_values(["farm", "event_id"], kind="stable")
    .reset_index(drop=True)
)

if len(EVENT_FILE_INVENTORY) != DATASET.expected_total_cases:
    raise RuntimeError(
        f"Found {len(EVENT_FILE_INVENTORY)} numeric event files; "
        f"expected {DATASET.expected_total_cases}."
    )

_expected_case_counts = dict(zip(DATASET.farms, DATASET.expected_cases_by_farm))
_observed_case_counts = EVENT_FILE_INVENTORY.groupby("farm").size().to_dict()
if _observed_case_counts != _expected_case_counts:
    raise RuntimeError(
        f"Per-farm event-file counts differ from CARE v6. "
        f"Observed={_observed_case_counts}, expected={_expected_case_counts}."
    )

_manifest_records: list[dict[str, Any]] = []
for _path in _all_csv_files:
    _is_event = bool(EVENT_FILE_PATTERN.fullmatch(_path.stem))
    _manifest_records.append(
        {
            "relative_file_path": _path.relative_to(CARE_DATA_ROOT).as_posix(),
            "farm": infer_farm(_path),
            "file_kind": (
                "event_data"
                if _is_event
                else "feature_metadata"
                if "feature" in _path.stem.lower()
                else "event_metadata"
                if "event" in _path.stem.lower()
                else "other_csv"
            ),
            "size_bytes": int(_path.stat().st_size),
            # Full hashes for small metadata; a clearly named 64-KiB head hash for
            # large event files. The manifest is not misrepresented as a full-data hash.
            "content_hash_scope": "first_65536_bytes" if _is_event else "full_file",
            "content_sha256": sha256_file(_path, 65_536 if _is_event else None),
        }
    )

DATASET_FILE_MANIFEST = pd.DataFrame(_manifest_records).sort_values(
    "relative_file_path", kind="stable"
).reset_index(drop=True)
DATASET_MANIFEST_SHA256 = dataframe_sha256(
    DATASET_FILE_MANIFEST, ("relative_file_path",)
)


# =============================================================================
# 3. Safe case registry and outcome lockbox
# =============================================================================

def normalize_event_label(value: Any) -> bool:
    text = re.sub(r"[^a-z0-9]+", "", str(value).strip().lower())
    if text in {"1", "true", "yes"} or any(
        token in text for token in ("anomal", "abnormal", "fault", "failure")
    ):
        return True
    if text in {"0", "false", "no"} or any(
        token in text for token in ("normal", "healthy")
    ):
        return False
    raise ValueError(f"Unrecognized CARE event label: {value!r}")


def build_case_registry_and_lockbox() -> tuple[pd.DataFrame, pd.DataFrame, dict[str, Any]]:
    metadata_frames: list[pd.DataFrame] = []

    for farm in DATASET.farms:
        _, frame = select_unique_metadata_file(
            _event_info_files.get(farm, []),
            {"event_id", "asset_id"},
            f"event-information file for {farm}",
        )
        label_column = find_column(frame, ("event_label", "label", "event_type", "class"))
        if label_column is None:
            raise RuntimeError(f"The event-information file for {farm} has no label column.")
        frame = frame.rename(columns={label_column: "event_label_raw"}).copy()
        frame["farm"] = farm
        frame["event_id"] = pd.to_numeric(frame["event_id"], errors="raise").astype(int)
        frame["source_asset_id"] = frame["asset_id"].map(normalize_identifier)
        frame["asset_id"] = farm + "::" + frame["source_asset_id"]
        frame["case_key"] = (
            farm + "::event_" + frame["event_id"].astype(str)
        )
        frame["is_anomaly"] = frame["event_label_raw"].map(normalize_event_label)

        for column in ("event_start", "event_end"):
            frame[column] = (
                parse_care_timestamps(frame[column])
                if column in frame.columns
                else pd.NaT
            )
        for column in ("event_start_id", "event_end_id"):
            frame[column] = (
                pd.to_numeric(frame[column], errors="coerce").astype("Int64")
                if column in frame.columns
                else pd.Series(pd.NA, index=frame.index, dtype="Int64")
            )
        if "event_description" not in frame.columns:
            frame["event_description"] = ""
        metadata_frames.append(frame)

    metadata = pd.concat(metadata_frames, ignore_index=True)
    if metadata.duplicated(["farm", "event_id"]).any():
        raise RuntimeError("Duplicate (farm, event_id) entries exist in event metadata.")

    merged = metadata.merge(
        EVENT_FILE_INVENTORY,
        on=["farm", "event_id"],
        how="outer",
        validate="one_to_one",
        indicator=True,
    )
    if not merged["_merge"].eq("both").all():
        mismatch = merged.loc[
            ~merged["_merge"].eq("both"), ["farm", "event_id", "_merge"]
        ]
        raise RuntimeError(f"Event metadata/files do not match:\n{mismatch.to_string(index=False)}")
    merged = merged.drop(columns="_merge").sort_values(
        ["farm", "event_id"], kind="stable"
    ).reset_index(drop=True)

    anomaly_by_farm = (
        merged.groupby("farm", sort=False)["is_anomaly"].sum().astype(int).to_dict()
    )
    normal_by_farm = (
        merged.groupby("farm", sort=False)["is_anomaly"]
        .apply(lambda values: int((~values.astype(bool)).sum()))
        .to_dict()
    )
    expected_anomaly = dict(zip(DATASET.farms, DATASET.expected_anomaly_by_farm))
    expected_normal = dict(zip(DATASET.farms, DATASET.expected_normal_by_farm))
    if anomaly_by_farm != expected_anomaly or normal_by_farm != expected_normal:
        raise RuntimeError(
            "CARE outcome counts differ from the frozen v6 contract. "
            f"Anomaly={anomaly_by_farm}, normal={normal_by_farm}."
        )

    anomaly_boundaries = merged.loc[
        merged["is_anomaly"], ["event_start", "event_end"]
    ]
    if anomaly_boundaries.isna().any().any():
        raise RuntimeError("At least one anomaly case lacks a timestamp event boundary.")
    if (anomaly_boundaries["event_end"] < anomaly_boundaries["event_start"]).any():
        raise RuntimeError("At least one anomaly event ends before it starts.")

    observed_assets = int(merged["asset_id"].nunique())
    if observed_assets != DATASET.expected_assets:
        raw_unique = int(merged["source_asset_id"].nunique())
        raise RuntimeError(
            f"Found {observed_assets} farm-qualified assets; expected "
            f"{DATASET.expected_assets}. Raw unqualified identifiers={raw_unique}."
        )

    safe = merged.loc[:, SAFE_CASE_COLUMNS].copy()
    if set(safe.columns) & set(FORBIDDEN_PREDICTOR_FIELDS):
        raise RuntimeError("Outcome fields escaped into the safe CASE_REGISTRY.")

    lockbox_columns = (
        "case_key",
        "farm",
        "asset_id",
        "source_asset_id",
        "event_id",
        *OUTCOME_ONLY_COLUMNS,
    )
    lockbox = merged.loc[:, lockbox_columns].copy()
    counts = {
        "cases": int(len(merged)),
        "assets": observed_assets,
        "anomaly_cases": int(merged["is_anomaly"].sum()),
        "normal_cases": int((~merged["is_anomaly"]).sum()),
        "anomaly_by_farm": anomaly_by_farm,
        "normal_by_farm": normal_by_farm,
    }
    return safe, lockbox, counts


CASE_REGISTRY, _OUTCOME_LOCKBOX, OUTCOME_COUNT_CHECK = build_case_registry_and_lockbox()
OUTCOME_LOCKBOX_SHA256 = dataframe_sha256(
    _OUTCOME_LOCKBOX, ("farm", "event_id")
)

if tuple(CASE_REGISTRY.columns) != SAFE_CASE_COLUMNS:
    raise RuntimeError("CASE_REGISTRY does not have the exact safe-column contract.")


# =============================================================================
# 4. Per-farm schemas and metadata-grounded signal taxonomy
# =============================================================================

STATISTIC_SUFFIXES = {
    "_avg": "avg",
    "_std": "std",
    "_max": "max",
    "_min": "min",
}


def split_signal_statistic(column: str) -> tuple[str, str]:
    for suffix, statistic in STATISTIC_SUFFIXES.items():
        if column.endswith(suffix):
            return column[: -len(suffix)], statistic
    return column, "unspecified"


def parse_metadata_boolean(value: Any) -> bool:
    return str(value).strip().lower() in {"1", "true", "yes", "y"}


def semantic_role(
    description: str,
    is_angle: bool,
    is_counter: bool,
) -> tuple[str, str]:
    lowered = re.sub(r"\s+", " ", str(description).strip().lower())
    # CARE metadata uses both "wind speed" and "windspeed" spellings.
    lowered = re.sub(r"\bwindspeed\b", "wind speed", lowered)
    driver_matches = [
        pattern
        for pattern in MEAN_MODEL.driver_description_patterns
        if pattern in lowered
    ]
    temperature = bool(re.search(r"\btemp(?:erature)?\b", lowered))

    if driver_matches:
        role = "operating_driver_candidate"
        reason = "; ".join(driver_matches)
    elif temperature:
        role = "temperature_target_candidate"
        reason = "temperature semantics"
    elif is_angle:
        role = "angle_target_candidate"
        reason = "metadata is_angle"
    elif is_counter:
        role = "counter_target_candidate"
        reason = "metadata is_counter"
    elif lowered:
        role = "other_target_candidate"
        reason = "described non-driver signal"
    else:
        role = "unclassified_target_candidate"
        reason = "missing feature description"
    return role, reason


_feature_records: list[dict[str, Any]] = []
_schema_records: list[dict[str, Any]] = []
FARM_SCHEMAS: dict[str, dict[str, Any]] = {}

for _farm in DATASET.farms:
    _farm_cases = CASE_REGISTRY.loc[CASE_REGISTRY["farm"].eq(_farm)]
    _header_specs = [read_care_header(Path(path)) for path in _farm_cases["file_path"]]
    _schema_fingerprints = {spec[1] for spec in _header_specs}
    if len(_schema_fingerprints) != 1:
        raise RuntimeError(f"{_farm} contains {len(_schema_fingerprints)} event schemas.")

    _raw_columns, _columns, _encoding, _separator = _header_specs[0]
    _missing_metadata = set(DATASET.metadata_columns) - set(_columns)
    if _missing_metadata:
        raise RuntimeError(f"{_farm} schema lacks {sorted(_missing_metadata)}")
    _signal_columns = tuple(
        column for column in _columns if column not in set(DATASET.metadata_columns)
    )
    _forbidden_signals = set(_signal_columns) & set(FORBIDDEN_PREDICTOR_FIELDS)
    if _forbidden_signals:
        raise RuntimeError(
            f"Outcome fields appear inside {_farm}'s signal schema: {_forbidden_signals}"
        )

    _, _feature_metadata = select_unique_metadata_file(
        _feature_description_files.get(_farm, []),
        {"feature_name", "feature_description"},
        f"feature-description file for {_farm}",
    )
    _feature_metadata = _feature_metadata.copy()
    _feature_metadata["base_sensor"] = _feature_metadata["feature_name"].map(
        standardize_column_name
    )
    if _feature_metadata["base_sensor"].duplicated().any():
        duplicates = _feature_metadata.loc[
            _feature_metadata["base_sensor"].duplicated(False), "base_sensor"
        ].tolist()
        raise RuntimeError(f"Duplicate feature-description entries in {_farm}: {duplicates}")
    _metadata_lookup = _feature_metadata.set_index("base_sensor", drop=False)

    for _column in _signal_columns:
        _base, _statistic = split_signal_statistic(_column)
        _described = _base in _metadata_lookup.index
        if _described:
            _metadata_row = _metadata_lookup.loc[_base]
            _description = str(_metadata_row.get("feature_description", "")).strip()
            _unit = str(_metadata_row.get("unit", "")).strip()
            _statistics_declared = str(_metadata_row.get("statistics_type", "")).strip()
            _is_angle = parse_metadata_boolean(_metadata_row.get("is_angle", False))
            _is_counter = parse_metadata_boolean(_metadata_row.get("is_counter", False))
        else:
            _description = ""
            _unit = ""
            _statistics_declared = ""
            _is_angle = False
            _is_counter = False

        _role, _role_reason = semantic_role(_description, _is_angle, _is_counter)
        _preprocessing = (
            "within_segment_first_difference"
            if _is_counter
            else "sine_cosine_encoding"
            if _is_angle
            else "identity"
        )
        _feature_records.append(
            {
                "farm": _farm,
                "column": _column,
                "base_sensor": _base,
                "statistic": _statistic,
                "primary_analysis": _statistic in QUALITY.primary_statistics,
                "sensitivity_analysis": _statistic in QUALITY.sensitivity_statistics,
                "description": _description,
                "unit": _unit,
                "statistics_declared": _statistics_declared,
                "metadata_described": _described,
                "is_angle": _is_angle,
                "is_counter": _is_counter,
                "preprocessing": _preprocessing,
                "role": _role,
                "role_reason": _role_reason,
            }
        )

    _farm_features = pd.DataFrame(
        record for record in _feature_records if record["farm"] == _farm
    )
    _primary = tuple(
        _farm_features.loc[_farm_features["primary_analysis"], "column"]
    )
    _sensitivity = tuple(
        _farm_features.loc[_farm_features["sensitivity_analysis"], "column"]
    )
    _drivers = tuple(
        _farm_features.loc[
            _farm_features["primary_analysis"]
            & _farm_features["role"].eq("operating_driver_candidate"),
            "column",
        ]
    )
    _temperature_targets = tuple(
        _farm_features.loc[
            _farm_features["primary_analysis"]
            & _farm_features["role"].eq("temperature_target_candidate"),
            "column",
        ]
    )
    _other_targets = tuple(
        _farm_features.loc[
            _farm_features["primary_analysis"]
            & _farm_features["role"].isin(
                (
                    "other_target_candidate",
                    "angle_target_candidate",
                    "counter_target_candidate",
                    "unclassified_target_candidate",
                )
            ),
            "column",
        ]
    )
    _angles = tuple(
        _farm_features.loc[_farm_features["primary_analysis"] & _farm_features["is_angle"], "column"]
    )
    _counters = tuple(
        _farm_features.loc[_farm_features["primary_analysis"] & _farm_features["is_counter"], "column"]
    )

    FARM_SCHEMAS[_farm] = {
        "all_columns": tuple(_columns),
        "signal_columns": _signal_columns,
        "primary_columns": _primary,
        "sensitivity_columns": _sensitivity,
        "driver_candidates": _drivers,
        "temperature_target_candidates": _temperature_targets,
        "other_target_candidates": _other_targets,
        "angle_columns": _angles,
        "counter_columns": _counters,
        "encoding": _encoding,
        "separator": _separator,
    }
    _schema_records.append(
        {
            "farm": _farm,
            "cases": int(len(_farm_cases)),
            "total_columns": len(_columns),
            "signal_columns": len(_signal_columns),
            "primary_avg_columns": len(_primary),
            "sensitivity_avg_std_columns": len(_sensitivity),
            "driver_candidates": len(_drivers),
            "temperature_target_candidates": len(_temperature_targets),
            "other_target_candidates": len(_other_targets),
            "angle_columns": len(_angles),
            "counter_columns": len(_counters),
            "metadata_description_coverage": float(_farm_features["metadata_described"].mean()),
        }
    )

CARE_FEATURE_REGISTRY = pd.DataFrame(_feature_records).sort_values(
    ["farm", "column"], kind="stable"
).reset_index(drop=True)
FARM_SCHEMA_SUMMARY = pd.DataFrame(_schema_records)

if CARE_FEATURE_REGISTRY.loc[
    CARE_FEATURE_REGISTRY["statistic"].eq("std"), "primary_analysis"
].any():
    raise RuntimeError("A standard-deviation channel was incorrectly admitted to primary analysis.")

if not CARE_FEATURE_REGISTRY.loc[
    CARE_FEATURE_REGISTRY["primary_analysis"], "statistic"
].eq("avg").all():
    raise RuntimeError("Primary analysis contains a non-average channel.")


# =============================================================================
# 5. Streaming quality audit on source training-normal rows only
# =============================================================================

_train_tokens = {str(value).strip().lower() for value in DATASET.source_train_labels}
_prediction_tokens = {
    str(value).strip().lower() for value in DATASET.source_prediction_labels
}
_train_tokens |= {"0.0"} if "0" in _train_tokens else set()
_prediction_tokens |= {"1.0"} if "1" in _prediction_tokens else set()


def normalized_partition(values: pd.Series) -> pd.Series:
    text = values.astype("string").str.strip().str.lower()
    result = pd.Series("unknown", index=values.index, dtype="string")
    result.loc[text.isin(_train_tokens)] = "train"
    result.loc[text.isin(_prediction_tokens)] = "prediction"
    return result


def update_zero_runs(
    zero_matrix: np.ndarray,
    break_before: np.ndarray,
    carry: np.ndarray,
    longest: np.ndarray,
) -> None:
    """Update per-channel zero runs, resetting at temporal discontinuities."""
    if zero_matrix.size == 0:
        return
    starts = [0] + [int(index) for index in np.flatnonzero(break_before) if index > 0]
    ends = starts[1:] + [len(zero_matrix)]
    for segment_number, (start, end) in enumerate(zip(starts, ends)):
        if segment_number > 0 or break_before[start]:
            carry[:] = 0
        segment = zero_matrix[start:end]
        if len(segment) == 0:
            continue
        for column_index in range(segment.shape[1]):
            mask = segment[:, column_index]
            false_indices = np.flatnonzero(~mask)
            if len(false_indices) == 0:
                carry[column_index] += len(mask)
                longest[column_index] = max(longest[column_index], carry[column_index])
                continue
            prefix = int(false_indices[0])
            suffix = int(len(mask) - false_indices[-1] - 1)
            run_lengths = np.diff(
                np.concatenate((np.array([-1]), false_indices, np.array([len(mask)])))
            ) - 1
            longest[column_index] = max(
                longest[column_index],
                carry[column_index] + prefix,
                int(run_lengths.max(initial=0)),
            )
            carry[column_index] = suffix


def audit_one_case(case_row: Any) -> tuple[dict[str, Any], list[dict[str, Any]]]:
    farm = str(case_row.farm)
    file_path = Path(case_row.file_path)
    channels = list(FARM_SCHEMAS[farm]["primary_columns"])
    required = [*DATASET.metadata_columns, *channels]

    finite_count = np.zeros(len(channels), dtype=np.int64)
    zero_count = np.zeros(len(channels), dtype=np.int64)
    longest_zero_run = np.zeros(len(channels), dtype=np.int64)
    zero_run_carry = np.zeros(len(channels), dtype=np.int64)

    total_rows = train_rows = prediction_rows = fit_rows = unknown_split_rows = 0
    missing_timestamps = duplicate_timestamps = nonmonotonic_timestamps = 0
    gap_count = 0
    maximum_gap_minutes = 0.0
    previous_timestamp_ns: int | None = None
    source_assets_seen: set[str] = set()
    status_values_seen: set[str] = set()

    nat_ns = np.iinfo(np.int64).min
    expected_delta_ns = int(DATASET.sampling_minutes * 60 * 1_000_000_000)

    for chunk in read_care_chunks(file_path, required):
        rows = len(chunk)
        total_rows += rows
        if rows == 0:
            continue

        partitions = normalized_partition(chunk["train_test"])
        is_train = partitions.eq("train").to_numpy()
        is_prediction = partitions.eq("prediction").to_numpy()
        unknown_split_rows += int(partitions.eq("unknown").sum())
        train_rows += int(is_train.sum())
        prediction_rows += int(is_prediction.sum())

        statuses = pd.to_numeric(chunk["status_type_id"], errors="coerce")
        is_normal_status = statuses.isin(DATASET.normal_status_ids).to_numpy()
        status_values_seen.update(
            str(int(value)) if float(value).is_integer() else str(float(value))
            for value in statuses.dropna().unique()
        )
        fit_mask = is_train & is_normal_status
        fit_rows += int(fit_mask.sum())

        for value in chunk["asset_id"].dropna().unique():
            source_assets_seen.add(normalize_identifier(value))

        timestamps_ns = chunk["time_stamp"].to_numpy(dtype="datetime64[ns]").astype(np.int64)
        missing_timestamps += int((timestamps_ns == nat_ns).sum())
        break_before = np.zeros(rows, dtype=bool)
        for index, current_ns in enumerate(timestamps_ns):
            if index == 0:
                prior_ns = previous_timestamp_ns
            else:
                prior_ns = int(timestamps_ns[index - 1])
            if prior_ns is None or prior_ns == nat_ns or current_ns == nat_ns:
                break_before[index] = prior_ns is not None
                continue
            delta_ns = int(current_ns) - int(prior_ns)
            if delta_ns == 0:
                duplicate_timestamps += 1
                break_before[index] = True
            elif delta_ns < 0:
                nonmonotonic_timestamps += 1
                break_before[index] = True
            elif delta_ns > expected_delta_ns:
                gap_count += 1
                break_before[index] = True
                maximum_gap_minutes = max(
                    maximum_gap_minutes, delta_ns / (60 * 1_000_000_000)
                )
        previous_timestamp_ns = int(timestamps_ns[-1])

        numeric = chunk[channels].apply(pd.to_numeric, errors="coerce").to_numpy(
            dtype=np.float64
        )
        eligible_values = numeric[fit_mask]
        finite = np.isfinite(eligible_values)
        zeros = finite & (eligible_values == 0.0)
        finite_count += finite.sum(axis=0, dtype=np.int64)
        zero_count += zeros.sum(axis=0, dtype=np.int64)

        # Non-training or non-normal rows are False and therefore terminate a run.
        zero_timeline = np.zeros_like(numeric, dtype=bool)
        zero_timeline[fit_mask] = zeros
        update_zero_runs(
            zero_timeline,
            break_before,
            zero_run_carry,
            longest_zero_run,
        )

    expected_source_asset = str(case_row.source_asset_id)
    asset_match = source_assets_seen == {expected_source_asset}
    segments = int(gap_count + 1) if total_rows else 0
    case_record = {
        "case_key": case_row.case_key,
        "farm": farm,
        "asset_id": case_row.asset_id,
        "source_asset_id": expected_source_asset,
        "event_id": int(case_row.event_id),
        "rows": total_rows,
        "train_rows": train_rows,
        "prediction_rows": prediction_rows,
        "training_normal_rows": fit_rows,
        "unknown_split_rows": unknown_split_rows,
        "status_ids_seen": ";".join(sorted(status_values_seen)),
        "source_asset_ids_seen": ";".join(sorted(source_assets_seen)),
        "asset_id_matches_metadata": asset_match,
        "missing_timestamps": missing_timestamps,
        "duplicate_timestamps": duplicate_timestamps,
        "nonmonotonic_timestamps": nonmonotonic_timestamps,
        "gap_count": gap_count,
        "continuous_segments": segments,
        "maximum_gap_minutes": maximum_gap_minutes,
    }

    channel_records: list[dict[str, Any]] = []
    for index, channel in enumerate(channels):
        availability = finite_count[index] / fit_rows if fit_rows else 0.0
        zero_fraction = zero_count[index] / finite_count[index] if finite_count[index] else np.nan
        all_missing = finite_count[index] == 0
        all_zero = finite_count[index] > 0 and zero_count[index] == finite_count[index]
        long_zero = longest_zero_run[index] >= QUALITY.constant_zero_run_min_steps
        low_availability = availability < QUALITY.minimum_sensor_availability

        if all_missing:
            review_flag = "all_missing_training_normal"
        elif all_zero:
            review_flag = "structurally_zero_training_normal"
        elif low_availability:
            review_flag = "low_availability_training_normal"
        elif long_zero:
            review_flag = "sustained_zero_run_review"
        else:
            review_flag = "pass"

        channel_records.append(
            {
                "case_key": case_row.case_key,
                "farm": farm,
                "asset_id": case_row.asset_id,
                "event_id": int(case_row.event_id),
                "column": channel,
                "training_normal_rows": fit_rows,
                "finite_rows": int(finite_count[index]),
                "missing_rows": int(fit_rows - finite_count[index]),
                "availability": float(availability),
                "zero_rows": int(zero_count[index]),
                "zero_fraction_of_finite": float(zero_fraction),
                "longest_zero_run_steps": int(longest_zero_run[index]),
                "all_missing": bool(all_missing),
                "all_zero": bool(all_zero),
                "low_availability": bool(low_availability),
                "sustained_zero_run": bool(long_zero),
                "review_flag": review_flag,
            }
        )
    return case_record, channel_records


_case_quality_records: list[dict[str, Any]] = []
_channel_quality_records: list[dict[str, Any]] = []

for _farm in DATASET.farms:
    _farm_registry = CASE_REGISTRY.loc[CASE_REGISTRY["farm"].eq(_farm)]
    print(
        f"Streaming training-normal quality audit — {_farm}: "
        f"{len(_farm_registry)} cases",
        flush=True,
    )
    for _case_row in _farm_registry.itertuples(index=False):
        _case_record, _channel_records = audit_one_case(_case_row)
        _case_quality_records.append(_case_record)
        _channel_quality_records.extend(_channel_records)

CASE_QUALITY_AUDIT = pd.DataFrame(_case_quality_records).sort_values(
    ["farm", "event_id"], kind="stable"
).reset_index(drop=True)
CHANNEL_QUALITY_AUDIT = pd.DataFrame(_channel_quality_records).sort_values(
    ["farm", "event_id", "column"], kind="stable"
).reset_index(drop=True)

_fatal_case_mask = (
    (CASE_QUALITY_AUDIT["rows"] <= 0)
    | (CASE_QUALITY_AUDIT["train_rows"] <= 0)
    | (CASE_QUALITY_AUDIT["prediction_rows"] <= 0)
    | (CASE_QUALITY_AUDIT["training_normal_rows"] < QUALITY.minimum_training_rows)
    | (CASE_QUALITY_AUDIT["unknown_split_rows"] > 0)
    | (~CASE_QUALITY_AUDIT["asset_id_matches_metadata"])
    | (CASE_QUALITY_AUDIT["missing_timestamps"] > 0)
    | (CASE_QUALITY_AUDIT["duplicate_timestamps"] > 0)
    | (CASE_QUALITY_AUDIT["nonmonotonic_timestamps"] > 0)
)
if _fatal_case_mask.any():
    _fatal_columns = [
        "case_key",
        "rows",
        "train_rows",
        "prediction_rows",
        "training_normal_rows",
        "unknown_split_rows",
        "asset_id_matches_metadata",
        "missing_timestamps",
        "duplicate_timestamps",
        "nonmonotonic_timestamps",
    ]
    raise RuntimeError(
        "Structural quality checks failed. No model may be fitted:\n"
        + CASE_QUALITY_AUDIT.loc[_fatal_case_mask, _fatal_columns].to_string(index=False)
    )

CHANNEL_QUALITY_SUMMARY = (
    CHANNEL_QUALITY_AUDIT.groupby(["farm", "column"], sort=False)
    .agg(
        cases=("case_key", "nunique"),
        assets=("asset_id", "nunique"),
        training_normal_rows=("training_normal_rows", "sum"),
        finite_rows=("finite_rows", "sum"),
        missing_rows=("missing_rows", "sum"),
        zero_rows=("zero_rows", "sum"),
        worst_case_availability=("availability", "min"),
        median_case_availability=("availability", "median"),
        longest_zero_run_steps=("longest_zero_run_steps", "max"),
        all_missing_cases=("all_missing", "sum"),
        all_zero_cases=("all_zero", "sum"),
        low_availability_cases=("low_availability", "sum"),
        sustained_zero_run_cases=("sustained_zero_run", "sum"),
    )
    .reset_index()
)
CHANNEL_QUALITY_SUMMARY["pooled_availability"] = (
    CHANNEL_QUALITY_SUMMARY["finite_rows"]
    / CHANNEL_QUALITY_SUMMARY["training_normal_rows"].clip(lower=1)
)
CHANNEL_QUALITY_SUMMARY["pooled_zero_fraction"] = (
    CHANNEL_QUALITY_SUMMARY["zero_rows"]
    / CHANNEL_QUALITY_SUMMARY["finite_rows"].replace(0, np.nan)
)

ZERO_AND_MISSINGNESS_REVIEW = CHANNEL_QUALITY_AUDIT.loc[
    ~CHANNEL_QUALITY_AUDIT["review_flag"].eq("pass")
].reset_index(drop=True)

QUALITY_FARM_SUMMARY = (
    CASE_QUALITY_AUDIT.groupby("farm", sort=False)
    .agg(
        cases=("case_key", "size"),
        assets=("asset_id", "nunique"),
        total_rows=("rows", "sum"),
        median_training_normal_rows=("training_normal_rows", "median"),
        median_prediction_rows=("prediction_rows", "median"),
        total_gaps=("gap_count", "sum"),
        median_segments=("continuous_segments", "median"),
        maximum_gap_minutes=("maximum_gap_minutes", "max"),
    )
    .reset_index()
)


# =============================================================================
# 6. Freeze the audit receipt, then write tables
# =============================================================================

_audit_component_hashes = {
    "safe_case_registry_sha256": dataframe_sha256(
        CASE_REGISTRY.assign(file_path=CASE_REGISTRY["file_path"].astype(str)),
        ("farm", "event_id"),
    ),
    "outcome_lockbox_sha256": OUTCOME_LOCKBOX_SHA256,
    "feature_registry_sha256": dataframe_sha256(
        CARE_FEATURE_REGISTRY, ("farm", "column")
    ),
    "case_quality_sha256": dataframe_sha256(
        CASE_QUALITY_AUDIT, ("farm", "event_id")
    ),
    "channel_quality_sha256": dataframe_sha256(
        CHANNEL_QUALITY_AUDIT, ("farm", "event_id", "column")
    ),
}
CELL2_AUDIT_SHA256 = sha256_json(
    {
        "contract_sha256": CONTRACT_SHA256,
        "dataset_manifest_sha256": DATASET_MANIFEST_SHA256,
        "component_hashes": _audit_component_hashes,
        "scope": "training-normal diagnostic audit; no global channel selection",
    }
)

CELL2_RECEIPT_PATH = Path(INVENTORY_DIR) / "cell2_inventory_quality_audit_receipt.json"
if CELL2_RECEIPT_PATH.exists():
    _existing_receipt = json.loads(CELL2_RECEIPT_PATH.read_text(encoding="utf-8"))
    if (
        _existing_receipt.get("contract_sha256") != CONTRACT_SHA256
        or _existing_receipt.get("dataset_manifest_sha256") != DATASET_MANIFEST_SHA256
        or _existing_receipt.get("cell2_audit_sha256") != CELL2_AUDIT_SHA256
    ):
        raise RuntimeError(
            "A different Cell 2 audit already exists under this experiment ID. "
            "Do not overwrite it; investigate dataset or code drift and start a "
            "new experiment ID if the change is intentional."
        )
    CELL2_AUDIT_STATE = "existing identical audit verified"
    _write_new_receipt = False
else:
    CELL2_AUDIT_STATE = "new audit frozen"
    _write_new_receipt = True

_safe_registry_for_csv = CASE_REGISTRY.copy()
_safe_registry_for_csv["file_path"] = _safe_registry_for_csv["file_path"].astype(str)

save_csv_atomic(DATASET_FILE_MANIFEST, Path(INVENTORY_DIR) / "dataset_file_manifest.csv")
save_csv_atomic(_safe_registry_for_csv, Path(INVENTORY_DIR) / "care_case_registry_safe.csv")
save_csv_atomic(
    _OUTCOME_LOCKBOX,
    Path(INVENTORY_DIR) / "outcome_lockbox_do_not_load_before_outer_predictions.csv",
)
save_csv_atomic(FARM_SCHEMA_SUMMARY, Path(INVENTORY_DIR) / "farm_schema_summary.csv")
save_csv_atomic(CARE_FEATURE_REGISTRY, Path(INVENTORY_DIR) / "care_feature_registry.csv")
save_json(FARM_SCHEMAS, Path(INVENTORY_DIR) / "farm_schemas.json")
save_csv_atomic(CASE_QUALITY_AUDIT, Path(QUALITY_DIR) / "case_quality_audit.csv")
save_csv_atomic(CHANNEL_QUALITY_AUDIT, Path(QUALITY_DIR) / "channel_quality_audit.csv")
save_csv_atomic(CHANNEL_QUALITY_SUMMARY, Path(QUALITY_DIR) / "channel_quality_summary.csv")
save_csv_atomic(ZERO_AND_MISSINGNESS_REVIEW, Path(QUALITY_DIR) / "zero_missingness_review.csv")

if _write_new_receipt:
    save_json(
        {
            "contract_sha256": CONTRACT_SHA256,
            "dataset_manifest_sha256": DATASET_MANIFEST_SHA256,
            "cell2_audit_sha256": CELL2_AUDIT_SHA256,
            "component_hashes": _audit_component_hashes,
            "outcome_count_check": OUTCOME_COUNT_CHECK,
            "quality_scope": (
                "Signal diagnostics use source training-partition rows with normal "
                "status only. They do not select globally retained channels."
            ),
            "safe_registry_columns": SAFE_CASE_COLUMNS,
            "outcome_lockbox_columns": (
                "case_key",
                "farm",
                "asset_id",
                "source_asset_id",
                "event_id",
                *OUTCOME_ONLY_COLUMNS,
            ),
            "created_at_utc": utc_now(),
        },
        CELL2_RECEIPT_PATH,
    )

# Remove the only global DataFrame containing outcome data. Later nested-scoring
# code must use a fold-gated accessor rather than an ambient notebook variable.
del _OUTCOME_LOCKBOX


# =============================================================================
# 7. Concise notebook report
# =============================================================================

print("\n" + "=" * 92)
print("UC-RCF-NBM CELL 2 — CARE V6 INVENTORY AND QUALITY AUDIT")
print("=" * 92)
display(FARM_SCHEMA_SUMMARY)

print("\nTRAINING-NORMAL QUALITY SUMMARY")
display(QUALITY_FARM_SUMMARY)

_review_counts = (
    ZERO_AND_MISSINGNESS_REVIEW.groupby(["farm", "review_flag"])
    .size()
    .rename("case_channel_pairs")
    .reset_index()
)
print("\nZERO/MISSINGNESS ITEMS FOR FOLD-LOCAL REVIEW")
if len(_review_counts):
    display(_review_counts)
else:
    print("No review flags were raised.")

print("\n" + "-" * 92)
print(f"Cases registered                 : {len(CASE_REGISTRY)}")
print(f"Farm-qualified assets           : {CASE_REGISTRY['asset_id'].nunique()}")
print(f"Anomaly/normal count check      : {OUTCOME_COUNT_CHECK['anomaly_cases']} / "
      f"{OUTCOME_COUNT_CHECK['normal_cases']}")
print(f"Event data size                 : {EVENT_FILE_INVENTORY['size_bytes'].sum() / 1024**3:.2f} GiB")
print(f"Dataset manifest SHA-256        : {DATASET_MANIFEST_SHA256}")
print(f"Outcome lockbox SHA-256         : {OUTCOME_LOCKBOX_SHA256}")
print(f"Cell 2 audit SHA-256            : {CELL2_AUDIT_SHA256}")
print(f"Audit state                     : {CELL2_AUDIT_STATE}")
print("Outcome fields in CASE_REGISTRY : No")
print("Global sensor selection applied : No — selection remains outer-fold local")
print("Model fitted                    : No")
print("Structural checks               : PASS")
print("=" * 92)
print("CELL 2 COMPLETED SUCCESSFULLY — CARE V6 INVENTORY AND QUALITY AUDIT LOCKED")


Streaming training-normal quality audit — Wind Farm A: 22 cases
Streaming training-normal quality audit — Wind Farm B: 15 cases
Streaming training-normal quality audit — Wind Farm C: 58 cases

UC-RCF-NBM CELL 2 — CARE V6 INVENTORY AND QUALITY AUDIT


,farm,cases,total_columns,signal_columns,primary_avg_columns,sensitivity_avg_std_columns,driver_candidates,temperature_target_candidates,other_target_candidates,angle_columns,counter_columns,metadata_description_coverage
0,Wind Farm A,22,86,81,46,55,7,24,15,4,0,1.0
1,Wind Farm B,15,257,252,63,126,14,25,24,3,4,1.0
2,Wind Farm C,58,957,952,238,476,21,75,142,12,0,1.0



TRAINING-NORMAL QUALITY SUMMARY


,farm,cases,assets,total_rows,median_training_normal_rows,median_prediction_rows,total_gaps,median_segments,maximum_gap_minutes
0,Wind Farm A,22,5,1196747,46725.5,2344.0,460,17.5,13730.0
1,Wind Farm B,15,9,859065,47844.0,3073.0,87,3.0,3110.0
2,Wind Farm C,58,22,3187136,47129.5,2511.0,40,1.0,14410.0



ZERO/MISSINGNESS ITEMS FOR FOLD-LOCAL REVIEW


,farm,review_flag,case_channel_pairs
0,Wind Farm A,sustained_zero_run_review,58
1,Wind Farm B,sustained_zero_run_review,107
2,Wind Farm C,structurally_zero_training_normal,136
3,Wind Farm C,sustained_zero_run_review,2186



--------------------------------------------------------------------------------------------
Cases registered                 : 95
Farm-qualified assets           : 36
Anomaly/normal count check      : 45 / 50
Event data size                 : 18.61 GiB
Dataset manifest SHA-256        : 62484bab1219888aa1d0788965ecd77db2b85f0bbb9b476cd3240f4143026f1f
Outcome lockbox SHA-256         : 69c7c75ea8157e2e12cb61c596375168a90c73bef7ee283c618dab080a447e10
Cell 2 audit SHA-256            : 83732caf4ad3e226b69a671287bb14c4ed72e1bfc3b7c26ca144310fce8e5990
Audit state                     : new audit frozen
Outcome fields in CASE_REGISTRY : No
Global sensor selection applied : No — selection remains outer-fold local
Model fitted                    : No
Structural checks               : PASS
CELL 2 COMPLETED SUCCESSFULLY — CARE V6 INVENTORY AND QUALITY AUDIT LOCKED


In [3]:
print("\nFARM SCHEMA SUMMARY")
print(FARM_SCHEMA_SUMMARY.to_string(index=False))

print("\nTRAINING-NORMAL QUALITY SUMMARY")
print(QUALITY_FARM_SUMMARY.to_string(index=False))

print("\nZERO/MISSINGNESS REVIEW COUNTS")
if ZERO_AND_MISSINGNESS_REVIEW.empty:
    print("No review flags")
else:
    print(
        ZERO_AND_MISSINGNESS_REVIEW
        .groupby(["farm", "review_flag"])
        .size()
        .rename("case_channel_pairs")
        .reset_index()
        .to_string(index=False)
    )

print("\nHASHES")
print("Dataset manifest :", DATASET_MANIFEST_SHA256)
print("Outcome lockbox  :", OUTCOME_LOCKBOX_SHA256)
print("Cell 2 audit     :", CELL2_AUDIT_SHA256)


FARM SCHEMA SUMMARY
       farm  cases  total_columns  signal_columns  primary_avg_columns  sensitivity_avg_std_columns  driver_candidates  temperature_target_candidates  other_target_candidates  angle_columns  counter_columns  metadata_description_coverage
Wind Farm A     22             86              81                   46                           55                  7                             24                       15              4                0                            1.0
Wind Farm B     15            257             252                   63                          126                 14                             25                       24              3                4                            1.0
Wind Farm C     58            957             952                  238                          476                 21                             75                      142             12                0                            1.0

TRAINING-NORMAL QUALITY SU

In [4]:
"""CELL 3 — asset folds and leakage-safe CARE preprocessing.

Paste this complete file into the third cell of the UC-RCF-NBM notebook and run
it only after Cells 1 and 2 have completed successfully.

This cell creates the 36 leave-one-asset-out outer folds, deterministic
farm-balanced inner asset folds, and label-free case caches. All fitted
preprocessing quantities are estimated from normal-status rows in each case's
source training partition. No event label, fault description, or event boundary
is loaded. Sensor eligibility is recorded per case; it is not selected globally.
"""

from __future__ import annotations

import hashlib
import json
from pathlib import Path
from typing import Any, Iterable, Iterator

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print


# =============================================================================
# 0. Bind this cell to Umar's completed Cell 1 and Cell 2 receipts
# =============================================================================

EXPECTED_CONTRACT_SHA256 = (
    "827641aecd8e807193ad193d64c274319756092faa53bdb5084f310f62041f49"
)
EXPECTED_DATASET_MANIFEST_SHA256 = (
    "62484bab1219888aa1d0788965ecd77db2b85f0bbb9b476cd3240f4143026f1f"
)
EXPECTED_OUTCOME_LOCKBOX_SHA256 = (
    "69c7c75ea8157e2e12cb61c596375168a90c73bef7ee283c618dab080a447e10"
)
EXPECTED_CELL2_AUDIT_SHA256 = (
    "83732caf4ad3e226b69a671287bb14c4ed72e1bfc3b7c26ca144310fce8e5990"
)

_required_objects = (
    "CONTRACT_SHA256",
    "DATASET_MANIFEST_SHA256",
    "OUTCOME_LOCKBOX_SHA256",
    "CELL2_AUDIT_SHA256",
    "DATASET",
    "QUALITY",
    "EVALUATION",
    "REPRODUCIBILITY",
    "CASE_REGISTRY",
    "CARE_FEATURE_REGISTRY",
    "FARM_SCHEMAS",
    "read_care_chunks",
    "normalized_partition",
    "save_csv_atomic",
    "save_json",
    "sha256_json",
    "utc_now",
    "CACHE_DIR",
    "INVENTORY_DIR",
    "QUALITY_DIR",
)
_missing_objects = [name for name in _required_objects if name not in globals()]
if _missing_objects:
    raise RuntimeError(
        "Run UC-RCF-NBM Cells 1 and 2 before Cell 3. Missing objects: "
        + ", ".join(_missing_objects)
    )

_observed_hashes = {
    "contract": CONTRACT_SHA256,
    "dataset_manifest": DATASET_MANIFEST_SHA256,
    "outcome_lockbox": OUTCOME_LOCKBOX_SHA256,
    "cell2_audit": CELL2_AUDIT_SHA256,
}
_expected_hashes = {
    "contract": EXPECTED_CONTRACT_SHA256,
    "dataset_manifest": EXPECTED_DATASET_MANIFEST_SHA256,
    "outcome_lockbox": EXPECTED_OUTCOME_LOCKBOX_SHA256,
    "cell2_audit": EXPECTED_CELL2_AUDIT_SHA256,
}
if _observed_hashes != _expected_hashes:
    raise RuntimeError(
        "Cell 3 is bound to the completed Cell 1/Cell 2 audit reported in this "
        f"experiment. Observed={_observed_hashes}, expected={_expected_hashes}."
    )

_safe_registry_expected = {
    "case_key",
    "farm",
    "asset_id",
    "source_asset_id",
    "event_id",
    "file_path",
    "relative_file_path",
    "size_bytes",
}
if set(CASE_REGISTRY.columns) != _safe_registry_expected:
    raise RuntimeError(
        "CASE_REGISTRY has changed or includes unsafe fields. "
        f"Observed={sorted(CASE_REGISTRY.columns)}"
    )

_forbidden_tokens = (
    "event_label",
    "is_anomaly",
    "event_start",
    "event_end",
    "event_description",
    "fault_type",
    "care_ground_truth",
)
if any(
    any(token in str(column).lower() for token in _forbidden_tokens)
    for column in CASE_REGISTRY.columns
):
    raise RuntimeError("An outcome field is present in CASE_REGISTRY.")

if EVALUATION.outer_strategy != "leave-one-asset-out":
    raise RuntimeError("Cell 3 requires the frozen leave-one-asset-out strategy.")
if EVALUATION.inner_group_column != "asset_id":
    raise RuntimeError("Inner resampling must remain grouped by asset_id.")

CELL3_VERSION = "1.0.0"
CELL3_CACHE_ROOT = Path(CACHE_DIR) / "cell3_case_preprocessing"
CELL3_FOLD_DIR = Path(INVENTORY_DIR) / "cell3_folds"
CELL3_QUALITY_DIR = Path(QUALITY_DIR) / "cell3_preprocessing"
for _directory in (CELL3_CACHE_ROOT, CELL3_FOLD_DIR, CELL3_QUALITY_DIR):
    _directory.mkdir(parents=True, exist_ok=True)


# =============================================================================
# 1. Frozen preprocessing policy derived from Cell 2 diagnostics
# =============================================================================

PREPROCESSING_POLICY = {
    "version": CELL3_VERSION,
    "input_statistics": tuple(QUALITY.primary_statistics),
    "source_fit_rows": "train partition AND normal status, independently per case",
    "time_order": "stable chronological order; no sorting quantity uses outcomes",
    "continuity": (
        f"new segment when delta is not exactly {DATASET.sampling_minutes} minutes; "
        "all temporal transformations restart"
    ),
    "numeric_missing": "non-finite values remain missing",
    "zero_handling": (
        "never convert sustained zero runs globally; convert zero to missing only "
        "when that case-channel is structurally zero across all finite fit rows"
    ),
    "availability": (
        f"case-channel usable only when fit availability >= "
        f"{QUALITY.minimum_sensor_availability:.2f}"
    ),
    "driver_imputation": (
        "case-local training-normal median; missingness indicators retained; "
        "imputation parameters frozen before prediction rows"
    ),
    "target_missing": "not imputed; masked from fitting, uncertainty calibration, and scoring",
    "angles": "degrees mapped to sine and cosine; reset-neutral and bounded",
    "counters": (
        "within-segment first difference; first row, negative resets, and "
        "missing-adjacent differences are missing"
    ),
    "global_sensor_selection": False,
    "event_outcomes_read": False,
    "outer_prediction_rows_used_to_fit_preprocessing": False,
    "cache_dtype": "float32 values, boolean masks, int32 segment identifiers",
}
PREPROCESSING_POLICY_SHA256 = sha256_json(PREPROCESSING_POLICY)


# =============================================================================
# 2. Deterministic leave-one-asset-out and farm-balanced inner folds
# =============================================================================

def stable_hash_int(text: str, seed: int) -> int:
    payload = f"{seed}|{text}".encode("utf-8")
    return int.from_bytes(hashlib.sha256(payload).digest()[:8], "big", signed=False)


ASSET_REGISTRY = (
    CASE_REGISTRY.groupby(["farm", "asset_id", "source_asset_id"], as_index=False)
    .agg(cases=("case_key", "nunique"))
    .sort_values(["farm", "asset_id"], kind="stable")
    .reset_index(drop=True)
)

if len(ASSET_REGISTRY) != DATASET.expected_assets:
    raise RuntimeError(
        f"Found {len(ASSET_REGISTRY)} farm-qualified assets; "
        f"expected {DATASET.expected_assets}."
    )
if ASSET_REGISTRY.groupby("asset_id")["farm"].nunique().max() != 1:
    raise RuntimeError("A canonical asset_id appears in more than one farm.")

_outer_records: list[dict[str, Any]] = []
_inner_records: list[dict[str, Any]] = []

for _outer_index, _outer_asset in enumerate(ASSET_REGISTRY["asset_id"], start=1):
    _outer_farm = str(
        ASSET_REGISTRY.loc[ASSET_REGISTRY["asset_id"].eq(_outer_asset), "farm"].iloc[0]
    )
    _outer_cases = CASE_REGISTRY.loc[CASE_REGISTRY["asset_id"].eq(_outer_asset)]
    _development_assets = ASSET_REGISTRY.loc[
        ~ASSET_REGISTRY["asset_id"].eq(_outer_asset)
    ].copy()

    _outer_records.append(
        {
            "outer_fold": _outer_index,
            "outer_asset_id": _outer_asset,
            "outer_farm": _outer_farm,
            "outer_cases": int(len(_outer_cases)),
            "development_assets": int(len(_development_assets)),
            "development_cases": int(
                CASE_REGISTRY["asset_id"].isin(_development_assets["asset_id"]).sum()
            ),
        }
    )

    # Balance each farm's assets across the global five folds. The start offset
    # changes by outer fold, preventing one numbered fold from repeatedly holding
    # the same farm mix while keeping the assignment deterministic.
    _inner_assignment: dict[str, int] = {}
    for _farm_index, _farm in enumerate(DATASET.farms):
        _farm_assets = _development_assets.loc[
            _development_assets["farm"].eq(_farm), "asset_id"
        ].tolist()
        _farm_assets.sort(
            key=lambda asset: (stable_hash_int(asset, REPRODUCIBILITY.seed + _outer_index), asset)
        )
        _offset = (_outer_index + _farm_index) % EVALUATION.inner_splits
        for _position, _asset in enumerate(_farm_assets):
            _inner_assignment[_asset] = (
                (_position + _offset) % EVALUATION.inner_splits
            ) + 1

    if set(_inner_assignment) != set(_development_assets["asset_id"]):
        raise RuntimeError(f"Incomplete inner assignment in outer fold {_outer_index}.")

    _used_folds = set(_inner_assignment.values())
    if _used_folds != set(range(1, EVALUATION.inner_splits + 1)):
        raise RuntimeError(
            f"Outer fold {_outer_index} does not use all inner fold numbers: {_used_folds}"
        )

    for _asset, _inner_fold in sorted(_inner_assignment.items()):
        _asset_row = _development_assets.loc[
            _development_assets["asset_id"].eq(_asset)
        ].iloc[0]
        _inner_records.append(
            {
                "outer_fold": _outer_index,
                "outer_asset_id": _outer_asset,
                "inner_fold": _inner_fold,
                "asset_id": _asset,
                "farm": _asset_row["farm"],
                "cases": int(_asset_row["cases"]),
            }
        )

OUTER_FOLDS = pd.DataFrame(_outer_records)
INNER_ASSET_FOLDS = pd.DataFrame(_inner_records).sort_values(
    ["outer_fold", "inner_fold", "farm", "asset_id"], kind="stable"
).reset_index(drop=True)

if OUTER_FOLDS["outer_asset_id"].nunique() != DATASET.expected_assets:
    raise RuntimeError("Each asset must occur exactly once as the outer test asset.")
if len(INNER_ASSET_FOLDS) != DATASET.expected_assets * (DATASET.expected_assets - 1):
    raise RuntimeError("Unexpected number of outer-development asset assignments.")

for _outer_fold in OUTER_FOLDS["outer_fold"]:
    _outer_asset = OUTER_FOLDS.loc[
        OUTER_FOLDS["outer_fold"].eq(_outer_fold), "outer_asset_id"
    ].iloc[0]
    _assignments = INNER_ASSET_FOLDS.loc[
        INNER_ASSET_FOLDS["outer_fold"].eq(_outer_fold)
    ]
    if _outer_asset in set(_assignments["asset_id"]):
        raise RuntimeError(f"Outer asset leaked into inner folds for outer fold {_outer_fold}.")
    if _assignments["asset_id"].duplicated().any():
        raise RuntimeError(f"An inner asset is duplicated for outer fold {_outer_fold}.")
    if _assignments["asset_id"].nunique() != DATASET.expected_assets - 1:
        raise RuntimeError(f"Outer fold {_outer_fold} has the wrong development asset count.")


def iter_outer_folds() -> Iterator[dict[str, Any]]:
    """Yield safe case registries for one leave-one-asset-out fold at a time."""
    for row in OUTER_FOLDS.itertuples(index=False):
        test_mask = CASE_REGISTRY["asset_id"].eq(row.outer_asset_id)
        development = CASE_REGISTRY.loc[~test_mask].copy()
        test = CASE_REGISTRY.loc[test_mask].copy()
        if set(development["asset_id"]) & set(test["asset_id"]):
            raise RuntimeError(f"Asset leakage in outer fold {row.outer_fold}.")
        yield {
            "outer_fold": int(row.outer_fold),
            "outer_asset_id": row.outer_asset_id,
            "development_cases": development,
            "test_cases": test,
            "inner_asset_folds": INNER_ASSET_FOLDS.loc[
                INNER_ASSET_FOLDS["outer_fold"].eq(row.outer_fold)
            ].copy(),
        }


def iter_inner_folds(outer_fold: int) -> Iterator[dict[str, Any]]:
    """Yield development/validation asset partitions inside one outer fold."""
    assignments = INNER_ASSET_FOLDS.loc[
        INNER_ASSET_FOLDS["outer_fold"].eq(outer_fold)
    ]
    if assignments.empty:
        raise KeyError(f"Unknown outer fold: {outer_fold}")
    outer_asset = OUTER_FOLDS.loc[
        OUTER_FOLDS["outer_fold"].eq(outer_fold), "outer_asset_id"
    ].iloc[0]
    outer_development = CASE_REGISTRY.loc[
        ~CASE_REGISTRY["asset_id"].eq(outer_asset)
    ]
    for inner_fold in range(1, EVALUATION.inner_splits + 1):
        validation_assets = set(
            assignments.loc[assignments["inner_fold"].eq(inner_fold), "asset_id"]
        )
        validation = outer_development.loc[
            outer_development["asset_id"].isin(validation_assets)
        ].copy()
        development = outer_development.loc[
            ~outer_development["asset_id"].isin(validation_assets)
        ].copy()
        if validation.empty or development.empty:
            raise RuntimeError(
                f"Empty inner partition: outer={outer_fold}, inner={inner_fold}."
            )
        if set(validation["asset_id"]) & set(development["asset_id"]):
            raise RuntimeError(
                f"Asset leakage: outer={outer_fold}, inner={inner_fold}."
            )
        yield {
            "outer_fold": outer_fold,
            "inner_fold": inner_fold,
            "development_cases": development,
            "validation_cases": validation,
        }


# =============================================================================
# 3. Case-local, training-normal preprocessing helpers
# =============================================================================

def _strict_segment_ids(timestamps: np.ndarray) -> np.ndarray:
    timestamps = np.asarray(timestamps, dtype="datetime64[ns]")
    if timestamps.ndim != 1:
        raise ValueError("timestamps must be one-dimensional")
    if len(timestamps) == 0:
        return np.empty(0, dtype=np.int32)
    values = timestamps.astype(np.int64)
    nat_ns = np.iinfo(np.int64).min
    expected_ns = int(DATASET.sampling_minutes * 60 * 1_000_000_000)
    new_segment = np.ones(len(values), dtype=bool)
    if len(values) > 1:
        delta = values[1:] - values[:-1]
        new_segment[1:] = (
            (values[1:] == nat_ns)
            | (values[:-1] == nat_ns)
            | (delta != expected_ns)
        )
    return (np.cumsum(new_segment, dtype=np.int64) - 1).astype(np.int32)


def _counter_differences(values: np.ndarray, segment_ids: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=np.float64)
    output = np.full_like(values, np.nan, dtype=np.float64)
    if len(values) <= 1 or values.shape[1] == 0:
        return output
    same_segment = segment_ids[1:] == segment_ids[:-1]
    differences = values[1:] - values[:-1]
    valid = (
        same_segment[:, None]
        & np.isfinite(values[1:])
        & np.isfinite(values[:-1])
        & np.isfinite(differences)
        & (differences >= 0.0)
    )
    output[1:] = np.where(valid, differences, np.nan)
    return output


def _safe_slug(text: str) -> str:
    cleaned = "".join(character if character.isalnum() else "_" for character in text)
    return "_".join(part for part in cleaned.split("_") if part)


def _atomic_save_npz(output_path: Path, **arrays: np.ndarray) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = output_path.with_name(output_path.name + ".tmp.npz")
    np.savez_compressed(temporary, **arrays)
    temporary.replace(output_path)


def _json_file_sha256(path: Path) -> str:
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def _first_pass_fit_statistics(case_row: Any, channels: list[str]) -> dict[str, Any]:
    count = np.zeros(len(channels), dtype=np.int64)
    zero_count = np.zeros(len(channels), dtype=np.int64)
    mean = np.zeros(len(channels), dtype=np.float64)
    m2 = np.zeros(len(channels), dtype=np.float64)
    total_rows = fit_rows = train_rows = prediction_rows = unknown_rows = 0
    asset_values: set[str] = set()

    required = [*DATASET.metadata_columns, *channels]
    for chunk in read_care_chunks(Path(case_row.file_path), required):
        total_rows += len(chunk)
        partition = normalized_partition(chunk["train_test"])
        is_train = partition.eq("train").to_numpy()
        is_prediction = partition.eq("prediction").to_numpy()
        unknown_rows += int(partition.eq("unknown").sum())
        train_rows += int(is_train.sum())
        prediction_rows += int(is_prediction.sum())
        status = pd.to_numeric(chunk["status_type_id"], errors="coerce")
        fit_mask = is_train & status.isin(DATASET.normal_status_ids).to_numpy()
        fit_rows += int(fit_mask.sum())
        asset_values.update(str(value).strip() for value in chunk["asset_id"].dropna().unique())

        values = chunk[channels].apply(pd.to_numeric, errors="coerce").to_numpy(np.float64)
        fit_values = values[fit_mask]
        finite = np.isfinite(fit_values)
        zero_count += (finite & (fit_values == 0.0)).sum(axis=0, dtype=np.int64)

        for column_index in range(len(channels)):
            observed = fit_values[finite[:, column_index], column_index]
            n_batch = len(observed)
            if n_batch == 0:
                continue
            batch_mean = float(observed.mean())
            batch_m2 = float(np.square(observed - batch_mean).sum())
            n_old = int(count[column_index])
            n_new = n_old + n_batch
            delta = batch_mean - mean[column_index]
            mean[column_index] += delta * n_batch / n_new
            m2[column_index] += batch_m2 + delta * delta * n_old * n_batch / n_new
            count[column_index] = n_new

    availability = count / max(fit_rows, 1)
    variance = np.divide(
        m2,
        np.maximum(count - 1, 1),
        out=np.zeros_like(m2),
        where=count > 1,
    )
    structurally_zero = (count > 0) & (zero_count == count)
    usable = (
        (availability >= QUALITY.minimum_sensor_availability)
        & (count > 1)
        & np.isfinite(variance)
        & (variance > 0.0)
        & ~structurally_zero
    )
    if fit_rows < QUALITY.minimum_training_rows:
        raise RuntimeError(
            f"{case_row.case_key}: {fit_rows} training-normal rows; "
            f"minimum is {QUALITY.minimum_training_rows}."
        )
    if unknown_rows:
        raise RuntimeError(f"{case_row.case_key}: {unknown_rows} unknown partition rows.")

    return {
        "total_rows": total_rows,
        "train_rows": train_rows,
        "prediction_rows": prediction_rows,
        "fit_rows": fit_rows,
        "asset_values": sorted(asset_values),
        "finite_count": count,
        "zero_count": zero_count,
        "mean": mean,
        "variance": variance,
        "availability": availability,
        "structurally_zero": structurally_zero,
        "usable": usable,
    }


def _second_pass_arrays(
    case_row: Any,
    channels: list[str],
    structurally_zero: np.ndarray,
) -> dict[str, np.ndarray]:
    timestamps_parts: list[np.ndarray] = []
    partition_parts: list[np.ndarray] = []
    normal_status_parts: list[np.ndarray] = []
    values_parts: list[np.ndarray] = []

    required = [*DATASET.metadata_columns, *channels]
    for chunk in read_care_chunks(Path(case_row.file_path), required):
        timestamps_parts.append(chunk["time_stamp"].to_numpy(dtype="datetime64[ns]"))
        partition = normalized_partition(chunk["train_test"])
        partition_code = np.full(len(chunk), -1, dtype=np.int8)
        partition_code[partition.eq("train").to_numpy()] = 0
        partition_code[partition.eq("prediction").to_numpy()] = 1
        partition_parts.append(partition_code)
        status = pd.to_numeric(chunk["status_type_id"], errors="coerce")
        normal_status_parts.append(
            status.isin(DATASET.normal_status_ids).to_numpy(dtype=bool)
        )
        values = chunk[channels].apply(pd.to_numeric, errors="coerce").to_numpy(np.float64)
        values[~np.isfinite(values)] = np.nan
        if structurally_zero.any():
            values[:, structurally_zero] = np.nan
        values_parts.append(values)

    timestamps = np.concatenate(timestamps_parts)
    partition_code = np.concatenate(partition_parts)
    normal_status = np.concatenate(normal_status_parts)
    values = np.concatenate(values_parts, axis=0)
    order = np.argsort(timestamps.astype(np.int64), kind="stable")
    timestamps = timestamps[order]
    partition_code = partition_code[order]
    normal_status = normal_status[order]
    values = values[order]

    segment_id = _strict_segment_ids(timestamps)
    fit_mask = (partition_code == 0) & normal_status
    prediction_mask = partition_code == 1
    return {
        "timestamp_ns": timestamps.astype(np.int64),
        "partition_code": partition_code,
        "normal_status": normal_status,
        "fit_mask": fit_mask,
        "prediction_mask": prediction_mask,
        "segment_id": segment_id,
        "values": values,
    }


def _fit_medians(values: np.ndarray, fit_mask: np.ndarray) -> np.ndarray:
    medians = np.full(values.shape[1], np.nan, dtype=np.float64)
    for index in range(values.shape[1]):
        observed = values[fit_mask, index]
        observed = observed[np.isfinite(observed)]
        if len(observed):
            medians[index] = float(np.median(observed))
    return medians


def prepare_case_cache(case_row: Any) -> dict[str, Any]:
    farm = str(case_row.farm)
    feature_table = CARE_FEATURE_REGISTRY.loc[
        CARE_FEATURE_REGISTRY["farm"].eq(farm)
        & CARE_FEATURE_REGISTRY["primary_analysis"]
    ].copy()
    feature_table = feature_table.sort_values("column", kind="stable").reset_index(drop=True)
    channels = feature_table["column"].tolist()
    if not channels:
        raise RuntimeError(f"{farm} has no primary average channels.")

    first_pass = _first_pass_fit_statistics(case_row, channels)
    arrays = _second_pass_arrays(case_row, channels, first_pass["structurally_zero"])
    values = arrays.pop("values")
    if len(values) != first_pass["total_rows"]:
        raise RuntimeError(f"{case_row.case_key}: first/second pass row mismatch.")

    metadata_source_asset = str(case_row.source_asset_id)
    observed_source_assets = {
        str(int(float(value))) if str(value).replace(".", "", 1).isdigit() and float(value).is_integer()
        else str(value)
        for value in first_pass["asset_values"]
    }
    if observed_source_assets != {metadata_source_asset}:
        raise RuntimeError(
            f"{case_row.case_key}: asset mismatch, source={observed_source_assets}, "
            f"metadata={metadata_source_asset}."
        )

    is_angle = feature_table["is_angle"].to_numpy(dtype=bool)
    is_counter = feature_table["is_counter"].to_numpy(dtype=bool)
    role = feature_table["role"].astype(str).to_numpy()
    usable = first_pass["usable"].copy()

    counter_values = values[:, is_counter]
    counter_names = [f"{name}__rate" for name in np.asarray(channels)[is_counter]]
    counter_differences = _counter_differences(counter_values, arrays["segment_id"])
    counter_rate_usable = np.zeros(counter_differences.shape[1], dtype=bool)
    for counter_index in range(counter_differences.shape[1]):
        observed = counter_differences[arrays["fit_mask"], counter_index]
        observed = observed[np.isfinite(observed)]
        rate_availability = len(observed) / max(int(arrays["fit_mask"].sum()), 1)
        counter_rate_usable[counter_index] = (
            rate_availability >= QUALITY.minimum_sensor_availability
            and len(observed) > 1
            and float(np.var(observed, ddof=1)) > 0.0
        )
    usable[is_counter] = counter_rate_usable

    noncounter = ~is_counter
    identity_mask = noncounter & ~is_angle
    angle_mask = noncounter & is_angle

    driver_identity = usable & identity_mask & (role == "operating_driver_candidate")
    driver_angles = usable & angle_mask & (role == "operating_driver_candidate")
    driver_counters = usable & is_counter & (role == "operating_driver_candidate")
    target_identity = usable & identity_mask & (role != "operating_driver_candidate")
    target_angles = usable & angle_mask & (role != "operating_driver_candidate")
    target_counters = usable & is_counter & (role != "operating_driver_candidate")

    driver_blocks: list[np.ndarray] = []
    driver_names: list[str] = []
    driver_imputation_medians: list[float] = []
    driver_missing_indicator_names: list[str] = []

    if driver_identity.any():
        block = values[:, driver_identity]
        names = list(np.asarray(channels)[driver_identity])
        medians = _fit_medians(block, arrays["fit_mask"])
        missing = ~np.isfinite(block)
        block = np.where(missing, medians[None, :], block)
        driver_blocks.append(block)
        driver_names.extend(names)
        driver_imputation_medians.extend(medians.tolist())
        # Add indicators only for channels that are actually missing anywhere.
        for column_index, name in enumerate(names):
            if missing[:, column_index].any():
                driver_blocks.append(missing[:, [column_index]].astype(np.float64))
                driver_names.append(f"{name}__missing")
                driver_imputation_medians.append(0.0)
                driver_missing_indicator_names.append(f"{name}__missing")

    if driver_angles.any():
        block = values[:, driver_angles]
        names = list(np.asarray(channels)[driver_angles])
        radians = np.deg2rad(block)
        for column_index, name in enumerate(names):
            pair = np.column_stack((np.sin(radians[:, column_index]), np.cos(radians[:, column_index])))
            missing = ~np.isfinite(pair).all(axis=1)
            pair[missing] = 0.0
            driver_blocks.append(pair)
            driver_names.extend((f"{name}__sin", f"{name}__cos"))
            driver_imputation_medians.extend((0.0, 0.0))
            if missing.any():
                driver_blocks.append(missing[:, None].astype(np.float64))
                driver_names.append(f"{name}__missing")
                driver_imputation_medians.append(0.0)
                driver_missing_indicator_names.append(f"{name}__missing")

    if driver_counters.any():
        counter_selected = driver_counters[is_counter]
        block = counter_differences[:, counter_selected]
        names = list(np.asarray(counter_names)[counter_selected])
        medians = _fit_medians(block, arrays["fit_mask"])
        missing = ~np.isfinite(block)
        block = np.where(missing, medians[None, :], block)
        driver_blocks.append(block)
        driver_names.extend(names)
        driver_imputation_medians.extend(medians.tolist())
        for column_index, name in enumerate(names):
            if missing[:, column_index].any():
                driver_blocks.append(missing[:, [column_index]].astype(np.float64))
                driver_names.append(f"{name}__missing")
                driver_imputation_medians.append(0.0)
                driver_missing_indicator_names.append(f"{name}__missing")

    target_blocks: list[np.ndarray] = []
    target_names: list[str] = []
    target_temperature: list[bool] = []

    if target_identity.any():
        target_blocks.append(values[:, target_identity])
        identity_names = list(np.asarray(channels)[target_identity])
        target_names.extend(identity_names)
        target_temperature.extend(
            (role[target_identity] == "temperature_target_candidate").tolist()
        )

    if target_angles.any():
        block = values[:, target_angles]
        names = list(np.asarray(channels)[target_angles])
        temp = role[target_angles] == "temperature_target_candidate"
        radians = np.deg2rad(block)
        for column_index, name in enumerate(names):
            target_blocks.append(
                np.column_stack((np.sin(radians[:, column_index]), np.cos(radians[:, column_index])))
            )
            target_names.extend((f"{name}__sin", f"{name}__cos"))
            target_temperature.extend((bool(temp[column_index]), bool(temp[column_index])))

    if target_counters.any():
        counter_selected = target_counters[is_counter]
        target_blocks.append(counter_differences[:, counter_selected])
        selected_names = list(np.asarray(counter_names)[counter_selected])
        target_names.extend(selected_names)
        target_temperature.extend([False] * len(selected_names))

    if not driver_blocks:
        raise RuntimeError(f"{case_row.case_key}: no usable operating drivers.")
    if not target_blocks:
        raise RuntimeError(f"{case_row.case_key}: no usable monitoring targets.")

    driver_matrix = np.concatenate(driver_blocks, axis=1).astype(np.float32)
    target_matrix = np.concatenate(target_blocks, axis=1).astype(np.float32)
    target_observed = np.isfinite(target_matrix)

    if not np.isfinite(driver_matrix).all():
        raise RuntimeError(f"{case_row.case_key}: driver imputation left non-finite values.")
    if len(driver_names) != driver_matrix.shape[1]:
        raise RuntimeError(f"{case_row.case_key}: driver-name mismatch.")
    if len(target_names) != target_matrix.shape[1]:
        raise RuntimeError(f"{case_row.case_key}: target-name mismatch.")

    cache_relative = Path(_safe_slug(farm)) / f"event_{int(case_row.event_id):03d}.npz"
    metadata_relative = Path(_safe_slug(farm)) / f"event_{int(case_row.event_id):03d}.json"
    cache_path = CELL3_CACHE_ROOT / cache_relative
    metadata_path = CELL3_CACHE_ROOT / metadata_relative

    case_metadata = {
        "cell3_version": CELL3_VERSION,
        "contract_sha256": CONTRACT_SHA256,
        "cell2_audit_sha256": CELL2_AUDIT_SHA256,
        "preprocessing_policy_sha256": PREPROCESSING_POLICY_SHA256,
        "case_key": case_row.case_key,
        "farm": farm,
        "asset_id": case_row.asset_id,
        "source_asset_id": metadata_source_asset,
        "event_id": int(case_row.event_id),
        "rows": int(len(driver_matrix)),
        "training_normal_rows": int(arrays["fit_mask"].sum()),
        "prediction_rows": int(arrays["prediction_mask"].sum()),
        "segments": int(arrays["segment_id"].max() + 1),
        "raw_primary_channels": channels,
        "raw_roles": role.tolist(),
        "raw_availability": first_pass["availability"].tolist(),
        "raw_finite_count": first_pass["finite_count"].tolist(),
        "raw_zero_count": first_pass["zero_count"].tolist(),
        "raw_structurally_zero": first_pass["structurally_zero"].tolist(),
        "raw_usable": usable.tolist(),
        "driver_names": driver_names,
        "driver_imputation_medians": driver_imputation_medians,
        "driver_missing_indicator_names": driver_missing_indicator_names,
        "target_names": target_names,
        "target_temperature": target_temperature,
        "outcome_fields_present": False,
    }
    case_preprocessing_sha256 = sha256_json(case_metadata)
    case_metadata["case_preprocessing_sha256"] = case_preprocessing_sha256

    if metadata_path.exists() and cache_path.exists():
        existing = json.loads(metadata_path.read_text(encoding="utf-8"))
        if existing.get("case_preprocessing_sha256") != case_preprocessing_sha256:
            raise RuntimeError(
                f"Preprocessing drift for {case_row.case_key}. Do not overwrite the cache."
            )
        cache_sha256 = _json_file_sha256(cache_path)
        if existing.get("cache_sha256") != cache_sha256:
            raise RuntimeError(
                f"Cached arrays failed their saved hash for {case_row.case_key}. "
                "Delete only that derived cache and rerun Cell 3."
            )
    else:
        _atomic_save_npz(
            cache_path,
            timestamp_ns=arrays["timestamp_ns"],
            partition_code=arrays["partition_code"],
            normal_status=arrays["normal_status"],
            fit_mask=arrays["fit_mask"],
            prediction_mask=arrays["prediction_mask"],
            segment_id=arrays["segment_id"],
            drivers=driver_matrix,
            targets=target_matrix,
            target_observed=target_observed,
            target_temperature=np.asarray(target_temperature, dtype=bool),
        )
        cache_sha256 = _json_file_sha256(cache_path)
        case_metadata["cache_sha256"] = cache_sha256
        save_json(case_metadata, metadata_path)

    return {
        "case_key": case_row.case_key,
        "farm": farm,
        "asset_id": case_row.asset_id,
        "event_id": int(case_row.event_id),
        "rows": int(len(driver_matrix)),
        "training_normal_rows": int(arrays["fit_mask"].sum()),
        "prediction_rows": int(arrays["prediction_mask"].sum()),
        "segments": int(arrays["segment_id"].max() + 1),
        "primary_channels": len(channels),
        "usable_driver_channels": int(
            driver_identity.sum() + driver_angles.sum() + driver_counters.sum()
        ),
        "usable_target_channels": int(
            target_identity.sum() + target_angles.sum() + target_counters.sum()
        ),
        "unusable_channels": int((~usable).sum()),
        "structurally_zero_channels": int(first_pass["structurally_zero"].sum()),
        "cache_relative_path": cache_relative.as_posix(),
        "metadata_relative_path": metadata_relative.as_posix(),
        "case_preprocessing_sha256": case_preprocessing_sha256,
        "cache_sha256": cache_sha256,
    }


def load_case_cache(case_key: str) -> dict[str, Any]:
    """Load a safe cached case by key; no outcomes are present in this cache."""
    match = CASE_CACHE_REGISTRY.loc[CASE_CACHE_REGISTRY["case_key"].eq(case_key)]
    if len(match) != 1:
        raise KeyError(f"Unknown or duplicate case_key: {case_key}")
    row = match.iloc[0]
    metadata_path = CELL3_CACHE_ROOT / row["metadata_relative_path"]
    cache_path = CELL3_CACHE_ROOT / row["cache_relative_path"]
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    if metadata.get("outcome_fields_present") is not False:
        raise RuntimeError(f"Unsafe cache metadata for {case_key}.")
    if _json_file_sha256(cache_path) != row["cache_sha256"]:
        raise RuntimeError(f"Cache hash mismatch for {case_key}.")
    arrays = dict(np.load(cache_path, allow_pickle=False))
    arrays["metadata"] = metadata
    return arrays


# =============================================================================
# 4. Materialize all label-free case caches
# =============================================================================

_cache_receipts: list[dict[str, Any]] = []
for _farm in DATASET.farms:
    _farm_cases = CASE_REGISTRY.loc[CASE_REGISTRY["farm"].eq(_farm)]
    print(f"Preparing label-free case caches — {_farm}: {len(_farm_cases)} cases", flush=True)
    for _row in _farm_cases.itertuples(index=False):
        _cache_receipts.append(prepare_case_cache(_row))

CASE_CACHE_REGISTRY = pd.DataFrame(_cache_receipts).sort_values(
    ["farm", "event_id"], kind="stable"
).reset_index(drop=True)

if len(CASE_CACHE_REGISTRY) != DATASET.expected_total_cases:
    raise RuntimeError("Not all 95 cases produced a preprocessing cache.")
if CASE_CACHE_REGISTRY["case_key"].duplicated().any():
    raise RuntimeError("Duplicate case cache keys were produced.")

# A small sample is loaded immediately to verify cache schema and hashes.
for _case_key in CASE_CACHE_REGISTRY.groupby("farm", sort=False).head(1)["case_key"]:
    _sample = load_case_cache(_case_key)
    _required_arrays = {
        "timestamp_ns",
        "partition_code",
        "normal_status",
        "fit_mask",
        "prediction_mask",
        "segment_id",
        "drivers",
        "targets",
        "target_observed",
        "target_temperature",
        "metadata",
    }
    if set(_sample) != _required_arrays:
        raise RuntimeError(f"Unexpected cache members for {_case_key}: {set(_sample)}")
    if len(_sample["drivers"]) != len(_sample["targets"]):
        raise RuntimeError(f"Row mismatch inside cache {_case_key}.")
    del _sample


# =============================================================================
# 5. Freeze Cell 3 receipt and write auditable tables
# =============================================================================

def dataframe_sha256(frame: pd.DataFrame, sort_columns: Iterable[str]) -> str:
    ordered = frame.sort_values(list(sort_columns), kind="stable").reset_index(drop=True)
    payload = ordered.to_csv(
        index=False,
        lineterminator="\n",
        na_rep="<NA>",
        float_format="%.12g",
    )
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


_component_hashes = {
    "asset_registry_sha256": dataframe_sha256(ASSET_REGISTRY, ("farm", "asset_id")),
    "outer_folds_sha256": dataframe_sha256(OUTER_FOLDS, ("outer_fold",)),
    "inner_asset_folds_sha256": dataframe_sha256(
        INNER_ASSET_FOLDS, ("outer_fold", "inner_fold", "farm", "asset_id")
    ),
    "case_cache_registry_sha256": dataframe_sha256(
        CASE_CACHE_REGISTRY, ("farm", "event_id")
    ),
}
CELL3_RECEIPT_SHA256 = sha256_json(
    {
        "contract_sha256": CONTRACT_SHA256,
        "cell2_audit_sha256": CELL2_AUDIT_SHA256,
        "preprocessing_policy_sha256": PREPROCESSING_POLICY_SHA256,
        "component_hashes": _component_hashes,
    }
)

CELL3_RECEIPT_PATH = CELL3_FOLD_DIR / "cell3_fold_and_preprocessing_receipt.json"
if CELL3_RECEIPT_PATH.exists():
    _existing = json.loads(CELL3_RECEIPT_PATH.read_text(encoding="utf-8"))
    if _existing.get("cell3_receipt_sha256") != CELL3_RECEIPT_SHA256:
        raise RuntimeError(
            "A different Cell 3 receipt exists for this experiment. Do not overwrite it."
        )
    CELL3_STATE = "existing identical Cell 3 receipt verified"
    _write_receipt = False
else:
    CELL3_STATE = "new Cell 3 receipt frozen"
    _write_receipt = True

save_csv_atomic(ASSET_REGISTRY, CELL3_FOLD_DIR / "asset_registry.csv")
save_csv_atomic(OUTER_FOLDS, CELL3_FOLD_DIR / "outer_leave_one_asset_out_folds.csv")
save_csv_atomic(INNER_ASSET_FOLDS, CELL3_FOLD_DIR / "inner_grouped_asset_folds.csv")
save_csv_atomic(CASE_CACHE_REGISTRY, CELL3_QUALITY_DIR / "case_cache_registry.csv")
save_json(PREPROCESSING_POLICY, CELL3_QUALITY_DIR / "preprocessing_policy.json")

if _write_receipt:
    save_json(
        {
            "cell3_version": CELL3_VERSION,
            "created_at_utc": utc_now(),
            "contract_sha256": CONTRACT_SHA256,
            "dataset_manifest_sha256": DATASET_MANIFEST_SHA256,
            "outcome_lockbox_sha256": OUTCOME_LOCKBOX_SHA256,
            "cell2_audit_sha256": CELL2_AUDIT_SHA256,
            "preprocessing_policy_sha256": PREPROCESSING_POLICY_SHA256,
            "component_hashes": _component_hashes,
            "cell3_receipt_sha256": CELL3_RECEIPT_SHA256,
            "outcomes_read": False,
            "global_sensor_selection": False,
        },
        CELL3_RECEIPT_PATH,
    )


# =============================================================================
# 6. Concise notebook report
# =============================================================================

_fold_balance = (
    INNER_ASSET_FOLDS.groupby(["outer_fold", "inner_fold"])
    .agg(validation_assets=("asset_id", "nunique"), validation_cases=("cases", "sum"))
    .reset_index()
)
INNER_FOLD_BALANCE_SUMMARY = (
    _fold_balance.groupby("inner_fold")
    .agg(
        min_validation_assets=("validation_assets", "min"),
        max_validation_assets=("validation_assets", "max"),
        min_validation_cases=("validation_cases", "min"),
        max_validation_cases=("validation_cases", "max"),
    )
    .reset_index()
)

CASE_CACHE_SUMMARY = (
    CASE_CACHE_REGISTRY.groupby("farm", sort=False)
    .agg(
        cases=("case_key", "size"),
        assets=("asset_id", "nunique"),
        rows=("rows", "sum"),
        median_drivers=("usable_driver_channels", "median"),
        median_targets=("usable_target_channels", "median"),
        median_unusable=("unusable_channels", "median"),
        structurally_zero_case_channels=("structurally_zero_channels", "sum"),
        median_segments=("segments", "median"),
    )
    .reset_index()
)

print("\n" + "=" * 92)
print("UC-RCF-NBM CELL 3 — ASSET FOLDS AND LEAKAGE-SAFE PREPROCESSING")
print("=" * 92)
print("\nASSET REGISTRY")
display(
    ASSET_REGISTRY.groupby("farm", sort=False)
    .agg(assets=("asset_id", "nunique"), cases=("cases", "sum"))
    .reset_index()
)
print("\nINNER-FOLD BALANCE ACROSS THE 36 OUTER FOLDS")
display(INNER_FOLD_BALANCE_SUMMARY)
print("\nLABEL-FREE CASE CACHE SUMMARY")
display(CASE_CACHE_SUMMARY)

print("\n" + "-" * 92)
print(f"Outer folds                     : {len(OUTER_FOLDS)}")
print(f"Outer test assets               : {OUTER_FOLDS['outer_asset_id'].nunique()}")
print(f"Inner folds per outer fold      : {EVALUATION.inner_splits}")
print(f"Cases cached                    : {len(CASE_CACHE_REGISTRY)}")
print(f"Preprocessing policy SHA-256    : {PREPROCESSING_POLICY_SHA256}")
print(f"Cell 3 receipt SHA-256          : {CELL3_RECEIPT_SHA256}")
print(f"Cell 3 state                    : {CELL3_STATE}")
print("Outcome metadata loaded         : No")
print("Case labels in cache            : No")
print("Global sensor selection applied : No")
print("Outer prediction used for fit   : No")
print("Fold leakage checks             : PASS")
print("=" * 92)
print("CELL 3 COMPLETED SUCCESSFULLY — ASSET FOLDS AND PREPROCESSING LOCKED")


Preparing label-free case caches — Wind Farm A: 22 cases
Preparing label-free case caches — Wind Farm B: 15 cases
Preparing label-free case caches — Wind Farm C: 58 cases

UC-RCF-NBM CELL 3 — ASSET FOLDS AND LEAKAGE-SAFE PREPROCESSING

ASSET REGISTRY


,farm,assets,cases
0,Wind Farm A,5,22
1,Wind Farm B,9,15
2,Wind Farm C,22,58



INNER-FOLD BALANCE ACROSS THE 36 OUTER FOLDS


,inner_fold,min_validation_assets,max_validation_assets,min_validation_cases,max_validation_cases
0,1,6,8,12,24
1,2,6,8,14,24
2,3,6,8,12,27
3,4,6,8,12,25
4,5,6,8,14,25



LABEL-FREE CASE CACHE SUMMARY


,farm,cases,assets,rows,median_drivers,median_targets,median_unusable,structurally_zero_case_channels,median_segments
0,Wind Farm A,22,5,1196747,7.0,39.0,0.0,0,17.5
1,Wind Farm B,15,9,859065,12.0,49.0,2.0,0,3.0
2,Wind Farm C,58,22,3187136,20.0,215.5,2.0,136,1.0



--------------------------------------------------------------------------------------------
Outer folds                     : 36
Outer test assets               : 36
Inner folds per outer fold      : 5
Cases cached                    : 95
Preprocessing policy SHA-256    : 67ccb2442d0a44393ca7e3cc0bc8030f7cc3fc69458c1dce9f9b36c57577dcc9
Cell 3 receipt SHA-256          : aded107bea4397babbf24f4b9ab5740d9cfdd117a3783b3f03bd1cbcfbc55762
Cell 3 state                    : new Cell 3 receipt frozen
Outcome metadata loaded         : No
Case labels in cache            : No
Global sensor selection applied : No
Outer prediction used for fit   : No
Fold leakage checks             : PASS
CELL 3 COMPLETED SUCCESSFULLY — ASSET FOLDS AND PREPROCESSING LOCKED


In [5]:
print("\nCOMPLETE CASE CACHE SUMMARY")
print(CASE_CACHE_SUMMARY.to_string(index=False))

print("\nCELL 3 HASHES")
print("Preprocessing policy :", PREPROCESSING_POLICY_SHA256)
print("Cell 3 receipt       :", CELL3_RECEIPT_SHA256)
print("Cell 3 state         :", CELL3_STATE)


COMPLETE CASE CACHE SUMMARY
       farm  cases  assets    rows  median_drivers  median_targets  median_unusable  structurally_zero_case_channels  median_segments
Wind Farm A     22       5 1196747             7.0            39.0              0.0                                0             17.5
Wind Farm B     15       9  859065            12.0            49.0              2.0                                0              3.0
Wind Farm C     58      22 3187136            20.0           215.5              2.0                              136              1.0

CELL 3 HASHES
Preprocessing policy : 67ccb2442d0a44393ca7e3cc0bc8030f7cc3fc69458c1dce9f9b36c57577dcc9
Cell 3 receipt       : aded107bea4397babbf24f4b9ab5740d9cfdd117a3783b3f03bd1cbcfbc55762
Cell 3 state         : new Cell 3 receipt frozen


In [6]:
"""CELL 4 — cross-fitted nonlinear ridge normal-behaviour mean model.

Paste this complete file into the fourth cell of the UC-RCF-NBM notebook and
run it only after Cells 1–3 have completed successfully.

For every CARE case, this cell constructs embargoed temporal folds from the
normal source-training history, refits all preprocessing inside each fold,
selects the ridge penalty by masked out-of-fold error, produces honest OOF
residuals, and fits the final normal-behaviour mean model. It never reads event
labels, event boundaries, failure descriptions, or CARE outcome scores.
"""

from __future__ import annotations

import hashlib
import json
import re
from collections import defaultdict
from itertools import combinations
from pathlib import Path
from typing import Any, Iterable, Iterator

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print


# =============================================================================
# 0. Bind Cell 4 to the completed experiment receipts
# =============================================================================

EXPECTED_CONTRACT_SHA256 = (
    "827641aecd8e807193ad193d64c274319756092faa53bdb5084f310f62041f49"
)
EXPECTED_DATASET_MANIFEST_SHA256 = (
    "62484bab1219888aa1d0788965ecd77db2b85f0bbb9b476cd3240f4143026f1f"
)
EXPECTED_CELL2_AUDIT_SHA256 = (
    "83732caf4ad3e226b69a671287bb14c4ed72e1bfc3b7c26ca144310fce8e5990"
)
EXPECTED_PREPROCESSING_POLICY_SHA256 = (
    "67ccb2442d0a44393ca7e3cc0bc8030f7cc3fc69458c1dce9f9b36c57577dcc9"
)
EXPECTED_CELL3_RECEIPT_SHA256 = (
    "aded107bea4397babbf24f4b9ab5740d9cfdd117a3783b3f03bd1cbcfbc55762"
)

_required_objects = (
    "CONTRACT_SHA256",
    "DATASET_MANIFEST_SHA256",
    "CELL2_AUDIT_SHA256",
    "PREPROCESSING_POLICY_SHA256",
    "CELL3_RECEIPT_SHA256",
    "DATASET",
    "MEAN_MODEL",
    "REPRODUCIBILITY",
    "CASE_REGISTRY",
    "CASE_CACHE_REGISTRY",
    "CARE_FEATURE_REGISTRY",
    "load_case_cache",
    "save_csv_atomic",
    "save_json",
    "sha256_json",
    "utc_now",
    "MODEL_DIR",
    "CACHE_DIR",
    "QUALITY_DIR",
)
_missing_objects = [name for name in _required_objects if name not in globals()]
if _missing_objects:
    raise RuntimeError(
        "Run UC-RCF-NBM Cells 1–3 before Cell 4. Missing objects: "
        + ", ".join(_missing_objects)
    )

_observed_receipts = {
    "contract": CONTRACT_SHA256,
    "dataset_manifest": DATASET_MANIFEST_SHA256,
    "cell2_audit": CELL2_AUDIT_SHA256,
    "preprocessing_policy": PREPROCESSING_POLICY_SHA256,
    "cell3_receipt": CELL3_RECEIPT_SHA256,
}
_expected_receipts = {
    "contract": EXPECTED_CONTRACT_SHA256,
    "dataset_manifest": EXPECTED_DATASET_MANIFEST_SHA256,
    "cell2_audit": EXPECTED_CELL2_AUDIT_SHA256,
    "preprocessing_policy": EXPECTED_PREPROCESSING_POLICY_SHA256,
    "cell3_receipt": EXPECTED_CELL3_RECEIPT_SHA256,
}
if _observed_receipts != _expected_receipts:
    raise RuntimeError(
        "Cell 4 is bound to the exact completed Cell 1–3 receipts. "
        f"Observed={_observed_receipts}, expected={_expected_receipts}."
    )

_forbidden_tokens = (
    "event_label",
    "is_anomaly",
    "event_start",
    "event_end",
    "event_description",
    "fault_type",
    "care_ground_truth",
)
if any(
    any(token in str(column).lower() for token in _forbidden_tokens)
    for column in CASE_REGISTRY.columns
):
    raise RuntimeError("Outcome information is present in the safe case registry.")

if MEAN_MODEL.crossfit_folds != 5:
    raise RuntimeError("Cell 4 requires the frozen five temporal cross-fitting folds.")
if MEAN_MODEL.temporal_block_steps <= MEAN_MODEL.embargo_steps:
    raise RuntimeError("Temporal blocks must be longer than the embargo.")
if tuple(MEAN_MODEL.ridge_grid) != tuple(sorted(MEAN_MODEL.ridge_grid)):
    raise RuntimeError("The frozen ridge grid must be ordered.")

CELL4_VERSION = "1.0.0"
CELL4_MODEL_ROOT = Path(MODEL_DIR) / "cell4_cross_fitted_mean_nbm"
CELL4_RESIDUAL_ROOT = Path(CACHE_DIR) / "cell4_cross_fitted_mean_residuals"
CELL4_QUALITY_ROOT = Path(QUALITY_DIR) / "cell4_cross_fitted_mean_nbm"
for _directory in (CELL4_MODEL_ROOT, CELL4_RESIDUAL_ROOT, CELL4_QUALITY_ROOT):
    _directory.mkdir(parents=True, exist_ok=True)


# =============================================================================
# 1. Freeze implementation choices that do not use outcome information
# =============================================================================

MEAN_MODEL_IMPLEMENTATION = {
    "version": CELL4_VERSION,
    "model": MEAN_MODEL.model,
    "basis": MEAN_MODEL.basis,
    "temporal_blocks": {
        "folds": MEAN_MODEL.crossfit_folds,
        "block_steps": MEAN_MODEL.temporal_block_steps,
        "embargo_steps": MEAN_MODEL.embargo_steps,
        "assignment": "round-robin complete blocks within continuous segments",
        "embargo": "two-sided in original row steps within each segment",
    },
    "fold_local_preprocessing": {
        "driver_imputation": "training-fold median reconstructed from missingness indicators",
        "driver_scaling": "training-fold median and IQR; standard-deviation fallback",
        "basis_scaling": "training-fold median and IQR; standard-deviation fallback",
        "target_scaling": "target-wise training-fold median and IQR; standard-deviation fallback",
    },
    "basis_terms": {
        "linear": "all nonconstant processed drivers and missingness indicators",
        "quadratic": "all non-indicator drivers",
        "wind_cubic": "non-indicator wind-speed drivers only",
        "interactions": (
            f"all pairs among at most {MEAN_MODEL.maximum_core_interaction_drivers} "
            "deterministically prioritized physical drivers"
        ),
    },
    "ridge": {
        "penalties": tuple(MEAN_MODEL.ridge_grid),
        "intercept_penalized": False,
        "selection": MEAN_MODEL.penalty_objective,
        "selection_residual_units": "fold-local robust target scales",
        "selection_residual_cap": MEAN_MODEL.residual_cap,
        "tie_break": "largest penalty among numerically tied minima",
    },
    "masked_targets": {
        "grouping": "targets sharing identical training-observation masks",
        "minimum_observed_rows": 200,
        "minimum_rows_above_design_dimension": 10,
        "missing_targets_imputed": False,
    },
    "outputs": {
        "oof_residuals": "raw target units on observed training-normal rows only",
        "final_model": "coefficients plus deterministic transformation state",
        "final_predictions_cached": False,
        "outcomes_read": False,
    },
}
MEAN_MODEL_IMPLEMENTATION_SHA256 = sha256_json(MEAN_MODEL_IMPLEMENTATION)

NUMERICAL_EPSILON = 1.0e-12
MINIMUM_TARGET_ROWS = 200
RIDGE_PENALTIES = np.asarray(MEAN_MODEL.ridge_grid, dtype=np.float64)


# =============================================================================
# 2. General integrity and serialization helpers
# =============================================================================

def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(1024 * 1024)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def dataframe_sha256(frame: pd.DataFrame, sort_columns: Iterable[str]) -> str:
    # JSON receipts are written with sorted keys. Canonical column ordering keeps
    # a first-run in-memory DataFrame identical to the same summaries reloaded
    # from those JSON files on a repeat run.
    ordered = (
        frame.loc[:, sorted(frame.columns)]
        .sort_values(list(sort_columns), kind="stable")
        .reset_index(drop=True)
    )
    payload = ordered.to_csv(
        index=False,
        lineterminator="\n",
        na_rep="<NA>",
        float_format="%.12g",
    )
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def atomic_save_npz(output_path: Path, **arrays: np.ndarray) -> None:
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = output_path.with_name(output_path.name + ".tmp.npz")
    np.savez_compressed(temporary, **arrays)
    temporary.replace(output_path)


def safe_slug(text: str) -> str:
    cleaned = "".join(character if character.isalnum() else "_" for character in text)
    return "_".join(part for part in cleaned.split("_") if part)


def robust_location_scale(values: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Column-wise median/IQR with standard-deviation fallback."""
    values = np.asarray(values, dtype=np.float64)
    if values.ndim != 2:
        raise ValueError("robust_location_scale expects a two-dimensional matrix")
    location = np.median(values, axis=0)
    q25, q75 = np.percentile(values, (25.0, 75.0), axis=0)
    scale = q75 - q25
    standard_deviation = np.std(values, axis=0, ddof=0)
    use_standard_deviation = (~np.isfinite(scale)) | (scale <= NUMERICAL_EPSILON)
    scale = np.where(use_standard_deviation, standard_deviation, scale)
    valid = np.isfinite(location) & np.isfinite(scale) & (scale > NUMERICAL_EPSILON)
    safe_scale = np.where(valid, scale, 1.0)
    return location, safe_scale, valid


def robust_target_parameters(values: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Robust parameters for a fully observed target group."""
    return robust_location_scale(values)


# =============================================================================
# 3. Embargoed temporal block construction
# =============================================================================

def make_temporal_fold_ids(segment_id: np.ndarray) -> np.ndarray:
    """Assign every row to one complete temporal block and cross-fit fold."""
    segment_id = np.asarray(segment_id, dtype=np.int64)
    fold_id = np.full(len(segment_id), -1, dtype=np.int8)
    next_global_block = 0

    for segment in np.unique(segment_id):
        rows = np.flatnonzero(segment_id == segment)
        if len(rows) == 0:
            continue
        local_blocks = np.arange(len(rows), dtype=np.int64) // MEAN_MODEL.temporal_block_steps
        for local_block in np.unique(local_blocks):
            block_rows = rows[local_blocks == local_block]
            fold_id[block_rows] = next_global_block % MEAN_MODEL.crossfit_folds
            next_global_block += 1

    if (fold_id < 0).any():
        raise RuntimeError("At least one row was not assigned to a temporal fold.")
    return fold_id


def embargo_dilation(
    fold_id: np.ndarray,
    validation_fold: int,
    segment_id: np.ndarray,
) -> np.ndarray:
    """Dilate validation blocks by the frozen two-sided embargo within segments."""
    fold_id = np.asarray(fold_id)
    segment_id = np.asarray(segment_id)
    excluded = np.zeros(len(fold_id), dtype=bool)
    radius = int(MEAN_MODEL.embargo_steps)

    for segment in np.unique(segment_id):
        rows = np.flatnonzero(segment_id == segment)
        local_validation = fold_id[rows] == validation_fold
        if not local_validation.any():
            continue
        padded = np.concatenate(([False], local_validation, [False]))
        changes = np.diff(padded.astype(np.int8))
        starts = np.flatnonzero(changes == 1)
        ends = np.flatnonzero(changes == -1)
        difference = np.zeros(len(rows) + 1, dtype=np.int64)
        for start, end in zip(starts, ends):
            left = max(0, int(start) - radius)
            right = min(len(rows), int(end) + radius)
            difference[left] += 1
            difference[right] -= 1
        excluded[rows] = np.cumsum(difference[:-1]) > 0
    return excluded


def iter_case_temporal_folds(
    fit_mask: np.ndarray,
    segment_id: np.ndarray,
) -> Iterator[dict[str, Any]]:
    fit_mask = np.asarray(fit_mask, dtype=bool)
    fold_id = make_temporal_fold_ids(segment_id)

    for validation_fold in range(MEAN_MODEL.crossfit_folds):
        validation_mask = fit_mask & (fold_id == validation_fold)
        dilated = embargo_dilation(fold_id, validation_fold, segment_id)
        candidate_training = fit_mask & (fold_id != validation_fold)
        training_mask = candidate_training & ~dilated
        if not validation_mask.any():
            raise RuntimeError(f"Temporal validation fold {validation_fold + 1} is empty.")
        if training_mask.sum() < MINIMUM_TARGET_ROWS:
            raise RuntimeError(
                f"Temporal training fold {validation_fold + 1} contains only "
                f"{int(training_mask.sum())} eligible rows."
            )
        yield {
            "fold": validation_fold,
            "fold_number": validation_fold + 1,
            "fold_id": fold_id,
            "training_mask": training_mask,
            "validation_mask": validation_mask,
            "embargoed_fit_rows": int((candidate_training & dilated).sum()),
        }


# =============================================================================
# 4. Fold-local nonlinear driver transformation
# =============================================================================

_feature_description_lookup = {
    (str(row.farm), str(row.column)): str(row.description)
    for row in CARE_FEATURE_REGISTRY.itertuples(index=False)
}


def raw_driver_name(processed_name: str) -> str:
    name = str(processed_name)
    for _ in range(3):
        changed = False
        for suffix in ("__missing", "__sin", "__cos", "__rate"):
            if name.endswith(suffix):
                name = name[: -len(suffix)]
                changed = True
                break
        if not changed:
            break
    return name


def semantic_text(farm: str, processed_name: str) -> str:
    raw = raw_driver_name(processed_name)
    description = _feature_description_lookup.get((farm, raw), "")
    text = f"{raw} {description}".lower().replace("windspeed", "wind speed")
    return re.sub(r"\s+", " ", text).strip()


def semantic_match(text: str, phrase: str) -> bool:
    phrase_pattern = re.escape(str(phrase).lower()).replace(r"\ ", r"\s+")
    return re.search(rf"\b{phrase_pattern}\b", text) is not None


def missing_indicator_name(processed_name: str) -> str:
    if processed_name.endswith("__sin") or processed_name.endswith("__cos"):
        return processed_name.rsplit("__", 1)[0] + "__missing"
    return processed_name + "__missing"


def choose_core_driver_indices(farm: str, driver_names: list[str]) -> list[int]:
    eligible = [
        index for index, name in enumerate(driver_names)
        if not name.endswith("__missing")
    ]
    selected: list[int] = []
    for pattern in MEAN_MODEL.driver_description_patterns:
        matches = [
            index for index in eligible
            if index not in selected and semantic_match(semantic_text(farm, driver_names[index]), pattern)
        ]
        matches.sort(key=lambda index: driver_names[index])
        for index in matches:
            selected.append(index)
            if len(selected) == MEAN_MODEL.maximum_core_interaction_drivers:
                return selected
    for index in sorted(eligible, key=lambda item: driver_names[item]):
        if index not in selected:
            selected.append(index)
        if len(selected) == MEAN_MODEL.maximum_core_interaction_drivers:
            break
    return selected


def fit_design_state(
    drivers: np.ndarray,
    driver_names: list[str],
    training_mask: np.ndarray,
    farm: str,
) -> tuple[dict[str, Any], np.ndarray]:
    """Fit all driver transformations using one temporal training fold only."""
    drivers = np.asarray(drivers, dtype=np.float64)
    training_mask = np.asarray(training_mask, dtype=bool)
    names = list(driver_names)
    if drivers.shape[1] != len(names):
        raise ValueError("Driver matrix and driver-name count differ.")

    indicator_lookup = {
        name: index for index, name in enumerate(names) if name.endswith("__missing")
    }
    is_indicator = np.asarray(
        [name.endswith("__missing") for name in names], dtype=bool
    )
    imputation = np.zeros(drivers.shape[1], dtype=np.float64)
    filled = drivers.copy()

    for index, name in enumerate(names):
        if is_indicator[index]:
            filled[:, index] = np.where(np.isfinite(filled[:, index]), filled[:, index], 0.0)
            continue
        indicator_index = indicator_lookup.get(missing_indicator_name(name))
        missing = ~np.isfinite(filled[:, index])
        if indicator_index is not None:
            missing |= filled[:, indicator_index] > 0.5
        observed_training = training_mask & ~missing & np.isfinite(filled[:, index])
        if not observed_training.any():
            raise RuntimeError(f"No fold-training observation for driver {name!r}.")
        imputation[index] = float(np.median(filled[observed_training, index]))
        filled[missing, index] = imputation[index]

    raw_location = np.zeros(drivers.shape[1], dtype=np.float64)
    raw_scale = np.ones(drivers.shape[1], dtype=np.float64)
    if (~is_indicator).any():
        location, scale, _ = robust_location_scale(
            filled[training_mask][:, ~is_indicator]
        )
        raw_location[~is_indicator] = location
        raw_scale[~is_indicator] = scale
    standardized = (filled - raw_location[None, :]) / raw_scale[None, :]

    term_kind: list[str] = []
    term_left: list[int] = []
    term_right: list[int] = []
    term_names: list[str] = []

    for index, name in enumerate(names):
        term_kind.append("linear")
        term_left.append(index)
        term_right.append(-1)
        term_names.append(f"linear::{name}")

    for index, name in enumerate(names):
        if is_indicator[index]:
            continue
        term_kind.append("quadratic")
        term_left.append(index)
        term_right.append(-1)
        term_names.append(f"quadratic::{name}")

    for index, name in enumerate(names):
        if is_indicator[index]:
            continue
        if semantic_match(semantic_text(farm, name), "wind speed"):
            term_kind.append("cubic")
            term_left.append(index)
            term_right.append(-1)
            term_names.append(f"wind_cubic::{name}")

    core_indices = choose_core_driver_indices(farm, names)
    for left, right in combinations(core_indices, 2):
        term_kind.append("interaction")
        term_left.append(left)
        term_right.append(right)
        term_names.append(f"interaction::{names[left]}*{names[right]}")

    basis = build_unscaled_basis(
        standardized,
        term_kind,
        np.asarray(term_left, dtype=np.int32),
        np.asarray(term_right, dtype=np.int32),
    )
    term_location, term_scale, active = robust_location_scale(basis[training_mask])
    if not active.any():
        raise RuntimeError("All nonlinear design terms are constant in a training fold.")
    design = (
        basis[:, active] - term_location[None, active]
    ) / term_scale[None, active]

    state = {
        "driver_names": names,
        "imputation": imputation,
        "raw_location": raw_location,
        "raw_scale": raw_scale,
        "is_indicator": is_indicator,
        "term_kind": term_kind,
        "term_left": np.asarray(term_left, dtype=np.int32),
        "term_right": np.asarray(term_right, dtype=np.int32),
        "term_names": term_names,
        "term_location": term_location,
        "term_scale": term_scale,
        "term_active": active,
        "core_driver_names": [names[index] for index in core_indices],
    }
    return state, design


def build_unscaled_basis(
    standardized_drivers: np.ndarray,
    term_kind: list[str],
    term_left: np.ndarray,
    term_right: np.ndarray,
) -> np.ndarray:
    columns: list[np.ndarray] = []
    for kind, left, right in zip(term_kind, term_left, term_right):
        if kind == "linear":
            columns.append(standardized_drivers[:, left])
        elif kind == "quadratic":
            columns.append(np.square(standardized_drivers[:, left]))
        elif kind == "cubic":
            columns.append(np.power(standardized_drivers[:, left], 3))
        elif kind == "interaction":
            columns.append(
                standardized_drivers[:, left] * standardized_drivers[:, right]
            )
        else:
            raise ValueError(f"Unknown design term kind: {kind}")
    return np.column_stack(columns)


def transform_design(drivers: np.ndarray, state: dict[str, Any]) -> np.ndarray:
    drivers = np.asarray(drivers, dtype=np.float64)
    names = list(state["driver_names"])
    indicator_lookup = {
        name: index for index, name in enumerate(names) if name.endswith("__missing")
    }
    is_indicator = np.asarray(state["is_indicator"], dtype=bool)
    filled = drivers.copy()
    imputation = np.asarray(state["imputation"], dtype=np.float64)

    for index, name in enumerate(names):
        if is_indicator[index]:
            filled[:, index] = np.where(np.isfinite(filled[:, index]), filled[:, index], 0.0)
            continue
        indicator_index = indicator_lookup.get(missing_indicator_name(name))
        missing = ~np.isfinite(filled[:, index])
        if indicator_index is not None:
            missing |= filled[:, indicator_index] > 0.5
        filled[missing, index] = imputation[index]

    standardized = (
        filled - np.asarray(state["raw_location"])[None, :]
    ) / np.asarray(state["raw_scale"])[None, :]
    basis = build_unscaled_basis(
        standardized,
        list(state["term_kind"]),
        np.asarray(state["term_left"], dtype=np.int32),
        np.asarray(state["term_right"], dtype=np.int32),
    )
    active = np.asarray(state["term_active"], dtype=bool)
    return (
        basis[:, active] - np.asarray(state["term_location"])[None, active]
    ) / np.asarray(state["term_scale"])[None, active]


# =============================================================================
# 5. Mask-aware grouped multi-target ridge fitting
# =============================================================================

def group_targets_by_observation_mask(observed: np.ndarray) -> list[tuple[np.ndarray, np.ndarray]]:
    """Group targets with identical row-observation masks."""
    observed = np.asarray(observed, dtype=bool)
    packed = np.packbits(observed, axis=0)
    groups: dict[bytes, list[int]] = defaultdict(list)
    for target_index in range(observed.shape[1]):
        groups[packed[:, target_index].tobytes()].append(target_index)
    output: list[tuple[np.ndarray, np.ndarray]] = []
    for target_indices in groups.values():
        indices = np.asarray(target_indices, dtype=np.int32)
        output.append((indices, observed[:, indices[0]]))
    return output


def augmented_design(design: np.ndarray) -> np.ndarray:
    return np.column_stack((np.ones(len(design), dtype=np.float64), design))


def solve_ridge(
    gram: np.ndarray,
    cross_product: np.ndarray,
    penalty: float,
) -> np.ndarray:
    regularized = gram.copy()
    diagonal = np.arange(len(regularized))
    regularized[diagonal[1:], diagonal[1:]] += float(penalty)
    try:
        return np.linalg.solve(regularized, cross_product)
    except np.linalg.LinAlgError:
        return np.linalg.lstsq(regularized, cross_product, rcond=None)[0]


def score_penalties_for_fold(
    design: np.ndarray,
    targets: np.ndarray,
    training_mask: np.ndarray,
    validation_mask: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, int]:
    training_rows = np.flatnonzero(training_mask)
    validation_rows = np.flatnonzero(validation_mask)
    x_train = augmented_design(design[training_rows])
    x_validation = augmented_design(design[validation_rows])
    y_train = np.asarray(targets[training_rows], dtype=np.float64)
    y_validation = np.asarray(targets[validation_rows], dtype=np.float64)
    observed_train = np.isfinite(y_train)

    squared_error = np.zeros(len(RIDGE_PENALTIES), dtype=np.float64)
    observation_count = np.zeros(len(RIDGE_PENALTIES), dtype=np.int64)
    fitted_targets = 0

    minimum_rows = max(MINIMUM_TARGET_ROWS, x_train.shape[1] + 10)
    for target_indices, group_observed in group_targets_by_observation_mask(observed_train):
        if int(group_observed.sum()) < minimum_rows:
            continue
        x_observed = x_train[group_observed]
        y_observed = y_train[group_observed][:, target_indices]
        location, scale, valid_target = robust_target_parameters(y_observed)
        if not valid_target.any():
            continue
        target_indices = target_indices[valid_target]
        location = location[valid_target]
        scale = scale[valid_target]
        y_standardized = (y_observed[:, valid_target] - location[None, :]) / scale[None, :]
        gram = x_observed.T @ x_observed
        cross_product = x_observed.T @ y_standardized
        validation_values = y_validation[:, target_indices]
        validation_observed = np.isfinite(validation_values)
        fitted_targets += len(target_indices)

        for penalty_index, penalty in enumerate(RIDGE_PENALTIES):
            coefficients = solve_ridge(gram, cross_product, float(penalty))
            prediction = (
                location[None, :] + scale[None, :] * (x_validation @ coefficients)
            )
            residual = (validation_values - prediction) / scale[None, :]
            valid = validation_observed & np.isfinite(residual)
            clipped = np.clip(
                residual[valid], -MEAN_MODEL.residual_cap, MEAN_MODEL.residual_cap
            )
            squared_error[penalty_index] += float(np.square(clipped).sum())
            observation_count[penalty_index] += int(len(clipped))

    return squared_error, observation_count, fitted_targets


def predict_one_fold(
    design: np.ndarray,
    targets: np.ndarray,
    training_mask: np.ndarray,
    validation_mask: np.ndarray,
    penalty: float,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Return raw predictions and fold-training mean baselines for validation rows."""
    training_rows = np.flatnonzero(training_mask)
    validation_rows = np.flatnonzero(validation_mask)
    x_train = augmented_design(design[training_rows])
    x_validation = augmented_design(design[validation_rows])
    y_train = np.asarray(targets[training_rows], dtype=np.float64)
    observed_train = np.isfinite(y_train)
    prediction = np.full((len(validation_rows), targets.shape[1]), np.nan, dtype=np.float64)
    baseline = np.full_like(prediction, np.nan)
    fitted = np.zeros(targets.shape[1], dtype=bool)
    minimum_rows = max(MINIMUM_TARGET_ROWS, x_train.shape[1] + 10)

    for target_indices, group_observed in group_targets_by_observation_mask(observed_train):
        if int(group_observed.sum()) < minimum_rows:
            continue
        x_observed = x_train[group_observed]
        y_observed = y_train[group_observed][:, target_indices]
        location, scale, valid_target = robust_target_parameters(y_observed)
        if not valid_target.any():
            continue
        target_indices = target_indices[valid_target]
        location = location[valid_target]
        scale = scale[valid_target]
        y_valid = y_observed[:, valid_target]
        y_standardized = (y_valid - location[None, :]) / scale[None, :]
        gram = x_observed.T @ x_observed
        cross_product = x_observed.T @ y_standardized
        coefficients = solve_ridge(gram, cross_product, penalty)
        prediction[:, target_indices] = (
            location[None, :] + scale[None, :] * (x_validation @ coefficients)
        )
        baseline[:, target_indices] = np.mean(y_valid, axis=0)[None, :]
        fitted[target_indices] = True
    return prediction, baseline, fitted


def fit_final_masked_ridge(
    design: np.ndarray,
    targets: np.ndarray,
    fit_mask: np.ndarray,
    penalty: float,
) -> dict[str, np.ndarray]:
    fit_rows = np.flatnonzero(fit_mask)
    x_fit = augmented_design(design[fit_rows])
    y_fit = np.asarray(targets[fit_rows], dtype=np.float64)
    observed = np.isfinite(y_fit)
    coefficients = np.full(
        (x_fit.shape[1], targets.shape[1]), np.nan, dtype=np.float64
    )
    target_location = np.full(targets.shape[1], np.nan, dtype=np.float64)
    target_scale = np.full(targets.shape[1], np.nan, dtype=np.float64)
    target_count = observed.sum(axis=0, dtype=np.int64)
    modeled = np.zeros(targets.shape[1], dtype=bool)
    minimum_rows = max(MINIMUM_TARGET_ROWS, x_fit.shape[1] + 10)

    for target_indices, group_observed in group_targets_by_observation_mask(observed):
        if int(group_observed.sum()) < minimum_rows:
            continue
        x_observed = x_fit[group_observed]
        y_observed = y_fit[group_observed][:, target_indices]
        location, scale, valid_target = robust_target_parameters(y_observed)
        if not valid_target.any():
            continue
        target_indices = target_indices[valid_target]
        location = location[valid_target]
        scale = scale[valid_target]
        y_standardized = (
            y_observed[:, valid_target] - location[None, :]
        ) / scale[None, :]
        gram = x_observed.T @ x_observed
        cross_product = x_observed.T @ y_standardized
        coefficients[:, target_indices] = solve_ridge(gram, cross_product, penalty)
        target_location[target_indices] = location
        target_scale[target_indices] = scale
        modeled[target_indices] = True

    return {
        "coefficients": coefficients,
        "target_location": target_location,
        "target_scale": target_scale,
        "target_training_count": target_count,
        "target_modeled": modeled,
    }


# =============================================================================
# 6. Per-case cross-fitting and final model artifacts
# =============================================================================

def case_artifact_paths(farm: str, event_id: int) -> tuple[Path, Path]:
    relative = Path(safe_slug(farm)) / f"event_{int(event_id):03d}"
    return (
        CELL4_MODEL_ROOT / relative.with_suffix(".json"),
        CELL4_RESIDUAL_ROOT / relative.with_suffix(".npz"),
    )


def case_input_signature(case_row: Any) -> str:
    return sha256_json(
        {
            "cell3_receipt_sha256": CELL3_RECEIPT_SHA256,
            "case_key": str(case_row.case_key),
            "cell3_cache_sha256": str(case_row.cache_sha256),
            "cell3_case_preprocessing_sha256": str(case_row.case_preprocessing_sha256),
            "mean_model_implementation_sha256": MEAN_MODEL_IMPLEMENTATION_SHA256,
        }
    )


def serialize_design_state(state: dict[str, Any]) -> tuple[dict[str, Any], dict[str, np.ndarray]]:
    metadata = {
        "driver_names": list(state["driver_names"]),
        "term_kind": list(state["term_kind"]),
        "term_names": list(state["term_names"]),
        "core_driver_names": list(state["core_driver_names"]),
    }
    arrays = {
        "design_imputation": np.asarray(state["imputation"], dtype=np.float64),
        "design_raw_location": np.asarray(state["raw_location"], dtype=np.float64),
        "design_raw_scale": np.asarray(state["raw_scale"], dtype=np.float64),
        "design_is_indicator": np.asarray(state["is_indicator"], dtype=bool),
        "design_term_left": np.asarray(state["term_left"], dtype=np.int32),
        "design_term_right": np.asarray(state["term_right"], dtype=np.int32),
        "design_term_location": np.asarray(state["term_location"], dtype=np.float64),
        "design_term_scale": np.asarray(state["term_scale"], dtype=np.float64),
        "design_term_active": np.asarray(state["term_active"], dtype=bool),
    }
    return metadata, arrays


def restore_design_state(metadata: dict[str, Any], arrays: dict[str, np.ndarray]) -> dict[str, Any]:
    return {
        "driver_names": list(metadata["driver_names"]),
        "imputation": arrays["design_imputation"],
        "raw_location": arrays["design_raw_location"],
        "raw_scale": arrays["design_raw_scale"],
        "is_indicator": arrays["design_is_indicator"],
        "term_kind": list(metadata["term_kind"]),
        "term_left": arrays["design_term_left"],
        "term_right": arrays["design_term_right"],
        "term_names": list(metadata["term_names"]),
        "term_location": arrays["design_term_location"],
        "term_scale": arrays["design_term_scale"],
        "term_active": arrays["design_term_active"],
        "core_driver_names": list(metadata["core_driver_names"]),
    }


def existing_case_summary(
    metadata_path: Path,
    artifact_path: Path,
    expected_input_signature: str,
) -> dict[str, Any] | None:
    if not metadata_path.exists() or not artifact_path.exists():
        return None
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    if metadata.get("case_input_signature") != expected_input_signature:
        raise RuntimeError(
            f"Mean-model input drift for {metadata.get('case_key', metadata_path.stem)}. "
            "Do not overwrite the existing artifact."
        )
    observed_hash = file_sha256(artifact_path)
    if metadata.get("artifact_sha256") != observed_hash:
        raise RuntimeError(f"Mean-model artifact hash failed: {artifact_path}")
    return dict(metadata["summary"])


def fit_one_case_mean_model(case_row: Any) -> dict[str, Any]:
    metadata_path, artifact_path = case_artifact_paths(case_row.farm, case_row.event_id)
    input_signature = case_input_signature(case_row)
    existing = existing_case_summary(metadata_path, artifact_path, input_signature)
    if existing is not None:
        return existing

    cache = load_case_cache(str(case_row.case_key))
    cache_metadata = cache["metadata"]
    if cache_metadata.get("outcome_fields_present") is not False:
        raise RuntimeError(f"Unsafe Cell 3 cache for {case_row.case_key}.")

    drivers = np.asarray(cache["drivers"], dtype=np.float64)
    targets = np.asarray(cache["targets"], dtype=np.float64)
    fit_mask = np.asarray(cache["fit_mask"], dtype=bool)
    segment_id = np.asarray(cache["segment_id"], dtype=np.int32)
    driver_names = list(cache_metadata["driver_names"])
    target_names = list(cache_metadata["target_names"])
    target_temperature = np.asarray(cache["target_temperature"], dtype=bool)

    if len(drivers) != len(targets) or len(fit_mask) != len(drivers):
        raise RuntimeError(f"Row mismatch in Cell 3 cache for {case_row.case_key}.")
    if drivers.shape[1] != len(driver_names) or targets.shape[1] != len(target_names):
        raise RuntimeError(f"Channel-name mismatch for {case_row.case_key}.")
    if not np.isfinite(drivers).all():
        raise RuntimeError(f"Non-finite driver values remain for {case_row.case_key}.")

    folds = list(iter_case_temporal_folds(fit_mask, segment_id))
    penalty_sse = np.zeros(len(RIDGE_PENALTIES), dtype=np.float64)
    penalty_count = np.zeros(len(RIDGE_PENALTIES), dtype=np.int64)
    fold_diagnostics: list[dict[str, Any]] = []

    # Pass 1: label-free ridge-penalty selection.
    for fold in folds:
        state, design = fit_design_state(
            drivers, driver_names, fold["training_mask"], str(case_row.farm)
        )
        fold_sse, fold_count, fitted_targets = score_penalties_for_fold(
            design,
            targets,
            fold["training_mask"],
            fold["validation_mask"],
        )
        penalty_sse += fold_sse
        penalty_count += fold_count
        fold_diagnostics.append(
            {
                "fold": int(fold["fold_number"]),
                "training_rows": int(fold["training_mask"].sum()),
                "validation_rows": int(fold["validation_mask"].sum()),
                "embargoed_fit_rows": int(fold["embargoed_fit_rows"]),
                "design_terms": int(design.shape[1]),
                "fitted_targets": int(fitted_targets),
            }
        )
        del state, design

    if (penalty_count == 0).any():
        raise RuntimeError(f"No OOF observations for one or more penalties in {case_row.case_key}.")
    penalty_mse = penalty_sse / penalty_count
    minimum_mse = float(np.min(penalty_mse))
    tied = np.flatnonzero(np.isclose(penalty_mse, minimum_mse, rtol=1e-12, atol=1e-12))
    selected_index = int(tied[-1])
    selected_penalty = float(RIDGE_PENALTIES[selected_index])

    # Pass 2: honest OOF predictions and residuals at the selected penalty.
    oof_prediction = np.full_like(targets, np.nan, dtype=np.float32)
    oof_baseline = np.full_like(targets, np.nan, dtype=np.float32)
    oof_fold_id = np.full(len(targets), -1, dtype=np.int8)
    fold_target_fit_count = np.zeros(targets.shape[1], dtype=np.int16)

    for fold in folds:
        state, design = fit_design_state(
            drivers, driver_names, fold["training_mask"], str(case_row.farm)
        )
        prediction, baseline, fitted_targets = predict_one_fold(
            design,
            targets,
            fold["training_mask"],
            fold["validation_mask"],
            selected_penalty,
        )
        validation_rows = np.flatnonzero(fold["validation_mask"])
        oof_prediction[validation_rows] = prediction.astype(np.float32)
        oof_baseline[validation_rows] = baseline.astype(np.float32)
        oof_fold_id[validation_rows] = int(fold["fold"])
        fold_target_fit_count += fitted_targets.astype(np.int16)
        del state, design, prediction, baseline

    oof_residual = targets - oof_prediction.astype(np.float64)
    observed_oof = fit_mask[:, None] & np.isfinite(targets) & np.isfinite(oof_prediction)
    baseline_valid = observed_oof & np.isfinite(oof_baseline)
    target_oof_count = observed_oof.sum(axis=0, dtype=np.int64)
    target_fit_observed = (fit_mask[:, None] & np.isfinite(targets)).sum(
        axis=0, dtype=np.int64
    )
    target_oof_coverage = target_oof_count / np.maximum(target_fit_observed, 1)
    target_crossfit_r2 = np.full(targets.shape[1], np.nan, dtype=np.float64)

    for target_index in range(targets.shape[1]):
        valid = observed_oof[:, target_index]
        valid_baseline = baseline_valid[:, target_index]
        if valid.sum() < MINIMUM_TARGET_ROWS or valid_baseline.sum() != valid.sum():
            continue
        model_sse = float(np.square(oof_residual[valid, target_index]).sum())
        baseline_error = (
            targets[valid, target_index]
            - oof_baseline[valid, target_index].astype(np.float64)
        )
        baseline_sse = float(np.square(baseline_error).sum())
        if baseline_sse > NUMERICAL_EPSILON:
            target_crossfit_r2[target_index] = 1.0 - model_sse / baseline_sse

    # Predictions and baselines have served their sole purpose. Releasing them
    # before the final fit keeps the Farm C peak memory bounded.
    del oof_prediction, oof_baseline

    # Final label-free model on every normal source-training row.
    final_state, final_design = fit_design_state(
        drivers, driver_names, fit_mask, str(case_row.farm)
    )
    final_model = fit_final_masked_ridge(
        final_design, targets, fit_mask, selected_penalty
    )
    state_metadata, state_arrays = serialize_design_state(final_state)

    fit_rows = np.flatnonzero(fit_mask).astype(np.int32)
    residual_fit = oof_residual[fit_mask].astype(np.float32)
    residual_fold_fit = oof_fold_id[fit_mask].astype(np.int8)
    if (residual_fold_fit < 0).any():
        raise RuntimeError(f"OOF fold assignment is incomplete for {case_row.case_key}.")

    artifact_arrays = {
        **state_arrays,
        "coefficients": final_model["coefficients"].astype(np.float32),
        "target_location": final_model["target_location"].astype(np.float64),
        "target_scale": final_model["target_scale"].astype(np.float64),
        "target_training_count": final_model["target_training_count"].astype(np.int64),
        "target_modeled": final_model["target_modeled"].astype(bool),
        "target_crossfit_r2": target_crossfit_r2.astype(np.float64),
        "target_oof_coverage": target_oof_coverage.astype(np.float64),
        "target_fold_fit_count": fold_target_fit_count.astype(np.int16),
        "target_temperature": target_temperature.astype(bool),
        "fit_row_indices": fit_rows,
        "oof_fold_fit": residual_fold_fit,
        "oof_residual_fit": residual_fit,
        "ridge_penalties": RIDGE_PENALTIES.astype(np.float64),
        "ridge_selection_mse": penalty_mse.astype(np.float64),
        "ridge_selection_count": penalty_count.astype(np.int64),
    }
    atomic_save_npz(artifact_path, **artifact_arrays)
    artifact_hash = file_sha256(artifact_path)

    finite_r2 = target_crossfit_r2[np.isfinite(target_crossfit_r2)]
    temperature_r2 = target_crossfit_r2[target_temperature & np.isfinite(target_crossfit_r2)]
    summary = {
        "case_key": str(case_row.case_key),
        "farm": str(case_row.farm),
        "asset_id": str(case_row.asset_id),
        "event_id": int(case_row.event_id),
        "rows": int(len(drivers)),
        "training_normal_rows": int(fit_mask.sum()),
        "drivers": int(drivers.shape[1]),
        "targets": int(targets.shape[1]),
        "final_design_terms": int(final_design.shape[1]),
        "selected_penalty": selected_penalty,
        "selection_mse": float(penalty_mse[selected_index]),
        "modeled_targets": int(final_model["target_modeled"].sum()),
        "complete_crossfit_targets": int((fold_target_fit_count == MEAN_MODEL.crossfit_folds).sum()),
        "median_crossfit_r2": float(np.median(finite_r2)) if len(finite_r2) else np.nan,
        "median_temperature_crossfit_r2": (
            float(np.median(temperature_r2)) if len(temperature_r2) else np.nan
        ),
        "fraction_crossfit_r2_positive": (
            float((finite_r2 > 0.0).mean()) if len(finite_r2) else np.nan
        ),
        "median_oof_coverage": float(np.median(target_oof_coverage)),
        "artifact_relative_path": artifact_path.relative_to(CELL4_RESIDUAL_ROOT).as_posix(),
        "metadata_relative_path": metadata_path.relative_to(CELL4_MODEL_ROOT).as_posix(),
        "case_input_signature": input_signature,
        "artifact_sha256": artifact_hash,
    }

    save_json(
        {
            "cell4_version": CELL4_VERSION,
            "created_at_utc": utc_now(),
            "contract_sha256": CONTRACT_SHA256,
            "cell3_receipt_sha256": CELL3_RECEIPT_SHA256,
            "mean_model_implementation_sha256": MEAN_MODEL_IMPLEMENTATION_SHA256,
            "case_input_signature": input_signature,
            "case_key": str(case_row.case_key),
            "farm": str(case_row.farm),
            "asset_id": str(case_row.asset_id),
            "event_id": int(case_row.event_id),
            "driver_names": driver_names,
            "target_names": target_names,
            "design_state": state_metadata,
            "fold_diagnostics": fold_diagnostics,
            "selected_penalty": selected_penalty,
            "ridge_penalties": RIDGE_PENALTIES.tolist(),
            "ridge_selection_mse": penalty_mse.tolist(),
            "artifact_sha256": artifact_hash,
            "artifact_relative_path": summary["artifact_relative_path"],
            "outcome_fields_present": False,
            "summary": summary,
        },
        metadata_path,
    )
    return summary


# =============================================================================
# 7. Safe model loading and reproducible mean prediction
# =============================================================================

def load_case_mean_model(case_key: str) -> dict[str, Any]:
    match = CASE_MEAN_MODEL_REGISTRY.loc[
        CASE_MEAN_MODEL_REGISTRY["case_key"].eq(case_key)
    ]
    if len(match) != 1:
        raise KeyError(f"Unknown or duplicate case_key: {case_key}")
    row = match.iloc[0]
    metadata_path = CELL4_MODEL_ROOT / row["metadata_relative_path"]
    artifact_path = CELL4_RESIDUAL_ROOT / row["artifact_relative_path"]
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    if metadata.get("outcome_fields_present") is not False:
        raise RuntimeError(f"Unsafe mean-model metadata for {case_key}.")
    if file_sha256(artifact_path) != row["artifact_sha256"]:
        raise RuntimeError(f"Mean-model artifact hash mismatch for {case_key}.")
    arrays = dict(np.load(artifact_path, allow_pickle=False))
    arrays["metadata"] = metadata
    arrays["design_state"] = restore_design_state(metadata["design_state"], arrays)
    return arrays


def predict_case_mean(
    case_key: str,
    row_selector: slice | np.ndarray | list[int] | None = None,
) -> np.ndarray:
    """Reproduce the final normal-behaviour mean without outcome information."""
    cache = load_case_cache(case_key)
    model = load_case_mean_model(case_key)
    drivers = np.asarray(cache["drivers"], dtype=np.float64)
    if row_selector is not None:
        drivers = drivers[row_selector]
    design = transform_design(drivers, model["design_state"])
    x = augmented_design(design)
    coefficients = np.asarray(model["coefficients"], dtype=np.float64)
    target_location = np.asarray(model["target_location"], dtype=np.float64)
    target_scale = np.asarray(model["target_scale"], dtype=np.float64)
    prediction = target_location[None, :] + target_scale[None, :] * (x @ coefficients)
    prediction[:, ~np.asarray(model["target_modeled"], dtype=bool)] = np.nan
    return prediction.astype(np.float32)


# =============================================================================
# 8. Fit or verify all 95 cases
# =============================================================================

_model_summaries: list[dict[str, Any]] = []
for _farm in DATASET.farms:
    _farm_cases = CASE_CACHE_REGISTRY.loc[CASE_CACHE_REGISTRY["farm"].eq(_farm)]
    print(
        f"Cross-fitting nonlinear ridge mean models — {_farm}: {len(_farm_cases)} cases",
        flush=True,
    )
    for _case_row in _farm_cases.itertuples(index=False):
        _summary = fit_one_case_mean_model(_case_row)
        _model_summaries.append(_summary)
        print(
            f"  event {_case_row.event_id:>3}: λ={_summary['selected_penalty']:<8g} "
            f"targets={_summary['modeled_targets']:>3}/{_summary['targets']:<3} "
            f"median OOF R²={_summary['median_crossfit_r2']:.3f}",
            flush=True,
        )

CASE_MEAN_MODEL_REGISTRY = pd.DataFrame(_model_summaries).sort_values(
    ["farm", "event_id"], kind="stable"
).reset_index(drop=True)

if len(CASE_MEAN_MODEL_REGISTRY) != DATASET.expected_total_cases:
    raise RuntimeError("Not all 95 cases produced a mean-model artifact.")
if CASE_MEAN_MODEL_REGISTRY["case_key"].duplicated().any():
    raise RuntimeError("Duplicate case keys exist in the mean-model registry.")
if (CASE_MEAN_MODEL_REGISTRY["modeled_targets"] <= 0).any():
    raise RuntimeError("At least one case has no fitted monitoring target.")

# Load one artifact per farm and reproduce a small prediction slice.
for _case_key in CASE_MEAN_MODEL_REGISTRY.groupby("farm", sort=False).head(1)["case_key"]:
    _model = load_case_mean_model(_case_key)
    _prediction = predict_case_mean(_case_key, slice(0, 32))
    if _prediction.shape[0] != 32:
        raise RuntimeError(f"Prediction smoke test failed for {_case_key}.")
    if _prediction.shape[1] != len(_model["metadata"]["target_names"]):
        raise RuntimeError(f"Target shape mismatch for {_case_key}.")
    del _model, _prediction


# =============================================================================
# 9. Freeze Cell 4 receipt and report modelability diagnostics
# =============================================================================

MEAN_MODEL_FARM_SUMMARY = (
    CASE_MEAN_MODEL_REGISTRY.groupby("farm", sort=False)
    .agg(
        cases=("case_key", "size"),
        median_selected_penalty=("selected_penalty", "median"),
        median_design_terms=("final_design_terms", "median"),
        median_modeled_targets=("modeled_targets", "median"),
        median_crossfit_r2=("median_crossfit_r2", "median"),
        median_temperature_crossfit_r2=("median_temperature_crossfit_r2", "median"),
        median_positive_r2_fraction=("fraction_crossfit_r2_positive", "median"),
        median_oof_coverage=("median_oof_coverage", "median"),
    )
    .reset_index()
)

RIDGE_SELECTION_SUMMARY = (
    CASE_MEAN_MODEL_REGISTRY.groupby(["farm", "selected_penalty"])
    .size()
    .rename("cases")
    .reset_index()
    .sort_values(["farm", "selected_penalty"], kind="stable")
)

_component_hashes = {
    "case_mean_model_registry_sha256": dataframe_sha256(
        CASE_MEAN_MODEL_REGISTRY, ("farm", "event_id")
    ),
    "farm_summary_sha256": dataframe_sha256(
        MEAN_MODEL_FARM_SUMMARY, ("farm",)
    ),
    "ridge_selection_summary_sha256": dataframe_sha256(
        RIDGE_SELECTION_SUMMARY, ("farm", "selected_penalty")
    ),
}
CELL4_RECEIPT_SHA256 = sha256_json(
    {
        "contract_sha256": CONTRACT_SHA256,
        "cell3_receipt_sha256": CELL3_RECEIPT_SHA256,
        "mean_model_implementation_sha256": MEAN_MODEL_IMPLEMENTATION_SHA256,
        "component_hashes": _component_hashes,
    }
)

CELL4_RECEIPT_PATH = CELL4_QUALITY_ROOT / "cell4_cross_fitted_mean_model_receipt.json"
if CELL4_RECEIPT_PATH.exists():
    _existing_receipt = json.loads(CELL4_RECEIPT_PATH.read_text(encoding="utf-8"))
    if _existing_receipt.get("cell4_receipt_sha256") != CELL4_RECEIPT_SHA256:
        raise RuntimeError(
            "A different Cell 4 receipt exists for this experiment. Do not overwrite it."
        )
    CELL4_STATE = "existing identical Cell 4 receipt verified"
    _write_cell4_receipt = False
else:
    CELL4_STATE = "new Cell 4 receipt frozen"
    _write_cell4_receipt = True

save_csv_atomic(
    CASE_MEAN_MODEL_REGISTRY,
    CELL4_QUALITY_ROOT / "case_mean_model_registry.csv",
)
save_csv_atomic(
    MEAN_MODEL_FARM_SUMMARY,
    CELL4_QUALITY_ROOT / "mean_model_farm_summary.csv",
)
save_csv_atomic(
    RIDGE_SELECTION_SUMMARY,
    CELL4_QUALITY_ROOT / "ridge_selection_summary.csv",
)
save_json(
    MEAN_MODEL_IMPLEMENTATION,
    CELL4_QUALITY_ROOT / "mean_model_implementation.json",
)

if _write_cell4_receipt:
    save_json(
        {
            "cell4_version": CELL4_VERSION,
            "created_at_utc": utc_now(),
            "contract_sha256": CONTRACT_SHA256,
            "dataset_manifest_sha256": DATASET_MANIFEST_SHA256,
            "cell2_audit_sha256": CELL2_AUDIT_SHA256,
            "cell3_receipt_sha256": CELL3_RECEIPT_SHA256,
            "preprocessing_policy_sha256": PREPROCESSING_POLICY_SHA256,
            "mean_model_implementation_sha256": MEAN_MODEL_IMPLEMENTATION_SHA256,
            "component_hashes": _component_hashes,
            "cell4_receipt_sha256": CELL4_RECEIPT_SHA256,
            "outcomes_read": False,
        },
        CELL4_RECEIPT_PATH,
    )


print("\n" + "=" * 92)
print("UC-RCF-NBM CELL 4 — CROSS-FITTED NONLINEAR RIDGE MEAN MODEL")
print("=" * 92)
print("\nMEAN-MODEL PERFORMANCE ON HONEST TRAINING-NORMAL OOF RESIDUALS")
display(MEAN_MODEL_FARM_SUMMARY)
print("\nRIDGE PENALTIES SELECTED BY MASKED OOF MSE")
display(RIDGE_SELECTION_SUMMARY)

print("\n" + "-" * 92)
print(f"Cases modeled                    : {len(CASE_MEAN_MODEL_REGISTRY)}")
print(f"Temporal folds per case          : {MEAN_MODEL.crossfit_folds}")
print(f"Temporal block / embargo         : {MEAN_MODEL.temporal_block_steps} / "
      f"{MEAN_MODEL.embargo_steps} steps")
print(f"Mean implementation SHA-256      : {MEAN_MODEL_IMPLEMENTATION_SHA256}")
print(f"Cell 4 receipt SHA-256           : {CELL4_RECEIPT_SHA256}")
print(f"Cell 4 state                     : {CELL4_STATE}")
print("Fold-local preprocessing         : Yes")
print("Masked targets imputed           : No")
print("Event outcomes accessed          : No")
print("Final prediction rows used in fit: No")
print("OOF/model artifact checks        : PASS")
print("=" * 92)
print("CELL 4 COMPLETED SUCCESSFULLY — CROSS-FITTED MEAN MODELS LOCKED")


Cross-fitting nonlinear ridge mean models — Wind Farm A: 22 cases
  event   0: λ=0.0001   targets= 43/43  median OOF R²=0.851
  event   3: λ=0.01     targets= 43/43  median OOF R²=0.822
  event  10: λ=0.01     targets= 43/43  median OOF R²=0.835
  event  13: λ=100      targets= 43/43  median OOF R²=0.819
  event  14: λ=1        targets= 43/43  median OOF R²=0.857
  event  17: λ=0.01     targets= 43/43  median OOF R²=0.835
  event  22: λ=100      targets= 43/43  median OOF R²=0.831
  event  24: λ=0.0001   targets= 43/43  median OOF R²=0.807
  event  25: λ=1        targets= 43/43  median OOF R²=0.830
  event  26: λ=1        targets= 43/43  median OOF R²=0.851
  event  38: λ=0.01     targets= 43/43  median OOF R²=0.846
  event  40: λ=1        targets= 43/43  median OOF R²=0.837
  event  42: λ=0.01     targets= 43/43  median OOF R²=0.832
  event  45: λ=0.01     targets= 43/43  median OOF R²=0.823
  event  51: λ=100      targets= 43/43  median OOF R²=0.830
  event  68: λ=1        targets= 4

,farm,cases,median_selected_penalty,median_design_terms,median_modeled_targets,median_crossfit_r2,median_temperature_crossfit_r2,median_positive_r2_fraction,median_oof_coverage
0,Wind Farm A,22,1.00,26.0,43.0,0.831847,0.859656,0.953488,1.0
1,Wind Farm B,15,10000.00,40.0,52.0,0.689352,0.469255,0.961538,1.0
2,Wind Farm C,58,0.01,53.0,226.0,0.733874,0.815076,0.932886,1.0



RIDGE PENALTIES SELECTED BY MASKED OOF MSE


,farm,selected_penalty,cases
0,Wind Farm A,0.0001,3
1,Wind Farm A,0.0100,7
2,Wind Farm A,1.0000,8
3,Wind Farm A,100.0000,4
4,Wind Farm B,0.0001,3
5,Wind Farm B,0.0100,1
6,Wind Farm B,1.0000,1
7,Wind Farm B,100.0000,2
8,Wind Farm B,10000.0000,8
9,Wind Farm C,0.0001,25



--------------------------------------------------------------------------------------------
Cases modeled                    : 95
Temporal folds per case          : 5
Temporal block / embargo         : 432 / 144 steps
Mean implementation SHA-256      : abbb47187122184be8a9fcc6bdb60e0b421e990e62b18ebe2dfa6b1c1a64e024
Cell 4 receipt SHA-256           : 680541be1ef4649872a888e37d8ac4e9648bd24fd90a55aeb8e9f80f606b249b
Cell 4 state                     : new Cell 4 receipt frozen
Fold-local preprocessing         : Yes
Masked targets imputed           : No
Event outcomes accessed          : No
Final prediction rows used in fit: No
OOF/model artifact checks        : PASS
CELL 4 COMPLETED SUCCESSFULLY — CROSS-FITTED MEAN MODELS LOCKED


In [7]:
modelability_records = []

for row in CASE_MEAN_MODEL_REGISTRY.itertuples(index=False):
    model = load_case_mean_model(row.case_key)

    r2 = np.asarray(model["target_crossfit_r2"], dtype=float)
    modeled = np.asarray(model["target_modeled"], dtype=bool)
    valid = modeled & np.isfinite(r2)

    modelability_records.append({
        "farm": row.farm,
        "event_id": row.event_id,
        "modeled_targets": int(valid.sum()),
        "r2_ge_0": int((valid & (r2 >= 0.0)).sum()),
        "r2_ge_0p1": int((valid & (r2 >= 0.1)).sum()),
        "r2_ge_0p3": int((valid & (r2 >= 0.3)).sum()),
        "fraction_r2_ge_0": float(
            (r2[valid] >= 0.0).mean() if valid.any() else np.nan
        ),
        "median_r2": float(
            np.median(r2[valid]) if valid.any() else np.nan
        ),
    })

modelability_diagnostic = pd.DataFrame(modelability_records)

print("\nMODELABILITY AVAILABILITY BY FARM")
print(
    modelability_diagnostic.groupby("farm").agg(
        cases=("event_id", "size"),
        minimum_positive_targets=("r2_ge_0", "min"),
        median_positive_targets=("r2_ge_0", "median"),
        minimum_fraction_positive=("fraction_r2_ge_0", "min"),
        median_fraction_positive=("fraction_r2_ge_0", "median"),
    ).to_string()
)

print("\nLOWEST-MODELABILITY CASES")
print(
    modelability_diagnostic
    .sort_values(["fraction_r2_ge_0", "median_r2"])
    .head(20)
    .to_string(index=False)
)

print("\nCELL 4 HASHES")
print("Mean implementation :", MEAN_MODEL_IMPLEMENTATION_SHA256)
print("Cell 4 receipt      :", CELL4_RECEIPT_SHA256)
print("Cell 4 state        :", CELL4_STATE)


MODELABILITY AVAILABILITY BY FARM
             cases  minimum_positive_targets  median_positive_targets  minimum_fraction_positive  median_fraction_positive
farm                                                                                                                      
Wind Farm A     22                        39                     41.0                   0.906977                  0.953488
Wind Farm B     15                         3                     50.0                   0.057692                  0.961538
Wind Farm C     58                        32                    210.0                   0.140351                  0.932886

LOWEST-MODELABILITY CASES
       farm  event_id  modeled_targets  r2_ge_0  r2_ge_0p1  r2_ge_0p3  fraction_r2_ge_0    median_r2
Wind Farm B        83               52        3          3          3          0.057692 -6531.523799
Wind Farm C        85              228       32         31         29          0.140351   -41.016739
Wind Farm C        3

In [8]:
"""CELL 5 — cross-fitted aleatoric and operating-support uncertainty.

Paste this complete file into the fifth cell of the UC-RCF-NBM notebook and
run it only after Cells 1–4 have completed successfully.

The cell learns a heteroscedastic scale model from the honest training-normal
OOF residuals produced by Cell 4 and quantifies operating-support uncertainty
with regularized nonlinear-design leverage. Both uncertainty components are
estimated without reading event outcomes. Target modelability is retained as
the frozen R² sensitivity grid; cases are never discarded because of a poor
case-level median R².
"""

from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print


# =============================================================================
# 0. Bind Cell 5 to the exact completed experiment state
# =============================================================================

EXPECTED_CONTRACT_SHA256 = (
    "827641aecd8e807193ad193d64c274319756092faa53bdb5084f310f62041f49"
)
EXPECTED_DATASET_MANIFEST_SHA256 = (
    "62484bab1219888aa1d0788965ecd77db2b85f0bbb9b476cd3240f4143026f1f"
)
EXPECTED_CELL2_AUDIT_SHA256 = (
    "83732caf4ad3e226b69a671287bb14c4ed72e1bfc3b7c26ca144310fce8e5990"
)
EXPECTED_PREPROCESSING_POLICY_SHA256 = (
    "67ccb2442d0a44393ca7e3cc0bc8030f7cc3fc69458c1dce9f9b36c57577dcc9"
)
EXPECTED_CELL3_RECEIPT_SHA256 = (
    "aded107bea4397babbf24f4b9ab5740d9cfdd117a3783b3f03bd1cbcfbc55762"
)
EXPECTED_MEAN_IMPLEMENTATION_SHA256 = (
    "abbb47187122184be8a9fcc6bdb60e0b421e990e62b18ebe2dfa6b1c1a64e024"
)
EXPECTED_CELL4_RECEIPT_SHA256 = (
    "680541be1ef4649872a888e37d8ac4e9648bd24fd90a55aeb8e9f80f606b249b"
)

_required_objects = (
    "CONTRACT_SHA256",
    "DATASET_MANIFEST_SHA256",
    "CELL2_AUDIT_SHA256",
    "PREPROCESSING_POLICY_SHA256",
    "CELL3_RECEIPT_SHA256",
    "MEAN_MODEL_IMPLEMENTATION_SHA256",
    "CELL4_RECEIPT_SHA256",
    "DATASET",
    "MEAN_MODEL",
    "UNCERTAINTY",
    "CASE_CACHE_REGISTRY",
    "CASE_MEAN_MODEL_REGISTRY",
    "load_case_cache",
    "load_case_mean_model",
    "iter_case_temporal_folds",
    "fit_design_state",
    "transform_design",
    "serialize_design_state",
    "restore_design_state",
    "fit_final_masked_ridge",
    "group_targets_by_observation_mask",
    "robust_target_parameters",
    "augmented_design",
    "solve_ridge",
    "atomic_save_npz",
    "file_sha256",
    "dataframe_sha256",
    "safe_slug",
    "save_csv_atomic",
    "save_json",
    "sha256_json",
    "utc_now",
    "MODEL_DIR",
    "QUALITY_DIR",
)
_missing_objects = [name for name in _required_objects if name not in globals()]
if _missing_objects:
    raise RuntimeError(
        "Run UC-RCF-NBM Cells 1–4 before Cell 5. Missing objects: "
        + ", ".join(_missing_objects)
    )

_observed_receipts = {
    "contract": CONTRACT_SHA256,
    "dataset_manifest": DATASET_MANIFEST_SHA256,
    "cell2_audit": CELL2_AUDIT_SHA256,
    "preprocessing_policy": PREPROCESSING_POLICY_SHA256,
    "cell3_receipt": CELL3_RECEIPT_SHA256,
    "mean_implementation": MEAN_MODEL_IMPLEMENTATION_SHA256,
    "cell4_receipt": CELL4_RECEIPT_SHA256,
}
_expected_receipts = {
    "contract": EXPECTED_CONTRACT_SHA256,
    "dataset_manifest": EXPECTED_DATASET_MANIFEST_SHA256,
    "cell2_audit": EXPECTED_CELL2_AUDIT_SHA256,
    "preprocessing_policy": EXPECTED_PREPROCESSING_POLICY_SHA256,
    "cell3_receipt": EXPECTED_CELL3_RECEIPT_SHA256,
    "mean_implementation": EXPECTED_MEAN_IMPLEMENTATION_SHA256,
    "cell4_receipt": EXPECTED_CELL4_RECEIPT_SHA256,
}
if _observed_receipts != _expected_receipts:
    raise RuntimeError(
        "Cell 5 is bound to the exact completed Cell 1–4 receipts. "
        f"Observed={_observed_receipts}, expected={_expected_receipts}."
    )

_forbidden_tokens = (
    "event_label",
    "is_anomaly",
    "event_start",
    "event_end",
    "event_description",
    "fault_type",
    "care_ground_truth",
)
for _safe_registry in (CASE_CACHE_REGISTRY, CASE_MEAN_MODEL_REGISTRY):
    if any(
        any(token in str(column).lower() for token in _forbidden_tokens)
        for column in _safe_registry.columns
    ):
        raise RuntimeError("Outcome information is present in a safe input registry.")

if UNCERTAINTY.scale_crossfit_folds != MEAN_MODEL.crossfit_folds:
    raise RuntimeError("Scale and mean cross-fitting must use the same five folds.")
if tuple(UNCERTAINTY.scale_ridge_grid) != tuple(sorted(UNCERTAINTY.scale_ridge_grid)):
    raise RuntimeError("The frozen scale ridge grid must be ordered.")
if tuple(MEAN_MODEL.modelability_r2_grid) != (0.0, 0.10, 0.30):
    raise RuntimeError("Unexpected modelability sensitivity grid.")
if not UNCERTAINTY.support_nonnegative_for_total_scale:
    raise RuntimeError("Cell 5 requires nonnegative support inflation.")

CELL5_VERSION = "1.0.0"
CELL5_MODEL_ROOT = Path(MODEL_DIR) / "cell5_cross_fitted_uncertainty"
CELL5_QUALITY_ROOT = Path(QUALITY_DIR) / "cell5_cross_fitted_uncertainty"
for _directory in (CELL5_MODEL_ROOT, CELL5_QUALITY_ROOT):
    _directory.mkdir(parents=True, exist_ok=True)


# =============================================================================
# 1. Freeze label-free implementation details
# =============================================================================

SCALE_RIDGE_PENALTIES = np.asarray(
    UNCERTAINTY.scale_ridge_grid, dtype=np.float64
)
MODELABILITY_THRESHOLDS = np.asarray(
    MEAN_MODEL.modelability_r2_grid, dtype=np.float64
)
SCALE_TARGET_EPSILON = float(UNCERTAINTY.scale_target_epsilon)
NUMERICAL_EPSILON_CELL5 = 1.0e-12
LOG_VARIANCE_MINIMUM = -60.0
LOG_VARIANCE_MAXIMUM = 60.0
MINIMUM_SCALE_ROWS = 200
LEVERAGE_CHUNK_ROWS = 65536

UNCERTAINTY_IMPLEMENTATION = {
    "version": CELL5_VERSION,
    "aleatoric": {
        "model": UNCERTAINTY.aleatoric_model,
        "response": "log(r_oof^2 + epsilon * s_reference^2)",
        "reference_scale": "target-wise training-fold median/IQR with standard-deviation fallback",
        "epsilon": SCALE_TARGET_EPSILON,
        "design": "the Cell 4 nonlinear operating-condition design, refit inside each scale fold",
        "crossfit_folds": UNCERTAINTY.scale_crossfit_folds,
        "penalties": tuple(UNCERTAINTY.scale_ridge_grid),
        "penalty_objective": "masked blocked OOF MSE of robust-standardized log squared residual",
        "objective_cap": MEAN_MODEL.residual_cap,
        "tie_break": "largest penalty among numerically tied minima",
        "conversion": "exp(0.5 * clipped predicted log variance)",
        "log_variance_numerical_clip": (
            LOG_VARIANCE_MINIMUM,
            LOG_VARIANCE_MAXIMUM,
        ),
        "absolute_scale_calibration": "deferred to embargoed blocked conformal calibration",
    },
    "support": {
        "measure": UNCERTAINTY.support_uncertainty,
        "ridge_penalty": "the case-specific Cell 4 mean-model penalty",
        "intercept_penalized": False,
        "sample_size_normalization": "n_reference * regularized leverage",
        "crossfit_reference": UNCERTAINTY.support_reference_scaling,
        "robust_fallback": "standard deviation if cross-fitted IQR is degenerate",
        "positive_part": "max(standardized cross-fitted leverage, 0)",
    },
    "total_scale": {
        "rule": UNCERTAINTY.total_scale_rule,
        "support_nonnegative": UNCERTAINTY.support_nonnegative_for_total_scale,
    },
    "modelability": {
        "unit": "target within case",
        "r2_thresholds": tuple(MEAN_MODEL.modelability_r2_grid),
        "hard_requirements": (
            "finite Cell 4 target OOF R2",
            "Cell 4 final mean target fitted",
            "Cell 5 final scale target fitted",
            "Cell 5 scale target fitted in all temporal folds",
        ),
        "case_exclusion": False,
        "threshold_selection": "not performed in Cell 5; retain full frozen sensitivity grid",
    },
    "outcomes_read": False,
}
UNCERTAINTY_IMPLEMENTATION_SHA256 = sha256_json(UNCERTAINTY_IMPLEMENTATION)


# =============================================================================
# 2. Numerical, serialization, and modelability helpers
# =============================================================================

def threshold_name(threshold: float) -> str:
    text = f"{float(threshold):g}".replace("-", "m").replace(".", "p")
    return f"r2_ge_{text}"


def finite_robust_location_scale(values: np.ndarray) -> tuple[float, float, bool]:
    finite = np.asarray(values, dtype=np.float64)
    finite = finite[np.isfinite(finite)]
    if len(finite) < 2:
        return np.nan, 1.0, False
    location = float(np.median(finite))
    q25, q75 = np.percentile(finite, (25.0, 75.0))
    scale = float(q75 - q25)
    if not np.isfinite(scale) or scale <= NUMERICAL_EPSILON_CELL5:
        scale = float(np.std(finite, ddof=0))
    valid = np.isfinite(location) and np.isfinite(scale) and scale > NUMERICAL_EPSILON_CELL5
    return location, scale if valid else 1.0, bool(valid)


def residual_reference_scale(
    residual_fit: np.ndarray,
    training_mask: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Target-wise robust residual scales using only the supplied training rows."""
    residual_fit = np.asarray(residual_fit, dtype=np.float64)
    training_mask = np.asarray(training_mask, dtype=bool)
    scales = np.full(residual_fit.shape[1], np.nan, dtype=np.float64)
    valid = np.zeros(residual_fit.shape[1], dtype=bool)
    for target_index in range(residual_fit.shape[1]):
        _, scale, usable = finite_robust_location_scale(
            residual_fit[training_mask, target_index]
        )
        if usable:
            scales[target_index] = scale
            valid[target_index] = True
    return scales, valid


def log_squared_residual_target(
    residual_fit: np.ndarray,
    reference_scale: np.ndarray,
) -> np.ndarray:
    residual_fit = np.asarray(residual_fit, dtype=np.float64)
    reference_scale = np.asarray(reference_scale, dtype=np.float64)
    floor = SCALE_TARGET_EPSILON * np.square(reference_scale)
    output = np.full_like(residual_fit, np.nan, dtype=np.float64)
    valid = np.isfinite(residual_fit) & np.isfinite(floor)[None, :] & (floor[None, :] > 0)
    squared = np.square(
        residual_fit,
        where=np.isfinite(residual_fit),
        out=np.zeros_like(residual_fit),
    )
    stabilized = squared + floor[None, :]
    output[valid] = np.log(stabilized[valid])
    return output


def predict_masked_ridge(
    design: np.ndarray,
    model: dict[str, np.ndarray],
) -> np.ndarray:
    x = augmented_design(np.asarray(design, dtype=np.float64))
    coefficients = np.asarray(model["coefficients"], dtype=np.float64)
    location = np.asarray(model["target_location"], dtype=np.float64)
    scale = np.asarray(model["target_scale"], dtype=np.float64)
    prediction = location[None, :] + scale[None, :] * (x @ coefficients)
    prediction[:, ~np.asarray(model["target_modeled"], dtype=bool)] = np.nan
    return prediction


def log_variance_to_aleatoric_scale(log_variance: np.ndarray) -> np.ndarray:
    clipped = np.clip(
        np.asarray(log_variance, dtype=np.float64),
        LOG_VARIANCE_MINIMUM,
        LOG_VARIANCE_MAXIMUM,
    )
    scale = np.exp(0.5 * clipped)
    scale[~np.isfinite(log_variance)] = np.nan
    return scale


def score_scale_penalties(
    design: np.ndarray,
    log_targets: np.ndarray,
    training_mask: np.ndarray,
    validation_mask: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, int]:
    """Masked fold score without changing the frozen Cell 4 ridge grid."""
    training_rows = np.flatnonzero(training_mask)
    validation_rows = np.flatnonzero(validation_mask)
    x_train = augmented_design(design[training_rows])
    x_validation = augmented_design(design[validation_rows])
    y_train = np.asarray(log_targets[training_rows], dtype=np.float64)
    y_validation = np.asarray(log_targets[validation_rows], dtype=np.float64)
    observed_train = np.isfinite(y_train)

    squared_error = np.zeros(len(SCALE_RIDGE_PENALTIES), dtype=np.float64)
    observation_count = np.zeros(len(SCALE_RIDGE_PENALTIES), dtype=np.int64)
    fitted_targets = 0
    minimum_rows = max(MINIMUM_SCALE_ROWS, x_train.shape[1] + 10)

    for target_indices, group_observed in group_targets_by_observation_mask(observed_train):
        if int(group_observed.sum()) < minimum_rows:
            continue
        x_observed = x_train[group_observed]
        y_observed = y_train[group_observed][:, target_indices]
        location, scale, valid_target = robust_target_parameters(y_observed)
        if not valid_target.any():
            continue
        target_indices = target_indices[valid_target]
        location = location[valid_target]
        scale = scale[valid_target]
        y_standardized = (
            y_observed[:, valid_target] - location[None, :]
        ) / scale[None, :]
        gram = x_observed.T @ x_observed
        cross_product = x_observed.T @ y_standardized
        validation_values = y_validation[:, target_indices]
        validation_observed = np.isfinite(validation_values)
        fitted_targets += len(target_indices)

        for penalty_index, penalty in enumerate(SCALE_RIDGE_PENALTIES):
            coefficients = solve_ridge(gram, cross_product, float(penalty))
            prediction = (
                location[None, :] + scale[None, :] * (x_validation @ coefficients)
            )
            standardized_error = (validation_values - prediction) / scale[None, :]
            valid = validation_observed & np.isfinite(standardized_error)
            clipped = np.clip(
                standardized_error[valid],
                -MEAN_MODEL.residual_cap,
                MEAN_MODEL.residual_cap,
            )
            squared_error[penalty_index] += float(np.square(clipped).sum())
            observation_count[penalty_index] += int(len(clipped))

    return squared_error, observation_count, fitted_targets


def regularized_design_precision(design: np.ndarray, penalty: float) -> np.ndarray:
    """Inverse regularized design Gram matrix using Cell 4 ridge geometry."""
    x = augmented_design(np.asarray(design, dtype=np.float64))
    gram = x.T @ x
    diagonal = np.arange(len(gram))
    gram[diagonal[1:], diagonal[1:]] += float(penalty)
    identity = np.eye(len(gram), dtype=np.float64)
    try:
        precision = np.linalg.solve(gram, identity)
    except np.linalg.LinAlgError:
        precision = np.linalg.pinv(gram, rcond=1.0e-12, hermitian=True)
    return 0.5 * (precision + precision.T)


def design_leverage(design: np.ndarray, precision: np.ndarray) -> np.ndarray:
    design = np.asarray(design, dtype=np.float64)
    precision = np.asarray(precision, dtype=np.float64)
    output = np.empty(len(design), dtype=np.float64)
    for start in range(0, len(design), LEVERAGE_CHUNK_ROWS):
        stop = min(start + LEVERAGE_CHUNK_ROWS, len(design))
        x = augmented_design(design[start:stop])
        output[start:stop] = np.einsum(
            "ij,jk,ik->i", x, precision, x, optimize=True
        )
    tiny_negative = (output < 0.0) & (output > -1.0e-10)
    output[tiny_negative] = 0.0
    if (output < 0.0).any() or not np.isfinite(output).all():
        raise RuntimeError("Regularized design leverage is invalid.")
    return output


def standardize_support(
    leverage: np.ndarray,
    reference_location: float,
    reference_scale: float,
) -> np.ndarray:
    standardized = (
        np.asarray(leverage, dtype=np.float64) - float(reference_location)
    ) / float(reference_scale)
    return standardized


def prefixed_state_arrays(
    state: dict[str, Any],
    prefix: str,
) -> tuple[dict[str, Any], dict[str, np.ndarray]]:
    metadata, arrays = serialize_design_state(state)
    return metadata, {f"{prefix}__{key}": value for key, value in arrays.items()}


def restore_prefixed_state(
    metadata: dict[str, Any],
    arrays: dict[str, np.ndarray],
    prefix: str,
) -> dict[str, Any]:
    marker = prefix + "__"
    selected = {
        key[len(marker):]: value
        for key, value in arrays.items()
        if key.startswith(marker)
    }
    return restore_design_state(metadata, selected)


def add_prefixed_model_arrays(
    destination: dict[str, np.ndarray],
    model: dict[str, np.ndarray],
    prefix: str,
) -> None:
    for key in (
        "coefficients",
        "target_location",
        "target_scale",
        "target_training_count",
        "target_modeled",
    ):
        value = np.asarray(model[key])
        if key == "coefficients":
            value = value.astype(np.float32)
        destination[f"{prefix}__{key}"] = value


def prefixed_model(
    arrays: dict[str, np.ndarray],
    prefix: str,
) -> dict[str, np.ndarray]:
    return {
        key: arrays[f"{prefix}__{key}"]
        for key in (
            "coefficients",
            "target_location",
            "target_scale",
            "target_training_count",
            "target_modeled",
        )
    }


# =============================================================================
# 3. Per-case cross-fitting and uncertainty artifacts
# =============================================================================

def case_uncertainty_paths(farm: str, event_id: int) -> tuple[Path, Path]:
    relative = Path(safe_slug(farm)) / f"event_{int(event_id):03d}"
    return (
        CELL5_MODEL_ROOT / relative.with_suffix(".json"),
        CELL5_MODEL_ROOT / relative.with_suffix(".npz"),
    )


def case_uncertainty_input_signature(case_row: Any) -> str:
    return sha256_json(
        {
            "cell4_receipt_sha256": CELL4_RECEIPT_SHA256,
            "case_key": str(case_row.case_key),
            "cell4_artifact_sha256": str(case_row.artifact_sha256),
            "uncertainty_implementation_sha256": UNCERTAINTY_IMPLEMENTATION_SHA256,
        }
    )


def existing_case_uncertainty_summary(
    metadata_path: Path,
    artifact_path: Path,
    expected_input_signature: str,
) -> dict[str, Any] | None:
    if not metadata_path.exists() or not artifact_path.exists():
        return None
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    if metadata.get("case_input_signature") != expected_input_signature:
        raise RuntimeError(
            f"Uncertainty-model input drift for {metadata.get('case_key', metadata_path.stem)}. "
            "Do not overwrite the existing artifact."
        )
    if metadata.get("artifact_sha256") != file_sha256(artifact_path):
        raise RuntimeError(f"Uncertainty artifact hash failed: {artifact_path}")
    return dict(metadata["summary"])


def fit_one_case_uncertainty(case_row: Any) -> dict[str, Any]:
    metadata_path, artifact_path = case_uncertainty_paths(
        case_row.farm, case_row.event_id
    )
    input_signature = case_uncertainty_input_signature(case_row)
    existing = existing_case_uncertainty_summary(
        metadata_path, artifact_path, input_signature
    )
    if existing is not None:
        return existing

    cache = load_case_cache(str(case_row.case_key))
    mean_model = load_case_mean_model(str(case_row.case_key))
    if cache["metadata"].get("outcome_fields_present") is not False:
        raise RuntimeError(f"Unsafe Cell 3 cache for {case_row.case_key}.")
    if mean_model["metadata"].get("outcome_fields_present") is not False:
        raise RuntimeError(f"Unsafe Cell 4 model for {case_row.case_key}.")

    drivers = np.asarray(cache["drivers"], dtype=np.float64)
    fit_mask = np.asarray(cache["fit_mask"], dtype=bool)
    segment_id = np.asarray(cache["segment_id"], dtype=np.int32)
    driver_names = list(cache["metadata"]["driver_names"])
    target_names = list(cache["metadata"]["target_names"])
    fit_rows = np.flatnonzero(fit_mask).astype(np.int32)
    stored_fit_rows = np.asarray(mean_model["fit_row_indices"], dtype=np.int32)
    if not np.array_equal(fit_rows, stored_fit_rows):
        raise RuntimeError(f"Cell 3/4 fit-row mismatch for {case_row.case_key}.")

    residual_fit = np.asarray(mean_model["oof_residual_fit"], dtype=np.float64)
    residual_fold_fit = np.asarray(mean_model["oof_fold_fit"], dtype=np.int8)
    if residual_fit.shape != (len(fit_rows), len(target_names)):
        raise RuntimeError(f"OOF residual shape mismatch for {case_row.case_key}.")
    if not np.isfinite(drivers).all():
        raise RuntimeError(f"Non-finite drivers remain for {case_row.case_key}.")

    folds = list(iter_case_temporal_folds(fit_mask, segment_id))
    if len(folds) != UNCERTAINTY.scale_crossfit_folds:
        raise RuntimeError(f"Unexpected temporal fold count for {case_row.case_key}.")

    fold_local_masks: list[tuple[dict[str, Any], np.ndarray, np.ndarray]] = []
    for fold in folds:
        training_local = np.asarray(fold["training_mask"][fit_rows], dtype=bool)
        validation_local = np.asarray(fold["validation_mask"][fit_rows], dtype=bool)
        if not np.all(residual_fold_fit[validation_local] == int(fold["fold"])):
            raise RuntimeError(f"Cell 4 OOF fold mismatch for {case_row.case_key}.")
        fold_local_masks.append((fold, training_local, validation_local))

    # Pass 1: select the heteroscedastic ridge penalty with blocked OOF error.
    penalty_sse = np.zeros(len(SCALE_RIDGE_PENALTIES), dtype=np.float64)
    penalty_count = np.zeros(len(SCALE_RIDGE_PENALTIES), dtype=np.int64)
    fold_diagnostics: list[dict[str, Any]] = []
    for fold, training_local, validation_local in fold_local_masks:
        reference_scale, reference_valid = residual_reference_scale(
            residual_fit, training_local
        )
        log_targets = log_squared_residual_target(residual_fit, reference_scale)
        state, design = fit_design_state(
            drivers,
            driver_names,
            fold["training_mask"],
            str(case_row.farm),
        )
        design_fit = design[fit_rows]
        fold_sse, fold_count, fitted_targets = score_scale_penalties(
            design_fit,
            log_targets,
            training_local,
            validation_local,
        )
        penalty_sse += fold_sse
        penalty_count += fold_count
        fold_diagnostics.append(
            {
                "fold": int(fold["fold_number"]),
                "training_rows": int(training_local.sum()),
                "validation_rows": int(validation_local.sum()),
                "embargoed_fit_rows": int(fold["embargoed_fit_rows"]),
                "design_terms": int(design.shape[1]),
                "reference_scale_targets": int(reference_valid.sum()),
                "fitted_scale_targets_during_selection": int(fitted_targets),
            }
        )
        del reference_scale, log_targets, state, design, design_fit

    if (penalty_count == 0).any():
        raise RuntimeError(
            f"No scale OOF observations for one or more penalties in {case_row.case_key}."
        )
    penalty_mse = penalty_sse / penalty_count
    minimum_mse = float(np.min(penalty_mse))
    tied = np.flatnonzero(
        np.isclose(penalty_mse, minimum_mse, rtol=1.0e-12, atol=1.0e-12)
    )
    selected_index = int(tied[-1])
    selected_scale_penalty = float(SCALE_RIDGE_PENALTIES[selected_index])
    mean_penalty = float(case_row.selected_penalty)

    # Pass 2: save fold models and obtain honest OOF scales/support leverage.
    oof_log_variance_fit = np.full(residual_fit.shape, np.nan, dtype=np.float32)
    oof_leverage_fit = np.full(len(fit_rows), np.nan, dtype=np.float64)
    fold_scale_fit_count = np.zeros(residual_fit.shape[1], dtype=np.int16)
    artifact_arrays: dict[str, np.ndarray] = {}
    fold_model_metadata: list[dict[str, Any]] = []

    for fold, training_local, validation_local in fold_local_masks:
        fold_number = int(fold["fold_number"])
        prefix = f"fold_{fold_number}"
        reference_scale, _ = residual_reference_scale(residual_fit, training_local)
        log_targets = log_squared_residual_target(residual_fit, reference_scale)
        state, design = fit_design_state(
            drivers,
            driver_names,
            fold["training_mask"],
            str(case_row.farm),
        )
        design_fit = design[fit_rows]
        scale_model = fit_final_masked_ridge(
            design_fit,
            log_targets,
            training_local,
            selected_scale_penalty,
        )
        prediction = predict_masked_ridge(
            design_fit[validation_local], scale_model
        )
        oof_log_variance_fit[validation_local] = prediction.astype(np.float32)
        fold_scale_fit_count += np.asarray(
            scale_model["target_modeled"], dtype=np.int16
        )

        support_precision = regularized_design_precision(
            design_fit[training_local], mean_penalty
        )
        oof_leverage_fit[validation_local] = design_leverage(
            design_fit[validation_local], support_precision
        ) * int(training_local.sum())

        state_metadata, state_arrays = prefixed_state_arrays(state, prefix)
        artifact_arrays.update(state_arrays)
        add_prefixed_model_arrays(artifact_arrays, scale_model, prefix)
        artifact_arrays[f"{prefix}__residual_reference_scale"] = (
            reference_scale.astype(np.float64)
        )
        artifact_arrays[f"{prefix}__support_precision"] = (
            support_precision.astype(np.float64)
        )
        fold_model_metadata.append(
            {
                "fold": fold_number,
                "fold_index": int(fold["fold"]),
                "prefix": prefix,
                "design_state": state_metadata,
                "modeled_scale_targets": int(
                    np.asarray(scale_model["target_modeled"], dtype=bool).sum()
                ),
                "support_training_rows": int(training_local.sum()),
            }
        )
        del (
            reference_scale,
            log_targets,
            state,
            design,
            design_fit,
            scale_model,
            prediction,
            support_precision,
        )

    if not np.isfinite(oof_leverage_fit).all():
        raise RuntimeError(f"Incomplete OOF leverage for {case_row.case_key}.")
    support_location, support_scale, support_valid = finite_robust_location_scale(
        oof_leverage_fit
    )
    if not support_valid:
        raise RuntimeError(f"Degenerate OOF support reference for {case_row.case_key}.")
    oof_support = standardize_support(
        oof_leverage_fit, support_location, support_scale
    )
    oof_positive_support = np.maximum(oof_support, 0.0)

    # Final uncertainty models use every training-normal OOF residual. The final
    # scale design is exactly the frozen final Cell 4 mean-design transformation.
    final_design = transform_design(drivers, mean_model["design_state"])
    final_design_fit = final_design[fit_rows]
    final_reference_scale, _ = residual_reference_scale(
        residual_fit, np.ones(len(fit_rows), dtype=bool)
    )
    final_log_targets = log_squared_residual_target(
        residual_fit, final_reference_scale
    )
    final_scale_model = fit_final_masked_ridge(
        final_design_fit,
        final_log_targets,
        np.ones(len(fit_rows), dtype=bool),
        selected_scale_penalty,
    )
    final_support_precision = regularized_design_precision(
        final_design_fit, mean_penalty
    )

    mean_modeled = np.asarray(mean_model["target_modeled"], dtype=bool)
    mean_r2 = np.asarray(mean_model["target_crossfit_r2"], dtype=np.float64)
    scale_modeled = np.asarray(final_scale_model["target_modeled"], dtype=bool)
    complete_scale_crossfit = fold_scale_fit_count == UNCERTAINTY.scale_crossfit_folds
    hard_eligible = (
        mean_modeled
        & scale_modeled
        & complete_scale_crossfit
        & np.isfinite(mean_r2)
    )
    modelability_masks = np.vstack(
        [hard_eligible & (mean_r2 >= threshold) for threshold in MODELABILITY_THRESHOLDS]
    )
    if not modelability_masks[0].any():
        raise RuntimeError(
            f"No target passes the fallback modelability gate in {case_row.case_key}."
        )

    # Label-free diagnostics. These do not calibrate or choose a threshold.
    oof_aleatoric = log_variance_to_aleatoric_scale(oof_log_variance_fit)
    oof_total_scale = oof_aleatoric * np.sqrt(1.0 + oof_positive_support[:, None])
    fallback = modelability_masks[0]
    valid_aleatoric = (
        np.isfinite(residual_fit[:, fallback])
        & np.isfinite(oof_aleatoric[:, fallback])
        & (oof_aleatoric[:, fallback] > 0.0)
    )
    standardized_aleatoric = np.full(valid_aleatoric.shape, np.nan, dtype=np.float64)
    standardized_total = np.full(valid_aleatoric.shape, np.nan, dtype=np.float64)
    absolute_residual = np.abs(residual_fit[:, fallback])
    standardized_aleatoric[valid_aleatoric] = (
        absolute_residual[valid_aleatoric]
        / oof_aleatoric[:, fallback][valid_aleatoric]
    )
    standardized_total[valid_aleatoric] = (
        absolute_residual[valid_aleatoric]
        / oof_total_scale[:, fallback][valid_aleatoric]
    )

    add_prefixed_model_arrays(artifact_arrays, final_scale_model, "final_scale")
    artifact_arrays.update(
        {
            "final_residual_reference_scale": final_reference_scale.astype(np.float64),
            "final_support_precision": final_support_precision.astype(np.float64),
            "final_support_training_rows": np.asarray(len(fit_rows), dtype=np.int64),
            "support_reference_location": np.asarray(support_location, dtype=np.float64),
            "support_reference_scale": np.asarray(support_scale, dtype=np.float64),
            "modelability_thresholds": MODELABILITY_THRESHOLDS.astype(np.float64),
            "modelability_masks": modelability_masks.astype(bool),
            "hard_eligible_targets": hard_eligible.astype(bool),
            "target_crossfit_r2": mean_r2.astype(np.float64),
            "target_temperature": np.asarray(mean_model["target_temperature"], dtype=bool),
            "target_scale_fold_fit_count": fold_scale_fit_count.astype(np.int16),
            "fit_row_indices": fit_rows.astype(np.int32),
            "scale_ridge_penalties": SCALE_RIDGE_PENALTIES.astype(np.float64),
            "scale_selection_mse": penalty_mse.astype(np.float64),
            "scale_selection_count": penalty_count.astype(np.int64),
        }
    )
    atomic_save_npz(artifact_path, **artifact_arrays)
    artifact_hash = file_sha256(artifact_path)

    threshold_counts = {
        f"targets_{threshold_name(threshold)}": int(modelability_masks[index].sum())
        for index, threshold in enumerate(MODELABILITY_THRESHOLDS)
    }
    threshold_fractions = {
        f"fraction_{threshold_name(threshold)}": float(modelability_masks[index].mean())
        for index, threshold in enumerate(MODELABILITY_THRESHOLDS)
    }
    summary = {
        "case_key": str(case_row.case_key),
        "farm": str(case_row.farm),
        "asset_id": str(case_row.asset_id),
        "event_id": int(case_row.event_id),
        "rows": int(len(drivers)),
        "training_normal_rows": int(len(fit_rows)),
        "targets": int(len(target_names)),
        "selected_mean_penalty": mean_penalty,
        "selected_scale_penalty": selected_scale_penalty,
        "scale_selection_mse": float(penalty_mse[selected_index]),
        "final_scale_modeled_targets": int(scale_modeled.sum()),
        "complete_scale_crossfit_targets": int(complete_scale_crossfit.sum()),
        "hard_eligible_targets": int(hard_eligible.sum()),
        **threshold_counts,
        **threshold_fractions,
        "median_oof_absolute_standardized_residual_aleatoric": float(
            np.nanmedian(standardized_aleatoric)
        ),
        "median_oof_absolute_standardized_residual_total": float(
            np.nanmedian(standardized_total)
        ),
        "oof_support_location": float(support_location),
        "oof_support_scale": float(support_scale),
        "oof_support_p95": float(np.quantile(oof_support, 0.95)),
        "oof_positive_support_fraction": float((oof_support > 0.0).mean()),
        "artifact_relative_path": artifact_path.relative_to(CELL5_MODEL_ROOT).as_posix(),
        "metadata_relative_path": metadata_path.relative_to(CELL5_MODEL_ROOT).as_posix(),
        "case_input_signature": input_signature,
        "artifact_sha256": artifact_hash,
    }

    save_json(
        {
            "cell5_version": CELL5_VERSION,
            "created_at_utc": utc_now(),
            "contract_sha256": CONTRACT_SHA256,
            "cell4_receipt_sha256": CELL4_RECEIPT_SHA256,
            "uncertainty_implementation_sha256": UNCERTAINTY_IMPLEMENTATION_SHA256,
            "case_input_signature": input_signature,
            "case_key": str(case_row.case_key),
            "farm": str(case_row.farm),
            "asset_id": str(case_row.asset_id),
            "event_id": int(case_row.event_id),
            "driver_names": driver_names,
            "target_names": target_names,
            "fold_models": fold_model_metadata,
            "selected_mean_penalty": mean_penalty,
            "selected_scale_penalty": selected_scale_penalty,
            "scale_ridge_penalties": SCALE_RIDGE_PENALTIES.tolist(),
            "scale_selection_mse": penalty_mse.tolist(),
            "artifact_sha256": artifact_hash,
            "artifact_relative_path": summary["artifact_relative_path"],
            "outcome_fields_present": False,
            "summary": summary,
        },
        metadata_path,
    )
    return summary


# =============================================================================
# 4. Safe loading and reproducible uncertainty prediction
# =============================================================================

def load_case_uncertainty_model(case_key: str) -> dict[str, Any]:
    match = CASE_UNCERTAINTY_REGISTRY.loc[
        CASE_UNCERTAINTY_REGISTRY["case_key"].eq(case_key)
    ]
    if len(match) != 1:
        raise KeyError(f"Unknown or duplicate case_key: {case_key}")
    row = match.iloc[0]
    metadata_path = CELL5_MODEL_ROOT / row["metadata_relative_path"]
    artifact_path = CELL5_MODEL_ROOT / row["artifact_relative_path"]
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    if metadata.get("outcome_fields_present") is not False:
        raise RuntimeError(f"Unsafe uncertainty metadata for {case_key}.")
    if file_sha256(artifact_path) != row["artifact_sha256"]:
        raise RuntimeError(f"Uncertainty artifact hash mismatch for {case_key}.")
    arrays = dict(np.load(artifact_path, allow_pickle=False))
    arrays["metadata"] = metadata
    return arrays


def predict_case_uncertainty(
    case_key: str,
    row_selector: slice | np.ndarray | list[int] | None = None,
) -> dict[str, np.ndarray]:
    """Predict final aleatoric, support, and combined scale for arbitrary rows."""
    cache = load_case_cache(case_key)
    mean_model = load_case_mean_model(case_key)
    uncertainty_model = load_case_uncertainty_model(case_key)
    drivers = np.asarray(cache["drivers"], dtype=np.float64)
    if row_selector is not None:
        drivers = drivers[row_selector]
    design = transform_design(drivers, mean_model["design_state"])
    log_variance = predict_masked_ridge(
        design, prefixed_model(uncertainty_model, "final_scale")
    )
    aleatoric_scale = log_variance_to_aleatoric_scale(log_variance)
    leverage = design_leverage(
        design, uncertainty_model["final_support_precision"]
    ) * int(uncertainty_model["final_support_training_rows"])
    support_uncertainty = standardize_support(
        leverage,
        float(uncertainty_model["support_reference_location"]),
        float(uncertainty_model["support_reference_scale"]),
    )
    positive_support = np.maximum(support_uncertainty, 0.0)
    total_predictive_scale = (
        aleatoric_scale * np.sqrt(1.0 + positive_support[:, None])
    )
    return {
        "aleatoric_scale": aleatoric_scale.astype(np.float32),
        "support_uncertainty": support_uncertainty.astype(np.float32),
        "positive_support_uncertainty": positive_support.astype(np.float32),
        "total_predictive_scale": total_predictive_scale.astype(np.float32),
        "modelability_thresholds": np.asarray(
            uncertainty_model["modelability_thresholds"], dtype=np.float64
        ),
        "modelability_masks": np.asarray(
            uncertainty_model["modelability_masks"], dtype=bool
        ),
    }


def predict_case_crossfit_uncertainty(case_key: str) -> dict[str, np.ndarray]:
    """Reconstruct training-normal OOF uncertainty without caching dense matrices."""
    cache = load_case_cache(case_key)
    model = load_case_uncertainty_model(case_key)
    drivers = np.asarray(cache["drivers"], dtype=np.float64)
    fit_mask = np.asarray(cache["fit_mask"], dtype=bool)
    segment_id = np.asarray(cache["segment_id"], dtype=np.int32)
    fit_rows = np.asarray(model["fit_row_indices"], dtype=np.int32)
    if not np.array_equal(fit_rows, np.flatnonzero(fit_mask).astype(np.int32)):
        raise RuntimeError(f"Fit-row mismatch while reconstructing {case_key}.")

    target_count = len(model["metadata"]["target_names"])
    log_variance = np.full((len(fit_rows), target_count), np.nan, dtype=np.float32)
    leverage = np.full(len(fit_rows), np.nan, dtype=np.float64)
    row_to_fit = np.full(len(fit_mask), -1, dtype=np.int64)
    row_to_fit[fit_rows] = np.arange(len(fit_rows), dtype=np.int64)
    folds_by_number = {
        int(fold["fold_number"]): fold
        for fold in iter_case_temporal_folds(fit_mask, segment_id)
    }

    for fold_metadata in model["metadata"]["fold_models"]:
        fold_number = int(fold_metadata["fold"])
        prefix = str(fold_metadata["prefix"])
        fold = folds_by_number[fold_number]
        validation_rows = np.flatnonzero(fold["validation_mask"])
        validation_local = row_to_fit[validation_rows]
        if (validation_local < 0).any():
            raise RuntimeError(f"Invalid validation alignment for {case_key}.")
        state = restore_prefixed_state(
            fold_metadata["design_state"], model, prefix
        )
        validation_design = transform_design(drivers[validation_rows], state)
        prediction = predict_masked_ridge(
            validation_design, prefixed_model(model, prefix)
        )
        log_variance[validation_local] = prediction.astype(np.float32)
        leverage[validation_local] = design_leverage(
            validation_design, model[f"{prefix}__support_precision"]
        ) * int(fold_metadata["support_training_rows"])

    if not np.isfinite(leverage).all():
        raise RuntimeError(f"Incomplete cross-fitted leverage for {case_key}.")
    aleatoric_scale = log_variance_to_aleatoric_scale(log_variance)
    support_uncertainty = standardize_support(
        leverage,
        float(model["support_reference_location"]),
        float(model["support_reference_scale"]),
    )
    positive_support = np.maximum(support_uncertainty, 0.0)
    total_predictive_scale = (
        aleatoric_scale * np.sqrt(1.0 + positive_support[:, None])
    )
    return {
        "fit_row_indices": fit_rows,
        "aleatoric_scale": aleatoric_scale.astype(np.float32),
        "support_uncertainty": support_uncertainty.astype(np.float32),
        "positive_support_uncertainty": positive_support.astype(np.float32),
        "total_predictive_scale": total_predictive_scale.astype(np.float32),
        "modelability_thresholds": np.asarray(
            model["modelability_thresholds"], dtype=np.float64
        ),
        "modelability_masks": np.asarray(model["modelability_masks"], dtype=bool),
    }


# =============================================================================
# 5. Fit or verify all 95 cases
# =============================================================================

_uncertainty_summaries: list[dict[str, Any]] = []
for _farm in DATASET.farms:
    _farm_cases = CASE_MEAN_MODEL_REGISTRY.loc[
        CASE_MEAN_MODEL_REGISTRY["farm"].eq(_farm)
    ]
    print(
        f"Cross-fitting aleatoric and support uncertainty — {_farm}: "
        f"{len(_farm_cases)} cases",
        flush=True,
    )
    for _case_row in _farm_cases.itertuples(index=False):
        _summary = fit_one_case_uncertainty(_case_row)
        _uncertainty_summaries.append(_summary)
        print(
            f"  event {_case_row.event_id:>3}: "
            f"λ_scale={_summary['selected_scale_penalty']:<8g} "
            f"eligible={_summary['targets_r2_ge_0']:>3}/"
            f"{_summary['targets']:<3} "
            f"support p95={_summary['oof_support_p95']:.3f}",
            flush=True,
        )

CASE_UNCERTAINTY_REGISTRY = pd.DataFrame(_uncertainty_summaries).sort_values(
    ["farm", "event_id"], kind="stable"
).reset_index(drop=True)

if len(CASE_UNCERTAINTY_REGISTRY) != DATASET.expected_total_cases:
    raise RuntimeError("Not all 95 cases produced an uncertainty artifact.")
if CASE_UNCERTAINTY_REGISTRY["case_key"].duplicated().any():
    raise RuntimeError("Duplicate case keys exist in the uncertainty registry.")
if (CASE_UNCERTAINTY_REGISTRY["targets_r2_ge_0"] <= 0).any():
    raise RuntimeError("At least one case has no target at the fallback R² gate.")

# One final-prediction slice per farm verifies dimensions and artifact hashes.
for _case_key in CASE_UNCERTAINTY_REGISTRY.groupby("farm", sort=False).head(1)["case_key"]:
    _loaded = load_case_uncertainty_model(_case_key)
    _prediction = predict_case_uncertainty(_case_key, slice(0, 32))
    if _prediction["total_predictive_scale"].shape[0] != 32:
        raise RuntimeError(f"Uncertainty prediction smoke test failed for {_case_key}.")
    if _prediction["modelability_masks"].shape[0] != len(MODELABILITY_THRESHOLDS):
        raise RuntimeError(f"Modelability-grid smoke test failed for {_case_key}.")
    del _loaded, _prediction


# =============================================================================
# 6. Freeze Cell 5 receipt and report diagnostics
# =============================================================================

UNCERTAINTY_FARM_SUMMARY = (
    CASE_UNCERTAINTY_REGISTRY.groupby("farm", sort=False)
    .agg(
        cases=("case_key", "size"),
        median_selected_scale_penalty=("selected_scale_penalty", "median"),
        median_scale_modeled_targets=("final_scale_modeled_targets", "median"),
        median_complete_scale_crossfit_targets=(
            "complete_scale_crossfit_targets", "median"
        ),
        median_oof_abs_standardized_aleatoric=(
            "median_oof_absolute_standardized_residual_aleatoric", "median"
        ),
        median_oof_abs_standardized_total=(
            "median_oof_absolute_standardized_residual_total", "median"
        ),
        median_support_p95=("oof_support_p95", "median"),
        median_positive_support_fraction=(
            "oof_positive_support_fraction", "median"
        ),
    )
    .reset_index()
)

_modelability_rows: list[dict[str, Any]] = []
for _row in CASE_UNCERTAINTY_REGISTRY.itertuples(index=False):
    for _threshold in MODELABILITY_THRESHOLDS:
        _count = int(getattr(_row, f"targets_{threshold_name(_threshold)}"))
        _modelability_rows.append(
            {
                "case_key": str(_row.case_key),
                "farm": str(_row.farm),
                "event_id": int(_row.event_id),
                "r2_threshold": float(_threshold),
                "eligible_targets": _count,
                "total_targets": int(_row.targets),
                "eligible_fraction": _count / int(_row.targets),
            }
        )
MODELABILITY_CASE_GRID = pd.DataFrame(_modelability_rows).sort_values(
    ["farm", "event_id", "r2_threshold"], kind="stable"
).reset_index(drop=True)

MODELABILITY_FARM_SUMMARY = (
    MODELABILITY_CASE_GRID.groupby(["farm", "r2_threshold"], sort=False)
    .agg(
        cases=("case_key", "size"),
        minimum_eligible_targets=("eligible_targets", "min"),
        median_eligible_targets=("eligible_targets", "median"),
        minimum_eligible_fraction=("eligible_fraction", "min"),
        median_eligible_fraction=("eligible_fraction", "median"),
    )
    .reset_index()
)

SCALE_RIDGE_SELECTION_SUMMARY = (
    CASE_UNCERTAINTY_REGISTRY.groupby(["farm", "selected_scale_penalty"])
    .size()
    .rename("cases")
    .reset_index()
    .sort_values(["farm", "selected_scale_penalty"], kind="stable")
)

LOWEST_MODELABILITY_CASES = (
    MODELABILITY_CASE_GRID.loc[
        MODELABILITY_CASE_GRID["r2_threshold"].eq(
            float(MEAN_MODEL.fallback_minimum_r2)
        )
    ]
    .sort_values(
        ["eligible_fraction", "eligible_targets", "farm", "event_id"],
        kind="stable",
    )
    .head(20)
    .reset_index(drop=True)
)

_component_hashes = {
    "case_uncertainty_registry_sha256": dataframe_sha256(
        CASE_UNCERTAINTY_REGISTRY, ("farm", "event_id")
    ),
    "uncertainty_farm_summary_sha256": dataframe_sha256(
        UNCERTAINTY_FARM_SUMMARY, ("farm",)
    ),
    "modelability_case_grid_sha256": dataframe_sha256(
        MODELABILITY_CASE_GRID, ("farm", "event_id", "r2_threshold")
    ),
    "modelability_farm_summary_sha256": dataframe_sha256(
        MODELABILITY_FARM_SUMMARY, ("farm", "r2_threshold")
    ),
    "scale_ridge_selection_summary_sha256": dataframe_sha256(
        SCALE_RIDGE_SELECTION_SUMMARY, ("farm", "selected_scale_penalty")
    ),
}
CELL5_RECEIPT_SHA256 = sha256_json(
    {
        "contract_sha256": CONTRACT_SHA256,
        "cell4_receipt_sha256": CELL4_RECEIPT_SHA256,
        "uncertainty_implementation_sha256": UNCERTAINTY_IMPLEMENTATION_SHA256,
        "component_hashes": _component_hashes,
    }
)

CELL5_RECEIPT_PATH = CELL5_QUALITY_ROOT / "cell5_uncertainty_receipt.json"
if CELL5_RECEIPT_PATH.exists():
    _existing_receipt = json.loads(CELL5_RECEIPT_PATH.read_text(encoding="utf-8"))
    if _existing_receipt.get("cell5_receipt_sha256") != CELL5_RECEIPT_SHA256:
        raise RuntimeError(
            "A different Cell 5 receipt exists for this experiment. Do not overwrite it."
        )
    CELL5_STATE = "existing identical Cell 5 receipt verified"
    _write_cell5_receipt = False
else:
    CELL5_STATE = "new Cell 5 receipt frozen"
    _write_cell5_receipt = True

save_csv_atomic(
    CASE_UNCERTAINTY_REGISTRY,
    CELL5_QUALITY_ROOT / "case_uncertainty_registry.csv",
)
save_csv_atomic(
    UNCERTAINTY_FARM_SUMMARY,
    CELL5_QUALITY_ROOT / "uncertainty_farm_summary.csv",
)
save_csv_atomic(
    MODELABILITY_CASE_GRID,
    CELL5_QUALITY_ROOT / "modelability_case_grid.csv",
)
save_csv_atomic(
    MODELABILITY_FARM_SUMMARY,
    CELL5_QUALITY_ROOT / "modelability_farm_summary.csv",
)
save_csv_atomic(
    SCALE_RIDGE_SELECTION_SUMMARY,
    CELL5_QUALITY_ROOT / "scale_ridge_selection_summary.csv",
)
save_json(
    UNCERTAINTY_IMPLEMENTATION,
    CELL5_QUALITY_ROOT / "uncertainty_implementation.json",
)

if _write_cell5_receipt:
    save_json(
        {
            "cell5_version": CELL5_VERSION,
            "created_at_utc": utc_now(),
            "contract_sha256": CONTRACT_SHA256,
            "dataset_manifest_sha256": DATASET_MANIFEST_SHA256,
            "cell2_audit_sha256": CELL2_AUDIT_SHA256,
            "cell3_receipt_sha256": CELL3_RECEIPT_SHA256,
            "cell4_receipt_sha256": CELL4_RECEIPT_SHA256,
            "uncertainty_implementation_sha256": UNCERTAINTY_IMPLEMENTATION_SHA256,
            "component_hashes": _component_hashes,
            "cell5_receipt_sha256": CELL5_RECEIPT_SHA256,
            "outcomes_read": False,
        },
        CELL5_RECEIPT_PATH,
    )


print("\n" + "=" * 92)
print("UC-RCF-NBM CELL 5 — CROSS-FITTED ALEATORIC AND SUPPORT UNCERTAINTY")
print("=" * 92)
print("\nUNCERTAINTY MODEL SUMMARY ON TRAINING-NORMAL OOF RESIDUALS")
display(UNCERTAINTY_FARM_SUMMARY)
print("\nTARGET-WISE MODELABILITY SENSITIVITY GRID")
display(MODELABILITY_FARM_SUMMARY)
print("\nLOWEST MODELABILITY CASES AT THE FALLBACK R² GATE")
display(LOWEST_MODELABILITY_CASES)
print("\nSCALE RIDGE PENALTIES SELECTED BY BLOCKED OOF MSE")
display(SCALE_RIDGE_SELECTION_SUMMARY)

print("\n" + "-" * 92)
print(f"Cases modeled                       : {len(CASE_UNCERTAINTY_REGISTRY)}")
print(f"Scale temporal folds                : {UNCERTAINTY.scale_crossfit_folds}")
print(
    "Modelability R² sensitivity grid    : "
    f"{tuple(float(value) for value in MODELABILITY_THRESHOLDS)}"
)
print(f"Fallback target gate                : R² >= {MEAN_MODEL.fallback_minimum_r2:g}")
print(f"Case-level exclusions               : 0")
print(f"Uncertainty implementation SHA-256  : {UNCERTAINTY_IMPLEMENTATION_SHA256}")
print(f"Cell 5 receipt SHA-256              : {CELL5_RECEIPT_SHA256}")
print(f"Cell 5 state                        : {CELL5_STATE}")
print("Fold-local scale preprocessing      : Yes")
print("Cross-fitted support reference      : Yes")
print("Absolute uncertainty calibrated here: No — reserved for Cell 6")
print("Event outcomes accessed             : No")
print("Artifact/prediction checks           : PASS")
print("=" * 92)
print("CELL 5 COMPLETED SUCCESSFULLY — UNCERTAINTY MODELS LOCKED")


Cross-fitting aleatoric and support uncertainty — Wind Farm A: 22 cases
  event   0: λ_scale=100      eligible= 41/43  support p95=4.146
  event   3: λ_scale=0.0001   eligible= 42/43  support p95=3.807
  event  10: λ_scale=0.01     eligible= 41/43  support p95=3.302
  event  13: λ_scale=100      eligible= 41/43  support p95=3.489
  event  14: λ_scale=0.0001   eligible= 42/43  support p95=3.387
  event  17: λ_scale=0.0001   eligible= 41/43  support p95=3.503
  event  22: λ_scale=100      eligible= 41/43  support p95=4.020
  event  24: λ_scale=100      eligible= 41/43  support p95=4.258
  event  25: λ_scale=1        eligible= 42/43  support p95=3.543
  event  26: λ_scale=1        eligible= 41/43  support p95=4.087
  event  38: λ_scale=0.01     eligible= 40/43  support p95=3.213
  event  40: λ_scale=0.0001   eligible= 42/43  support p95=3.867
  event  42: λ_scale=0.0001   eligible= 41/43  support p95=3.297
  event  45: λ_scale=0.01     eligible= 39/43  support p95=3.276
  event  51: λ_sca

,farm,cases,median_selected_scale_penalty,median_scale_modeled_targets,median_complete_scale_crossfit_targets,median_oof_abs_standardized_aleatoric,median_oof_abs_standardized_total,median_support_p95,median_positive_support_fraction
0,Wind Farm A,22,1.0000,43.0,43.0,1.245838,1.051348,3.708758,0.50000
1,Wind Farm B,15,0.0001,52.0,52.0,1.236743,1.069197,2.906330,0.49999
2,Wind Farm C,58,0.0001,226.0,226.0,1.222572,1.043704,3.463316,0.49999



TARGET-WISE MODELABILITY SENSITIVITY GRID


,farm,r2_threshold,cases,minimum_eligible_targets,median_eligible_targets,minimum_eligible_fraction,median_eligible_fraction
0,Wind Farm A,0.0,22,39,41.0,0.906977,0.953488
1,Wind Farm A,0.1,22,37,39.0,0.860465,0.906977
2,Wind Farm A,0.3,22,33,35.0,0.767442,0.813953
3,Wind Farm B,0.0,15,3,50.0,0.057692,0.961538
4,Wind Farm B,0.1,15,3,47.0,0.057692,0.903846
5,Wind Farm B,0.3,15,3,38.0,0.057692,0.730769
6,Wind Farm C,0.0,58,32,210.0,0.140351,0.929359
7,Wind Farm C,0.1,58,31,188.0,0.135965,0.828190
8,Wind Farm C,0.3,58,22,170.5,0.096916,0.750556



LOWEST MODELABILITY CASES AT THE FALLBACK R² GATE


,case_key,farm,event_id,r2_threshold,eligible_targets,total_targets,eligible_fraction
0,Wind Farm B::event_83,Wind Farm B,83,0.0,3,52,0.057692
1,Wind Farm C::event_85,Wind Farm C,85,0.0,32,228,0.140351
2,Wind Farm C::event_33,Wind Farm C,33,0.0,35,227,0.154185
3,Wind Farm C::event_79,Wind Farm C,79,0.0,38,223,0.170404
4,Wind Farm C::event_9,Wind Farm C,9,0.0,42,227,0.185022
5,Wind Farm C::event_39,Wind Farm C,39,0.0,42,225,0.186667
6,Wind Farm C::event_91,Wind Farm C,91,0.0,47,227,0.207048
7,Wind Farm C::event_43,Wind Farm C,43,0.0,47,225,0.208889
8,Wind Farm C::event_32,Wind Farm C,32,0.0,48,227,0.211454
9,Wind Farm C::event_88,Wind Farm C,88,0.0,52,229,0.227074



SCALE RIDGE PENALTIES SELECTED BY BLOCKED OOF MSE


,farm,selected_scale_penalty,cases
0,Wind Farm A,0.0001,5
1,Wind Farm A,0.0100,5
2,Wind Farm A,1.0000,6
3,Wind Farm A,100.0000,6
4,Wind Farm B,0.0001,14
5,Wind Farm B,10000.0000,1
6,Wind Farm C,0.0001,34
7,Wind Farm C,0.0100,15
8,Wind Farm C,1.0000,5
9,Wind Farm C,100.0000,3



--------------------------------------------------------------------------------------------
Cases modeled                       : 95
Scale temporal folds                : 5
Modelability R² sensitivity grid    : (0.0, 0.1, 0.3)
Fallback target gate                : R² >= 0
Case-level exclusions               : 0
Uncertainty implementation SHA-256  : 9dc8d186817dbf669fa93fd31c2c966f1a01aa3eda3c635af4087e224026d728
Cell 5 receipt SHA-256              : 00e6ba8f0a41a108e5ac2a8151f37f2f377bfab295fd01f2db8f1ab8f28ccf20
Cell 5 state                        : new Cell 5 receipt frozen
Fold-local scale preprocessing      : Yes
Cross-fitted support reference      : Yes
Absolute uncertainty calibrated here: No — reserved for Cell 6
Event outcomes accessed             : No
Artifact/prediction checks           : PASS
CELL 5 COMPLETED SUCCESSFULLY — UNCERTAINTY MODELS LOCKED


In [10]:
"""CELL 6 — embargoed blocked cross-conformal calibration.

Paste this complete file into the sixth cell of the UC-RCF-NBM notebook and
run it only after Cells 1–5 have completed successfully.

This cell converts the honest cross-fitted residuals and predictive scales from
Cells 4–5 into target-wise conformal multipliers. Calibration uses complete,
non-overlapping 144-step blocks that never cross a temporal gap, an excluded
training-status run, or an OOF fold. The primary interval targets block-balanced
marginal timestamp coverage; a block-maximum simultaneous envelope is retained
as a predeclared sensitivity analysis. The finite-sample probability correction
uses the number of eligible temporal blocks, not the number of correlated
timestamps. Exact exchangeability is explicitly not claimed. No event label,
event boundary, fault description, or CARE outcome is read.
"""

from __future__ import annotations

import json
import math
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print


# =============================================================================
# 0. Bind Cell 6 to the exact completed experiment state
# =============================================================================

EXPECTED_CONTRACT_SHA256 = (
    "827641aecd8e807193ad193d64c274319756092faa53bdb5084f310f62041f49"
)
EXPECTED_DATASET_MANIFEST_SHA256 = (
    "62484bab1219888aa1d0788965ecd77db2b85f0bbb9b476cd3240f4143026f1f"
)
EXPECTED_CELL2_AUDIT_SHA256 = (
    "83732caf4ad3e226b69a671287bb14c4ed72e1bfc3b7c26ca144310fce8e5990"
)
EXPECTED_PREPROCESSING_POLICY_SHA256 = (
    "67ccb2442d0a44393ca7e3cc0bc8030f7cc3fc69458c1dce9f9b36c57577dcc9"
)
EXPECTED_CELL3_RECEIPT_SHA256 = (
    "aded107bea4397babbf24f4b9ab5740d9cfdd117a3783b3f03bd1cbcfbc55762"
)
EXPECTED_MEAN_IMPLEMENTATION_SHA256 = (
    "abbb47187122184be8a9fcc6bdb60e0b421e990e62b18ebe2dfa6b1c1a64e024"
)
EXPECTED_CELL4_RECEIPT_SHA256 = (
    "680541be1ef4649872a888e37d8ac4e9648bd24fd90a55aeb8e9f80f606b249b"
)
EXPECTED_UNCERTAINTY_IMPLEMENTATION_SHA256 = (
    "9dc8d186817dbf669fa93fd31c2c966f1a01aa3eda3c635af4087e224026d728"
)
EXPECTED_CELL5_RECEIPT_SHA256 = (
    "00e6ba8f0a41a108e5ac2a8151f37f2f377bfab295fd01f2db8f1ab8f28ccf20"
)

_required_objects = (
    "CONTRACT_SHA256",
    "DATASET_MANIFEST_SHA256",
    "CELL2_AUDIT_SHA256",
    "PREPROCESSING_POLICY_SHA256",
    "CELL3_RECEIPT_SHA256",
    "MEAN_MODEL_IMPLEMENTATION_SHA256",
    "CELL4_RECEIPT_SHA256",
    "UNCERTAINTY_IMPLEMENTATION_SHA256",
    "CELL5_RECEIPT_SHA256",
    "DATASET",
    "QUALITY",
    "MEAN_MODEL",
    "UNCERTAINTY",
    "CASE_UNCERTAINTY_REGISTRY",
    "load_case_cache",
    "load_case_mean_model",
    "load_case_uncertainty_model",
    "predict_case_mean",
    "predict_case_uncertainty",
    "predict_case_crossfit_uncertainty",
    "atomic_save_npz",
    "file_sha256",
    "dataframe_sha256",
    "safe_slug",
    "threshold_name",
    "save_csv_atomic",
    "save_json",
    "sha256_json",
    "utc_now",
    "MODEL_DIR",
    "QUALITY_DIR",
)
_missing_objects = [name for name in _required_objects if name not in globals()]
if _missing_objects:
    raise RuntimeError(
        "Run UC-RCF-NBM Cells 1–5 before Cell 6. Missing objects: "
        + ", ".join(_missing_objects)
    )

_observed_receipts = {
    "contract": CONTRACT_SHA256,
    "dataset_manifest": DATASET_MANIFEST_SHA256,
    "cell2_audit": CELL2_AUDIT_SHA256,
    "preprocessing_policy": PREPROCESSING_POLICY_SHA256,
    "cell3_receipt": CELL3_RECEIPT_SHA256,
    "mean_implementation": MEAN_MODEL_IMPLEMENTATION_SHA256,
    "cell4_receipt": CELL4_RECEIPT_SHA256,
    "uncertainty_implementation": UNCERTAINTY_IMPLEMENTATION_SHA256,
    "cell5_receipt": CELL5_RECEIPT_SHA256,
}
_expected_receipts = {
    "contract": EXPECTED_CONTRACT_SHA256,
    "dataset_manifest": EXPECTED_DATASET_MANIFEST_SHA256,
    "cell2_audit": EXPECTED_CELL2_AUDIT_SHA256,
    "preprocessing_policy": EXPECTED_PREPROCESSING_POLICY_SHA256,
    "cell3_receipt": EXPECTED_CELL3_RECEIPT_SHA256,
    "mean_implementation": EXPECTED_MEAN_IMPLEMENTATION_SHA256,
    "cell4_receipt": EXPECTED_CELL4_RECEIPT_SHA256,
    "uncertainty_implementation": EXPECTED_UNCERTAINTY_IMPLEMENTATION_SHA256,
    "cell5_receipt": EXPECTED_CELL5_RECEIPT_SHA256,
}
if _observed_receipts != _expected_receipts:
    raise RuntimeError(
        "Cell 6 is bound to the exact completed Cell 1–5 receipts. "
        f"Observed={_observed_receipts}, expected={_expected_receipts}."
    )

_forbidden_tokens = (
    "event_label",
    "is_anomaly",
    "event_start",
    "event_end",
    "event_description",
    "fault_type",
    "care_ground_truth",
)
if any(
    any(token in str(column).lower() for token in _forbidden_tokens)
    for column in CASE_UNCERTAINTY_REGISTRY.columns
):
    raise RuntimeError("Outcome information is present in the uncertainty registry.")

if UNCERTAINTY.conformal_method != "embargoed blocked cross-conformal calibration":
    raise RuntimeError("Unexpected conformal-calibration contract.")
if UNCERTAINTY.conformal_block_steps != MEAN_MODEL.embargo_steps:
    raise RuntimeError("The conformal block and temporal embargo must both be 144 steps.")
if UNCERTAINTY.minimum_conformal_blocks < 2:
    raise RuntimeError("At least two conformal blocks are required.")
if not UNCERTAINTY.finite_sample_quantile_correction:
    raise RuntimeError("The frozen experiment requires finite-sample rank correction.")
if UNCERTAINTY.exact_exchangeability_claimed:
    raise RuntimeError("Exact exchangeability cannot be claimed for CARE time series.")

CELL6_VERSION = "1.0.0"
CELL6_MODEL_ROOT = Path(MODEL_DIR) / "cell6_blocked_cross_conformal"
CELL6_QUALITY_ROOT = Path(QUALITY_DIR) / "cell6_blocked_cross_conformal"
for _directory in (CELL6_MODEL_ROOT, CELL6_QUALITY_ROOT):
    _directory.mkdir(parents=True, exist_ok=True)


# =============================================================================
# 1. Freeze the dependence-aware calibration protocol
# =============================================================================

CONFORMAL_COVERAGES = np.asarray(
    UNCERTAINTY.conformal_coverage_grid, dtype=np.float64
)
CONFORMAL_BLOCK_STEPS = int(UNCERTAINTY.conformal_block_steps)
MINIMUM_CONFORMAL_BLOCKS = int(UNCERTAINTY.minimum_conformal_blocks)
MINIMUM_TARGET_OBSERVATIONS_PER_BLOCK = int(
    math.ceil(QUALITY.minimum_sensor_availability * CONFORMAL_BLOCK_STEPS)
)
MODELABILITY_THRESHOLDS_CELL6 = np.asarray(
    MEAN_MODEL.modelability_r2_grid, dtype=np.float64
)

CONFORMAL_IMPLEMENTATION = {
    "version": CELL6_VERSION,
    "input": {
        "residual": "Cell 4 embargoed OOF residual in raw target units",
        "primary_scale": "Cell 5 cross-fitted total predictive scale",
        "ablation_scale": "Cell 5 cross-fitted aleatoric scale",
        "normal_only": True,
        "outcomes_read": False,
    },
    "nonconformity": {
        "timestamp_score": "absolute OOF residual / positive cross-fitted scale",
        "primary_variant": "total_predictive_scale",
        "ablation_variant": "aleatoric_scale",
    },
    "blocks": {
        "steps": CONFORMAL_BLOCK_STEPS,
        "duration_hours": CONFORMAL_BLOCK_STEPS * DATASET.sampling_minutes / 60.0,
        "complete_blocks_only": True,
        "never_cross": (
            "timestamp-continuity segment",
            "nonconsecutive training-normal fit rows",
            "Cell 4/5 OOF fold",
        ),
        "minimum_target_observations": MINIMUM_TARGET_OBSERVATIONS_PER_BLOCK,
        "minimum_target_observation_fraction": QUALITY.minimum_sensor_availability,
        "minimum_blocks_per_target": MINIMUM_CONFORMAL_BLOCKS,
        "incomplete_run_tail": "discarded",
        "embargo": "inherited from the 144-step Cell 4/5 temporal cross-fitting embargo",
    },
    "quantile": {
        "coverages": tuple(UNCERTAINTY.conformal_coverage_grid),
        "primary_estimator": (
            "pooled observed timestamp quantile within eligible equal-length blocks, "
            "with the corrected probability determined by eligible block count"
        ),
        "block_rank": "ceil((number_of_eligible_blocks + 1) * coverage)",
        "corrected_probability": "min(block_rank / number_of_eligible_blocks, 1)",
        "finite_sample_correction": True,
        "fold_diagnostic": (
            "block-balanced empirical coverage reported separately in each honest OOF fold"
        ),
        "simultaneous_sensitivity": (
            "standard corrected quantile of target-wise maximum timestamp score per block"
        ),
    },
    "interval": {
        "form": "mean prediction +/- conformal multiplier * final total predictive scale",
        "primary_target": "block-balanced marginal coverage of observed timestamps",
        "interval_exceedance": "max(abs residual / calibrated half-width - 1, 0)",
    },
    "modelability": {
        "r2_thresholds": tuple(MEAN_MODEL.modelability_r2_grid),
        "calibrated_mask": "Cell 5 modelability mask AND at least minimum conformal blocks",
        "case_exclusion": False,
    },
    "validity_statement": {
        "exact_exchangeability_claimed": False,
        "interpretation": (
            "block-count probability correction on cross-fitted scores; empirical "
            "dependence-aware calibration rather than an exact distribution-free "
            "time-series guarantee"
        ),
    },
}
CONFORMAL_IMPLEMENTATION_SHA256 = sha256_json(CONFORMAL_IMPLEMENTATION)


# =============================================================================
# 2. Blocks, block scores, and corrected order statistics
# =============================================================================

def coverage_name(coverage: float) -> str:
    return f"c{int(round(float(coverage) * 1000)):03d}"


def make_complete_conformal_blocks(
    fit_row_indices: np.ndarray,
    segment_id: np.ndarray,
    oof_fold_fit: np.ndarray,
) -> tuple[list[np.ndarray], np.ndarray, dict[str, int]]:
    """Build complete blocks in fit-local coordinates without crossing boundaries."""
    fit_rows = np.asarray(fit_row_indices, dtype=np.int64)
    segment_id = np.asarray(segment_id, dtype=np.int64)
    fold_fit = np.asarray(oof_fold_fit, dtype=np.int8)
    if len(fit_rows) != len(fold_fit):
        raise ValueError("Fit rows and OOF-fold assignments differ in length.")
    if len(fit_rows) == 0 or (fold_fit < 0).any():
        raise ValueError("Fit rows or OOF-fold assignments are incomplete.")

    boundary = np.ones(len(fit_rows), dtype=bool)
    boundary[1:] = (
        (np.diff(fit_rows) != 1)
        | (segment_id[fit_rows[1:]] != segment_id[fit_rows[:-1]])
        | (fold_fit[1:] != fold_fit[:-1])
    )
    run_starts = np.flatnonzero(boundary)
    run_stops = np.r_[run_starts[1:], len(fit_rows)]
    blocks: list[np.ndarray] = []
    block_folds: list[int] = []
    discarded_tail_rows = 0
    rows_in_blocks = 0

    for start, stop in zip(run_starts, run_stops):
        run_length = int(stop - start)
        complete = run_length // CONFORMAL_BLOCK_STEPS
        for block_index in range(complete):
            block_start = int(start + block_index * CONFORMAL_BLOCK_STEPS)
            block_stop = block_start + CONFORMAL_BLOCK_STEPS
            block = np.arange(block_start, block_stop, dtype=np.int32)
            original_rows = fit_rows[block]
            if not np.all(np.diff(original_rows) == 1):
                raise RuntimeError("A conformal block crosses nonconsecutive rows.")
            if len(np.unique(segment_id[original_rows])) != 1:
                raise RuntimeError("A conformal block crosses a temporal segment.")
            if len(np.unique(fold_fit[block])) != 1:
                raise RuntimeError("A conformal block crosses an OOF fold.")
            blocks.append(block)
            block_folds.append(int(fold_fit[block[0]]))
            rows_in_blocks += CONFORMAL_BLOCK_STEPS
        discarded_tail_rows += run_length - complete * CONFORMAL_BLOCK_STEPS

    diagnostics = {
        "fit_rows": int(len(fit_rows)),
        "homogeneous_runs": int(len(run_starts)),
        "complete_blocks": int(len(blocks)),
        "rows_in_complete_blocks": int(rows_in_blocks),
        "discarded_tail_rows": int(discarded_tail_rows),
    }
    return blocks, np.asarray(block_folds, dtype=np.int8), diagnostics


def normalized_residual_scores(
    residual_fit: np.ndarray,
    scale_fit: np.ndarray,
) -> np.ndarray:
    residual = np.asarray(residual_fit, dtype=np.float32)
    scale = np.asarray(scale_fit, dtype=np.float32)
    if residual.shape != scale.shape:
        raise ValueError("Residual and uncertainty-scale matrices differ in shape.")
    score = np.full(residual.shape, np.nan, dtype=np.float32)
    valid = np.isfinite(residual) & np.isfinite(scale) & (scale > 0.0)
    np.divide(
        np.abs(residual),
        scale,
        out=score,
        where=valid,
    )
    return score


def maximum_block_scores(
    timestamp_scores: np.ndarray,
    blocks: list[np.ndarray],
) -> tuple[np.ndarray, np.ndarray]:
    """Return block maxima and target observation counts for complete blocks."""
    timestamp_scores = np.asarray(timestamp_scores, dtype=np.float32)
    block_scores = np.full(
        (len(blocks), timestamp_scores.shape[1]), np.nan, dtype=np.float32
    )
    block_observation_count = np.zeros(
        (len(blocks), timestamp_scores.shape[1]), dtype=np.int16
    )
    for block_index, block in enumerate(blocks):
        values = timestamp_scores[block]
        observed = np.isfinite(values)
        counts = observed.sum(axis=0, dtype=np.int16)
        safe_values = np.where(observed, values, -np.inf)
        maxima = np.max(safe_values, axis=0)
        usable = counts >= MINIMUM_TARGET_OBSERVATIONS_PER_BLOCK
        block_scores[block_index, usable] = maxima[usable]
        block_observation_count[block_index] = counts
    return block_scores, block_observation_count


def pooled_scores_from_eligible_blocks(
    timestamp_scores: np.ndarray,
    blocks: list[np.ndarray],
    usable_block_target: np.ndarray,
) -> np.ndarray:
    """Pool timestamps while excluding target-blocks below availability policy."""
    pieces: list[np.ndarray] = []
    for block_index, block in enumerate(blocks):
        values = np.asarray(timestamp_scores[block], dtype=np.float32).copy()
        values[:, ~usable_block_target[block_index]] = np.nan
        pieces.append(values)
    return np.vstack(pieces)


def corrected_block_probability(
    number_of_blocks: int,
    coverage: float,
) -> tuple[float, int]:
    if number_of_blocks < MINIMUM_CONFORMAL_BLOCKS:
        return np.nan, 0
    rank = int(math.ceil((number_of_blocks + 1) * float(coverage)))
    rank = min(max(rank, 1), number_of_blocks)
    return rank / number_of_blocks, rank


def pooled_quantiles_with_block_correction(
    pooled_scores: np.ndarray,
    block_count: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Timestamp quantiles whose probability correction uses temporal blocks."""
    pooled_scores = np.asarray(pooled_scores, dtype=np.float32)
    block_count = np.asarray(block_count, dtype=np.int32)
    target_count = pooled_scores.shape[1]
    quantiles = np.full(
        (len(CONFORMAL_COVERAGES), target_count), np.nan, dtype=np.float64
    )
    ranks = np.zeros((len(CONFORMAL_COVERAGES), target_count), dtype=np.int32)
    probabilities = np.full_like(quantiles, np.nan, dtype=np.float64)

    valid_counts = sorted(
        int(value)
        for value in np.unique(block_count)
        if int(value) >= MINIMUM_CONFORMAL_BLOCKS
    )
    for count in valid_counts:
        target_indices = np.flatnonzero(block_count == count)
        values = pooled_scores[:, target_indices]
        for coverage_index, coverage in enumerate(CONFORMAL_COVERAGES):
            probability, rank = corrected_block_probability(count, float(coverage))
            quantiles[coverage_index, target_indices] = np.nanquantile(
                values,
                probability,
                axis=0,
                method="higher",
            )
            ranks[coverage_index, target_indices] = rank
            probabilities[coverage_index, target_indices] = probability
    return quantiles, ranks, probabilities


def block_balanced_coverage_sums(
    timestamp_scores: np.ndarray,
    blocks: list[np.ndarray],
    usable_block_target: np.ndarray,
    quantiles: np.ndarray,
    selected_blocks: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Each eligible block contributes one target-wise timestamp-coverage rate."""
    quantiles = np.asarray(quantiles, dtype=np.float64)
    coverage_sum = np.zeros_like(quantiles, dtype=np.float64)
    coverage_count = np.zeros_like(quantiles, dtype=np.int32)
    for block_index in np.flatnonzero(selected_blocks):
        values = np.asarray(timestamp_scores[blocks[block_index]], dtype=np.float64)
        observed = np.isfinite(values)
        usable_targets = np.flatnonzero(usable_block_target[block_index])
        if len(usable_targets) == 0:
            continue
        for coverage_index in range(len(CONFORMAL_COVERAGES)):
            covered = observed[:, usable_targets] & (
                values[:, usable_targets]
                <= quantiles[coverage_index, usable_targets][None, :]
            )
            observation_count = observed[:, usable_targets].sum(axis=0)
            block_coverage = covered.sum(axis=0) / observation_count
            finite = np.isfinite(block_coverage)
            selected_targets = usable_targets[finite]
            coverage_sum[coverage_index, selected_targets] += block_coverage[finite]
            coverage_count[coverage_index, selected_targets] += 1
    return coverage_sum, coverage_count


def calibrate_block_balanced_timestamp_scores(
    timestamp_scores: np.ndarray,
    blocks: list[np.ndarray],
    block_folds: np.ndarray,
) -> dict[str, np.ndarray]:
    """Primary marginal calibration plus simultaneous block-max sensitivity."""
    block_maxima, block_observation_count = maximum_block_scores(
        timestamp_scores, blocks
    )
    usable_block_target = np.isfinite(block_maxima)
    block_count = usable_block_target.sum(axis=0, dtype=np.int32)
    valid_target = block_count >= MINIMUM_CONFORMAL_BLOCKS
    pooled_scores = pooled_scores_from_eligible_blocks(
        timestamp_scores, blocks, usable_block_target
    )
    quantiles, ranks, probabilities = pooled_quantiles_with_block_correction(
        pooled_scores, block_count
    )
    del pooled_scores

    fold_values = np.arange(MEAN_MODEL.crossfit_folds, dtype=np.int8)
    fold_coverage = np.full(
        (
            len(fold_values),
            len(CONFORMAL_COVERAGES),
            timestamp_scores.shape[1],
        ),
        np.nan,
        dtype=np.float64,
    )
    for fold_index, fold in enumerate(fold_values):
        selected = np.asarray(block_folds == fold, dtype=bool)
        coverage_sum, coverage_count = block_balanced_coverage_sums(
            timestamp_scores,
            blocks,
            usable_block_target,
            quantiles,
            selected,
        )
        np.divide(
            coverage_sum,
            coverage_count,
            out=fold_coverage[fold_index],
            where=coverage_count > 0,
        )

    overall_sum, overall_count = block_balanced_coverage_sums(
        timestamp_scores,
        blocks,
        usable_block_target,
        quantiles,
        np.ones(len(blocks), dtype=bool),
    )
    overall_coverage = np.full_like(quantiles, np.nan, dtype=np.float64)
    np.divide(
        overall_sum,
        overall_count,
        out=overall_coverage,
        where=overall_count > 0,
    )
    simultaneous = calibrate_block_score_matrix(block_maxima)
    return {
        "quantiles": quantiles,
        "ranks": ranks,
        "corrected_probabilities": probabilities,
        "overall_block_balanced_coverage": overall_coverage,
        "fold_block_balanced_coverage": fold_coverage,
        "block_count": block_count,
        "valid_target": valid_target,
        "block_observation_count": block_observation_count,
        "simultaneous_quantiles": simultaneous["quantiles"],
        "simultaneous_lobo_coverage": simultaneous["lobo_coverage"],
    }


def corrected_conformal_quantile(
    values: np.ndarray,
    coverage: float,
) -> tuple[float, int]:
    finite = np.sort(np.asarray(values, dtype=np.float64)[np.isfinite(values)])
    if len(finite) < MINIMUM_CONFORMAL_BLOCKS:
        return np.nan, 0
    rank = int(math.ceil((len(finite) + 1) * float(coverage)))
    rank = min(max(rank, 1), len(finite))
    return float(finite[rank - 1]), rank


def leave_one_block_out_coverage(values: np.ndarray, coverage: float) -> float:
    """Efficient leave-one-block-out coverage for an order-statistic rule."""
    finite = np.sort(np.asarray(values, dtype=np.float64)[np.isfinite(values)])
    n = len(finite)
    if n < MINIMUM_CONFORMAL_BLOCKS or n < 2:
        return np.nan
    rank = int(math.ceil(n * float(coverage)))
    rank = min(max(rank, 1), n - 1)
    positions = np.arange(n)
    threshold = np.where(positions < rank, finite[rank], finite[rank - 1])
    return float(np.mean(finite <= threshold))


def calibrate_block_score_matrix(
    block_scores: np.ndarray,
) -> dict[str, np.ndarray]:
    block_scores = np.asarray(block_scores, dtype=np.float32)
    target_count = block_scores.shape[1]
    quantiles = np.full(
        (len(CONFORMAL_COVERAGES), target_count), np.nan, dtype=np.float64
    )
    ranks = np.zeros(
        (len(CONFORMAL_COVERAGES), target_count), dtype=np.int32
    )
    lobo_coverage = np.full_like(quantiles, np.nan, dtype=np.float64)
    block_count = np.isfinite(block_scores).sum(axis=0, dtype=np.int32)
    valid_target = block_count >= MINIMUM_CONFORMAL_BLOCKS

    for target_index in np.flatnonzero(valid_target):
        values = block_scores[:, target_index]
        for coverage_index, coverage in enumerate(CONFORMAL_COVERAGES):
            quantile, rank = corrected_conformal_quantile(values, float(coverage))
            quantiles[coverage_index, target_index] = quantile
            ranks[coverage_index, target_index] = rank
            lobo_coverage[coverage_index, target_index] = (
                leave_one_block_out_coverage(values, float(coverage))
            )

    return {
        "quantiles": quantiles,
        "ranks": ranks,
        "lobo_coverage": lobo_coverage,
        "block_count": block_count,
        "valid_target": valid_target,
    }


def validate_primary_conformal_quantiles(
    quantiles: np.ndarray,
    primary_inference_mask: np.ndarray,
    case_key: str,
) -> None:
    """Validate only channels that can enter the primary inference head."""
    mask = np.asarray(primary_inference_mask, dtype=bool)
    selected = np.asarray(quantiles, dtype=np.float64)[:, mask]
    if selected.shape[1] == 0:
        raise RuntimeError(
            f"No target enters primary conformal inference for {case_key}."
        )
    if (
        not np.isfinite(selected).all()
        or (selected <= 0.0).any()
        or (np.diff(selected, axis=0) < -1.0e-12).any()
    ):
        raise RuntimeError(
            f"Primary conformal quantiles are invalid for {case_key}."
        )


# =============================================================================
# 3. Per-case blocked cross-conformal calibration
# =============================================================================

def case_conformal_paths(farm: str, event_id: int) -> tuple[Path, Path]:
    relative = Path(safe_slug(farm)) / f"event_{int(event_id):03d}"
    return (
        CELL6_MODEL_ROOT / relative.with_suffix(".json"),
        CELL6_MODEL_ROOT / relative.with_suffix(".npz"),
    )


def case_conformal_input_signature(case_row: Any) -> str:
    return sha256_json(
        {
            "cell5_receipt_sha256": CELL5_RECEIPT_SHA256,
            "case_key": str(case_row.case_key),
            "cell5_artifact_sha256": str(case_row.artifact_sha256),
            "conformal_implementation_sha256": CONFORMAL_IMPLEMENTATION_SHA256,
        }
    )


def existing_case_conformal_summary(
    metadata_path: Path,
    artifact_path: Path,
    expected_input_signature: str,
) -> dict[str, Any] | None:
    if not metadata_path.exists() or not artifact_path.exists():
        return None
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    if metadata.get("case_input_signature") != expected_input_signature:
        raise RuntimeError(
            f"Conformal-calibration input drift for "
            f"{metadata.get('case_key', metadata_path.stem)}. "
            "Do not overwrite the existing artifact."
        )
    if metadata.get("artifact_sha256") != file_sha256(artifact_path):
        raise RuntimeError(f"Conformal artifact hash failed: {artifact_path}")
    return dict(metadata["summary"])


def fit_one_case_conformal_calibration(case_row: Any) -> dict[str, Any]:
    metadata_path, artifact_path = case_conformal_paths(
        case_row.farm, case_row.event_id
    )
    input_signature = case_conformal_input_signature(case_row)
    existing = existing_case_conformal_summary(
        metadata_path, artifact_path, input_signature
    )
    if existing is not None:
        return existing

    cache = load_case_cache(str(case_row.case_key))
    mean_model = load_case_mean_model(str(case_row.case_key))
    uncertainty_model = load_case_uncertainty_model(str(case_row.case_key))
    for source, metadata in (
        ("Cell 3", cache["metadata"]),
        ("Cell 4", mean_model["metadata"]),
        ("Cell 5", uncertainty_model["metadata"]),
    ):
        if metadata.get("outcome_fields_present") is not False:
            raise RuntimeError(f"Unsafe {source} artifact for {case_row.case_key}.")

    fit_rows = np.asarray(mean_model["fit_row_indices"], dtype=np.int32)
    residual_fit = np.asarray(mean_model["oof_residual_fit"], dtype=np.float32)
    oof_fold_fit = np.asarray(mean_model["oof_fold_fit"], dtype=np.int8)
    segment_id = np.asarray(cache["segment_id"], dtype=np.int32)
    target_names = list(mean_model["metadata"]["target_names"])
    if not np.array_equal(
        fit_rows, np.flatnonzero(np.asarray(cache["fit_mask"], dtype=bool))
    ):
        raise RuntimeError(f"Cell 3/4 fit-row mismatch for {case_row.case_key}.")
    if residual_fit.shape != (len(fit_rows), len(target_names)):
        raise RuntimeError(f"OOF residual shape mismatch for {case_row.case_key}.")

    blocks, block_folds, block_diagnostics = make_complete_conformal_blocks(
        fit_rows, segment_id, oof_fold_fit
    )
    if len(blocks) < MINIMUM_CONFORMAL_BLOCKS:
        raise RuntimeError(
            f"Only {len(blocks)} complete conformal blocks in {case_row.case_key}; "
            f"at least {MINIMUM_CONFORMAL_BLOCKS} are required."
        )

    crossfit_uncertainty = predict_case_crossfit_uncertainty(
        str(case_row.case_key)
    )
    if not np.array_equal(
        fit_rows,
        np.asarray(crossfit_uncertainty["fit_row_indices"], dtype=np.int32),
    ):
        raise RuntimeError(f"Cell 5 cross-fit alignment failed for {case_row.case_key}.")

    total_timestamp_scores = normalized_residual_scores(
        residual_fit,
        crossfit_uncertainty["total_predictive_scale"],
    )
    total_calibration = calibrate_block_balanced_timestamp_scores(
        total_timestamp_scores, blocks, block_folds
    )
    del total_timestamp_scores

    aleatoric_timestamp_scores = normalized_residual_scores(
        residual_fit,
        crossfit_uncertainty["aleatoric_scale"],
    )
    aleatoric_calibration = calibrate_block_balanced_timestamp_scores(
        aleatoric_timestamp_scores, blocks, block_folds
    )
    del aleatoric_timestamp_scores, crossfit_uncertainty

    modelability_thresholds = np.asarray(
        uncertainty_model["modelability_thresholds"], dtype=np.float64
    )
    modelability_masks = np.asarray(
        uncertainty_model["modelability_masks"], dtype=bool
    )
    if not np.array_equal(
        modelability_thresholds, MODELABILITY_THRESHOLDS_CELL6
    ):
        raise RuntimeError(f"Modelability grid drift for {case_row.case_key}.")
    calibration_valid = np.asarray(
        total_calibration["valid_target"], dtype=bool
    )
    aleatoric_calibration_valid = np.asarray(
        aleatoric_calibration["valid_target"], dtype=bool
    )
    calibrated_modelability_masks = modelability_masks & calibration_valid[None, :]
    if not calibrated_modelability_masks[0].any():
        raise RuntimeError(
            f"No target is conformally calibratable at the fallback gate in "
            f"{case_row.case_key}."
        )
    # Some statistically calibratable channels are intentionally excluded by
    # the frozen modelability gate. A rejected near-deterministic channel can
    # legitimately have a zero residual quantile; it must not invalidate the
    # intervals of the targets that are actually eligible for inference.
    primary_inference_mask = calibrated_modelability_masks[0]
    validate_primary_conformal_quantiles(
        total_calibration["quantiles"],
        primary_inference_mask,
        str(case_row.case_key),
    )

    artifact_arrays = {
        "conformal_coverages": CONFORMAL_COVERAGES.astype(np.float64),
        "modelability_thresholds": modelability_thresholds.astype(np.float64),
        "modelability_masks": modelability_masks.astype(bool),
        "calibrated_modelability_masks": calibrated_modelability_masks.astype(bool),
        "calibration_valid_targets": calibration_valid.astype(bool),
        "aleatoric_calibration_valid_targets": (
            aleatoric_calibration_valid.astype(bool)
        ),
        "target_crossfit_r2": np.asarray(
            uncertainty_model["target_crossfit_r2"], dtype=np.float64
        ),
        "target_temperature": np.asarray(
            uncertainty_model["target_temperature"], dtype=bool
        ),
        "total_conformal_quantiles": np.asarray(
            total_calibration["quantiles"], dtype=np.float64
        ),
        "total_conformal_ranks": np.asarray(
            total_calibration["ranks"], dtype=np.int32
        ),
        "total_corrected_probabilities": np.asarray(
            total_calibration["corrected_probabilities"], dtype=np.float64
        ),
        "total_block_balanced_coverage": np.asarray(
            total_calibration["overall_block_balanced_coverage"], dtype=np.float64
        ),
        "total_fold_block_balanced_coverage": np.asarray(
            total_calibration["fold_block_balanced_coverage"], dtype=np.float64
        ),
        "total_calibration_block_count": np.asarray(
            total_calibration["block_count"], dtype=np.int32
        ),
        "total_simultaneous_block_quantiles": np.asarray(
            total_calibration["simultaneous_quantiles"], dtype=np.float64
        ),
        "total_simultaneous_lobo_coverage": np.asarray(
            total_calibration["simultaneous_lobo_coverage"], dtype=np.float64
        ),
        "aleatoric_conformal_quantiles": np.asarray(
            aleatoric_calibration["quantiles"], dtype=np.float64
        ),
        "aleatoric_conformal_ranks": np.asarray(
            aleatoric_calibration["ranks"], dtype=np.int32
        ),
        "aleatoric_corrected_probabilities": np.asarray(
            aleatoric_calibration["corrected_probabilities"], dtype=np.float64
        ),
        "aleatoric_block_balanced_coverage": np.asarray(
            aleatoric_calibration["overall_block_balanced_coverage"], dtype=np.float64
        ),
        "aleatoric_fold_block_balanced_coverage": np.asarray(
            aleatoric_calibration["fold_block_balanced_coverage"], dtype=np.float64
        ),
        "aleatoric_calibration_block_count": np.asarray(
            aleatoric_calibration["block_count"], dtype=np.int32
        ),
        "aleatoric_simultaneous_block_quantiles": np.asarray(
            aleatoric_calibration["simultaneous_quantiles"], dtype=np.float64
        ),
        "complete_block_target_observation_count": np.asarray(
            total_calibration["block_observation_count"], dtype=np.int16
        ),
        "complete_block_oof_fold": block_folds.astype(np.int8),
    }
    atomic_save_npz(artifact_path, **artifact_arrays)
    artifact_hash = file_sha256(artifact_path)

    fallback_mask = calibrated_modelability_masks[0]
    fallback_block_counts = np.asarray(
        total_calibration["block_count"], dtype=np.int32
    )[fallback_mask]
    summary: dict[str, Any] = {
        "case_key": str(case_row.case_key),
        "farm": str(case_row.farm),
        "asset_id": str(case_row.asset_id),
        "event_id": int(case_row.event_id),
        "training_normal_rows": int(len(fit_rows)),
        "targets": int(len(target_names)),
        "complete_blocks": int(block_diagnostics["complete_blocks"]),
        "homogeneous_runs": int(block_diagnostics["homogeneous_runs"]),
        "rows_in_complete_blocks": int(
            block_diagnostics["rows_in_complete_blocks"]
        ),
        "discarded_tail_rows": int(block_diagnostics["discarded_tail_rows"]),
        "calibration_valid_targets": int(calibration_valid.sum()),
        "minimum_fallback_target_blocks": int(np.min(fallback_block_counts)),
        "median_fallback_target_blocks": float(np.median(fallback_block_counts)),
        "artifact_relative_path": artifact_path.relative_to(
            CELL6_MODEL_ROOT
        ).as_posix(),
        "metadata_relative_path": metadata_path.relative_to(
            CELL6_MODEL_ROOT
        ).as_posix(),
        "case_input_signature": input_signature,
        "artifact_sha256": artifact_hash,
    }
    for threshold_index, threshold in enumerate(modelability_thresholds):
        mask = calibrated_modelability_masks[threshold_index]
        summary[f"targets_{threshold_name(float(threshold))}"] = int(mask.sum())
        summary[f"fraction_{threshold_name(float(threshold))}"] = float(mask.mean())
    for coverage_index, coverage in enumerate(CONFORMAL_COVERAGES):
        name = coverage_name(float(coverage))
        total_quantiles = np.asarray(
            total_calibration["quantiles"], dtype=np.float64
        )[coverage_index, fallback_mask]
        aleatoric_quantiles = np.asarray(
            aleatoric_calibration["quantiles"], dtype=np.float64
        )[coverage_index, fallback_mask]
        total_coverage = np.asarray(
            total_calibration["overall_block_balanced_coverage"], dtype=np.float64
        )[coverage_index, fallback_mask]
        total_fold_coverage = np.asarray(
            total_calibration["fold_block_balanced_coverage"], dtype=np.float64
        )[:, coverage_index, fallback_mask]
        simultaneous_quantiles = np.asarray(
            total_calibration["simultaneous_quantiles"], dtype=np.float64
        )[coverage_index, fallback_mask]
        summary[f"median_total_q_{name}"] = float(np.nanmedian(total_quantiles))
        summary[f"median_aleatoric_q_{name}"] = float(
            np.nanmedian(aleatoric_quantiles)
        )
        summary[f"median_total_block_coverage_{name}"] = float(
            np.nanmedian(total_coverage)
        )
        summary[f"median_total_block_abs_error_{name}"] = float(
            np.nanmedian(np.abs(total_coverage - float(coverage)))
        )
        summary[f"median_total_fold_coverage_range_{name}"] = float(
            np.nanmedian(
                np.nanmax(total_fold_coverage, axis=0)
                - np.nanmin(total_fold_coverage, axis=0)
            )
        )
        summary[f"median_simultaneous_total_q_{name}"] = float(
            np.nanmedian(simultaneous_quantiles)
        )

    save_json(
        {
            "cell6_version": CELL6_VERSION,
            "created_at_utc": utc_now(),
            "contract_sha256": CONTRACT_SHA256,
            "cell5_receipt_sha256": CELL5_RECEIPT_SHA256,
            "conformal_implementation_sha256": CONFORMAL_IMPLEMENTATION_SHA256,
            "case_input_signature": input_signature,
            "case_key": str(case_row.case_key),
            "farm": str(case_row.farm),
            "asset_id": str(case_row.asset_id),
            "event_id": int(case_row.event_id),
            "target_names": target_names,
            "block_diagnostics": block_diagnostics,
            "artifact_sha256": artifact_hash,
            "artifact_relative_path": summary["artifact_relative_path"],
            "outcome_fields_present": False,
            "summary": summary,
        },
        metadata_path,
    )
    return summary


# =============================================================================
# 4. Safe loading, interval prediction, and interval-exceedance scoring
# =============================================================================

def load_case_conformal_calibration(case_key: str) -> dict[str, Any]:
    match = CASE_CONFORMAL_REGISTRY.loc[
        CASE_CONFORMAL_REGISTRY["case_key"].eq(case_key)
    ]
    if len(match) != 1:
        raise KeyError(f"Unknown or duplicate case_key: {case_key}")
    row = match.iloc[0]
    metadata_path = CELL6_MODEL_ROOT / row["metadata_relative_path"]
    artifact_path = CELL6_MODEL_ROOT / row["artifact_relative_path"]
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    if metadata.get("outcome_fields_present") is not False:
        raise RuntimeError(f"Unsafe conformal metadata for {case_key}.")
    if file_sha256(artifact_path) != row["artifact_sha256"]:
        raise RuntimeError(f"Conformal artifact hash mismatch for {case_key}.")
    arrays = dict(np.load(artifact_path, allow_pickle=False))
    arrays["metadata"] = metadata
    return arrays


def exact_grid_index(values: np.ndarray, requested: float, label: str) -> int:
    values = np.asarray(values, dtype=np.float64)
    matches = np.flatnonzero(np.isclose(values, float(requested), rtol=0.0, atol=1.0e-12))
    if len(matches) != 1:
        raise ValueError(
            f"{label}={requested} is not in the frozen grid {tuple(values)}."
        )
    return int(matches[0])


def predict_case_conformal_interval(
    case_key: str,
    row_selector: slice | np.ndarray | list[int] | None = None,
    coverage: float = 0.95,
    minimum_r2: float = 0.0,
) -> dict[str, np.ndarray]:
    """Return primary final intervals without reading event outcomes."""
    calibration = load_case_conformal_calibration(case_key)
    coverage_index = exact_grid_index(
        calibration["conformal_coverages"], coverage, "coverage"
    )
    modelability_index = exact_grid_index(
        calibration["modelability_thresholds"], minimum_r2, "minimum_r2"
    )
    eligible = np.asarray(
        calibration["calibrated_modelability_masks"], dtype=bool
    )[modelability_index]
    multiplier = np.asarray(
        calibration["total_conformal_quantiles"], dtype=np.float64
    )[coverage_index]
    mean_prediction = np.asarray(
        predict_case_mean(case_key, row_selector), dtype=np.float64
    )
    uncertainty = predict_case_uncertainty(case_key, row_selector)
    total_scale = np.asarray(
        uncertainty["total_predictive_scale"], dtype=np.float64
    )
    half_width = total_scale * multiplier[None, :]
    invalid = ~eligible
    mean_prediction[:, invalid] = np.nan
    half_width[:, invalid] = np.nan
    lower = mean_prediction - half_width
    upper = mean_prediction + half_width
    return {
        "mean": mean_prediction.astype(np.float32),
        "lower": lower.astype(np.float32),
        "upper": upper.astype(np.float32),
        "half_width": half_width.astype(np.float32),
        "total_predictive_scale": total_scale.astype(np.float32),
        "conformal_multiplier": multiplier.astype(np.float64),
        "eligible_targets": eligible,
        "coverage": np.asarray(float(coverage), dtype=np.float64),
        "minimum_r2": np.asarray(float(minimum_r2), dtype=np.float64),
    }


def score_case_conformal_exceedance(
    case_key: str,
    row_selector: slice | np.ndarray | list[int] | None = None,
    coverage: float = 0.95,
    minimum_r2: float = 0.0,
) -> dict[str, np.ndarray]:
    """Score measured channels against frozen conformal intervals, without labels."""
    interval = predict_case_conformal_interval(
        case_key,
        row_selector=row_selector,
        coverage=coverage,
        minimum_r2=minimum_r2,
    )
    cache = load_case_cache(case_key)
    targets = np.asarray(cache["targets"], dtype=np.float64)
    if row_selector is not None:
        targets = targets[row_selector]
    residual = targets - np.asarray(interval["mean"], dtype=np.float64)
    half_width = np.asarray(interval["half_width"], dtype=np.float64)
    observed = (
        np.isfinite(residual)
        & np.isfinite(half_width)
        & (half_width > 0.0)
    )
    normalized_to_interval = np.full(residual.shape, np.nan, dtype=np.float64)
    np.divide(
        np.abs(residual),
        half_width,
        out=normalized_to_interval,
        where=observed,
    )
    interval_exceedance = np.full(residual.shape, np.nan, dtype=np.float64)
    interval_exceedance[observed] = np.maximum(
        normalized_to_interval[observed] - 1.0, 0.0
    )
    covered = np.zeros(residual.shape, dtype=bool)
    covered[observed] = normalized_to_interval[observed] <= 1.0
    return {
        **interval,
        "observed": observed,
        "covered": covered,
        "normalized_to_interval": normalized_to_interval.astype(np.float32),
        "interval_exceedance": interval_exceedance.astype(np.float32),
    }


# =============================================================================
# 5. Fit or verify all 95 case calibrators
# =============================================================================

_conformal_summaries: list[dict[str, Any]] = []
for _farm in DATASET.farms:
    _farm_cases = CASE_UNCERTAINTY_REGISTRY.loc[
        CASE_UNCERTAINTY_REGISTRY["farm"].eq(_farm)
    ]
    print(
        f"Calibrating blocked cross-conformal envelopes — {_farm}: "
        f"{len(_farm_cases)} cases",
        flush=True,
    )
    for _case_row in _farm_cases.itertuples(index=False):
        _summary = fit_one_case_conformal_calibration(_case_row)
        _conformal_summaries.append(_summary)
        print(
            f"  event {_case_row.event_id:>3}: "
            f"blocks={_summary['complete_blocks']:>3} "
            f"eligible={_summary['targets_r2_ge_0']:>3}/"
            f"{_summary['targets']:<3} "
            f"median q95={_summary['median_total_q_c950']:.3f}",
            flush=True,
        )

CASE_CONFORMAL_REGISTRY = pd.DataFrame(_conformal_summaries).sort_values(
    ["farm", "event_id"], kind="stable"
).reset_index(drop=True)

if len(CASE_CONFORMAL_REGISTRY) != DATASET.expected_total_cases:
    raise RuntimeError("Not all 95 cases produced a conformal-calibration artifact.")
if CASE_CONFORMAL_REGISTRY["case_key"].duplicated().any():
    raise RuntimeError("Duplicate case keys exist in the conformal registry.")
if (CASE_CONFORMAL_REGISTRY["targets_r2_ge_0"] <= 0).any():
    raise RuntimeError("At least one case has no calibrated fallback target.")
if (CASE_CONFORMAL_REGISTRY["minimum_fallback_target_blocks"] < MINIMUM_CONFORMAL_BLOCKS).any():
    raise RuntimeError("At least one fallback target has too few conformal blocks.")

# Smoke-test final intervals and exceedance scores for one case from each farm.
for _case_key in CASE_CONFORMAL_REGISTRY.groupby("farm", sort=False).head(1)["case_key"]:
    _calibration = load_case_conformal_calibration(_case_key)
    _interval = predict_case_conformal_interval(
        _case_key, slice(0, 32), coverage=0.95, minimum_r2=0.0
    )
    _score = score_case_conformal_exceedance(
        _case_key, slice(0, 32), coverage=0.95, minimum_r2=0.0
    )
    if _interval["mean"].shape != _score["interval_exceedance"].shape:
        raise RuntimeError(f"Conformal scoring smoke test failed for {_case_key}.")
    if not np.all(
        _interval["upper"][:, _interval["eligible_targets"]]
        >= _interval["lower"][:, _interval["eligible_targets"]]
    ):
        raise RuntimeError(f"Invalid conformal interval ordering for {_case_key}.")
    del _calibration, _interval, _score


# =============================================================================
# 6. Freeze Cell 6 receipt and report calibration diagnostics
# =============================================================================

_farm_aggregations: dict[str, tuple[str, str]] = {
    "cases": ("case_key", "size"),
    "median_complete_blocks": ("complete_blocks", "median"),
    "minimum_complete_blocks": ("complete_blocks", "min"),
    "median_minimum_fallback_target_blocks": (
        "minimum_fallback_target_blocks", "median"
    ),
}
for _coverage in CONFORMAL_COVERAGES:
    _name = coverage_name(float(_coverage))
    _farm_aggregations[f"median_total_q_{_name}"] = (
        f"median_total_q_{_name}", "median"
    )
    _farm_aggregations[f"median_aleatoric_q_{_name}"] = (
        f"median_aleatoric_q_{_name}", "median"
    )
    _farm_aggregations[f"median_block_coverage_{_name}"] = (
        f"median_total_block_coverage_{_name}", "median"
    )
    _farm_aggregations[f"median_block_abs_error_{_name}"] = (
        f"median_total_block_abs_error_{_name}", "median"
    )
    _farm_aggregations[f"median_fold_coverage_range_{_name}"] = (
        f"median_total_fold_coverage_range_{_name}", "median"
    )
    _farm_aggregations[f"median_simultaneous_q_{_name}"] = (
        f"median_simultaneous_total_q_{_name}", "median"
    )

CONFORMAL_FARM_SUMMARY = (
    CASE_CONFORMAL_REGISTRY.groupby("farm", sort=False)
    .agg(**_farm_aggregations)
    .reset_index()
)

_sensitivity_rows: list[dict[str, Any]] = []
for _row in CASE_CONFORMAL_REGISTRY.itertuples(index=False):
    for _threshold in MODELABILITY_THRESHOLDS_CELL6:
        _count = int(getattr(_row, f"targets_{threshold_name(float(_threshold))}"))
        _sensitivity_rows.append(
            {
                "case_key": str(_row.case_key),
                "farm": str(_row.farm),
                "event_id": int(_row.event_id),
                "r2_threshold": float(_threshold),
                "eligible_targets": _count,
                "total_targets": int(_row.targets),
                "eligible_fraction": _count / int(_row.targets),
            }
        )
CALIBRATED_MODELABILITY_CASE_GRID = pd.DataFrame(_sensitivity_rows).sort_values(
    ["farm", "event_id", "r2_threshold"], kind="stable"
).reset_index(drop=True)

CALIBRATED_MODELABILITY_FARM_SUMMARY = (
    CALIBRATED_MODELABILITY_CASE_GRID.groupby(
        ["farm", "r2_threshold"], sort=False
    )
    .agg(
        cases=("case_key", "size"),
        minimum_eligible_targets=("eligible_targets", "min"),
        median_eligible_targets=("eligible_targets", "median"),
        minimum_eligible_fraction=("eligible_fraction", "min"),
        median_eligible_fraction=("eligible_fraction", "median"),
    )
    .reset_index()
)

LOWEST_CALIBRATION_CASES = (
    CASE_CONFORMAL_REGISTRY.sort_values(
        [
            "minimum_fallback_target_blocks",
            "targets_r2_ge_0",
            "complete_blocks",
            "farm",
            "event_id",
        ],
        kind="stable",
    )
    .loc[
        :,
        [
            "case_key",
            "farm",
            "event_id",
            "complete_blocks",
            "minimum_fallback_target_blocks",
            "median_fallback_target_blocks",
            "targets_r2_ge_0",
            "targets",
            "discarded_tail_rows",
        ],
    ]
    .head(20)
    .reset_index(drop=True)
)

_component_hashes = {
    "case_conformal_registry_sha256": dataframe_sha256(
        CASE_CONFORMAL_REGISTRY, ("farm", "event_id")
    ),
    "conformal_farm_summary_sha256": dataframe_sha256(
        CONFORMAL_FARM_SUMMARY, ("farm",)
    ),
    "calibrated_modelability_case_grid_sha256": dataframe_sha256(
        CALIBRATED_MODELABILITY_CASE_GRID,
        ("farm", "event_id", "r2_threshold"),
    ),
    "calibrated_modelability_farm_summary_sha256": dataframe_sha256(
        CALIBRATED_MODELABILITY_FARM_SUMMARY, ("farm", "r2_threshold")
    ),
}
CELL6_RECEIPT_SHA256 = sha256_json(
    {
        "contract_sha256": CONTRACT_SHA256,
        "cell5_receipt_sha256": CELL5_RECEIPT_SHA256,
        "conformal_implementation_sha256": CONFORMAL_IMPLEMENTATION_SHA256,
        "component_hashes": _component_hashes,
    }
)

CELL6_RECEIPT_PATH = CELL6_QUALITY_ROOT / "cell6_blocked_conformal_receipt.json"
if CELL6_RECEIPT_PATH.exists():
    _existing_receipt = json.loads(CELL6_RECEIPT_PATH.read_text(encoding="utf-8"))
    if _existing_receipt.get("cell6_receipt_sha256") != CELL6_RECEIPT_SHA256:
        raise RuntimeError(
            "A different Cell 6 receipt exists for this experiment. Do not overwrite it."
        )
    CELL6_STATE = "existing identical Cell 6 receipt verified"
    _write_cell6_receipt = False
else:
    CELL6_STATE = "new Cell 6 receipt frozen"
    _write_cell6_receipt = True

save_csv_atomic(
    CASE_CONFORMAL_REGISTRY,
    CELL6_QUALITY_ROOT / "case_conformal_registry.csv",
)
save_csv_atomic(
    CONFORMAL_FARM_SUMMARY,
    CELL6_QUALITY_ROOT / "conformal_farm_summary.csv",
)
save_csv_atomic(
    CALIBRATED_MODELABILITY_CASE_GRID,
    CELL6_QUALITY_ROOT / "calibrated_modelability_case_grid.csv",
)
save_csv_atomic(
    CALIBRATED_MODELABILITY_FARM_SUMMARY,
    CELL6_QUALITY_ROOT / "calibrated_modelability_farm_summary.csv",
)
save_json(
    CONFORMAL_IMPLEMENTATION,
    CELL6_QUALITY_ROOT / "conformal_implementation.json",
)

if _write_cell6_receipt:
    save_json(
        {
            "cell6_version": CELL6_VERSION,
            "created_at_utc": utc_now(),
            "contract_sha256": CONTRACT_SHA256,
            "dataset_manifest_sha256": DATASET_MANIFEST_SHA256,
            "cell2_audit_sha256": CELL2_AUDIT_SHA256,
            "cell3_receipt_sha256": CELL3_RECEIPT_SHA256,
            "cell4_receipt_sha256": CELL4_RECEIPT_SHA256,
            "cell5_receipt_sha256": CELL5_RECEIPT_SHA256,
            "conformal_implementation_sha256": CONFORMAL_IMPLEMENTATION_SHA256,
            "component_hashes": _component_hashes,
            "cell6_receipt_sha256": CELL6_RECEIPT_SHA256,
            "outcomes_read": False,
            "exact_exchangeability_claimed": False,
        },
        CELL6_RECEIPT_PATH,
    )


print("\n" + "=" * 92)
print("UC-RCF-NBM CELL 6 — EMBARGOED BLOCKED CROSS-CONFORMAL CALIBRATION")
print("=" * 92)
print("\nBLOCK-CALIBRATION SUMMARY")
display(CONFORMAL_FARM_SUMMARY)
print("\nCALIBRATED TARGET-WISE MODELABILITY SENSITIVITY GRID")
display(CALIBRATED_MODELABILITY_FARM_SUMMARY)
print("\nLOWEST BLOCK-AVAILABILITY CASES")
display(LOWEST_CALIBRATION_CASES)

print("\n" + "-" * 92)
print(f"Cases calibrated                    : {len(CASE_CONFORMAL_REGISTRY)}")
print(f"Conformal block steps / hours       : {CONFORMAL_BLOCK_STEPS} / "
      f"{CONFORMAL_BLOCK_STEPS * DATASET.sampling_minutes / 60:g}")
print(f"Minimum observations per block      : {MINIMUM_TARGET_OBSERVATIONS_PER_BLOCK}")
print(f"Minimum blocks per target           : {MINIMUM_CONFORMAL_BLOCKS}")
print(f"Coverage grid                       : "
      f"{tuple(float(value) for value in CONFORMAL_COVERAGES)}")
print("Primary calibration                 : block-count-corrected timestamp quantile")
print("Primary coverage target             : block-balanced marginal timestamp coverage")
print("Simultaneous block-max envelope      : Stored sensitivity analysis")
print("Finite-sample rank correction       : Yes")
print("Exact time-series exchangeability   : No")
print(f"Conformal implementation SHA-256    : {CONFORMAL_IMPLEMENTATION_SHA256}")
print(f"Cell 6 receipt SHA-256              : {CELL6_RECEIPT_SHA256}")
print(f"Cell 6 state                        : {CELL6_STATE}")
print("Case-level exclusions               : 0")
print("Event outcomes accessed             : No")
print("Artifact/interval checks            : PASS")
print("=" * 92)
print("CELL 6 COMPLETED SUCCESSFULLY — BLOCKED CONFORMAL CALIBRATION LOCKED")


Calibrating blocked cross-conformal envelopes — Wind Farm A: 22 cases
  event   0: blocks=317 eligible= 41/43  median q95=3.383
  event   3: blocks=277 eligible= 42/43  median q95=3.408
  event  10: blocks=305 eligible= 41/43  median q95=3.354
  event  13: blocks=274 eligible= 41/43  median q95=3.370
  event  14: blocks=336 eligible= 42/43  median q95=3.265
  event  17: blocks=304 eligible= 41/43  median q95=3.364
  event  22: blocks=295 eligible= 41/43  median q95=3.401
  event  24: blocks=312 eligible= 41/43  median q95=3.384
  event  25: blocks=340 eligible= 42/43  median q95=3.261
  event  26: blocks=311 eligible= 41/43  median q95=3.391
  event  38: blocks=321 eligible= 40/43  median q95=3.293
  event  40: blocks=289 eligible= 42/43  median q95=3.320
  event  42: blocks=309 eligible= 41/43  median q95=3.376
  event  45: blocks=335 eligible= 39/43  median q95=3.354
  event  51: blocks=315 eligible= 41/43  median q95=3.422
  event  68: blocks=353 eligible= 41/43  median q95=3.329
  

,farm,cases,median_complete_blocks,minimum_complete_blocks,median_minimum_fallback_target_blocks,median_total_q_c950,median_aleatoric_q_c950,median_block_coverage_c950,median_block_abs_error_c950,median_fold_coverage_range_c950,...,median_block_coverage_c980,median_block_abs_error_c980,median_fold_coverage_range_c980,median_simultaneous_q_c980,median_total_q_c990,median_aleatoric_q_c990,median_block_coverage_c990,median_block_abs_error_c990,median_fold_coverage_range_c990,median_simultaneous_q_c990
0,Wind Farm A,22,315.0,274,315.0,3.354149,3.798159,0.955198,0.005198,0.020109,...,0.984274,0.004274,0.009294,9.851931,5.447936,6.767790,0.993762,0.003762,0.004963,11.428979
1,Wind Farm B,15,222.0,198,222.0,3.261682,3.624134,0.957111,0.007111,0.033341,...,0.986578,0.006578,0.017659,9.774072,5.617873,6.935168,0.995547,0.005547,0.007102,12.945373
2,Wind Farm C,58,164.0,116,164.0,3.634292,4.049043,0.958375,0.008375,0.037412,...,0.988345,0.008345,0.015128,9.787114,12.704988,21.680674,1.000000,0.010000,0.000000,12.704988



CALIBRATED TARGET-WISE MODELABILITY SENSITIVITY GRID


,farm,r2_threshold,cases,minimum_eligible_targets,median_eligible_targets,minimum_eligible_fraction,median_eligible_fraction
0,Wind Farm A,0.0,22,39,41.0,0.906977,0.953488
1,Wind Farm A,0.1,22,37,39.0,0.860465,0.906977
2,Wind Farm A,0.3,22,33,35.0,0.767442,0.813953
3,Wind Farm B,0.0,15,3,50.0,0.057692,0.961538
4,Wind Farm B,0.1,15,3,47.0,0.057692,0.903846
5,Wind Farm B,0.3,15,3,38.0,0.057692,0.730769
6,Wind Farm C,0.0,58,32,210.0,0.140351,0.929359
7,Wind Farm C,0.1,58,31,188.0,0.135965,0.828190
8,Wind Farm C,0.3,58,22,170.5,0.096916,0.750556



LOWEST BLOCK-AVAILABILITY CASES


,case_key,farm,event_id,complete_blocks,minimum_fallback_target_blocks,median_fallback_target_blocks,targets_r2_ge_0,targets,discarded_tail_rows
0,Wind Farm C::event_60,Wind Farm C,60,126,97,126.0,216,228,23134
1,Wind Farm C::event_48,Wind Farm C,48,116,116,116.0,216,228,22399
2,Wind Farm C::event_29,Wind Farm C,29,140,140,140.0,214,224,20656
3,Wind Farm C::event_28,Wind Farm C,28,142,142,142.0,211,225,24673
4,Wind Farm C::event_43,Wind Farm C,43,143,143,143.0,47,225,22021
5,Wind Farm C::event_1,Wind Farm C,1,143,143,143.0,179,228,24551
6,Wind Farm C::event_54,Wind Farm C,54,143,143,143.0,209,224,24168
7,Wind Farm C::event_11,Wind Farm C,11,145,145,145.0,73,228,23041
8,Wind Farm C::event_35,Wind Farm C,35,146,146,146.0,180,228,25208
9,Wind Farm C::event_61,Wind Farm C,61,146,146,146.0,214,227,26203



--------------------------------------------------------------------------------------------
Cases calibrated                    : 95
Conformal block steps / hours       : 144 / 24
Minimum observations per block      : 101
Minimum blocks per target           : 30
Coverage grid                       : (0.95, 0.98, 0.99)
Primary calibration                 : block-count-corrected timestamp quantile
Primary coverage target             : block-balanced marginal timestamp coverage
Simultaneous block-max envelope      : Stored sensitivity analysis
Finite-sample rank correction       : Yes
Exact time-series exchangeability   : No
Conformal implementation SHA-256    : 082d2cbdc738e2d48e6e95b466dee2377232b98d7e0d43d4c729a53f8fe6396a
Cell 6 receipt SHA-256              : 6cc14f1dfa64f7a2f5414bf748c2d6dc49ad454e83af650e730c9e0d9e2678f3
Cell 6 state                        : new Cell 6 receipt frozen
Case-level exclusions               : 0
Event outcomes accessed             : No
Artifact/interval

In [12]:
"""CELL 7 — cross-fitted health evidence and label-free alarm thresholds.

Paste this complete file into the seventh cell of the UC-RCF-NBM notebook and
run it only after Cells 1–6 have completed successfully.

This cell converts conformal interval exceedances into multichannel health
indicators. Sensor weights combine cross-fitted mean-model modelability with
fold-stratified conformal calibration stability. Health evidence is smoothed
causally within continuous runs, and alarm-threshold grids are estimated from
cross-fitted training-normal health indicators only. No configuration is
selected here and no event label, boundary, description, or CARE outcome is
read. Label-based selection remains reserved for nested asset-level validation.
"""

from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print


# =============================================================================
# 0. Bind Cell 7 to the exact completed experiment state
# =============================================================================

EXPECTED_CONTRACT_SHA256 = (
    "827641aecd8e807193ad193d64c274319756092faa53bdb5084f310f62041f49"
)
EXPECTED_DATASET_MANIFEST_SHA256 = (
    "62484bab1219888aa1d0788965ecd77db2b85f0bbb9b476cd3240f4143026f1f"
)
EXPECTED_CELL2_AUDIT_SHA256 = (
    "83732caf4ad3e226b69a671287bb14c4ed72e1bfc3b7c26ca144310fce8e5990"
)
EXPECTED_PREPROCESSING_POLICY_SHA256 = (
    "67ccb2442d0a44393ca7e3cc0bc8030f7cc3fc69458c1dce9f9b36c57577dcc9"
)
EXPECTED_CELL3_RECEIPT_SHA256 = (
    "aded107bea4397babbf24f4b9ab5740d9cfdd117a3783b3f03bd1cbcfbc55762"
)
EXPECTED_MEAN_IMPLEMENTATION_SHA256 = (
    "abbb47187122184be8a9fcc6bdb60e0b421e990e62b18ebe2dfa6b1c1a64e024"
)
EXPECTED_CELL4_RECEIPT_SHA256 = (
    "680541be1ef4649872a888e37d8ac4e9648bd24fd90a55aeb8e9f80f606b249b"
)
EXPECTED_UNCERTAINTY_IMPLEMENTATION_SHA256 = (
    "9dc8d186817dbf669fa93fd31c2c966f1a01aa3eda3c635af4087e224026d728"
)
EXPECTED_CELL5_RECEIPT_SHA256 = (
    "00e6ba8f0a41a108e5ac2a8151f37f2f377bfab295fd01f2db8f1ab8f28ccf20"
)
EXPECTED_CONFORMAL_IMPLEMENTATION_SHA256 = (
    "082d2cbdc738e2d48e6e95b466dee2377232b98d7e0d43d4c729a53f8fe6396a"
)
EXPECTED_CELL6_RECEIPT_SHA256 = (
    "6cc14f1dfa64f7a2f5414bf748c2d6dc49ad454e83af650e730c9e0d9e2678f3"
)

_required_objects = (
    "CONTRACT_SHA256",
    "DATASET_MANIFEST_SHA256",
    "CELL2_AUDIT_SHA256",
    "PREPROCESSING_POLICY_SHA256",
    "CELL3_RECEIPT_SHA256",
    "MEAN_MODEL_IMPLEMENTATION_SHA256",
    "CELL4_RECEIPT_SHA256",
    "UNCERTAINTY_IMPLEMENTATION_SHA256",
    "CELL5_RECEIPT_SHA256",
    "CONFORMAL_IMPLEMENTATION_SHA256",
    "CELL6_RECEIPT_SHA256",
    "DATASET",
    "QUALITY",
    "MEAN_MODEL",
    "UNCERTAINTY",
    "ALARM",
    "CASE_CONFORMAL_REGISTRY",
    "load_case_cache",
    "load_case_mean_model",
    "load_case_conformal_calibration",
    "predict_case_crossfit_uncertainty",
    "score_case_conformal_exceedance",
    "make_complete_conformal_blocks",
    "normalized_residual_scores",
    "maximum_block_scores",
    "pooled_scores_from_eligible_blocks",
    "pooled_quantiles_with_block_correction",
    "block_balanced_coverage_sums",
    "atomic_save_npz",
    "file_sha256",
    "dataframe_sha256",
    "safe_slug",
    "save_csv_atomic",
    "save_json",
    "sha256_json",
    "utc_now",
    "MODEL_DIR",
    "QUALITY_DIR",
)
_missing_objects = [name for name in _required_objects if name not in globals()]
if _missing_objects:
    raise RuntimeError(
        "Run UC-RCF-NBM Cells 1–6 before Cell 7. Missing objects: "
        + ", ".join(_missing_objects)
    )

_observed_receipts = {
    "contract": CONTRACT_SHA256,
    "dataset_manifest": DATASET_MANIFEST_SHA256,
    "cell2_audit": CELL2_AUDIT_SHA256,
    "preprocessing_policy": PREPROCESSING_POLICY_SHA256,
    "cell3_receipt": CELL3_RECEIPT_SHA256,
    "mean_implementation": MEAN_MODEL_IMPLEMENTATION_SHA256,
    "cell4_receipt": CELL4_RECEIPT_SHA256,
    "uncertainty_implementation": UNCERTAINTY_IMPLEMENTATION_SHA256,
    "cell5_receipt": CELL5_RECEIPT_SHA256,
    "conformal_implementation": CONFORMAL_IMPLEMENTATION_SHA256,
    "cell6_receipt": CELL6_RECEIPT_SHA256,
}
_expected_receipts = {
    "contract": EXPECTED_CONTRACT_SHA256,
    "dataset_manifest": EXPECTED_DATASET_MANIFEST_SHA256,
    "cell2_audit": EXPECTED_CELL2_AUDIT_SHA256,
    "preprocessing_policy": EXPECTED_PREPROCESSING_POLICY_SHA256,
    "cell3_receipt": EXPECTED_CELL3_RECEIPT_SHA256,
    "mean_implementation": EXPECTED_MEAN_IMPLEMENTATION_SHA256,
    "cell4_receipt": EXPECTED_CELL4_RECEIPT_SHA256,
    "uncertainty_implementation": EXPECTED_UNCERTAINTY_IMPLEMENTATION_SHA256,
    "cell5_receipt": EXPECTED_CELL5_RECEIPT_SHA256,
    "conformal_implementation": EXPECTED_CONFORMAL_IMPLEMENTATION_SHA256,
    "cell6_receipt": EXPECTED_CELL6_RECEIPT_SHA256,
}
if _observed_receipts != _expected_receipts:
    raise RuntimeError(
        "Cell 7 is bound to the exact completed Cell 1–6 receipts. "
        f"Observed={_observed_receipts}, expected={_expected_receipts}."
    )

_forbidden_tokens = (
    "event_label",
    "is_anomaly",
    "event_start",
    "event_end",
    "event_description",
    "fault_type",
    "care_ground_truth",
)
if any(
    any(token in str(column).lower() for token in _forbidden_tokens)
    for column in CASE_CONFORMAL_REGISTRY.columns
):
    raise RuntimeError("Outcome information is present in the conformal registry.")

if ALARM.sensor_weighting != "cross-fitted modelability and calibration quality":
    raise RuntimeError("Unexpected sensor-weighting contract.")
if ALARM.health_indicator != "weighted mean of positive conformal interval excess":
    raise RuntimeError("Unexpected health-indicator contract.")
if ALARM.smoothing != "causal rolling median within continuous segments":
    raise RuntimeError("Unexpected health-smoothing contract.")
if ALARM.post_alarm_latching or ALARM.criticality_feedback_into_predictions:
    raise RuntimeError("Cell 7 prohibits latching and criticality feedback.")

CELL7_VERSION = "1.0.1"
# Versioned roots deliberately preserve any partial v1.0.0 artifacts produced
# before a configuration-availability edge case was encountered. The corrected
# cell never mutates or silently reuses those incomplete artifacts.
CELL7_ARTIFACT_NAMESPACE = "cell7_health_evidence_v1_0_1"
CELL7_MODEL_ROOT = Path(MODEL_DIR) / CELL7_ARTIFACT_NAMESPACE
CELL7_QUALITY_ROOT = Path(QUALITY_DIR) / CELL7_ARTIFACT_NAMESPACE
for _directory in (CELL7_MODEL_ROOT, CELL7_QUALITY_ROOT):
    _directory.mkdir(parents=True, exist_ok=True)


# =============================================================================
# 1. Freeze outcome-free health-evidence implementation details
# =============================================================================

HEALTH_COVERAGES = np.asarray(
    UNCERTAINTY.conformal_coverage_grid, dtype=np.float64
)
HEALTH_MODELABILITY_THRESHOLDS = np.asarray(
    MEAN_MODEL.modelability_r2_grid, dtype=np.float64
)
EVIDENCE_HEADS = tuple(ALARM.evidence_head_grid)
SMOOTHING_STEPS = np.asarray(ALARM.smoothing_steps_grid, dtype=np.int32)
ALARM_THRESHOLD_QUANTILES = np.asarray(
    ALARM.health_threshold_quantile_grid, dtype=np.float64
)
MINIMUM_OBSERVED_WEIGHT = float(QUALITY.minimum_sensor_availability)
MINIMUM_HEALTH_BLOCK_OBSERVATIONS = int(
    np.ceil(QUALITY.minimum_sensor_availability * UNCERTAINTY.conformal_block_steps)
)
MINIMUM_HEALTH_BLOCKS = int(UNCERTAINTY.minimum_conformal_blocks)
NUMERICAL_EPSILON_CELL7 = 1.0e-12

if EVIDENCE_HEADS != ("all_modellable", "temperature_modellable"):
    raise RuntimeError("Unexpected evidence-head grid.")
if tuple(int(value) for value in SMOOTHING_STEPS) != (36, 72, 144):
    raise RuntimeError("Unexpected smoothing grid.")
if tuple(float(value) for value in ALARM_THRESHOLD_QUANTILES) != (
    0.95,
    0.98,
    0.99,
    0.995,
):
    raise RuntimeError("Unexpected alarm-threshold grid.")

HEALTH_IMPLEMENTATION = {
    "version": CELL7_VERSION,
    "sensor_reliability": {
        "modelability_component": "clip(cross-fitted target R2, 0, 1)",
        "calibration_error": (
            "target-wise RMSE of five fold-held-out block-balanced coverage "
            "estimates around nominal coverage"
        ),
        "calibration_component": "exp(-coverage_RMSE / (1 - nominal_coverage))",
        "raw_weight": "modelability_component * calibration_component",
        "normalization": "unit sum within each modelability gate and evidence head",
        "zero_sum_fallback": "uniform over eligible targets; explicitly recorded",
        "outcomes_used": False,
    },
    "health_indicator": {
        "channel_evidence": "max(abs residual / conformal half-width - 1, 0)",
        "crossfit_conformal_multiplier": (
            "each OOF temporal fold uses a conformal multiplier calibrated "
            "from complete blocks in the other four folds"
        ),
        "aggregation": "reliability-weighted mean over observed eligible targets",
        "missing_target_policy": "not imputed; weights renormalized over observed targets",
        "minimum_observed_weight": MINIMUM_OBSERVED_WEIGHT,
        "heads": tuple(ALARM.evidence_head_grid),
    },
    "smoothing": {
        "method": ALARM.smoothing,
        "windows": tuple(ALARM.smoothing_steps_grid),
        "full_window_required": True,
        "crossfit_boundaries": (
            "timestamp-continuity segment",
            "nonconsecutive training-normal rows",
            "OOF fold",
            "missing raw health evidence",
        ),
        "final_inference_boundaries": (
            "timestamp-continuity segment",
            "missing raw health evidence",
        ),
    },
    "alarm_thresholds": {
        "source": "cross-fitted training-normal smoothed health only",
        "quantiles": tuple(ALARM.health_threshold_quantile_grid),
        "quantile_method": "equal-block-weighted higher empirical quantile",
        "eligible_blocks": (
            "complete Cell 6 blocks with at least the frozen minimum number "
            "of finite smoothed-health observations"
        ),
        "minimum_block_observations": MINIMUM_HEALTH_BLOCK_OBSERVATIONS,
        "minimum_blocks": MINIMUM_HEALTH_BLOCKS,
        "timestamp_rule": ALARM.timestamp_rule,
        "post_alarm_latching": False,
    },
    "configuration_selection": {
        "performed": False,
        "reason": "reserved for nested grouped validation on development assets",
    },
    "outcomes_read": False,
}
HEALTH_IMPLEMENTATION_SHA256 = sha256_json(HEALTH_IMPLEMENTATION)


# =============================================================================
# 2. Reliability weights, health aggregation, smoothing, and thresholds
# =============================================================================

def exact_health_grid_index(values: np.ndarray, requested: float, label: str) -> int:
    values = np.asarray(values, dtype=np.float64)
    matches = np.flatnonzero(
        np.isclose(values, float(requested), rtol=0.0, atol=1.0e-12)
    )
    if len(matches) != 1:
        raise ValueError(
            f"{label}={requested} is not in the frozen grid "
            f"{tuple(float(value) for value in values)}."
        )
    return int(matches[0])


def head_target_mask(
    eligible: np.ndarray,
    target_temperature: np.ndarray,
    evidence_head: str,
) -> np.ndarray:
    eligible = np.asarray(eligible, dtype=bool)
    target_temperature = np.asarray(target_temperature, dtype=bool)
    if evidence_head == "all_modellable":
        return eligible.copy()
    if evidence_head == "temperature_modellable":
        return eligible & target_temperature
    raise ValueError(f"Unknown evidence head: {evidence_head}")


def fold_calibration_rmse(
    fold_coverage: np.ndarray,
    nominal_coverage: float,
) -> np.ndarray:
    values = np.asarray(fold_coverage, dtype=np.float64)
    valid = np.isfinite(values)
    squared = np.where(
        valid,
        np.square(values - float(nominal_coverage)),
        0.0,
    )
    count = valid.sum(axis=0)
    rmse = np.full(values.shape[1], np.nan, dtype=np.float64)
    usable = count > 0
    rmse[usable] = np.sqrt(squared[:, usable].sum(axis=0) / count[usable])
    return rmse


def reliability_weights(
    target_crossfit_r2: np.ndarray,
    fold_coverage: np.ndarray,
    eligible: np.ndarray,
    target_temperature: np.ndarray,
    evidence_head: str,
    nominal_coverage: float,
) -> dict[str, np.ndarray | float | bool | int]:
    r2 = np.asarray(target_crossfit_r2, dtype=np.float64)
    head_mask = head_target_mask(eligible, target_temperature, evidence_head)
    rmse = fold_calibration_rmse(fold_coverage, nominal_coverage)
    modelability_quality = np.clip(r2, 0.0, 1.0)
    error_budget = max(1.0 - float(nominal_coverage), NUMERICAL_EPSILON_CELL7)
    calibration_quality = np.zeros_like(rmse, dtype=np.float64)
    finite_rmse = np.isfinite(rmse)
    calibration_quality[finite_rmse] = np.exp(-rmse[finite_rmse] / error_budget)
    raw = np.where(
        head_mask,
        modelability_quality * calibration_quality,
        0.0,
    )
    raw[~np.isfinite(raw)] = 0.0
    uniform_fallback = False
    if raw.sum() <= NUMERICAL_EPSILON_CELL7 and head_mask.any():
        raw = head_mask.astype(np.float64)
        uniform_fallback = True
    weights = np.zeros_like(raw, dtype=np.float64)
    if raw.sum() > NUMERICAL_EPSILON_CELL7:
        weights = raw / raw.sum()
    positive = weights > 0.0
    effective_targets = (
        float(1.0 / np.square(weights[positive]).sum())
        if positive.any()
        else 0.0
    )
    return {
        "weights": weights,
        "eligible_mask": head_mask,
        "calibration_rmse": rmse,
        "calibration_quality": calibration_quality,
        "uniform_fallback": uniform_fallback,
        "eligible_targets": int(head_mask.sum()),
        "positive_weight_targets": int(positive.sum()),
        "effective_targets": effective_targets,
    }


def weighted_positive_health(
    interval_exceedance: np.ndarray,
    weights: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    evidence = np.asarray(interval_exceedance, dtype=np.float64)
    weights = np.asarray(weights, dtype=np.float64)
    if evidence.shape[1] != len(weights):
        raise ValueError("Health evidence and sensor weights differ in target count.")
    observed = np.isfinite(evidence) & (weights[None, :] > 0.0)
    observed_weight = (observed * weights[None, :]).sum(axis=1)
    weighted_sum = np.where(observed, evidence * weights[None, :], 0.0).sum(axis=1)
    health = np.full(len(evidence), np.nan, dtype=np.float64)
    sufficient = observed_weight >= MINIMUM_OBSERVED_WEIGHT - NUMERICAL_EPSILON_CELL7
    health[sufficient] = weighted_sum[sufficient] / observed_weight[sufficient]
    return health, observed_weight


def crossfit_continuity_id(
    fit_row_indices: np.ndarray,
    segment_id: np.ndarray,
    oof_fold_fit: np.ndarray,
) -> np.ndarray:
    fit_rows = np.asarray(fit_row_indices, dtype=np.int64)
    segment_id = np.asarray(segment_id, dtype=np.int64)
    fold = np.asarray(oof_fold_fit, dtype=np.int8)
    if len(fit_rows) != len(fold):
        raise ValueError("Cross-fit continuity inputs differ in length.")
    boundary = np.ones(len(fit_rows), dtype=bool)
    boundary[1:] = (
        (np.diff(fit_rows) != 1)
        | (segment_id[fit_rows[1:]] != segment_id[fit_rows[:-1]])
        | (fold[1:] != fold[:-1])
    )
    return np.cumsum(boundary, dtype=np.int64) - 1


def causal_rolling_median(
    values: np.ndarray,
    continuity_id: np.ndarray,
    window: int,
) -> np.ndarray:
    """Full-window causal median that restarts at gaps and missing health."""
    values = np.asarray(values, dtype=np.float64)
    continuity_id = np.asarray(continuity_id, dtype=np.int64)
    if len(values) != len(continuity_id):
        raise ValueError("Health values and continuity identifiers differ in length.")
    if int(window) <= 0:
        raise ValueError("Smoothing window must be positive.")
    output = np.full(len(values), np.nan, dtype=np.float64)
    finite = np.isfinite(values)
    boundary = np.ones(len(values), dtype=bool)
    boundary[1:] = (
        (continuity_id[1:] != continuity_id[:-1])
        | (~finite[1:])
        | (~finite[:-1])
    )
    starts = np.flatnonzero(boundary)
    stops = np.r_[starts[1:], len(values)]
    for start, stop in zip(starts, stops):
        if not finite[start] or int(stop - start) < int(window):
            continue
        rolled = (
            pd.Series(values[start:stop])
            .rolling(window=int(window), min_periods=int(window))
            .median()
            .to_numpy(dtype=np.float64)
        )
        output[start:stop] = rolled
    return output


def positive_interval_exceedance(
    residual: np.ndarray,
    scale: np.ndarray,
    multiplier: np.ndarray,
) -> np.ndarray:
    residual = np.asarray(residual, dtype=np.float32)
    scale = np.asarray(scale, dtype=np.float32)
    multiplier = np.asarray(multiplier, dtype=np.float64)
    if residual.shape != scale.shape or residual.shape[1] != len(multiplier):
        raise ValueError("Residual, scale, and multiplier shapes are inconsistent.")
    half_width = scale.astype(np.float64) * multiplier[None, :]
    evidence = np.full(residual.shape, np.nan, dtype=np.float32)
    valid = (
        np.isfinite(residual)
        & np.isfinite(half_width)
        & (half_width > 0.0)
    )
    normalized = np.zeros(residual.shape, dtype=np.float64)
    np.divide(
        np.abs(residual),
        half_width,
        out=normalized,
        where=valid,
    )
    evidence[valid] = np.maximum(normalized[valid] - 1.0, 0.0).astype(np.float32)
    return evidence


def fold_excluded_conformal_state(
    timestamp_scores: np.ndarray,
    blocks: list[np.ndarray],
    block_folds: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Calibrate each fold from the other four and score held-out coverage."""
    scores = np.asarray(timestamp_scores, dtype=np.float32)
    folds = np.asarray(block_folds, dtype=np.int8)
    fold_values = np.arange(MEAN_MODEL.crossfit_folds, dtype=np.int8)
    quantiles = np.full(
        (
            len(fold_values),
            len(HEALTH_COVERAGES),
            scores.shape[1],
        ),
        np.nan,
        dtype=np.float64,
    )
    heldout_coverage = np.full_like(quantiles, np.nan, dtype=np.float64)
    block_maxima, _ = maximum_block_scores(scores, blocks)
    block_usable = np.isfinite(block_maxima)

    for fold_index, fold in enumerate(fold_values):
        calibration_selected = folds != fold
        calibration_blocks = [
            blocks[index] for index in np.flatnonzero(calibration_selected)
        ]
        calibration_usable = block_usable[calibration_selected]
        calibration_block_count = calibration_usable.sum(axis=0, dtype=np.int32)
        pooled = pooled_scores_from_eligible_blocks(
            scores,
            calibration_blocks,
            calibration_usable,
        )
        fold_quantiles, _, _ = pooled_quantiles_with_block_correction(
            pooled,
            calibration_block_count,
        )
        del pooled
        quantiles[fold_index] = fold_quantiles

        validation_selected = folds == fold
        quantile_valid = np.isfinite(fold_quantiles).all(axis=0)
        evaluation_usable = block_usable & quantile_valid[None, :]
        coverage_sum, coverage_count = block_balanced_coverage_sums(
            scores,
            blocks,
            evaluation_usable,
            fold_quantiles,
            validation_selected,
        )
        np.divide(
            coverage_sum,
            coverage_count,
            out=heldout_coverage[fold_index],
            where=coverage_count > 0,
        )
    return quantiles, heldout_coverage


def equal_block_weighted_higher_quantile(
    block_values: list[np.ndarray],
    probability: float,
) -> float:
    """Higher empirical quantile with total mass one for every block."""
    if not block_values:
        return np.nan
    values = np.concatenate(block_values).astype(np.float64, copy=False)
    weights = np.concatenate(
        [
            np.full(len(block), 1.0 / len(block), dtype=np.float64)
            for block in block_values
        ]
    )
    order = np.argsort(values, kind="stable")
    sorted_values = values[order]
    cumulative = np.cumsum(weights[order])
    target_mass = float(probability) * float(len(block_values))
    index = int(np.searchsorted(cumulative, target_mass, side="left"))
    index = min(max(index, 0), len(sorted_values) - 1)
    return float(sorted_values[index])


def health_threshold_from_complete_blocks(
    smoothed_health: np.ndarray,
    blocks: list[np.ndarray],
    threshold_quantile: float,
) -> tuple[float, int, int]:
    pieces: list[np.ndarray] = []
    eligible_blocks = 0
    for block in blocks:
        values = np.asarray(smoothed_health[block], dtype=np.float64)
        finite = values[np.isfinite(values)]
        if len(finite) < MINIMUM_HEALTH_BLOCK_OBSERVATIONS:
            continue
        pieces.append(finite)
        eligible_blocks += 1
    if eligible_blocks < MINIMUM_HEALTH_BLOCKS or not pieces:
        return np.nan, eligible_blocks, 0
    threshold = equal_block_weighted_higher_quantile(
        pieces,
        float(threshold_quantile),
    )
    return threshold, eligible_blocks, int(sum(len(piece) for piece in pieces))


def apply_row_selector(
    values: np.ndarray,
    row_selector: slice | np.ndarray | list[int] | None,
) -> np.ndarray:
    return values if row_selector is None else values[row_selector]


def validate_primary_family_availability(
    configuration_available: np.ndarray,
    fallback_index: int,
    all_head_index: int,
    case_key: str,
) -> int:
    """Require usable fallback evidence, without requiring every smoother.

    Long causal windows can legitimately be unavailable in highly segmented
    cases because full-window smoothing and the minimum-block policy are both
    enforced. Such configurations remain NaN and are excluded downstream.
    Every conformal coverage must nevertheless retain at least one complete
    all-modelable fallback configuration.
    """
    available = np.asarray(configuration_available, dtype=bool)
    primary = available[:, int(fallback_index), int(all_head_index), :, :]
    coverage_has_candidate = primary.reshape(primary.shape[0], -1).any(axis=1)
    if not coverage_has_candidate.all():
        missing = tuple(
            float(HEALTH_COVERAGES[index])
            for index in np.flatnonzero(~coverage_has_candidate)
        )
        raise RuntimeError(
            f"No all-modelable fallback alarm configuration remains at "
            f"coverage(s) {missing} for {case_key}."
        )
    return int(primary.sum())


# =============================================================================
# 3. Per-case cross-fitted health and threshold artifacts
# =============================================================================

def case_health_paths(farm: str, event_id: int) -> tuple[Path, Path]:
    relative = Path(safe_slug(farm)) / f"event_{int(event_id):03d}"
    return (
        CELL7_MODEL_ROOT / relative.with_suffix(".json"),
        CELL7_MODEL_ROOT / relative.with_suffix(".npz"),
    )


def case_health_input_signature(case_row: Any) -> str:
    return sha256_json(
        {
            "cell6_receipt_sha256": CELL6_RECEIPT_SHA256,
            "case_key": str(case_row.case_key),
            "cell6_artifact_sha256": str(case_row.artifact_sha256),
            "health_implementation_sha256": HEALTH_IMPLEMENTATION_SHA256,
        }
    )


def existing_case_health_summary(
    metadata_path: Path,
    artifact_path: Path,
    expected_input_signature: str,
) -> dict[str, Any] | None:
    if not metadata_path.exists() or not artifact_path.exists():
        return None
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    if metadata.get("case_input_signature") != expected_input_signature:
        raise RuntimeError(
            f"Health-model input drift for "
            f"{metadata.get('case_key', metadata_path.stem)}. "
            "Do not overwrite the existing artifact."
        )
    if metadata.get("artifact_sha256") != file_sha256(artifact_path):
        raise RuntimeError(f"Health artifact hash failed: {artifact_path}")
    return dict(metadata["summary"])


def fit_one_case_health_thresholds(case_row: Any) -> dict[str, Any]:
    metadata_path, artifact_path = case_health_paths(
        case_row.farm, case_row.event_id
    )
    input_signature = case_health_input_signature(case_row)
    existing = existing_case_health_summary(
        metadata_path, artifact_path, input_signature
    )
    if existing is not None:
        return existing

    cache = load_case_cache(str(case_row.case_key))
    mean_model = load_case_mean_model(str(case_row.case_key))
    conformal = load_case_conformal_calibration(str(case_row.case_key))
    for source, metadata in (
        ("Cell 3", cache["metadata"]),
        ("Cell 4", mean_model["metadata"]),
        ("Cell 6", conformal["metadata"]),
    ):
        if metadata.get("outcome_fields_present") is not False:
            raise RuntimeError(f"Unsafe {source} artifact for {case_row.case_key}.")

    fit_rows = np.asarray(mean_model["fit_row_indices"], dtype=np.int32)
    residual_fit = np.asarray(mean_model["oof_residual_fit"], dtype=np.float32)
    oof_fold_fit = np.asarray(mean_model["oof_fold_fit"], dtype=np.int8)
    segment_id = np.asarray(cache["segment_id"], dtype=np.int32)
    if not np.array_equal(
        fit_rows, np.flatnonzero(np.asarray(cache["fit_mask"], dtype=bool))
    ):
        raise RuntimeError(f"Cell 3/4 fit-row mismatch for {case_row.case_key}.")

    coverages = np.asarray(conformal["conformal_coverages"], dtype=np.float64)
    modelability_thresholds = np.asarray(
        conformal["modelability_thresholds"], dtype=np.float64
    )
    if not np.array_equal(coverages, HEALTH_COVERAGES):
        raise RuntimeError(f"Coverage-grid drift for {case_row.case_key}.")
    if not np.array_equal(
        modelability_thresholds, HEALTH_MODELABILITY_THRESHOLDS
    ):
        raise RuntimeError(f"Modelability-grid drift for {case_row.case_key}.")

    target_r2 = np.asarray(conformal["target_crossfit_r2"], dtype=np.float64)
    target_temperature = np.asarray(conformal["target_temperature"], dtype=bool)
    calibrated_masks = np.asarray(
        conformal["calibrated_modelability_masks"], dtype=bool
    )
    crossfit_uncertainty = predict_case_crossfit_uncertainty(
        str(case_row.case_key)
    )
    if not np.array_equal(
        fit_rows,
        np.asarray(crossfit_uncertainty["fit_row_indices"], dtype=np.int32),
    ):
        raise RuntimeError(f"Cell 5 cross-fit alignment failed for {case_row.case_key}.")
    total_scale_fit = np.asarray(
        crossfit_uncertainty["total_predictive_scale"], dtype=np.float32
    )
    del crossfit_uncertainty

    blocks, block_folds, block_diagnostics = make_complete_conformal_blocks(
        fit_rows, segment_id, oof_fold_fit
    )
    continuity_id = crossfit_continuity_id(
        fit_rows, segment_id, oof_fold_fit
    )
    total_timestamp_scores = normalized_residual_scores(
        residual_fit,
        total_scale_fit,
    )
    crossfit_quantiles, crossfit_fold_coverage = fold_excluded_conformal_state(
        total_timestamp_scores,
        blocks,
        block_folds,
    )
    del total_timestamp_scores
    crossfit_calibratable = (
        np.isfinite(crossfit_quantiles).all(axis=(0, 1))
        & (crossfit_quantiles > 0.0).all(axis=(0, 1))
    )
    health_modelability_masks = (
        calibrated_masks & crossfit_calibratable[None, :]
    )
    if not health_modelability_masks[0].any():
        raise RuntimeError(
            f"No fold-excluded conformal target remains for {case_row.case_key}."
        )

    shape_weights = (
        len(coverages),
        len(modelability_thresholds),
        len(EVIDENCE_HEADS),
        len(target_r2),
    )
    sensor_weights = np.zeros(shape_weights, dtype=np.float32)
    calibration_rmse = np.full(
        (len(coverages), len(target_r2)), np.nan, dtype=np.float64
    )
    eligible_target_count = np.zeros(shape_weights[:3], dtype=np.int16)
    positive_weight_target_count = np.zeros(shape_weights[:3], dtype=np.int16)
    effective_target_count = np.zeros(shape_weights[:3], dtype=np.float32)
    uniform_weight_fallback = np.zeros(shape_weights[:3], dtype=bool)
    head_available = np.zeros(shape_weights[:3], dtype=bool)

    threshold_shape = (
        len(coverages),
        len(modelability_thresholds),
        len(EVIDENCE_HEADS),
        len(SMOOTHING_STEPS),
        len(ALARM_THRESHOLD_QUANTILES),
    )
    health_thresholds = np.full(threshold_shape, np.nan, dtype=np.float64)
    threshold_block_count = np.zeros(threshold_shape[:-1], dtype=np.int16)
    threshold_timestamp_count = np.zeros(threshold_shape[:-1], dtype=np.int32)
    raw_health_finite_count = np.zeros(shape_weights[:3], dtype=np.int32)
    raw_health_positive_fraction = np.full(shape_weights[:3], np.nan, dtype=np.float64)
    raw_health_median = np.full(shape_weights[:3], np.nan, dtype=np.float64)
    raw_health_p95 = np.full(shape_weights[:3], np.nan, dtype=np.float64)
    smoothed_health_positive_fraction = np.full(
        threshold_shape[:-1], np.nan, dtype=np.float64
    )

    for coverage_index, coverage in enumerate(coverages):
        calibration_rmse[coverage_index] = fold_calibration_rmse(
            crossfit_fold_coverage[:, coverage_index, :], float(coverage)
        )
        channel_evidence = np.full(residual_fit.shape, np.nan, dtype=np.float32)
        for fold_index in range(MEAN_MODEL.crossfit_folds):
            fold_rows = oof_fold_fit == fold_index
            channel_evidence[fold_rows] = positive_interval_exceedance(
                residual_fit[fold_rows],
                total_scale_fit[fold_rows],
                crossfit_quantiles[fold_index, coverage_index],
            )
        for modelability_index in range(len(modelability_thresholds)):
            eligible = health_modelability_masks[modelability_index]
            for head_index, evidence_head in enumerate(EVIDENCE_HEADS):
                weight_state = reliability_weights(
                    target_r2,
                    crossfit_fold_coverage[:, coverage_index, :],
                    eligible,
                    target_temperature,
                    evidence_head,
                    float(coverage),
                )
                weights = np.asarray(weight_state["weights"], dtype=np.float64)
                sensor_weights[
                    coverage_index, modelability_index, head_index
                ] = weights.astype(np.float32)
                eligible_target_count[
                    coverage_index, modelability_index, head_index
                ] = int(weight_state["eligible_targets"])
                positive_weight_target_count[
                    coverage_index, modelability_index, head_index
                ] = int(weight_state["positive_weight_targets"])
                effective_target_count[
                    coverage_index, modelability_index, head_index
                ] = float(weight_state["effective_targets"])
                uniform_weight_fallback[
                    coverage_index, modelability_index, head_index
                ] = bool(weight_state["uniform_fallback"])
                if weights.sum() <= NUMERICAL_EPSILON_CELL7:
                    continue

                raw_health, _ = weighted_positive_health(
                    channel_evidence, weights
                )
                finite_raw = raw_health[np.isfinite(raw_health)]
                if len(finite_raw) == 0:
                    continue
                raw_health_finite_count[
                    coverage_index, modelability_index, head_index
                ] = len(finite_raw)
                raw_health_positive_fraction[
                    coverage_index, modelability_index, head_index
                ] = float((finite_raw > 0.0).mean())
                raw_health_median[
                    coverage_index, modelability_index, head_index
                ] = float(np.median(finite_raw))
                raw_health_p95[
                    coverage_index, modelability_index, head_index
                ] = float(np.quantile(finite_raw, 0.95, method="higher"))

                for smoothing_index, smoothing in enumerate(SMOOTHING_STEPS):
                    smoothed = causal_rolling_median(
                        raw_health, continuity_id, int(smoothing)
                    )
                    finite_smoothed = smoothed[np.isfinite(smoothed)]
                    if len(finite_smoothed):
                        smoothed_health_positive_fraction[
                            coverage_index,
                            modelability_index,
                            head_index,
                            smoothing_index,
                        ] = float((finite_smoothed > 0.0).mean())
                    first_threshold, block_count, timestamp_count = (
                        health_threshold_from_complete_blocks(
                            smoothed,
                            blocks,
                            float(ALARM_THRESHOLD_QUANTILES[0]),
                        )
                    )
                    threshold_block_count[
                        coverage_index,
                        modelability_index,
                        head_index,
                        smoothing_index,
                    ] = block_count
                    threshold_timestamp_count[
                        coverage_index,
                        modelability_index,
                        head_index,
                        smoothing_index,
                    ] = timestamp_count
                    if not np.isfinite(first_threshold):
                        continue
                    health_thresholds[
                        coverage_index,
                        modelability_index,
                        head_index,
                        smoothing_index,
                        0,
                    ] = first_threshold
                    for threshold_index in range(1, len(ALARM_THRESHOLD_QUANTILES)):
                        threshold, repeated_blocks, repeated_rows = (
                            health_threshold_from_complete_blocks(
                                smoothed,
                                blocks,
                                float(ALARM_THRESHOLD_QUANTILES[threshold_index]),
                            )
                        )
                        if repeated_blocks != block_count or repeated_rows != timestamp_count:
                            raise RuntimeError("Threshold calibration rows changed by quantile.")
                        health_thresholds[
                            coverage_index,
                            modelability_index,
                            head_index,
                            smoothing_index,
                            threshold_index,
                        ] = threshold
                    head_available[
                        coverage_index, modelability_index, head_index
                    ] = True

    configuration_available = np.isfinite(health_thresholds)
    fallback_index = exact_health_grid_index(
        modelability_thresholds,
        float(MEAN_MODEL.fallback_minimum_r2),
        "minimum_r2",
    )
    all_head_index = EVIDENCE_HEADS.index("all_modellable")
    available_fallback_all_configurations = validate_primary_family_availability(
        configuration_available,
        fallback_index,
        all_head_index,
        str(case_row.case_key),
    )
    available_thresholds = health_thresholds[configuration_available]
    if (
        (available_thresholds < 0.0).any()
        or not np.isfinite(available_thresholds).all()
    ):
        raise RuntimeError(f"Invalid health threshold for {case_row.case_key}.")

    artifact_arrays = {
        "health_coverages": coverages.astype(np.float64),
        "modelability_thresholds": modelability_thresholds.astype(np.float64),
        "smoothing_steps": SMOOTHING_STEPS.astype(np.int32),
        "alarm_threshold_quantiles": ALARM_THRESHOLD_QUANTILES.astype(np.float64),
        "target_temperature": target_temperature.astype(bool),
        "target_crossfit_r2": target_r2.astype(np.float64),
        "health_modelability_masks": health_modelability_masks.astype(bool),
        "crossfit_conformal_quantiles": crossfit_quantiles.astype(np.float64),
        "crossfit_fold_block_balanced_coverage": (
            crossfit_fold_coverage.astype(np.float64)
        ),
        "sensor_weights": sensor_weights.astype(np.float32),
        "calibration_rmse": calibration_rmse.astype(np.float64),
        "eligible_target_count": eligible_target_count.astype(np.int16),
        "positive_weight_target_count": positive_weight_target_count.astype(np.int16),
        "effective_target_count": effective_target_count.astype(np.float32),
        "uniform_weight_fallback": uniform_weight_fallback.astype(bool),
        "head_available": head_available.astype(bool),
        "configuration_available": configuration_available.astype(bool),
        "health_thresholds": health_thresholds.astype(np.float64),
        "threshold_block_count": threshold_block_count.astype(np.int16),
        "threshold_timestamp_count": threshold_timestamp_count.astype(np.int32),
        "raw_health_finite_count": raw_health_finite_count.astype(np.int32),
        "raw_health_positive_fraction": raw_health_positive_fraction.astype(np.float64),
        "raw_health_median": raw_health_median.astype(np.float64),
        "raw_health_p95": raw_health_p95.astype(np.float64),
        "smoothed_health_positive_fraction": (
            smoothed_health_positive_fraction.astype(np.float64)
        ),
    }
    atomic_save_npz(artifact_path, **artifact_arrays)
    artifact_hash = file_sha256(artifact_path)

    coverage95_index = exact_health_grid_index(coverages, 0.95, "coverage")
    temperature_head_index = EVIDENCE_HEADS.index("temperature_modellable")
    available_block_counts = threshold_block_count[
        np.any(configuration_available, axis=-1)
    ]
    summary = {
        "case_key": str(case_row.case_key),
        "farm": str(case_row.farm),
        "asset_id": str(case_row.asset_id),
        "event_id": int(case_row.event_id),
        "training_normal_rows": int(len(fit_rows)),
        "targets": int(len(target_r2)),
        "complete_health_blocks": int(block_diagnostics["complete_blocks"]),
        "base_health_families": int(np.prod(shape_weights[:3])),
        "available_base_health_families": int(head_available.sum()),
        "alarm_configurations": int(np.prod(threshold_shape)),
        "available_alarm_configurations": int(configuration_available.sum()),
        "fallback_all_configurations": int(
            len(coverages)
            * len(SMOOTHING_STEPS)
            * len(ALARM_THRESHOLD_QUANTILES)
        ),
        "available_fallback_all_configurations": int(
            available_fallback_all_configurations
        ),
        "minimum_available_threshold_blocks": int(np.min(available_block_counts)),
        "uniform_weight_fallback_families": int(uniform_weight_fallback.sum()),
        "fallback_all_targets": int(
            eligible_target_count[coverage95_index, fallback_index, all_head_index]
        ),
        "fallback_temperature_targets": int(
            eligible_target_count[
                coverage95_index, fallback_index, temperature_head_index
            ]
        ),
        "fallback_all_effective_targets": float(
            effective_target_count[coverage95_index, fallback_index, all_head_index]
        ),
        "fallback_temperature_effective_targets": float(
            effective_target_count[
                coverage95_index, fallback_index, temperature_head_index
            ]
        ),
        "fallback_all_raw_positive_fraction": float(
            raw_health_positive_fraction[
                coverage95_index, fallback_index, all_head_index
            ]
        ),
        "fallback_temperature_raw_positive_fraction": float(
            raw_health_positive_fraction[
                coverage95_index, fallback_index, temperature_head_index
            ]
        ),
        "artifact_relative_path": artifact_path.relative_to(
            CELL7_MODEL_ROOT
        ).as_posix(),
        "metadata_relative_path": metadata_path.relative_to(
            CELL7_MODEL_ROOT
        ).as_posix(),
        "case_input_signature": input_signature,
        "artifact_sha256": artifact_hash,
    }

    save_json(
        {
            "cell7_version": CELL7_VERSION,
            "created_at_utc": utc_now(),
            "contract_sha256": CONTRACT_SHA256,
            "cell6_receipt_sha256": CELL6_RECEIPT_SHA256,
            "health_implementation_sha256": HEALTH_IMPLEMENTATION_SHA256,
            "case_input_signature": input_signature,
            "case_key": str(case_row.case_key),
            "farm": str(case_row.farm),
            "asset_id": str(case_row.asset_id),
            "event_id": int(case_row.event_id),
            "target_names": list(mean_model["metadata"]["target_names"]),
            "evidence_heads": list(EVIDENCE_HEADS),
            "artifact_sha256": artifact_hash,
            "artifact_relative_path": summary["artifact_relative_path"],
            "outcome_fields_present": False,
            "summary": summary,
        },
        metadata_path,
    )
    return summary


# =============================================================================
# 4. Safe loading and final online health scoring
# =============================================================================

def load_case_health_model(case_key: str) -> dict[str, Any]:
    match = CASE_HEALTH_REGISTRY.loc[
        CASE_HEALTH_REGISTRY["case_key"].eq(case_key)
    ]
    if len(match) != 1:
        raise KeyError(f"Unknown or duplicate case_key: {case_key}")
    row = match.iloc[0]
    metadata_path = CELL7_MODEL_ROOT / row["metadata_relative_path"]
    artifact_path = CELL7_MODEL_ROOT / row["artifact_relative_path"]
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    if metadata.get("outcome_fields_present") is not False:
        raise RuntimeError(f"Unsafe health metadata for {case_key}.")
    if file_sha256(artifact_path) != row["artifact_sha256"]:
        raise RuntimeError(f"Health artifact hash mismatch for {case_key}.")
    arrays = dict(np.load(artifact_path, allow_pickle=False))
    arrays["metadata"] = metadata
    return arrays


def score_case_health(
    case_key: str,
    row_selector: slice | np.ndarray | list[int] | None = None,
    coverage: float = 0.95,
    minimum_r2: float = 0.0,
    evidence_head: str = "all_modellable",
    smoothing_steps: int = 72,
    threshold_quantile: float = 0.99,
) -> dict[str, np.ndarray]:
    """Score one frozen health/alarm configuration without reading outcomes."""
    model = load_case_health_model(case_key)
    coverage_index = exact_health_grid_index(
        model["health_coverages"], coverage, "coverage"
    )
    modelability_index = exact_health_grid_index(
        model["modelability_thresholds"], minimum_r2, "minimum_r2"
    )
    if evidence_head not in model["metadata"]["evidence_heads"]:
        raise ValueError(f"Unknown evidence head: {evidence_head}")
    head_index = model["metadata"]["evidence_heads"].index(evidence_head)
    smoothing_index = exact_health_grid_index(
        model["smoothing_steps"], smoothing_steps, "smoothing_steps"
    )
    threshold_index = exact_health_grid_index(
        model["alarm_threshold_quantiles"],
        threshold_quantile,
        "threshold_quantile",
    )
    threshold = float(
        model["health_thresholds"][
            coverage_index,
            modelability_index,
            head_index,
            smoothing_index,
            threshold_index,
        ]
    )
    if not np.isfinite(threshold):
        raise RuntimeError(
            f"The requested health configuration is unavailable for {case_key}."
        )
    weights = np.asarray(
        model["sensor_weights"][
            coverage_index, modelability_index, head_index
        ],
        dtype=np.float64,
    )
    cache = load_case_cache(case_key)
    evidence_state = score_case_conformal_exceedance(
        case_key,
        row_selector=None,
        coverage=coverage,
        minimum_r2=minimum_r2,
    )
    raw_health, observed_weight = weighted_positive_health(
        evidence_state["interval_exceedance"], weights
    )
    segment_id = np.asarray(cache["segment_id"], dtype=np.int64)
    smoothed_health = causal_rolling_median(
        raw_health, segment_id, int(smoothing_steps)
    )
    alarm = np.isfinite(smoothed_health) & (smoothed_health > threshold)
    timestamp_ns = np.asarray(cache["timestamp_ns"], dtype=np.int64)
    return {
        "timestamp_ns": apply_row_selector(timestamp_ns, row_selector),
        "raw_health": apply_row_selector(raw_health, row_selector).astype(np.float32),
        "observed_weight_fraction": apply_row_selector(
            observed_weight, row_selector
        ).astype(np.float32),
        "smoothed_health": apply_row_selector(
            smoothed_health, row_selector
        ).astype(np.float32),
        "alarm": apply_row_selector(alarm, row_selector).astype(bool),
        "threshold": np.asarray(threshold, dtype=np.float64),
        "sensor_weights": weights.astype(np.float32),
        "eligible_targets": weights > 0.0,
        "coverage": np.asarray(float(coverage), dtype=np.float64),
        "minimum_r2": np.asarray(float(minimum_r2), dtype=np.float64),
        "smoothing_steps": np.asarray(int(smoothing_steps), dtype=np.int32),
        "threshold_quantile": np.asarray(
            float(threshold_quantile), dtype=np.float64
        ),
    }


# =============================================================================
# 5. Fit or verify all 95 health models
# =============================================================================

_health_summaries: list[dict[str, Any]] = []
for _farm in DATASET.farms:
    _farm_cases = CASE_CONFORMAL_REGISTRY.loc[
        CASE_CONFORMAL_REGISTRY["farm"].eq(_farm)
    ]
    print(
        f"Constructing cross-fitted health evidence — {_farm}: "
        f"{len(_farm_cases)} cases",
        flush=True,
    )
    for _case_row in _farm_cases.itertuples(index=False):
        _summary = fit_one_case_health_thresholds(_case_row)
        _health_summaries.append(_summary)
        print(
            f"  event {_case_row.event_id:>3}: "
            f"configs={_summary['available_alarm_configurations']:>3}/"
            f"{_summary['alarm_configurations']:<3} "
            f"all/temperature targets="
            f"{_summary['fallback_all_targets']:>3}/"
            f"{_summary['fallback_temperature_targets']:<3} "
            f"min blocks={_summary['minimum_available_threshold_blocks']:>3}",
            flush=True,
        )

CASE_HEALTH_REGISTRY = pd.DataFrame(_health_summaries).sort_values(
    ["farm", "event_id"], kind="stable"
).reset_index(drop=True)

if len(CASE_HEALTH_REGISTRY) != DATASET.expected_total_cases:
    raise RuntimeError("Not all 95 cases produced a health-evidence artifact.")
if CASE_HEALTH_REGISTRY["case_key"].duplicated().any():
    raise RuntimeError("Duplicate case keys exist in the health registry.")
if (CASE_HEALTH_REGISTRY["available_alarm_configurations"] <= 0).any():
    raise RuntimeError("At least one case has no available alarm configuration.")
if (
    CASE_HEALTH_REGISTRY["minimum_available_threshold_blocks"]
    < MINIMUM_HEALTH_BLOCKS
).any():
    raise RuntimeError("At least one health threshold uses too few blocks.")

# Smoke-test final prediction scoring for one case per farm.
for _case_key in CASE_HEALTH_REGISTRY.groupby("farm", sort=False).head(1)["case_key"]:
    _cache = load_case_cache(_case_key)
    _prediction_rows = np.flatnonzero(
        np.asarray(_cache["prediction_mask"], dtype=bool)
    )[:64]
    _score = score_case_health(
        _case_key,
        row_selector=_prediction_rows,
        coverage=0.95,
        minimum_r2=0.0,
        evidence_head="all_modellable",
        smoothing_steps=72,
        threshold_quantile=0.99,
    )
    if len(_score["alarm"]) != len(_prediction_rows):
        raise RuntimeError(f"Health-scoring smoke test failed for {_case_key}.")
    if bool(ALARM.post_alarm_latching):
        raise RuntimeError("Unexpected post-alarm latching state.")
    del _cache, _prediction_rows, _score


# =============================================================================
# 6. Configuration grid, diagnostics, and frozen Cell 7 receipt
# =============================================================================

_configuration_records: list[dict[str, Any]] = []
_configuration_id = 0
for _coverage in HEALTH_COVERAGES:
    for _minimum_r2 in HEALTH_MODELABILITY_THRESHOLDS:
        for _head in EVIDENCE_HEADS:
            for _smoothing in SMOOTHING_STEPS:
                for _threshold_quantile in ALARM_THRESHOLD_QUANTILES:
                    _configuration_id += 1
                    _configuration_records.append(
                        {
                            "configuration_id": _configuration_id,
                            "coverage": float(_coverage),
                            "minimum_r2": float(_minimum_r2),
                            "evidence_head": _head,
                            "smoothing_steps": int(_smoothing),
                            "threshold_quantile": float(_threshold_quantile),
                        }
                    )
HEALTH_CONFIGURATION_GRID = pd.DataFrame(_configuration_records)

_threshold_records: list[dict[str, Any]] = []
for _case_row in CASE_HEALTH_REGISTRY.itertuples(index=False):
    _model = load_case_health_model(str(_case_row.case_key))
    for _configuration in HEALTH_CONFIGURATION_GRID.itertuples(index=False):
        _ci = exact_health_grid_index(
            _model["health_coverages"], _configuration.coverage, "coverage"
        )
        _ri = exact_health_grid_index(
            _model["modelability_thresholds"],
            _configuration.minimum_r2,
            "minimum_r2",
        )
        _hi = _model["metadata"]["evidence_heads"].index(
            _configuration.evidence_head
        )
        _si = exact_health_grid_index(
            _model["smoothing_steps"],
            _configuration.smoothing_steps,
            "smoothing_steps",
        )
        _qi = exact_health_grid_index(
            _model["alarm_threshold_quantiles"],
            _configuration.threshold_quantile,
            "threshold_quantile",
        )
        _threshold = float(_model["health_thresholds"][_ci, _ri, _hi, _si, _qi])
        _threshold_records.append(
            {
                "case_key": str(_case_row.case_key),
                "farm": str(_case_row.farm),
                "asset_id": str(_case_row.asset_id),
                "event_id": int(_case_row.event_id),
                "configuration_id": int(_configuration.configuration_id),
                "coverage": float(_configuration.coverage),
                "minimum_r2": float(_configuration.minimum_r2),
                "evidence_head": str(_configuration.evidence_head),
                "smoothing_steps": int(_configuration.smoothing_steps),
                "threshold_quantile": float(_configuration.threshold_quantile),
                "configuration_available": bool(np.isfinite(_threshold)),
                "health_threshold": _threshold,
                "eligible_targets": int(
                    _model["eligible_target_count"][_ci, _ri, _hi]
                ),
                "effective_targets": float(
                    _model["effective_target_count"][_ci, _ri, _hi]
                ),
                "threshold_blocks": int(
                    _model["threshold_block_count"][_ci, _ri, _hi, _si]
                ),
                "threshold_timestamps": int(
                    _model["threshold_timestamp_count"][_ci, _ri, _hi, _si]
                ),
                "smoothed_positive_fraction": float(
                    _model["smoothed_health_positive_fraction"][
                        _ci, _ri, _hi, _si
                    ]
                ),
            }
        )
HEALTH_THRESHOLD_CASE_GRID = pd.DataFrame(_threshold_records).sort_values(
    ["farm", "event_id", "configuration_id"], kind="stable"
).reset_index(drop=True)

HEALTH_FARM_SUMMARY = (
    CASE_HEALTH_REGISTRY.groupby("farm", sort=False)
    .agg(
        cases=("case_key", "size"),
        median_available_alarm_configurations=(
            "available_alarm_configurations", "median"
        ),
        minimum_available_alarm_configurations=(
            "available_alarm_configurations", "min"
        ),
        median_available_fallback_all_configurations=(
            "available_fallback_all_configurations", "median"
        ),
        minimum_available_fallback_all_configurations=(
            "available_fallback_all_configurations", "min"
        ),
        median_minimum_threshold_blocks=(
            "minimum_available_threshold_blocks", "median"
        ),
        minimum_threshold_blocks=("minimum_available_threshold_blocks", "min"),
        median_fallback_all_targets=("fallback_all_targets", "median"),
        median_fallback_temperature_targets=(
            "fallback_temperature_targets", "median"
        ),
        median_fallback_all_effective_targets=(
            "fallback_all_effective_targets", "median"
        ),
        median_fallback_temperature_effective_targets=(
            "fallback_temperature_effective_targets", "median"
        ),
        median_fallback_all_raw_positive_fraction=(
            "fallback_all_raw_positive_fraction", "median"
        ),
        median_fallback_temperature_raw_positive_fraction=(
            "fallback_temperature_raw_positive_fraction", "median"
        ),
        total_uniform_weight_fallback_families=(
            "uniform_weight_fallback_families", "sum"
        ),
    )
    .reset_index()
)

HEALTH_CONFIGURATION_SUMMARY = (
    HEALTH_THRESHOLD_CASE_GRID.groupby(
        [
            "configuration_id",
            "coverage",
            "minimum_r2",
            "evidence_head",
            "smoothing_steps",
            "threshold_quantile",
        ],
        sort=False,
    )
    .agg(
        available_cases=("configuration_available", "sum"),
        median_threshold=("health_threshold", "median"),
        median_eligible_targets=("eligible_targets", "median"),
        median_effective_targets=("effective_targets", "median"),
        minimum_threshold_blocks=("threshold_blocks", "min"),
        median_threshold_blocks=("threshold_blocks", "median"),
        median_smoothed_positive_fraction=(
            "smoothed_positive_fraction", "median"
        ),
    )
    .reset_index()
)

LOWEST_HEALTH_AVAILABILITY_CASES = (
    CASE_HEALTH_REGISTRY.sort_values(
        [
            "available_alarm_configurations",
            "minimum_available_threshold_blocks",
            "fallback_all_targets",
            "farm",
            "event_id",
        ],
        kind="stable",
    )
    .loc[
        :,
        [
            "case_key",
            "farm",
            "event_id",
            "available_alarm_configurations",
            "alarm_configurations",
            "available_fallback_all_configurations",
            "fallback_all_configurations",
            "minimum_available_threshold_blocks",
            "fallback_all_targets",
            "fallback_temperature_targets",
            "fallback_all_effective_targets",
            "fallback_temperature_effective_targets",
        ],
    ]
    .head(20)
    .reset_index(drop=True)
)

_component_hashes = {
    "case_health_registry_sha256": dataframe_sha256(
        CASE_HEALTH_REGISTRY, ("farm", "event_id")
    ),
    "health_configuration_grid_sha256": dataframe_sha256(
        HEALTH_CONFIGURATION_GRID, ("configuration_id",)
    ),
    "health_threshold_case_grid_sha256": dataframe_sha256(
        HEALTH_THRESHOLD_CASE_GRID,
        ("farm", "event_id", "configuration_id"),
    ),
    "health_farm_summary_sha256": dataframe_sha256(
        HEALTH_FARM_SUMMARY, ("farm",)
    ),
    "health_configuration_summary_sha256": dataframe_sha256(
        HEALTH_CONFIGURATION_SUMMARY, ("configuration_id",)
    ),
}
CELL7_RECEIPT_SHA256 = sha256_json(
    {
        "contract_sha256": CONTRACT_SHA256,
        "cell6_receipt_sha256": CELL6_RECEIPT_SHA256,
        "health_implementation_sha256": HEALTH_IMPLEMENTATION_SHA256,
        "component_hashes": _component_hashes,
    }
)

CELL7_RECEIPT_PATH = CELL7_QUALITY_ROOT / "cell7_health_evidence_receipt.json"
if CELL7_RECEIPT_PATH.exists():
    _existing_receipt = json.loads(CELL7_RECEIPT_PATH.read_text(encoding="utf-8"))
    if _existing_receipt.get("cell7_receipt_sha256") != CELL7_RECEIPT_SHA256:
        raise RuntimeError(
            "A different Cell 7 receipt exists for this experiment. Do not overwrite it."
        )
    CELL7_STATE = "existing identical Cell 7 receipt verified"
    _write_cell7_receipt = False
else:
    CELL7_STATE = "new Cell 7 receipt frozen"
    _write_cell7_receipt = True

save_csv_atomic(
    CASE_HEALTH_REGISTRY,
    CELL7_QUALITY_ROOT / "case_health_registry.csv",
)
save_csv_atomic(
    HEALTH_CONFIGURATION_GRID,
    CELL7_QUALITY_ROOT / "health_configuration_grid.csv",
)
save_csv_atomic(
    HEALTH_THRESHOLD_CASE_GRID,
    CELL7_QUALITY_ROOT / "health_threshold_case_grid.csv",
)
save_csv_atomic(
    HEALTH_FARM_SUMMARY,
    CELL7_QUALITY_ROOT / "health_farm_summary.csv",
)
save_csv_atomic(
    HEALTH_CONFIGURATION_SUMMARY,
    CELL7_QUALITY_ROOT / "health_configuration_summary.csv",
)
save_json(
    HEALTH_IMPLEMENTATION,
    CELL7_QUALITY_ROOT / "health_implementation.json",
)

if _write_cell7_receipt:
    save_json(
        {
            "cell7_version": CELL7_VERSION,
            "created_at_utc": utc_now(),
            "contract_sha256": CONTRACT_SHA256,
            "dataset_manifest_sha256": DATASET_MANIFEST_SHA256,
            "cell2_audit_sha256": CELL2_AUDIT_SHA256,
            "cell3_receipt_sha256": CELL3_RECEIPT_SHA256,
            "cell4_receipt_sha256": CELL4_RECEIPT_SHA256,
            "cell5_receipt_sha256": CELL5_RECEIPT_SHA256,
            "cell6_receipt_sha256": CELL6_RECEIPT_SHA256,
            "health_implementation_sha256": HEALTH_IMPLEMENTATION_SHA256,
            "component_hashes": _component_hashes,
            "cell7_receipt_sha256": CELL7_RECEIPT_SHA256,
            "outcomes_read": False,
            "configuration_selected": False,
        },
        CELL7_RECEIPT_PATH,
    )


print("\n" + "=" * 92)
print("UC-RCF-NBM CELL 7 — CROSS-FITTED HEALTH EVIDENCE AND ALARM THRESHOLDS")
print("=" * 92)
print("\nHEALTH-EVIDENCE AVAILABILITY BY FARM")
display(HEALTH_FARM_SUMMARY)
print("\nLOWEST HEALTH-CONFIGURATION AVAILABILITY CASES")
display(LOWEST_HEALTH_AVAILABILITY_CASES)
print("\nLABEL-FREE HEALTH/ALARM CONFIGURATION GRID")
display(HEALTH_CONFIGURATION_GRID)

print("\n" + "-" * 92)
print(f"Cases processed                     : {len(CASE_HEALTH_REGISTRY)}")
print(f"Configurations retained             : {len(HEALTH_CONFIGURATION_GRID)}")
print(f"Coverage grid                       : "
      f"{tuple(float(value) for value in HEALTH_COVERAGES)}")
print(f"Modelability R² grid                : "
      f"{tuple(float(value) for value in HEALTH_MODELABILITY_THRESHOLDS)}")
print(f"Evidence heads                      : {EVIDENCE_HEADS}")
print(f"Causal smoothing steps              : "
      f"{tuple(int(value) for value in SMOOTHING_STEPS)}")
print(f"Alarm-threshold quantiles           : "
      f"{tuple(float(value) for value in ALARM_THRESHOLD_QUANTILES)}")
print(f"Minimum observed sensor weight      : {MINIMUM_OBSERVED_WEIGHT:g}")
print(f"Minimum health-threshold blocks     : {MINIMUM_HEALTH_BLOCKS}")
print(f"Health implementation SHA-256       : {HEALTH_IMPLEMENTATION_SHA256}")
print(f"Cell 7 receipt SHA-256              : {CELL7_RECEIPT_SHA256}")
print(f"Cell 7 state                        : {CELL7_STATE}")
print("Configuration selected              : No — reserved for nested validation")
print("Post-alarm latching                 : No")
print("Event outcomes accessed             : No")
print("Artifact/online-score checks        : PASS")
print("=" * 92)
print("CELL 7 COMPLETED SUCCESSFULLY — HEALTH EVIDENCE AND THRESHOLDS LOCKED")


Constructing cross-fitted health evidence — Wind Farm A: 22 cases
  event   0: configs=216/216 all/temperature targets= 41/24  min blocks=207
  event   3: configs=216/216 all/temperature targets= 42/24  min blocks=177
  event  10: configs=216/216 all/temperature targets= 41/24  min blocks=200
  event  13: configs=216/216 all/temperature targets= 41/24  min blocks=178
  event  14: configs=216/216 all/temperature targets= 42/24  min blocks=219
  event  17: configs=216/216 all/temperature targets= 41/24  min blocks=198
  event  22: configs=216/216 all/temperature targets= 41/24  min blocks=193
  event  24: configs=216/216 all/temperature targets= 41/24  min blocks=202
  event  25: configs=216/216 all/temperature targets= 42/24  min blocks=221
  event  26: configs=216/216 all/temperature targets= 41/24  min blocks=201
  event  38: configs=216/216 all/temperature targets= 40/24  min blocks=209
  event  40: configs=216/216 all/temperature targets= 42/24  min blocks=186
  event  42: configs=2

,farm,cases,median_available_alarm_configurations,minimum_available_alarm_configurations,median_available_fallback_all_configurations,minimum_available_fallback_all_configurations,median_minimum_threshold_blocks,minimum_threshold_blocks,median_fallback_all_targets,median_fallback_temperature_targets,median_fallback_all_effective_targets,median_fallback_temperature_effective_targets,median_fallback_all_raw_positive_fraction,median_fallback_temperature_raw_positive_fraction,total_uniform_weight_fallback_families
0,Wind Farm A,22,216.0,216,36.0,36,206.0,177,41.0,24.0,35.250065,23.44606,0.547791,0.338490,0
1,Wind Farm B,15,216.0,108,36.0,36,92.0,77,50.0,25.0,38.636684,18.46973,0.591957,0.366720,0
2,Wind Farm C,58,216.0,72,36.0,12,41.0,31,210.0,75.0,168.339584,71.51038,0.848735,0.459812,0



LOWEST HEALTH-CONFIGURATION AVAILABILITY CASES


,case_key,farm,event_id,available_alarm_configurations,alarm_configurations,available_fallback_all_configurations,fallback_all_configurations,minimum_available_threshold_blocks,fallback_all_targets,fallback_temperature_targets,fallback_all_effective_targets,fallback_temperature_effective_targets
0,Wind Farm C::event_60,Wind Farm C,60,72,216,12,36,126,216,74,164.014572,71.631058
1,Wind Farm B::event_83,Wind Farm B,83,108,216,36,36,77,3,0,2.908002,0.000000
2,Wind Farm C::event_62,Wind Farm C,62,216,216,36,36,31,211,75,167.445740,72.187714
3,Wind Farm C::event_48,Wind Farm C,48,216,216,36,36,31,216,75,174.265930,71.430763
4,Wind Farm C::event_67,Wind Farm C,67,216,216,36,36,32,198,75,170.971634,72.166054
5,Wind Farm C::event_28,Wind Farm C,28,216,216,36,36,32,211,75,170.390671,71.589607
6,Wind Farm C::event_1,Wind Farm C,1,216,216,36,36,33,179,74,148.332428,71.299095
7,Wind Farm C::event_35,Wind Farm C,35,216,216,36,36,33,180,74,146.220062,71.173981
8,Wind Farm C::event_39,Wind Farm C,39,216,216,36,36,34,42,12,31.572079,11.360116
9,Wind Farm C::event_54,Wind Farm C,54,216,216,36,36,34,209,75,170.645554,72.286232



LABEL-FREE HEALTH/ALARM CONFIGURATION GRID


,configuration_id,coverage,minimum_r2,evidence_head,smoothing_steps,threshold_quantile
0,1,0.95,0.0,all_modellable,36,0.950
1,2,0.95,0.0,all_modellable,36,0.980
2,3,0.95,0.0,all_modellable,36,0.990
3,4,0.95,0.0,all_modellable,36,0.995
4,5,0.95,0.0,all_modellable,72,0.950
...,...,...,...,...,...,...
211,212,0.99,0.3,temperature_modellable,72,0.995
212,213,0.99,0.3,temperature_modellable,144,0.950
213,214,0.99,0.3,temperature_modellable,144,0.980
214,215,0.99,0.3,temperature_modellable,144,0.990



--------------------------------------------------------------------------------------------
Cases processed                     : 95
Configurations retained             : 216
Coverage grid                       : (0.95, 0.98, 0.99)
Modelability R² grid                : (0.0, 0.1, 0.3)
Evidence heads                      : ('all_modellable', 'temperature_modellable')
Causal smoothing steps              : (36, 72, 144)
Alarm-threshold quantiles           : (0.95, 0.98, 0.99, 0.995)
Minimum observed sensor weight      : 0.7
Minimum health-threshold blocks     : 30
Health implementation SHA-256       : dbff972179e40bdf6b5586d47fca4b3f99992df23e814dc94a07db803cde5cc0
Cell 7 receipt SHA-256              : bbecdbd66abc26fcc86bd543b63cc7b757655216082d277993b44ead0857c61a
Cell 7 state                        : new Cell 7 receipt frozen
Configuration selected              : No — reserved for nested validation
Post-alarm latching                 : No
Event outcomes accessed             : No
Arti

In [13]:
"""CELL 8 — nested asset selection and outer-prediction freezing.

Paste this complete file into the eighth UC-RCF-NBM notebook cell and run it
only after the corrected Cell 7 v1.0.1 has completed successfully.

The cell first identifies configurations available for every CARE case and
freezes their raw timestamp predictions without opening event outcomes. Only
after that prediction receipt exists are labels used for five-fold grouped
selection on the development assets of each leave-one-asset-out fold. The
selected outer predictions are copied from the pre-frozen candidate matrix;
outer outcomes are not scored here. The single-use pooled outer evaluation is
therefore reserved for Cell 9.
"""

from __future__ import annotations

import json
import hashlib
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print


# =============================================================================
# 0. Bind Cell 8 to the exact completed experiment state
# =============================================================================

EXPECTED_CONTRACT_SHA256 = (
    "827641aecd8e807193ad193d64c274319756092faa53bdb5084f310f62041f49"
)
EXPECTED_DATASET_MANIFEST_SHA256 = (
    "62484bab1219888aa1d0788965ecd77db2b85f0bbb9b476cd3240f4143026f1f"
)
EXPECTED_OUTCOME_LOCKBOX_SHA256 = (
    "69c7c75ea8157e2e12cb61c596375168a90c73bef7ee283c618dab080a447e10"
)
EXPECTED_CELL2_AUDIT_SHA256 = (
    "83732caf4ad3e226b69a671287bb14c4ed72e1bfc3b7c26ca144310fce8e5990"
)
EXPECTED_PREPROCESSING_POLICY_SHA256 = (
    "67ccb2442d0a44393ca7e3cc0bc8030f7cc3fc69458c1dce9f9b36c57577dcc9"
)
EXPECTED_CELL3_RECEIPT_SHA256 = (
    "aded107bea4397babbf24f4b9ab5740d9cfdd117a3783b3f03bd1cbcfbc55762"
)
EXPECTED_MEAN_IMPLEMENTATION_SHA256 = (
    "abbb47187122184be8a9fcc6bdb60e0b421e990e62b18ebe2dfa6b1c1a64e024"
)
EXPECTED_CELL4_RECEIPT_SHA256 = (
    "680541be1ef4649872a888e37d8ac4e9648bd24fd90a55aeb8e9f80f606b249b"
)
EXPECTED_UNCERTAINTY_IMPLEMENTATION_SHA256 = (
    "9dc8d186817dbf669fa93fd31c2c966f1a01aa3eda3c635af4087e224026d728"
)
EXPECTED_CELL5_RECEIPT_SHA256 = (
    "00e6ba8f0a41a108e5ac2a8151f37f2f377bfab295fd01f2db8f1ab8f28ccf20"
)
EXPECTED_CONFORMAL_IMPLEMENTATION_SHA256 = (
    "082d2cbdc738e2d48e6e95b466dee2377232b98d7e0d43d4c729a53f8fe6396a"
)
EXPECTED_CELL6_RECEIPT_SHA256 = (
    "6cc14f1dfa64f7a2f5414bf748c2d6dc49ad454e83af650e730c9e0d9e2678f3"
)
EXPECTED_HEALTH_IMPLEMENTATION_SHA256 = (
    "dbff972179e40bdf6b5586d47fca4b3f99992df23e814dc94a07db803cde5cc0"
)
EXPECTED_CELL7_RECEIPT_SHA256 = (
    "bbecdbd66abc26fcc86bd543b63cc7b757655216082d277993b44ead0857c61a"
)

_required_objects_cell8 = (
    "CONTRACT_SHA256",
    "DATASET_MANIFEST_SHA256",
    "OUTCOME_LOCKBOX_SHA256",
    "CELL2_AUDIT_SHA256",
    "PREPROCESSING_POLICY_SHA256",
    "CELL3_RECEIPT_SHA256",
    "MEAN_MODEL_IMPLEMENTATION_SHA256",
    "CELL4_RECEIPT_SHA256",
    "UNCERTAINTY_IMPLEMENTATION_SHA256",
    "CELL5_RECEIPT_SHA256",
    "CONFORMAL_IMPLEMENTATION_SHA256",
    "CELL6_RECEIPT_SHA256",
    "HEALTH_IMPLEMENTATION_SHA256",
    "CELL7_RECEIPT_SHA256",
    "CELL7_RECEIPT_PATH",
    "DATASET",
    "QUALITY",
    "MEAN_MODEL",
    "UNCERTAINTY",
    "ALARM",
    "EVALUATION",
    "CARE",
    "REPRODUCIBILITY",
    "CASE_REGISTRY",
    "OUTER_FOLDS",
    "INNER_ASSET_FOLDS",
    "CASE_HEALTH_REGISTRY",
    "HEALTH_CONFIGURATION_GRID",
    "HEALTH_THRESHOLD_CASE_GRID",
    "HEALTH_FARM_SUMMARY",
    "HEALTH_CONFIGURATION_SUMMARY",
    "load_case_cache",
    "load_case_mean_model",
    "predict_case_mean",
    "predict_case_uncertainty",
    "load_case_conformal_calibration",
    "load_case_health_model",
    "score_case_health",
    "positive_interval_exceedance",
    "weighted_positive_health",
    "causal_rolling_median",
    "exact_health_grid_index",
    "atomic_save_npz",
    "file_sha256",
    "dataframe_sha256",
    "safe_slug",
    "save_csv_atomic",
    "save_json",
    "sha256_json",
    "utc_now",
    "INVENTORY_DIR",
    "QUALITY_DIR",
    "PREDICTION_DIR",
)
_missing_objects_cell8 = [
    name for name in _required_objects_cell8 if name not in globals()
]
if _missing_objects_cell8:
    raise RuntimeError(
        "Run UC-RCF-NBM Cells 1–7 before Cell 8. Missing objects: "
        + ", ".join(_missing_objects_cell8)
    )

_observed_receipts_cell8 = {
    "contract": CONTRACT_SHA256,
    "dataset_manifest": DATASET_MANIFEST_SHA256,
    "outcome_lockbox": OUTCOME_LOCKBOX_SHA256,
    "cell2_audit": CELL2_AUDIT_SHA256,
    "preprocessing_policy": PREPROCESSING_POLICY_SHA256,
    "cell3_receipt": CELL3_RECEIPT_SHA256,
    "mean_implementation": MEAN_MODEL_IMPLEMENTATION_SHA256,
    "cell4_receipt": CELL4_RECEIPT_SHA256,
    "uncertainty_implementation": UNCERTAINTY_IMPLEMENTATION_SHA256,
    "cell5_receipt": CELL5_RECEIPT_SHA256,
    "conformal_implementation": CONFORMAL_IMPLEMENTATION_SHA256,
    "cell6_receipt": CELL6_RECEIPT_SHA256,
    "health_implementation": HEALTH_IMPLEMENTATION_SHA256,
    "cell7_receipt": CELL7_RECEIPT_SHA256,
}
_expected_receipts_cell8 = {
    "contract": EXPECTED_CONTRACT_SHA256,
    "dataset_manifest": EXPECTED_DATASET_MANIFEST_SHA256,
    "outcome_lockbox": EXPECTED_OUTCOME_LOCKBOX_SHA256,
    "cell2_audit": EXPECTED_CELL2_AUDIT_SHA256,
    "preprocessing_policy": EXPECTED_PREPROCESSING_POLICY_SHA256,
    "cell3_receipt": EXPECTED_CELL3_RECEIPT_SHA256,
    "mean_implementation": EXPECTED_MEAN_IMPLEMENTATION_SHA256,
    "cell4_receipt": EXPECTED_CELL4_RECEIPT_SHA256,
    "uncertainty_implementation": EXPECTED_UNCERTAINTY_IMPLEMENTATION_SHA256,
    "cell5_receipt": EXPECTED_CELL5_RECEIPT_SHA256,
    "conformal_implementation": EXPECTED_CONFORMAL_IMPLEMENTATION_SHA256,
    "cell6_receipt": EXPECTED_CELL6_RECEIPT_SHA256,
    "health_implementation": EXPECTED_HEALTH_IMPLEMENTATION_SHA256,
    "cell7_receipt": EXPECTED_CELL7_RECEIPT_SHA256,
}
if _observed_receipts_cell8 != _expected_receipts_cell8:
    raise RuntimeError(
        "Cell 8 is bound to the exact completed Cell 1–7 state. "
        f"Observed={_observed_receipts_cell8}, expected={_expected_receipts_cell8}."
    )

_cell7_receipt_document_cell8 = json.loads(
    Path(CELL7_RECEIPT_PATH).read_text(encoding="utf-8")
)
if _cell7_receipt_document_cell8.get("cell7_receipt_sha256") != CELL7_RECEIPT_SHA256:
    raise RuntimeError("The on-disk Cell 7 receipt does not match notebook state.")
_live_cell7_hashes_cell8 = {
    "case_health_registry_sha256": dataframe_sha256(
        CASE_HEALTH_REGISTRY, ("farm", "event_id")
    ),
    "health_configuration_grid_sha256": dataframe_sha256(
        HEALTH_CONFIGURATION_GRID, ("configuration_id",)
    ),
    "health_threshold_case_grid_sha256": dataframe_sha256(
        HEALTH_THRESHOLD_CASE_GRID,
        ("farm", "event_id", "configuration_id"),
    ),
    "health_farm_summary_sha256": dataframe_sha256(
        HEALTH_FARM_SUMMARY, ("farm",)
    ),
    "health_configuration_summary_sha256": dataframe_sha256(
        HEALTH_CONFIGURATION_SUMMARY, ("configuration_id",)
    ),
}
if _cell7_receipt_document_cell8.get("component_hashes") != _live_cell7_hashes_cell8:
    raise RuntimeError("Live Cell 7 tables differ from their frozen receipt.")

if EVALUATION.outer_strategy != "leave-one-asset-out":
    raise RuntimeError("Cell 8 requires leave-one-asset-out evaluation.")
if EVALUATION.inner_splits != 5:
    raise RuntimeError("Cell 8 requires five grouped inner folds.")
if EVALUATION.outer_labels_available_during_selection:
    raise RuntimeError("Outer labels cannot be available during selection.")
if not REPRODUCIBILITY.save_outer_predictions_before_label_scoring:
    raise RuntimeError("Predictions must be frozen before outcome access.")
if not CARE.raw_timestamp_predictions_only:
    raise RuntimeError("Official CARE requires raw timestamp predictions.")
if ALARM.post_alarm_latching or ALARM.criticality_feedback_into_predictions:
    raise RuntimeError("Cell 8 prohibits latching and prediction feedback.")

CELL8_VERSION = "1.0.1"
CELL8_NAMESPACE = "cell8_nested_selection_v1_0_1"
CELL8_CANDIDATE_ROOT = Path(PREDICTION_DIR) / CELL8_NAMESPACE / "candidates"
CELL8_SELECTED_ROOT = Path(PREDICTION_DIR) / CELL8_NAMESPACE / "selected_outer"
CELL8_QUALITY_ROOT = Path(QUALITY_DIR) / CELL8_NAMESPACE
for _directory_cell8 in (
    CELL8_CANDIDATE_ROOT,
    CELL8_SELECTED_ROOT,
    CELL8_QUALITY_ROOT,
):
    _directory_cell8.mkdir(parents=True, exist_ok=True)


# =============================================================================
# 1. Freeze the selection and official CARE implementation
# =============================================================================

CARE_BETA_CELL8 = float(CARE.beta)
CARE_COMPONENT_WEIGHTS_CELL8 = tuple(
    float(value) for value in CARE.component_weights
)
CARE_CRITICALITY_THRESHOLD_CELL8 = int(CARE.criticality_threshold)
EXPECTED_STEP_NS_CELL8 = int(
    pd.Timedelta(minutes=DATASET.sampling_minutes).value
)
NUMERICAL_EPSILON_CELL8 = 1.0e-12

CELL8_IMPLEMENTATION = {
    "version": CELL8_VERSION,
    "candidate_eligibility": (
        "configuration available for every one of the 95 cases; determined "
        "before outcome access"
    ),
    "candidate_prediction": {
        "unit": "raw CARE prediction timestamp",
        "generated_once": True,
        "all_candidates_frozen_before_outcomes": True,
        "timestamp_alarm": ALARM.timestamp_rule,
        "post_alarm_latching": False,
        "criticality_feedback": False,
    },
    "inner_selection": {
        "outer_strategy": EVALUATION.outer_strategy,
        "inner_strategy": EVALUATION.inner_strategy,
        "inner_splits": EVALUATION.inner_splits,
        "objective": EVALUATION.selection_objective,
        "aggregation": "pooled out-of-inner-fold development cases",
        "metric_cache": (
            "immutable case-configuration sufficient statistics computed "
            "once after prediction freeze; outer-asset mask applied before "
            "every selection objective"
        ),
        "tie_break": tuple(EVALUATION.deterministic_tie_break),
        "outer_case_metrics_enter_objective": False,
    },
    "official_care": {
        "beta": CARE_BETA_CELL8,
        "component_weights": CARE_COMPONENT_WEIGHTS_CELL8,
        "criticality_threshold": CARE_CRITICALITY_THRESHOLD_CELL8,
        "criticality_update": CARE.criticality_update_state,
        "farm_a_anomaly_status": CARE.farm_a_anomaly_prediction_status_policy,
        "farm_a_normal_status": CARE.farm_a_normal_prediction_status_policy,
        "farms_b_c_status": CARE.farms_b_c_prediction_status_policy,
        "earliness": "flat first half of event, then linear to zero",
        "boundary_rules": (
            "zero if no event alarm",
            "normal accuracy if below 0.5",
            "otherwise weighted CARE average",
        ),
    },
    "outer_handoff": {
        "selected_predictions_copied_from_frozen_candidate_artifacts": True,
        "outer_outcomes_scored": False,
        "pooled_outer_endpoint_reserved_for": "Cell 9",
    },
}
CELL8_IMPLEMENTATION_SHA256 = sha256_json(CELL8_IMPLEMENTATION)


# =============================================================================
# 2. Universally deployable candidate grid
# =============================================================================

_availability_cell8 = (
    HEALTH_THRESHOLD_CASE_GRID.groupby("configuration_id", sort=False)
    .agg(
        available_cases=("configuration_available", "sum"),
        cases=("case_key", "nunique"),
    )
    .reset_index()
)
if not _availability_cell8["cases"].eq(DATASET.expected_total_cases).all():
    raise RuntimeError("The Cell 7 availability grid does not cover all cases.")

UNIVERSAL_CONFIGURATION_GRID = (
    HEALTH_CONFIGURATION_GRID.merge(
        _availability_cell8,
        on="configuration_id",
        how="left",
        validate="one_to_one",
    )
    .loc[lambda frame: frame["available_cases"].eq(DATASET.expected_total_cases)]
    .drop(columns="cases")
    .sort_values("configuration_id", kind="stable")
    .reset_index(drop=True)
)
if UNIVERSAL_CONFIGURATION_GRID.empty:
    raise RuntimeError("No health configuration is available for every CARE case.")
if UNIVERSAL_CONFIGURATION_GRID["configuration_id"].duplicated().any():
    raise RuntimeError("Duplicate universal configuration identifiers exist.")
if not UNIVERSAL_CONFIGURATION_GRID["available_cases"].eq(
    DATASET.expected_total_cases
).all():
    raise RuntimeError("A non-universal configuration entered Cell 8.")

UNIVERSAL_CONFIGURATION_IDS = UNIVERSAL_CONFIGURATION_GRID[
    "configuration_id"
].to_numpy(dtype=np.int32)
UNIVERSAL_CONFIGURATION_SHA256 = dataframe_sha256(
    UNIVERSAL_CONFIGURATION_GRID,
    ("configuration_id",),
)


# =============================================================================
# 3. Label-free candidate-prediction helpers
# =============================================================================

def cell8_weighted_row_mean(
    values: np.ndarray,
    weights: np.ndarray,
) -> np.ndarray:
    matrix = np.asarray(values, dtype=np.float64)
    weights = np.asarray(weights, dtype=np.float64)
    if matrix.ndim != 2 or matrix.shape[1] != len(weights):
        raise ValueError("Weighted row inputs have incompatible shapes.")
    observed = np.isfinite(matrix) & (weights[None, :] > 0.0)
    observed_weight = (observed * weights[None, :]).sum(axis=1)
    numerator = np.where(observed, matrix * weights[None, :], 0.0).sum(axis=1)
    result = np.full(len(matrix), np.nan, dtype=np.float64)
    usable = observed_weight >= float(QUALITY.minimum_sensor_availability) - 1e-12
    result[usable] = numerator[usable] / observed_weight[usable]
    return result


def cell8_candidate_paths(farm: str, event_id: int) -> tuple[Path, Path]:
    relative = Path(safe_slug(farm)) / f"event_{int(event_id):03d}"
    return (
        CELL8_CANDIDATE_ROOT / relative.with_suffix(".json"),
        CELL8_CANDIDATE_ROOT / relative.with_suffix(".npz"),
    )


def cell8_candidate_input_signature(case_row: Any) -> str:
    health_row = CASE_HEALTH_REGISTRY.loc[
        CASE_HEALTH_REGISTRY["case_key"].eq(str(case_row.case_key))
    ]
    if len(health_row) != 1:
        raise KeyError(f"Unknown health case: {case_row.case_key}")
    return sha256_json(
        {
            "cell7_receipt_sha256": CELL7_RECEIPT_SHA256,
            "case_key": str(case_row.case_key),
            "cell7_artifact_sha256": str(health_row.iloc[0]["artifact_sha256"]),
            "universal_configuration_sha256": UNIVERSAL_CONFIGURATION_SHA256,
            "cell8_implementation_sha256": CELL8_IMPLEMENTATION_SHA256,
        }
    )


def cell8_existing_candidate_summary(
    metadata_path: Path,
    artifact_path: Path,
    expected_signature: str,
) -> dict[str, Any] | None:
    if not metadata_path.exists() or not artifact_path.exists():
        return None
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    if metadata.get("case_input_signature") != expected_signature:
        raise RuntimeError(
            f"Cell 8 candidate input drift for {metadata_path.stem}. "
            "Do not overwrite the existing artifact."
        )
    if metadata.get("artifact_sha256") != file_sha256(artifact_path):
        raise RuntimeError(f"Candidate artifact hash failed: {artifact_path}")
    return dict(metadata["summary"])


def cell8_fit_one_candidate_matrix(case_row: Any) -> dict[str, Any]:
    metadata_path, artifact_path = cell8_candidate_paths(
        case_row.farm, case_row.event_id
    )
    input_signature = cell8_candidate_input_signature(case_row)
    existing = cell8_existing_candidate_summary(
        metadata_path, artifact_path, input_signature
    )
    if existing is not None:
        return existing

    case_key = str(case_row.case_key)
    cache = load_case_cache(case_key)
    mean_model = load_case_mean_model(case_key)
    conformal = load_case_conformal_calibration(case_key)
    health_model = load_case_health_model(case_key)
    for source, metadata in (
        ("Cell 3", cache["metadata"]),
        ("Cell 4", mean_model["metadata"]),
        ("Cell 6", conformal["metadata"]),
        ("Cell 7", health_model["metadata"]),
    ):
        if metadata.get("outcome_fields_present") is not False:
            raise RuntimeError(f"Unsafe {source} artifact for {case_key}.")

    prediction_rows = np.flatnonzero(
        np.asarray(cache["prediction_mask"], dtype=bool)
    ).astype(np.int32)
    if len(prediction_rows) == 0:
        raise RuntimeError(f"No prediction rows for {case_key}.")
    timestamp_ns = np.asarray(cache["timestamp_ns"], dtype=np.int64)
    segment_id = np.asarray(cache["segment_id"], dtype=np.int32)
    normal_status = np.asarray(cache["normal_status"], dtype=bool)
    targets = np.asarray(cache["targets"], dtype=np.float64)

    mean_prediction = np.asarray(predict_case_mean(case_key), dtype=np.float64)
    uncertainty = predict_case_uncertainty(case_key)
    total_scale = np.asarray(
        uncertainty["total_predictive_scale"], dtype=np.float64
    )
    del uncertainty
    if mean_prediction.shape != targets.shape or total_scale.shape != targets.shape:
        raise RuntimeError(f"Prediction alignment failed for {case_key}.")
    residual = targets - mean_prediction
    target_scale = np.asarray(mean_model["target_scale"], dtype=np.float64)
    if len(target_scale) != targets.shape[1]:
        raise RuntimeError(f"Target-scale alignment failed for {case_key}.")

    configuration_count = len(UNIVERSAL_CONFIGURATION_GRID)
    output_shape = (len(prediction_rows), configuration_count)
    raw_health_matrix = np.full(output_shape, np.nan, dtype=np.float32)
    observed_weight_matrix = np.full(output_shape, np.nan, dtype=np.float32)
    smoothed_health_matrix = np.full(output_shape, np.nan, dtype=np.float32)
    alarm_matrix = np.zeros(output_shape, dtype=bool)
    normalized_width_matrix = np.full(output_shape, np.nan, dtype=np.float32)
    interval_coverage_matrix = np.full(output_shape, np.nan, dtype=np.float32)
    interval_score_matrix = np.full(output_shape, np.nan, dtype=np.float32)
    thresholds = np.full(configuration_count, np.nan, dtype=np.float64)

    base_columns = (
        UNIVERSAL_CONFIGURATION_GRID.groupby(
            ["coverage", "minimum_r2", "evidence_head", "smoothing_steps"],
            sort=False,
        )
        .size()
        .reset_index(name="configurations")
    )
    for base in base_columns.itertuples(index=False):
        coverage_index = exact_health_grid_index(
            health_model["health_coverages"], base.coverage, "coverage"
        )
        r2_index = exact_health_grid_index(
            health_model["modelability_thresholds"], base.minimum_r2, "minimum_r2"
        )
        head_index = health_model["metadata"]["evidence_heads"].index(
            str(base.evidence_head)
        )
        smoothing_index = exact_health_grid_index(
            health_model["smoothing_steps"], base.smoothing_steps, "smoothing_steps"
        )
        weights = np.asarray(
            health_model["sensor_weights"][coverage_index, r2_index, head_index],
            dtype=np.float64,
        )
        if weights.sum() <= NUMERICAL_EPSILON_CELL8:
            raise RuntimeError(f"Zero universal sensor weight for {case_key}.")
        multiplier = np.asarray(
            conformal["total_conformal_quantiles"][coverage_index],
            dtype=np.float64,
        )
        evidence = positive_interval_exceedance(
            residual.astype(np.float32),
            total_scale.astype(np.float32),
            multiplier,
        )
        raw_health, observed_weight = weighted_positive_health(evidence, weights)
        smoothed_health = causal_rolling_median(
            raw_health,
            segment_id,
            int(base.smoothing_steps),
        )

        half_width = total_scale * multiplier[None, :]
        valid_target_scale = np.isfinite(target_scale) & (target_scale > 0.0)
        standardized_half_width = np.full_like(half_width, np.nan, dtype=np.float64)
        np.divide(
            half_width,
            target_scale[None, :],
            out=standardized_half_width,
            where=valid_target_scale[None, :],
        )
        standardized_width = 2.0 * standardized_half_width
        normalized_width = cell8_weighted_row_mean(standardized_width, weights)

        normalized_residual = np.full_like(residual, np.nan, dtype=np.float64)
        np.divide(
            np.abs(residual),
            target_scale[None, :],
            out=normalized_residual,
            where=valid_target_scale[None, :],
        )
        covered_values = np.where(
            np.isfinite(normalized_residual) & np.isfinite(standardized_half_width),
            (normalized_residual <= standardized_half_width).astype(np.float64),
            np.nan,
        )
        interval_coverage = cell8_weighted_row_mean(covered_values, weights)
        alpha = 1.0 - float(base.coverage)
        winkler = standardized_width + (2.0 / alpha) * np.maximum(
            normalized_residual - standardized_half_width,
            0.0,
        )
        winkler[
            ~np.isfinite(normalized_residual) | ~np.isfinite(standardized_half_width)
        ] = np.nan
        interval_score = cell8_weighted_row_mean(winkler, weights)

        matching = UNIVERSAL_CONFIGURATION_GRID.index[
            UNIVERSAL_CONFIGURATION_GRID["coverage"].eq(float(base.coverage))
            & UNIVERSAL_CONFIGURATION_GRID["minimum_r2"].eq(float(base.minimum_r2))
            & UNIVERSAL_CONFIGURATION_GRID["evidence_head"].eq(str(base.evidence_head))
            & UNIVERSAL_CONFIGURATION_GRID["smoothing_steps"].eq(
                int(base.smoothing_steps)
            )
        ].to_numpy(dtype=np.int64)
        for column_index in matching:
            configuration = UNIVERSAL_CONFIGURATION_GRID.iloc[column_index]
            quantile_index = exact_health_grid_index(
                health_model["alarm_threshold_quantiles"],
                float(configuration["threshold_quantile"]),
                "threshold_quantile",
            )
            threshold = float(
                health_model["health_thresholds"]
                [
                    coverage_index,
                    r2_index,
                    head_index,
                    smoothing_index,
                    quantile_index,
                ]
            )
            if not np.isfinite(threshold):
                raise RuntimeError(
                    f"A universal threshold is unavailable for {case_key}."
                )
            thresholds[column_index] = threshold
            raw_health_matrix[:, column_index] = raw_health[prediction_rows]
            observed_weight_matrix[:, column_index] = observed_weight[prediction_rows]
            smoothed_health_matrix[:, column_index] = smoothed_health[prediction_rows]
            alarm_matrix[:, column_index] = (
                np.isfinite(smoothed_health[prediction_rows])
                & (smoothed_health[prediction_rows] > threshold)
            )
            normalized_width_matrix[:, column_index] = normalized_width[prediction_rows]
            interval_coverage_matrix[:, column_index] = (
                interval_coverage[prediction_rows]
            )
            interval_score_matrix[:, column_index] = interval_score[prediction_rows]

    if not np.isfinite(thresholds).all():
        raise RuntimeError(f"Incomplete universal thresholds for {case_key}.")
    if not np.array_equal(
        alarm_matrix,
        np.isfinite(smoothed_health_matrix)
        & (smoothed_health_matrix > thresholds[None, :]),
    ):
        raise RuntimeError(f"Strict timestamp rule failed for {case_key}.")

    artifact_arrays = {
        "prediction_row_indices": prediction_rows.astype(np.int32),
        "timestamp_ns": timestamp_ns[prediction_rows].astype(np.int64),
        "segment_id": segment_id[prediction_rows].astype(np.int32),
        "normal_status": normal_status[prediction_rows].astype(bool),
        "configuration_ids": UNIVERSAL_CONFIGURATION_IDS.astype(np.int32),
        "thresholds": thresholds.astype(np.float64),
        "raw_health": raw_health_matrix.astype(np.float32),
        "observed_weight_fraction": observed_weight_matrix.astype(np.float32),
        "smoothed_health": smoothed_health_matrix.astype(np.float32),
        "raw_alarm": alarm_matrix.astype(bool),
        "normalized_interval_width": normalized_width_matrix.astype(np.float32),
        "interval_covered_fraction": interval_coverage_matrix.astype(np.float32),
        "standardized_interval_score": interval_score_matrix.astype(np.float32),
    }
    atomic_save_npz(artifact_path, **artifact_arrays)
    artifact_hash = file_sha256(artifact_path)
    summary = {
        "case_key": case_key,
        "farm": str(case_row.farm),
        "asset_id": str(case_row.asset_id),
        "event_id": int(case_row.event_id),
        "prediction_rows": int(len(prediction_rows)),
        "configurations": int(configuration_count),
        "finite_smoothed_fraction": float(np.isfinite(smoothed_health_matrix).mean()),
        "raw_alarm_fraction": float(alarm_matrix.mean()),
        "artifact_relative_path": artifact_path.relative_to(
            CELL8_CANDIDATE_ROOT
        ).as_posix(),
        "metadata_relative_path": metadata_path.relative_to(
            CELL8_CANDIDATE_ROOT
        ).as_posix(),
        "case_input_signature": input_signature,
        "artifact_sha256": artifact_hash,
    }
    save_json(
        {
            "cell8_version": CELL8_VERSION,
            "created_at_utc": utc_now(),
            "contract_sha256": CONTRACT_SHA256,
            "cell7_receipt_sha256": CELL7_RECEIPT_SHA256,
            "cell8_implementation_sha256": CELL8_IMPLEMENTATION_SHA256,
            "universal_configuration_sha256": UNIVERSAL_CONFIGURATION_SHA256,
            "case_input_signature": input_signature,
            "case_key": case_key,
            "farm": str(case_row.farm),
            "asset_id": str(case_row.asset_id),
            "event_id": int(case_row.event_id),
            "artifact_relative_path": summary["artifact_relative_path"],
            "artifact_sha256": artifact_hash,
            "outcome_fields_present": False,
            "status_used_as_predictor": False,
            "summary": summary,
        },
        metadata_path,
    )
    del (
        targets,
        mean_prediction,
        total_scale,
        residual,
        raw_health_matrix,
        observed_weight_matrix,
        smoothed_health_matrix,
        alarm_matrix,
    )
    return summary


def load_cell8_candidate_predictions(case_key: str) -> dict[str, Any]:
    match = CANDIDATE_PREDICTION_REGISTRY.loc[
        CANDIDATE_PREDICTION_REGISTRY["case_key"].eq(case_key)
    ]
    if len(match) != 1:
        raise KeyError(f"Unknown candidate-prediction case: {case_key}")
    row = match.iloc[0]
    metadata_path = CELL8_CANDIDATE_ROOT / row["metadata_relative_path"]
    artifact_path = CELL8_CANDIDATE_ROOT / row["artifact_relative_path"]
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    if metadata.get("outcome_fields_present") is not False:
        raise RuntimeError(f"Unsafe candidate metadata for {case_key}.")
    if file_sha256(artifact_path) != str(row["artifact_sha256"]):
        raise RuntimeError(f"Candidate artifact hash mismatch for {case_key}.")
    arrays = dict(np.load(artifact_path, allow_pickle=False))
    arrays["metadata"] = metadata
    return arrays


# =============================================================================
# 4. Generate and freeze every candidate prediction before outcome access
# =============================================================================

_candidate_summaries_cell8: list[dict[str, Any]] = []
for _farm_cell8 in DATASET.farms:
    _farm_cases_cell8 = CASE_HEALTH_REGISTRY.loc[
        CASE_HEALTH_REGISTRY["farm"].eq(_farm_cell8)
    ]
    print(
        f"Freezing label-free candidate predictions — {_farm_cell8}: "
        f"{len(_farm_cases_cell8)} cases",
        flush=True,
    )
    for _case_row_cell8 in _farm_cases_cell8.itertuples(index=False):
        _summary_cell8 = cell8_fit_one_candidate_matrix(_case_row_cell8)
        _candidate_summaries_cell8.append(_summary_cell8)
        print(
            f"  event {_case_row_cell8.event_id:>3}: "
            f"rows={_summary_cell8['prediction_rows']:>5} "
            f"configs={_summary_cell8['configurations']:>3} "
            f"alarm rate={_summary_cell8['raw_alarm_fraction']:.4f}",
            flush=True,
        )

CANDIDATE_PREDICTION_REGISTRY = pd.DataFrame(
    _candidate_summaries_cell8
).sort_values(["farm", "event_id"], kind="stable").reset_index(drop=True)
if len(CANDIDATE_PREDICTION_REGISTRY) != DATASET.expected_total_cases:
    raise RuntimeError("Not all CARE cases produced candidate predictions.")
if CANDIDATE_PREDICTION_REGISTRY["case_key"].duplicated().any():
    raise RuntimeError("Duplicate candidate-prediction cases exist.")
if not CANDIDATE_PREDICTION_REGISTRY["configurations"].eq(
    len(UNIVERSAL_CONFIGURATION_GRID)
).all():
    raise RuntimeError("A candidate artifact has the wrong configuration count.")

# Reproduce Cell 7's public scorer for one case/configuration before freezing.
_smoke_case_cell8 = str(CANDIDATE_PREDICTION_REGISTRY.iloc[0]["case_key"])
_smoke_candidate_cell8 = load_cell8_candidate_predictions(_smoke_case_cell8)
_smoke_configuration_cell8 = UNIVERSAL_CONFIGURATION_GRID.iloc[0]
_smoke_score_cell8 = score_case_health(
    _smoke_case_cell8,
    row_selector=np.asarray(_smoke_candidate_cell8["prediction_row_indices"]),
    coverage=float(_smoke_configuration_cell8["coverage"]),
    minimum_r2=float(_smoke_configuration_cell8["minimum_r2"]),
    evidence_head=str(_smoke_configuration_cell8["evidence_head"]),
    smoothing_steps=int(_smoke_configuration_cell8["smoothing_steps"]),
    threshold_quantile=float(_smoke_configuration_cell8["threshold_quantile"]),
)
_smoke_column_cell8 = int(
    np.flatnonzero(
        np.asarray(_smoke_candidate_cell8["configuration_ids"], dtype=np.int32)
        == int(_smoke_configuration_cell8["configuration_id"])
    )[0]
)
if not np.allclose(
    np.asarray(_smoke_score_cell8["smoothed_health"], dtype=np.float32),
    np.asarray(_smoke_candidate_cell8["smoothed_health"], dtype=np.float32)[
        :, _smoke_column_cell8
    ],
    equal_nan=True,
    rtol=1e-5,
    atol=1e-6,
):
    raise RuntimeError("Cell 7/8 smoothed-health reproduction failed.")
if not np.array_equal(
    np.asarray(_smoke_score_cell8["alarm"], dtype=bool),
    np.asarray(_smoke_candidate_cell8["raw_alarm"], dtype=bool)[
        :, _smoke_column_cell8
    ],
):
    raise RuntimeError("Cell 7/8 alarm reproduction failed.")
del _smoke_candidate_cell8, _smoke_score_cell8

_candidate_component_hashes_cell8 = {
    "universal_configuration_sha256": UNIVERSAL_CONFIGURATION_SHA256,
    "candidate_prediction_registry_sha256": dataframe_sha256(
        CANDIDATE_PREDICTION_REGISTRY,
        ("farm", "event_id"),
    ),
}
CANDIDATE_PREDICTION_FREEZE_SHA256 = sha256_json(
    {
        "contract_sha256": CONTRACT_SHA256,
        "cell7_receipt_sha256": CELL7_RECEIPT_SHA256,
        "cell8_implementation_sha256": CELL8_IMPLEMENTATION_SHA256,
        "component_hashes": _candidate_component_hashes_cell8,
        "outcomes_accessed": False,
    }
)
CANDIDATE_PREDICTION_FREEZE_PATH = (
    CELL8_QUALITY_ROOT / "candidate_prediction_freeze_receipt.json"
)
if CANDIDATE_PREDICTION_FREEZE_PATH.exists():
    _existing_freeze_cell8 = json.loads(
        CANDIDATE_PREDICTION_FREEZE_PATH.read_text(encoding="utf-8")
    )
    if _existing_freeze_cell8.get("candidate_prediction_freeze_sha256") != (
        CANDIDATE_PREDICTION_FREEZE_SHA256
    ):
        raise RuntimeError("A different Cell 8 candidate freeze receipt exists.")
    CANDIDATE_FREEZE_STATE = "existing identical candidate freeze verified"
else:
    save_json(
        {
            "cell8_version": CELL8_VERSION,
            "created_at_utc": utc_now(),
            "contract_sha256": CONTRACT_SHA256,
            "cell7_receipt_sha256": CELL7_RECEIPT_SHA256,
            "cell8_implementation_sha256": CELL8_IMPLEMENTATION_SHA256,
            "component_hashes": _candidate_component_hashes_cell8,
            "candidate_prediction_freeze_sha256": (
                CANDIDATE_PREDICTION_FREEZE_SHA256
            ),
            "outcomes_accessed": False,
            "labels_used": False,
        },
        CANDIDATE_PREDICTION_FREEZE_PATH,
    )
    CANDIDATE_FREEZE_STATE = "new candidate prediction freeze created"

save_csv_atomic(
    UNIVERSAL_CONFIGURATION_GRID,
    CELL8_QUALITY_ROOT / "universal_configuration_grid.csv",
)
save_csv_atomic(
    CANDIDATE_PREDICTION_REGISTRY,
    CELL8_QUALITY_ROOT / "candidate_prediction_registry.csv",
)


# =============================================================================
# 5. Official CARE numerical helpers and reference tests
# =============================================================================

def cell8_confusion_counts(
    ground_truth: np.ndarray,
    prediction: np.ndarray,
) -> tuple[int, int, int, int]:
    truth = np.asarray(ground_truth, dtype=bool).reshape(-1)
    predicted = np.asarray(prediction, dtype=bool).reshape(-1)
    if truth.shape != predicted.shape or truth.size == 0:
        raise ValueError("CARE confusion arrays must be aligned and nonempty.")
    return (
        int(np.sum(~truth & ~predicted)),
        int(np.sum(~truth & predicted)),
        int(np.sum(truth & ~predicted)),
        int(np.sum(truth & predicted)),
    )


def cell8_fbeta_from_counts(
    true_positive: int,
    false_positive: int,
    false_negative: int,
    beta: float = CARE_BETA_CELL8,
) -> float:
    beta_squared = float(beta) ** 2
    numerator = (1.0 + beta_squared) * int(true_positive)
    denominator = (
        numerator
        + beta_squared * int(false_negative)
        + int(false_positive)
    )
    return float(numerator / denominator) if denominator > 0.0 else 0.0


def cell8_earliness_weights(relative_positions: np.ndarray) -> np.ndarray:
    positions = np.asarray(relative_positions, dtype=np.float64).reshape(-1)
    if positions.size == 0 or not np.isfinite(positions).all():
        raise ValueError("Earliness positions must be finite and nonempty.")
    if (positions < -1e-9).any() or (positions > 1.0 + 1e-9).any():
        raise ValueError("Earliness positions must lie in [0, 1].")
    positions = np.clip(positions, 0.0, 1.0)
    weights = np.where(positions <= 0.5, 1.0, 2.0 * (1.0 - positions))
    return np.clip(weights, 0.0, 1.0)


def cell8_criticality_trajectory(
    raw_alarm: np.ndarray,
    actionable: np.ndarray,
    timestamp_ns: np.ndarray,
    segment_id: np.ndarray,
) -> np.ndarray:
    alarm = np.asarray(raw_alarm, dtype=bool).reshape(-1)
    actionable = np.asarray(actionable, dtype=bool).reshape(-1)
    timestamps = np.asarray(timestamp_ns, dtype=np.int64).reshape(-1)
    segments = np.asarray(segment_id, dtype=np.int64).reshape(-1)
    if not (
        len(alarm) == len(actionable) == len(timestamps) == len(segments)
    ) or len(alarm) == 0:
        raise ValueError("Criticality inputs must be aligned and nonempty.")
    result = np.zeros(len(alarm), dtype=np.int32)
    current = 0
    for index in range(len(alarm)):
        if index > 0 and (
            segments[index] != segments[index - 1]
            or timestamps[index] - timestamps[index - 1] != EXPECTED_STEP_NS_CELL8
        ):
            current = 0
        if actionable[index]:
            current += 1 if alarm[index] else -1
            current = max(0, current)
        result[index] = current
    return result


def cell8_care_score(
    mean_coverage: float,
    mean_earliness: float,
    reliability: float,
    mean_normal_accuracy: float,
    event_alarm_count: int,
) -> tuple[float, str]:
    if int(event_alarm_count) == 0:
        return 0.0, "no_event_alarm"
    if float(mean_normal_accuracy) < 0.5:
        return float(mean_normal_accuracy), "normal_accuracy_not_above_random"
    weights = CARE_COMPONENT_WEIGHTS_CELL8
    score = (
        weights[0] * float(mean_coverage)
        + weights[1] * float(mean_earliness)
        + weights[2] * float(reliability)
        + weights[3] * float(mean_normal_accuracy)
    ) / sum(weights)
    return float(score), "weighted_average"


def cell8_run_official_reference_tests() -> None:
    if cell8_confusion_counts(
        np.asarray([False, False, True, True]),
        np.asarray([False, True, False, True]),
    ) != (1, 1, 1, 1):
        raise RuntimeError("CARE confusion reference test failed.")
    expected_f05 = 1.25 * 4.0 / (1.25 * 4.0 + 0.25 * 2.0 + 1.0)
    if not np.isclose(cell8_fbeta_from_counts(4, 1, 2), expected_f05):
        raise RuntimeError("CARE F0.5 reference test failed.")
    if not np.allclose(
        cell8_earliness_weights(np.asarray([0.0, 0.5, 0.75, 1.0])),
        np.asarray([1.0, 1.0, 0.5, 0.0]),
    ):
        raise RuntimeError("CARE earliness reference test failed.")
    trajectory = cell8_criticality_trajectory(
        np.asarray([True, True, False, True, True]),
        np.asarray([True, True, True, False, True]),
        np.asarray([0, 1, 2, 3, 5], dtype=np.int64) * EXPECTED_STEP_NS_CELL8,
        np.asarray([0, 0, 0, 0, 0]),
    )
    if not np.array_equal(trajectory, np.asarray([1, 2, 1, 1, 1])):
        raise RuntimeError("CARE criticality reference test failed.")
    if cell8_care_score(1.0, 1.0, 1.0, 1.0, 0) != (
        0.0,
        "no_event_alarm",
    ):
        raise RuntimeError("CARE no-alarm boundary test failed.")
    if cell8_care_score(1.0, 1.0, 1.0, 0.4, 1) != (
        0.4,
        "normal_accuracy_not_above_random",
    ):
        raise RuntimeError("CARE accuracy boundary test failed.")
    score, rule = cell8_care_score(0.5, 0.5, 0.5, 1.0, 1)
    if rule != "weighted_average" or not np.isclose(score, 0.7):
        raise RuntimeError("CARE weighted-score reference test failed.")


cell8_run_official_reference_tests()


# =============================================================================
# 6. Open the outcome lockbox only after candidate predictions are frozen
# =============================================================================

OUTCOME_LOCKBOX_PATH_CELL8 = (
    Path(INVENTORY_DIR) / "outcome_lockbox_do_not_load_before_outer_predictions.csv"
)
if not CANDIDATE_PREDICTION_FREEZE_PATH.exists():
    raise RuntimeError("Candidate predictions were not frozen before outcome access.")
if not OUTCOME_LOCKBOX_PATH_CELL8.exists():
    raise FileNotFoundError(f"Outcome lockbox not found: {OUTCOME_LOCKBOX_PATH_CELL8}")


def cell8_load_outcome_lockbox() -> pd.DataFrame:
    frame = pd.read_csv(OUTCOME_LOCKBOX_PATH_CELL8)
    expected_columns = (
        "case_key",
        "farm",
        "asset_id",
        "source_asset_id",
        "event_id",
        "event_label_raw",
        "is_anomaly",
        "event_start",
        "event_start_id",
        "event_end",
        "event_end_id",
        "event_description",
    )
    if tuple(frame.columns) != expected_columns:
        raise RuntimeError("Outcome-lockbox schema drift was detected.")
    for column in ("case_key", "farm", "asset_id", "source_asset_id"):
        frame[column] = frame[column].astype(str)
    frame["event_id"] = pd.to_numeric(frame["event_id"], errors="raise").astype(int)
    anomaly_text = frame["is_anomaly"].astype(str).str.strip().str.lower()
    mapping = {"true": True, "false": False, "1": True, "0": False}
    if not anomaly_text.isin(mapping).all():
        raise RuntimeError("Invalid anomaly label in the outcome lockbox.")
    frame["is_anomaly"] = anomaly_text.map(mapping).astype(bool)
    for column in ("event_start", "event_end"):
        frame[column] = pd.to_datetime(frame[column], errors="coerce")
    for column in ("event_start_id", "event_end_id"):
        frame[column] = pd.to_numeric(frame[column], errors="coerce").astype("Int64")
    # Preserve blank descriptions as missing exactly as Cell 2 serialized them;
    # converting them to empty strings changes the canonical DataFrame hash.
    ordered_lockbox = frame.sort_values(
        ["farm", "event_id"], kind="stable"
    ).reset_index(drop=True)
    lockbox_payload = ordered_lockbox.to_csv(
        index=False,
        lineterminator="\n",
        na_rep="<NA>",
        float_format="%.12g",
    )
    observed_lockbox_hash = hashlib.sha256(
        lockbox_payload.encode("utf-8")
    ).hexdigest()
    if observed_lockbox_hash != OUTCOME_LOCKBOX_SHA256:
        raise RuntimeError(
            "Outcome-lockbox hash verification failed. "
            f"Observed={observed_lockbox_hash}, expected={OUTCOME_LOCKBOX_SHA256}."
        )
    if len(frame) != DATASET.expected_total_cases:
        raise RuntimeError("Outcome-lockbox case count drift was detected.")
    if int(frame["is_anomaly"].sum()) != DATASET.expected_anomaly_cases:
        raise RuntimeError("Outcome-lockbox anomaly count drift was detected.")
    if int((~frame["is_anomaly"]).sum()) != DATASET.expected_normal_cases:
        raise RuntimeError("Outcome-lockbox normal count drift was detected.")
    return frame.sort_values(["farm", "event_id"], kind="stable").reset_index(drop=True)


_OUTCOME_LOCKBOX_CELL8 = cell8_load_outcome_lockbox()
OUTCOME_ACCESS_BEGAN_AFTER_CANDIDATE_FREEZE = True


# =============================================================================
# 7. Case metrics and pooled official CARE aggregation
# =============================================================================

def cell8_configuration_column(
    artifact: dict[str, Any],
    configuration_id: int,
) -> int:
    matches = np.flatnonzero(
        np.asarray(artifact["configuration_ids"], dtype=np.int32)
        == int(configuration_id)
    )
    if len(matches) != 1:
        raise KeyError(f"Unknown or duplicate configuration: {configuration_id}")
    return int(matches[0])


def cell8_case_configuration_metrics(
    outcome_row: Any,
    artifact: dict[str, Any],
    configuration_id: int,
) -> dict[str, Any]:
    column = cell8_configuration_column(artifact, configuration_id)
    timestamps = np.asarray(artifact["timestamp_ns"], dtype=np.int64)
    segments = np.asarray(artifact["segment_id"], dtype=np.int32)
    normal_status = np.asarray(artifact["normal_status"], dtype=bool)
    raw_alarm = np.asarray(artifact["raw_alarm"], dtype=bool)[:, column]
    normalized_width = np.asarray(
        artifact["normalized_interval_width"], dtype=np.float64
    )[:, column]
    interval_covered = np.asarray(
        artifact["interval_covered_fraction"], dtype=np.float64
    )[:, column]
    interval_score = np.asarray(
        artifact["standardized_interval_score"], dtype=np.float64
    )[:, column]
    is_anomaly = bool(outcome_row.is_anomaly)
    timestamp_values = pd.to_datetime(timestamps, unit="ns")

    if is_anomaly:
        event_start = pd.Timestamp(outcome_row.event_start)
        event_end = pd.Timestamp(outcome_row.event_end)
        if pd.isna(event_start) or pd.isna(event_end) or event_end < event_start:
            raise RuntimeError(f"Invalid event boundaries for {outcome_row.case_key}.")
        ground_truth = np.asarray(
            (timestamp_values >= event_start) & (timestamp_values <= event_end),
            dtype=bool,
        )
        if not ground_truth.any():
            raise RuntimeError(
                f"Anomaly interval has no prediction timestamp for {outcome_row.case_key}."
            )
    else:
        event_start = pd.NaT
        event_end = pd.NaT
        ground_truth = np.zeros(len(timestamps), dtype=bool)

    if str(outcome_row.farm) == "Wind Farm A" and is_anomaly:
        actionable = np.ones(len(timestamps), dtype=bool)
    else:
        actionable = normal_status.copy()
    if not actionable.any():
        raise RuntimeError(f"No CARE-actionable timestamp for {outcome_row.case_key}.")

    tn, fp, fn, tp = cell8_confusion_counts(
        ground_truth[actionable],
        raw_alarm[actionable],
    )
    point_accuracy = float((tn + tp) / (tn + fp + fn + tp))
    coverage_fbeta = (
        cell8_fbeta_from_counts(tp, fp, fn) if is_anomaly else np.nan
    )

    if is_anomaly:
        event_indices = np.flatnonzero(ground_truth)
        event_span_ns = int(event_end.value - event_start.value)
        if event_span_ns <= 0:
            relative_positions = np.zeros(len(event_indices), dtype=np.float64)
        else:
            relative_positions = (
                timestamps[event_indices].astype(np.float64) - float(event_start.value)
            ) / float(event_span_ns)
        earliness_weights = cell8_earliness_weights(relative_positions)
        earliness_score = float(
            np.sum(earliness_weights * raw_alarm[event_indices].astype(np.float64))
            / np.sum(earliness_weights)
        )
        criticality_scope = timestamps <= int(event_end.value)
    else:
        earliness_score = np.nan
        criticality_scope = np.ones(len(timestamps), dtype=bool)

    scoped_criticality = cell8_criticality_trajectory(
        raw_alarm[criticality_scope],
        actionable[criticality_scope],
        timestamps[criticality_scope],
        segments[criticality_scope],
    )
    maximum_criticality = int(scoped_criticality.max())
    event_alarm = maximum_criticality >= CARE_CRITICALITY_THRESHOLD_CELL8
    lead_hours = np.nan
    if is_anomaly and event_alarm:
        crossing = np.flatnonzero(
            scoped_criticality >= CARE_CRITICALITY_THRESHOLD_CELL8
        )
        first_timestamp_ns = timestamps[criticality_scope][int(crossing[0])]
        lead_hours = float(
            (int(event_end.value) - int(first_timestamp_ns)) / 3.6e12
        )

    interval_mask = actionable & np.isfinite(normalized_width)
    return {
        "case_key": str(outcome_row.case_key),
        "farm": str(outcome_row.farm),
        "asset_id": str(outcome_row.asset_id),
        "event_id": int(outcome_row.event_id),
        "configuration_id": int(configuration_id),
        "is_anomaly": is_anomaly,
        "prediction_timestamps": int(len(timestamps)),
        "care_actionable_timestamps": int(actionable.sum()),
        "true_negative": tn,
        "false_positive": fp,
        "false_negative": fn,
        "true_positive": tp,
        "point_accuracy": point_accuracy,
        "coverage_fbeta": coverage_fbeta,
        "earliness_score": earliness_score,
        "maximum_criticality": maximum_criticality,
        "event_alarm": bool(event_alarm),
        "lead_hours_to_event_end": lead_hours,
        "mean_normalized_interval_width": (
            float(np.nanmean(normalized_width[interval_mask]))
            if interval_mask.any()
            else np.nan
        ),
        "mean_interval_covered_fraction": (
            float(np.nanmean(interval_covered[actionable]))
            if np.isfinite(interval_covered[actionable]).any()
            else np.nan
        ),
        "mean_standardized_interval_score": (
            float(np.nanmean(interval_score[actionable]))
            if np.isfinite(interval_score[actionable]).any()
            else np.nan
        ),
    }


def cell8_case_all_configuration_metrics(
    outcome_row: Any,
    artifact: dict[str, Any],
) -> pd.DataFrame:
    """Vectorized CARE case metrics for the complete universal grid."""
    configuration_ids = np.asarray(
        artifact["configuration_ids"], dtype=np.int32
    )
    if not np.array_equal(configuration_ids, UNIVERSAL_CONFIGURATION_IDS):
        raise RuntimeError(
            f"Universal configuration order drift for {outcome_row.case_key}."
        )
    timestamps = np.asarray(artifact["timestamp_ns"], dtype=np.int64)
    segments = np.asarray(artifact["segment_id"], dtype=np.int32)
    normal_status = np.asarray(artifact["normal_status"], dtype=bool)
    raw_alarm = np.asarray(artifact["raw_alarm"], dtype=bool)
    normalized_width = np.asarray(
        artifact["normalized_interval_width"], dtype=np.float64
    )
    interval_covered = np.asarray(
        artifact["interval_covered_fraction"], dtype=np.float64
    )
    interval_score = np.asarray(
        artifact["standardized_interval_score"], dtype=np.float64
    )
    if raw_alarm.shape != (len(timestamps), len(configuration_ids)):
        raise RuntimeError(f"Candidate matrix shape drift for {outcome_row.case_key}.")
    is_anomaly = bool(outcome_row.is_anomaly)
    timestamp_values = pd.to_datetime(timestamps, unit="ns")
    if is_anomaly:
        event_start = pd.Timestamp(outcome_row.event_start)
        event_end = pd.Timestamp(outcome_row.event_end)
        if pd.isna(event_start) or pd.isna(event_end) or event_end < event_start:
            raise RuntimeError(f"Invalid event boundaries for {outcome_row.case_key}.")
        ground_truth = np.asarray(
            (timestamp_values >= event_start) & (timestamp_values <= event_end),
            dtype=bool,
        )
        if not ground_truth.any():
            raise RuntimeError(
                f"Anomaly interval has no prediction timestamp for {outcome_row.case_key}."
            )
    else:
        event_start = pd.NaT
        event_end = pd.NaT
        ground_truth = np.zeros(len(timestamps), dtype=bool)
    actionable = (
        np.ones(len(timestamps), dtype=bool)
        if str(outcome_row.farm) == "Wind Farm A" and is_anomaly
        else normal_status.copy()
    )
    if not actionable.any():
        raise RuntimeError(f"No CARE-actionable timestamp for {outcome_row.case_key}.")

    truth = ground_truth[actionable, None]
    predicted = raw_alarm[actionable]
    true_negative = np.sum(~truth & ~predicted, axis=0, dtype=np.int64)
    false_positive = np.sum(~truth & predicted, axis=0, dtype=np.int64)
    false_negative = np.sum(truth & ~predicted, axis=0, dtype=np.int64)
    true_positive = np.sum(truth & predicted, axis=0, dtype=np.int64)
    point_accuracy = (true_negative + true_positive) / float(actionable.sum())
    if is_anomaly:
        beta_squared = CARE_BETA_CELL8**2
        fbeta_numerator = (1.0 + beta_squared) * true_positive
        fbeta_denominator = (
            fbeta_numerator
            + beta_squared * false_negative
            + false_positive
        )
        coverage_fbeta = np.divide(
            fbeta_numerator,
            fbeta_denominator,
            out=np.zeros(len(configuration_ids), dtype=np.float64),
            where=fbeta_denominator > 0,
        )
        event_indices = np.flatnonzero(ground_truth)
        event_span_ns = int(event_end.value - event_start.value)
        relative_positions = (
            np.zeros(len(event_indices), dtype=np.float64)
            if event_span_ns <= 0
            else (
                timestamps[event_indices].astype(np.float64)
                - float(event_start.value)
            )
            / float(event_span_ns)
        )
        earliness_weights = cell8_earliness_weights(relative_positions)
        earliness_score = (
            earliness_weights[:, None]
            * raw_alarm[event_indices].astype(np.float64)
        ).sum(axis=0) / earliness_weights.sum()
        criticality_scope = timestamps <= int(event_end.value)
    else:
        coverage_fbeta = np.full(len(configuration_ids), np.nan, dtype=np.float64)
        earliness_score = np.full(len(configuration_ids), np.nan, dtype=np.float64)
        criticality_scope = np.ones(len(timestamps), dtype=bool)

    scoped_alarm = raw_alarm[criticality_scope]
    scoped_actionable = actionable[criticality_scope]
    scoped_timestamps = timestamps[criticality_scope]
    scoped_segments = segments[criticality_scope]
    current = np.zeros(len(configuration_ids), dtype=np.int32)
    maximum = np.zeros(len(configuration_ids), dtype=np.int32)
    first_crossing = np.full(len(configuration_ids), -1, dtype=np.int64)
    for row_index in range(len(scoped_timestamps)):
        if row_index > 0 and (
            scoped_segments[row_index] != scoped_segments[row_index - 1]
            or scoped_timestamps[row_index] - scoped_timestamps[row_index - 1]
            != EXPECTED_STEP_NS_CELL8
        ):
            current.fill(0)
        if scoped_actionable[row_index]:
            current += np.where(scoped_alarm[row_index], 1, -1).astype(np.int32)
            np.maximum(current, 0, out=current)
        maximum = np.maximum(maximum, current)
        newly_crossed = (
            (first_crossing < 0)
            & (current >= CARE_CRITICALITY_THRESHOLD_CELL8)
        )
        first_crossing[newly_crossed] = row_index
    event_alarm = maximum >= CARE_CRITICALITY_THRESHOLD_CELL8
    lead_hours = np.full(len(configuration_ids), np.nan, dtype=np.float64)
    if is_anomaly:
        detected = np.flatnonzero(event_alarm)
        if len(detected):
            crossing_timestamps = scoped_timestamps[first_crossing[detected]]
            lead_hours[detected] = (
                int(event_end.value) - crossing_timestamps
            ) / 3.6e12

    def _column_nanmean(values: np.ndarray, row_mask: np.ndarray) -> np.ndarray:
        selected = np.asarray(values, dtype=np.float64)[row_mask]
        finite = np.isfinite(selected)
        count = finite.sum(axis=0)
        result = np.full(selected.shape[1], np.nan, dtype=np.float64)
        np.divide(
            np.where(finite, selected, 0.0).sum(axis=0),
            count,
            out=result,
            where=count > 0,
        )
        return result

    mean_width = _column_nanmean(normalized_width, actionable)
    mean_interval_coverage = _column_nanmean(interval_covered, actionable)
    mean_interval_score = _column_nanmean(interval_score, actionable)
    configuration_count = len(configuration_ids)
    return pd.DataFrame(
        {
            "case_key": [str(outcome_row.case_key)] * configuration_count,
            "farm": [str(outcome_row.farm)] * configuration_count,
            "asset_id": [str(outcome_row.asset_id)] * configuration_count,
            "event_id": [int(outcome_row.event_id)] * configuration_count,
            "configuration_id": configuration_ids,
            "is_anomaly": [is_anomaly] * configuration_count,
            "prediction_timestamps": [len(timestamps)] * configuration_count,
            "care_actionable_timestamps": [int(actionable.sum())] * configuration_count,
            "true_negative": true_negative,
            "false_positive": false_positive,
            "false_negative": false_negative,
            "true_positive": true_positive,
            "point_accuracy": point_accuracy,
            "coverage_fbeta": coverage_fbeta,
            "earliness_score": earliness_score,
            "maximum_criticality": maximum,
            "event_alarm": event_alarm,
            "lead_hours_to_event_end": lead_hours,
            "mean_normalized_interval_width": mean_width,
            "mean_interval_covered_fraction": mean_interval_coverage,
            "mean_standardized_interval_score": mean_interval_score,
        }
    )


def cell8_aggregate_care(case_metrics: pd.DataFrame) -> dict[str, Any]:
    if case_metrics.empty:
        raise ValueError("CARE aggregation requires at least one case.")
    anomaly = case_metrics["is_anomaly"].astype(bool)
    normal = ~anomaly
    if not anomaly.any() or not normal.any():
        return {
            "care_defined": False,
            "coverage_fbeta": np.nan,
            "earliness_weighted_score": np.nan,
            "event_reliability_fbeta": np.nan,
            "normal_accuracy": np.nan,
            "care_score": np.nan,
            "care_boundary_rule": "missing_anomaly_or_normal_case",
            "event_true_negative": 0,
            "event_false_positive": 0,
            "event_false_negative": 0,
            "event_true_positive": 0,
            "event_false_alarm_count": 0,
            "detected_anomaly_cases": 0,
            "median_lead_hours": np.nan,
            "mean_normalized_normal_interval_width": np.nan,
            "normal_interval_coverage": np.nan,
            "normal_interval_score": np.nan,
        }
    event_tn, event_fp, event_fn, event_tp = cell8_confusion_counts(
        anomaly.to_numpy(),
        case_metrics["event_alarm"].astype(bool).to_numpy(),
    )
    reliability = cell8_fbeta_from_counts(event_tp, event_fp, event_fn)
    coverage = float(case_metrics.loc[anomaly, "coverage_fbeta"].mean())
    earliness = float(case_metrics.loc[anomaly, "earliness_score"].mean())
    normal_accuracy = float(case_metrics.loc[normal, "point_accuracy"].mean())
    event_alarm_count = int(case_metrics["event_alarm"].sum())
    care_score, rule = cell8_care_score(
        coverage,
        earliness,
        reliability,
        normal_accuracy,
        event_alarm_count,
    )
    detected_leads = case_metrics.loc[
        anomaly & case_metrics["event_alarm"].astype(bool),
        "lead_hours_to_event_end",
    ]
    return {
        "care_defined": True,
        "coverage_fbeta": coverage,
        "earliness_weighted_score": earliness,
        "event_reliability_fbeta": reliability,
        "normal_accuracy": normal_accuracy,
        "care_score": care_score,
        "care_boundary_rule": rule,
        "event_true_negative": event_tn,
        "event_false_positive": event_fp,
        "event_false_negative": event_fn,
        "event_true_positive": event_tp,
        "event_false_alarm_count": event_fp,
        "detected_anomaly_cases": event_tp,
        "median_lead_hours": (
            float(detected_leads.median()) if detected_leads.notna().any() else np.nan
        ),
        "mean_normalized_normal_interval_width": float(
            case_metrics.loc[normal, "mean_normalized_interval_width"].mean()
        ),
        "normal_interval_coverage": float(
            case_metrics.loc[normal, "mean_interval_covered_fraction"].mean()
        ),
        "normal_interval_score": float(
            case_metrics.loc[normal, "mean_standardized_interval_score"].mean()
        ),
    }


# =============================================================================
# 8. Five-fold grouped development scoring and one selection per outer asset
# =============================================================================

_case_metric_frames_cell8: list[pd.DataFrame] = []
for _outcome_cell8 in _OUTCOME_LOCKBOX_CELL8.itertuples(index=False):
    _artifact_cell8 = load_cell8_candidate_predictions(str(_outcome_cell8.case_key))
    _case_metric_frames_cell8.append(
        cell8_case_all_configuration_metrics(_outcome_cell8, _artifact_cell8)
    )
    del _artifact_cell8
CASE_CONFIGURATION_METRICS_CACHE = pd.concat(
    _case_metric_frames_cell8,
    ignore_index=True,
)
del _case_metric_frames_cell8
if len(CASE_CONFIGURATION_METRICS_CACHE) != (
    DATASET.expected_total_cases * len(UNIVERSAL_CONFIGURATION_GRID)
):
    raise RuntimeError("The case-configuration metric cache is incomplete.")
if CASE_CONFIGURATION_METRICS_CACHE.duplicated(
    ["case_key", "configuration_id"]
).any():
    raise RuntimeError("The case-configuration metric cache contains duplicates.")

_outer_grid_records_cell8: list[dict[str, Any]] = []
_inner_audit_records_cell8: list[dict[str, Any]] = []
_selected_records_cell8: list[dict[str, Any]] = []
_label_access_records_cell8: list[dict[str, Any]] = []

for _outer_row_cell8 in OUTER_FOLDS.sort_values("outer_fold").itertuples(index=False):
    _outer_fold_cell8 = int(_outer_row_cell8.outer_fold)
    _outer_asset_cell8 = str(_outer_row_cell8.outer_asset_id)
    _development_outcomes_cell8 = _OUTCOME_LOCKBOX_CELL8.loc[
        ~_OUTCOME_LOCKBOX_CELL8["asset_id"].eq(_outer_asset_cell8)
    ].copy()
    _outer_outcome_rows_cell8 = _OUTCOME_LOCKBOX_CELL8.loc[
        _OUTCOME_LOCKBOX_CELL8["asset_id"].eq(_outer_asset_cell8)
    ]
    if _development_outcomes_cell8.empty or _outer_outcome_rows_cell8.empty:
        raise RuntimeError(f"Empty outer partition in fold {_outer_fold_cell8}.")
    if _development_outcomes_cell8["asset_id"].eq(_outer_asset_cell8).any():
        raise RuntimeError(f"Outer-label leakage in fold {_outer_fold_cell8}.")
    _inner_assignments_cell8 = INNER_ASSET_FOLDS.loc[
        INNER_ASSET_FOLDS["outer_fold"].eq(_outer_fold_cell8),
        ["asset_id", "inner_fold"],
    ].copy()
    _inner_map_cell8 = dict(
        zip(
            _inner_assignments_cell8["asset_id"].astype(str),
            _inner_assignments_cell8["inner_fold"].astype(int),
        )
    )
    if set(_development_outcomes_cell8["asset_id"]) != set(_inner_map_cell8):
        raise RuntimeError(f"Inner assignment drift in outer fold {_outer_fold_cell8}.")

    _development_metrics_cell8 = CASE_CONFIGURATION_METRICS_CACHE.loc[
        ~CASE_CONFIGURATION_METRICS_CACHE["asset_id"].eq(_outer_asset_cell8)
    ].copy()
    _development_metrics_cell8["inner_fold"] = (
        _development_metrics_cell8["asset_id"].map(_inner_map_cell8)
    )
    if _development_metrics_cell8["inner_fold"].isna().any():
        raise RuntimeError(f"Inner-fold mapping failed in outer fold {_outer_fold_cell8}.")
    _development_metrics_cell8["inner_fold"] = (
        _development_metrics_cell8["inner_fold"].astype(int)
    )
    expected_rows = len(_development_outcomes_cell8) * len(
        UNIVERSAL_CONFIGURATION_GRID
    )
    if len(_development_metrics_cell8) != expected_rows:
        raise RuntimeError(f"Incomplete development metrics in fold {_outer_fold_cell8}.")
    if _development_metrics_cell8["asset_id"].eq(_outer_asset_cell8).any():
        raise RuntimeError(f"Outer asset entered the objective in fold {_outer_fold_cell8}.")

    for _configuration_cell8 in UNIVERSAL_CONFIGURATION_GRID.itertuples(index=False):
        _configuration_metrics_cell8 = _development_metrics_cell8.loc[
            _development_metrics_cell8["configuration_id"].eq(
                int(_configuration_cell8.configuration_id)
            )
        ]
        _pooled_cell8 = cell8_aggregate_care(_configuration_metrics_cell8)
        if not _pooled_cell8["care_defined"]:
            raise RuntimeError(
                f"Pooled CARE is undefined in outer fold {_outer_fold_cell8}."
            )
        _fold_scores_cell8: list[float] = []
        _defined_inner_cell8 = 0
        for _inner_fold_cell8 in range(1, EVALUATION.inner_splits + 1):
            _inner_metrics_cell8 = _configuration_metrics_cell8.loc[
                _configuration_metrics_cell8["inner_fold"].eq(_inner_fold_cell8)
            ]
            _inner_result_cell8 = cell8_aggregate_care(_inner_metrics_cell8)
            if _inner_result_cell8["care_defined"]:
                _defined_inner_cell8 += 1
                _fold_scores_cell8.append(float(_inner_result_cell8["care_score"]))
            _inner_audit_records_cell8.append(
                {
                    "outer_fold": _outer_fold_cell8,
                    "outer_asset_id": _outer_asset_cell8,
                    "inner_fold": _inner_fold_cell8,
                    "configuration_id": int(_configuration_cell8.configuration_id),
                    "validation_assets": int(
                        _inner_metrics_cell8["asset_id"].nunique()
                    ),
                    "validation_cases": int(len(_inner_metrics_cell8)),
                    **_inner_result_cell8,
                }
            )
        _outer_grid_records_cell8.append(
            {
                "outer_fold": _outer_fold_cell8,
                "outer_asset_id": _outer_asset_cell8,
                "outer_farm": str(_outer_row_cell8.outer_farm),
                "development_assets": int(
                    _development_metrics_cell8["asset_id"].nunique()
                ),
                "development_cases": int(
                    _development_metrics_cell8["case_key"].nunique()
                ),
                "configuration_id": int(_configuration_cell8.configuration_id),
                "coverage": float(_configuration_cell8.coverage),
                "minimum_r2": float(_configuration_cell8.minimum_r2),
                "evidence_head": str(_configuration_cell8.evidence_head),
                "smoothing_steps": int(_configuration_cell8.smoothing_steps),
                "threshold_quantile": float(_configuration_cell8.threshold_quantile),
                "defined_inner_folds": int(_defined_inner_cell8),
                "mean_inner_care": (
                    float(np.mean(_fold_scores_cell8))
                    if _fold_scores_cell8
                    else np.nan
                ),
                "minimum_inner_care": (
                    float(np.min(_fold_scores_cell8))
                    if _fold_scores_cell8
                    else np.nan
                ),
                **_pooled_cell8,
            }
        )

    _fold_grid_cell8 = pd.DataFrame(
        record
        for record in _outer_grid_records_cell8
        if int(record["outer_fold"]) == _outer_fold_cell8
    )
    _selection_order_cell8 = _fold_grid_cell8.sort_values(
        [
            "care_score",
            "normal_accuracy",
            "event_reliability_fbeta",
            "mean_normalized_normal_interval_width",
            "coverage",
            "minimum_r2",
            "evidence_head",
            "smoothing_steps",
            "threshold_quantile",
            "configuration_id",
        ],
        ascending=[False, False, False, True, True, True, True, True, True, True],
        kind="stable",
    ).reset_index(drop=True)
    _selected_cell8 = _selection_order_cell8.iloc[0].to_dict()
    _selected_cell8["selection_rank"] = 1
    _selected_cell8["selection_input_case_sha256"] = sha256_json(
        sorted(_development_outcomes_cell8["case_key"].astype(str).tolist())
    )
    _selected_cell8["outer_case_key_sha256"] = sha256_json(
        sorted(_outer_outcome_rows_cell8["case_key"].astype(str).tolist())
    )
    _selected_cell8["outer_labels_used"] = False
    _selected_records_cell8.append(_selected_cell8)
    _label_access_records_cell8.append(
        {
            "outer_fold": _outer_fold_cell8,
            "outer_asset_id": _outer_asset_cell8,
            "development_assets": int(_development_outcomes_cell8["asset_id"].nunique()),
            "development_cases": int(len(_development_outcomes_cell8)),
            "development_case_sha256": _selected_cell8[
                "selection_input_case_sha256"
            ],
            "excluded_outer_cases": int(len(_outer_outcome_rows_cell8)),
            "excluded_outer_case_sha256": _selected_cell8["outer_case_key_sha256"],
            "outer_case_entered_objective": False,
        }
    )
    print(
        f"Nested selection — outer {_outer_fold_cell8:>2}/"
        f"{len(OUTER_FOLDS)} asset={_outer_asset_cell8}: "
        f"config={int(_selected_cell8['configuration_id']):>3} "
        f"development CARE={float(_selected_cell8['care_score']):.4f}",
        flush=True,
    )
    del _development_metrics_cell8

OUTER_SELECTION_GRID = pd.DataFrame(_outer_grid_records_cell8).sort_values(
    ["outer_fold", "configuration_id"], kind="stable"
).reset_index(drop=True)
INNER_FOLD_SCORE_AUDIT = pd.DataFrame(_inner_audit_records_cell8).sort_values(
    ["outer_fold", "inner_fold", "configuration_id"], kind="stable"
).reset_index(drop=True)
OUTER_SELECTED_CONFIGURATIONS = pd.DataFrame(_selected_records_cell8).sort_values(
    "outer_fold", kind="stable"
).reset_index(drop=True)
LABEL_ACCESS_AUDIT = pd.DataFrame(_label_access_records_cell8).sort_values(
    "outer_fold", kind="stable"
).reset_index(drop=True)

if len(OUTER_SELECTED_CONFIGURATIONS) != DATASET.expected_assets:
    raise RuntimeError("Not every outer fold selected one configuration.")
if OUTER_SELECTED_CONFIGURATIONS["outer_asset_id"].duplicated().any():
    raise RuntimeError("An outer asset selected more than once.")
if OUTER_SELECTED_CONFIGURATIONS["outer_labels_used"].any():
    raise RuntimeError("Outer labels entered configuration selection.")
if LABEL_ACCESS_AUDIT["outer_case_entered_objective"].any():
    raise RuntimeError("The label-access audit detected outer leakage.")

SELECTED_CONFIGURATION_FREQUENCY = (
    OUTER_SELECTED_CONFIGURATIONS.groupby(
        [
            "configuration_id",
            "coverage",
            "minimum_r2",
            "evidence_head",
            "smoothing_steps",
            "threshold_quantile",
        ],
        as_index=False,
        sort=False,
    )
    .agg(
        outer_folds=("outer_fold", "size"),
        median_development_care=("care_score", "median"),
        minimum_development_care=("care_score", "min"),
    )
    .sort_values(["outer_folds", "configuration_id"], ascending=[False, True])
    .reset_index(drop=True)
)


# =============================================================================
# 9. Copy selected outer predictions from the pre-frozen candidate artifacts
# =============================================================================

def cell8_selected_paths(farm: str, event_id: int) -> tuple[Path, Path]:
    relative = Path(safe_slug(farm)) / f"event_{int(event_id):03d}"
    return (
        CELL8_SELECTED_ROOT / relative.with_suffix(".json"),
        CELL8_SELECTED_ROOT / relative.with_suffix(".npz"),
    )


def cell8_freeze_selected_case(case_row: Any) -> dict[str, Any]:
    selection_match = OUTER_SELECTED_CONFIGURATIONS.loc[
        OUTER_SELECTED_CONFIGURATIONS["outer_asset_id"].eq(str(case_row.asset_id))
    ]
    if len(selection_match) != 1:
        raise RuntimeError(f"Missing outer selection for {case_row.asset_id}.")
    selection = selection_match.iloc[0]
    candidate_row = CANDIDATE_PREDICTION_REGISTRY.loc[
        CANDIDATE_PREDICTION_REGISTRY["case_key"].eq(str(case_row.case_key))
    ]
    if len(candidate_row) != 1:
        raise RuntimeError(f"Missing candidate artifact for {case_row.case_key}.")
    candidate_hash = str(candidate_row.iloc[0]["artifact_sha256"])
    configuration_id = int(selection["configuration_id"])
    input_signature = sha256_json(
        {
            "candidate_prediction_freeze_sha256": CANDIDATE_PREDICTION_FREEZE_SHA256,
            "candidate_artifact_sha256": candidate_hash,
            "outer_fold": int(selection["outer_fold"]),
            "outer_asset_id": str(selection["outer_asset_id"]),
            "configuration_id": configuration_id,
            "selection_input_case_sha256": str(
                selection["selection_input_case_sha256"]
            ),
        }
    )
    metadata_path, artifact_path = cell8_selected_paths(
        str(case_row.farm), int(case_row.event_id)
    )
    if metadata_path.exists() and artifact_path.exists():
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        if metadata.get("case_input_signature") != input_signature:
            raise RuntimeError(
                f"Selected-prediction input drift for {case_row.case_key}."
            )
        if metadata.get("artifact_sha256") != file_sha256(artifact_path):
            raise RuntimeError(f"Selected artifact hash failed: {artifact_path}")
        return dict(metadata["summary"])

    candidate = load_cell8_candidate_predictions(str(case_row.case_key))
    column = cell8_configuration_column(candidate, configuration_id)
    artifact_arrays = {
        "prediction_row_indices": np.asarray(
            candidate["prediction_row_indices"], dtype=np.int32
        ),
        "timestamp_ns": np.asarray(candidate["timestamp_ns"], dtype=np.int64),
        "segment_id": np.asarray(candidate["segment_id"], dtype=np.int32),
        "normal_status": np.asarray(candidate["normal_status"], dtype=bool),
        "configuration_id": np.asarray(configuration_id, dtype=np.int32),
        "coverage": np.asarray(float(selection["coverage"]), dtype=np.float64),
        "minimum_r2": np.asarray(float(selection["minimum_r2"]), dtype=np.float64),
        "smoothing_steps": np.asarray(
            int(selection["smoothing_steps"]), dtype=np.int32
        ),
        "threshold_quantile": np.asarray(
            float(selection["threshold_quantile"]), dtype=np.float64
        ),
        "threshold": np.asarray(
            np.asarray(candidate["thresholds"], dtype=np.float64)[column],
            dtype=np.float64,
        ),
        "raw_health": np.asarray(candidate["raw_health"], dtype=np.float32)[:, column],
        "observed_weight_fraction": np.asarray(
            candidate["observed_weight_fraction"], dtype=np.float32
        )[:, column],
        "smoothed_health": np.asarray(
            candidate["smoothed_health"], dtype=np.float32
        )[:, column],
        "raw_alarm": np.asarray(candidate["raw_alarm"], dtype=bool)[:, column],
        "normalized_interval_width": np.asarray(
            candidate["normalized_interval_width"], dtype=np.float32
        )[:, column],
        "interval_covered_fraction": np.asarray(
            candidate["interval_covered_fraction"], dtype=np.float32
        )[:, column],
        "standardized_interval_score": np.asarray(
            candidate["standardized_interval_score"], dtype=np.float32
        )[:, column],
    }
    atomic_save_npz(artifact_path, **artifact_arrays)
    artifact_hash = file_sha256(artifact_path)
    summary = {
        "case_key": str(case_row.case_key),
        "farm": str(case_row.farm),
        "asset_id": str(case_row.asset_id),
        "event_id": int(case_row.event_id),
        "outer_fold": int(selection["outer_fold"]),
        "configuration_id": configuration_id,
        "prediction_rows": int(len(artifact_arrays["timestamp_ns"])),
        "raw_alarms": int(artifact_arrays["raw_alarm"].sum()),
        "artifact_relative_path": artifact_path.relative_to(
            CELL8_SELECTED_ROOT
        ).as_posix(),
        "metadata_relative_path": metadata_path.relative_to(
            CELL8_SELECTED_ROOT
        ).as_posix(),
        "case_input_signature": input_signature,
        "artifact_sha256": artifact_hash,
    }
    save_json(
        {
            "cell8_version": CELL8_VERSION,
            "created_at_utc": utc_now(),
            "contract_sha256": CONTRACT_SHA256,
            "candidate_prediction_freeze_sha256": (
                CANDIDATE_PREDICTION_FREEZE_SHA256
            ),
            "case_input_signature": input_signature,
            "case_key": str(case_row.case_key),
            "farm": str(case_row.farm),
            "asset_id": str(case_row.asset_id),
            "event_id": int(case_row.event_id),
            "outer_fold": int(selection["outer_fold"]),
            "configuration": {
                "configuration_id": configuration_id,
                "coverage": float(selection["coverage"]),
                "minimum_r2": float(selection["minimum_r2"]),
                "evidence_head": str(selection["evidence_head"]),
                "smoothing_steps": int(selection["smoothing_steps"]),
                "threshold_quantile": float(selection["threshold_quantile"]),
            },
            "artifact_relative_path": summary["artifact_relative_path"],
            "artifact_sha256": artifact_hash,
            "copied_from_frozen_candidate_artifact": True,
            "outer_outcome_fields_used": False,
            "outer_performance_scored": False,
            "summary": summary,
        },
        metadata_path,
    )
    return summary


_selected_prediction_summaries_cell8: list[dict[str, Any]] = []
for _case_row_cell8 in CASE_REGISTRY.sort_values(
    ["farm", "event_id"], kind="stable"
).itertuples(index=False):
    _selected_prediction_summaries_cell8.append(
        cell8_freeze_selected_case(_case_row_cell8)
    )

SELECTED_OUTER_PREDICTION_REGISTRY = pd.DataFrame(
    _selected_prediction_summaries_cell8
).sort_values(["outer_fold", "farm", "event_id"], kind="stable").reset_index(
    drop=True
)
if len(SELECTED_OUTER_PREDICTION_REGISTRY) != DATASET.expected_total_cases:
    raise RuntimeError("Not all outer cases received a selected prediction.")
if SELECTED_OUTER_PREDICTION_REGISTRY["case_key"].duplicated().any():
    raise RuntimeError("A selected outer case was duplicated.")
if SELECTED_OUTER_PREDICTION_REGISTRY["outer_fold"].nunique() != (
    DATASET.expected_assets
):
    raise RuntimeError("Selected predictions do not cover all outer folds.")

# Remove the only ambient DataFrame containing event outcomes. Cell 9 must
# deliberately reopen the lockbox after verifying the selected-prediction receipt.
del _OUTCOME_LOCKBOX_CELL8
del CASE_CONFIGURATION_METRICS_CACHE


# =============================================================================
# 10. Freeze the selection receipt and concise notebook report
# =============================================================================

_cell8_component_hashes = {
    **_candidate_component_hashes_cell8,
    "outer_selection_grid_sha256": dataframe_sha256(
        OUTER_SELECTION_GRID,
        ("outer_fold", "configuration_id"),
    ),
    "inner_fold_score_audit_sha256": dataframe_sha256(
        INNER_FOLD_SCORE_AUDIT,
        ("outer_fold", "inner_fold", "configuration_id"),
    ),
    "outer_selected_configurations_sha256": dataframe_sha256(
        OUTER_SELECTED_CONFIGURATIONS,
        ("outer_fold",),
    ),
    "label_access_audit_sha256": dataframe_sha256(
        LABEL_ACCESS_AUDIT,
        ("outer_fold",),
    ),
    "selected_outer_prediction_registry_sha256": dataframe_sha256(
        SELECTED_OUTER_PREDICTION_REGISTRY,
        ("outer_fold", "farm", "event_id"),
    ),
    "selected_configuration_frequency_sha256": dataframe_sha256(
        SELECTED_CONFIGURATION_FREQUENCY,
        ("configuration_id",),
    ),
}
CELL8_RECEIPT_SHA256 = sha256_json(
    {
        "contract_sha256": CONTRACT_SHA256,
        "cell7_receipt_sha256": CELL7_RECEIPT_SHA256,
        "outcome_lockbox_sha256": OUTCOME_LOCKBOX_SHA256,
        "candidate_prediction_freeze_sha256": CANDIDATE_PREDICTION_FREEZE_SHA256,
        "cell8_implementation_sha256": CELL8_IMPLEMENTATION_SHA256,
        "component_hashes": _cell8_component_hashes,
        "outer_outcomes_scored": False,
    }
)
CELL8_RECEIPT_PATH = CELL8_QUALITY_ROOT / "cell8_nested_selection_receipt.json"
if CELL8_RECEIPT_PATH.exists():
    _existing_receipt_cell8 = json.loads(
        CELL8_RECEIPT_PATH.read_text(encoding="utf-8")
    )
    if _existing_receipt_cell8.get("cell8_receipt_sha256") != CELL8_RECEIPT_SHA256:
        raise RuntimeError("A different Cell 8 receipt exists. Do not overwrite it.")
    CELL8_STATE = "existing identical Cell 8 receipt verified"
    _write_cell8_receipt = False
else:
    CELL8_STATE = "new Cell 8 receipt frozen"
    _write_cell8_receipt = True

save_csv_atomic(
    OUTER_SELECTION_GRID,
    CELL8_QUALITY_ROOT / "outer_selection_grid.csv",
)
save_csv_atomic(
    INNER_FOLD_SCORE_AUDIT,
    CELL8_QUALITY_ROOT / "inner_fold_score_audit.csv",
)
save_csv_atomic(
    OUTER_SELECTED_CONFIGURATIONS,
    CELL8_QUALITY_ROOT / "outer_selected_configurations.csv",
)
save_csv_atomic(
    LABEL_ACCESS_AUDIT,
    CELL8_QUALITY_ROOT / "label_access_audit.csv",
)
save_csv_atomic(
    SELECTED_OUTER_PREDICTION_REGISTRY,
    CELL8_QUALITY_ROOT / "selected_outer_prediction_registry.csv",
)
save_csv_atomic(
    SELECTED_CONFIGURATION_FREQUENCY,
    CELL8_QUALITY_ROOT / "selected_configuration_frequency.csv",
)
save_json(
    CELL8_IMPLEMENTATION,
    CELL8_QUALITY_ROOT / "cell8_implementation.json",
)

if _write_cell8_receipt:
    save_json(
        {
            "cell8_version": CELL8_VERSION,
            "created_at_utc": utc_now(),
            "contract_sha256": CONTRACT_SHA256,
            "dataset_manifest_sha256": DATASET_MANIFEST_SHA256,
            "outcome_lockbox_sha256": OUTCOME_LOCKBOX_SHA256,
            "cell7_receipt_sha256": CELL7_RECEIPT_SHA256,
            "cell8_implementation_sha256": CELL8_IMPLEMENTATION_SHA256,
            "candidate_prediction_freeze_sha256": (
                CANDIDATE_PREDICTION_FREEZE_SHA256
            ),
            "component_hashes": _cell8_component_hashes,
            "cell8_receipt_sha256": CELL8_RECEIPT_SHA256,
            "candidate_predictions_frozen_before_outcomes": True,
            "outer_labels_used_for_selection": False,
            "outer_outcomes_scored": False,
            "pooled_outer_endpoint_reserved_for_cell9": True,
        },
        CELL8_RECEIPT_PATH,
    )

print("\n" + "=" * 92)
print("UC-RCF-NBM CELL 8 — NESTED ASSET SELECTION AND OUTER PREDICTION FREEZE")
print("=" * 92)
print("\nUNIVERSALLY DEPLOYABLE CONFIGURATION GRID")
display(UNIVERSAL_CONFIGURATION_GRID)
print("\nSELECTED CONFIGURATION FREQUENCY ACROSS OUTER FOLDS")
display(SELECTED_CONFIGURATION_FREQUENCY)
print("\nOUTER-FOLD SELECTION AUDIT")
display(
    OUTER_SELECTED_CONFIGURATIONS.loc[
        :,
        [
            "outer_fold",
            "outer_asset_id",
            "outer_farm",
            "configuration_id",
            "coverage",
            "minimum_r2",
            "evidence_head",
            "smoothing_steps",
            "threshold_quantile",
            "care_score",
            "normal_accuracy",
            "event_reliability_fbeta",
        ],
    ]
)

print("\n" + "-" * 92)
print(f"Cases with frozen candidates          : {len(CANDIDATE_PREDICTION_REGISTRY)}")
print(f"Universally available configurations  : {len(UNIVERSAL_CONFIGURATION_GRID)}")
print(f"Outer leave-one-asset-out folds        : {len(OUTER_SELECTED_CONFIGURATIONS)}")
print(f"Inner grouped folds per outer fold     : {EVALUATION.inner_splits}")
print(f"Selected outer prediction cases        : {len(SELECTED_OUTER_PREDICTION_REGISTRY)}")
print(f"Official CARE criticality threshold    : {CARE_CRITICALITY_THRESHOLD_CELL8}")
print(f"Candidate prediction freeze SHA-256    : {CANDIDATE_PREDICTION_FREEZE_SHA256}")
print(f"Cell 8 implementation SHA-256          : {CELL8_IMPLEMENTATION_SHA256}")
print(f"Cell 8 receipt SHA-256                 : {CELL8_RECEIPT_SHA256}")
print(f"Cell 8 state                           : {CELL8_STATE}")
print("Candidate predictions before outcomes : Yes")
print("Outer labels used for selection       : No")
print("Outer outcomes scored                 : No — reserved for Cell 9")
print("Official CARE reference checks        : PASS")
print("Artifact/prediction checks             : PASS")
print("=" * 92)
print("CELL 8 COMPLETED SUCCESSFULLY — OUTER PREDICTIONS FROZEN, UNSCORED")


Freezing label-free candidate predictions — Wind Farm A: 22 cases
  event   0: rows= 2838 configs= 36 alarm rate=0.0174
  event   3: rows= 3302 configs= 36 alarm rate=0.0163
  event  10: rows= 1269 configs= 36 alarm rate=0.0018
  event  13: rows= 3583 configs= 36 alarm rate=0.0013
  event  14: rows= 1901 configs= 36 alarm rate=0.0071
  event  17: rows= 2781 configs= 36 alarm rate=0.0057
  event  22: rows= 1148 configs= 36 alarm rate=0.0189
  event  24: rows= 2714 configs= 36 alarm rate=0.0061
  event  25: rows= 2423 configs= 36 alarm rate=0.0026
  event  26: rows= 1441 configs= 36 alarm rate=0.0154
  event  38: rows= 2544 configs= 36 alarm rate=0.0640
  event  40: rows= 4939 configs= 36 alarm rate=0.2465
  event  42: rows= 1583 configs= 36 alarm rate=0.0173
  event  45: rows= 1440 configs= 36 alarm rate=0.0009
  event  51: rows= 2393 configs= 36 alarm rate=0.0013
  event  68: rows= 2295 configs= 36 alarm rate=0.0315
  event  69: rows= 2593 configs= 36 alarm rate=0.0115
  event  71: row

,configuration_id,coverage,minimum_r2,evidence_head,smoothing_steps,threshold_quantile,available_cases
0,1,0.95,0.0,all_modellable,36,0.950,95
1,2,0.95,0.0,all_modellable,36,0.980,95
2,3,0.95,0.0,all_modellable,36,0.990,95
3,4,0.95,0.0,all_modellable,36,0.995,95
4,25,0.95,0.1,all_modellable,36,0.950,95
5,26,0.95,0.1,all_modellable,36,0.980,95
6,27,0.95,0.1,all_modellable,36,0.990,95
7,28,0.95,0.1,all_modellable,36,0.995,95
8,49,0.95,0.3,all_modellable,36,0.950,95
9,50,0.95,0.3,all_modellable,36,0.980,95



SELECTED CONFIGURATION FREQUENCY ACROSS OUTER FOLDS


,configuration_id,coverage,minimum_r2,evidence_head,smoothing_steps,threshold_quantile,outer_folds,median_development_care,minimum_development_care
0,121,0.98,0.3,all_modellable,36,0.95,36,0.580905,0.571035



OUTER-FOLD SELECTION AUDIT


,outer_fold,outer_asset_id,outer_farm,configuration_id,coverage,minimum_r2,evidence_head,smoothing_steps,threshold_quantile,care_score,normal_accuracy,event_reliability_fbeta
0,1,Wind Farm A::0,Wind Farm A,121,0.98,0.3,all_modellable,36,0.95,0.590946,0.828469,0.553097
1,2,Wind Farm A::10,Wind Farm A,121,0.98,0.3,all_modellable,36,0.95,0.582741,0.828523,0.530973
2,3,Wind Farm A::11,Wind Farm A,121,0.98,0.3,all_modellable,36,0.95,0.575940,0.822451,0.526316
3,4,Wind Farm A::13,Wind Farm A,121,0.98,0.3,all_modellable,36,0.95,0.587110,0.827752,0.541126
4,5,Wind Farm A::21,Wind Farm A,121,0.98,0.3,all_modellable,36,0.95,0.591150,0.828867,0.543478
5,6,Wind Farm B::0,Wind Farm B,121,0.98,0.3,all_modellable,36,0.95,0.580124,0.828731,0.536481
6,7,Wind Farm B::11,Wind Farm B,121,0.98,0.3,all_modellable,36,0.95,0.578761,0.829163,0.526316
7,8,Wind Farm B::12,Wind Farm B,121,0.98,0.3,all_modellable,36,0.95,0.580951,0.831853,0.535714
8,9,Wind Farm B::13,Wind Farm B,121,0.98,0.3,all_modellable,36,0.95,0.580563,0.831650,0.535714
9,10,Wind Farm B::14,Wind Farm B,121,0.98,0.3,all_modellable,36,0.95,0.579901,0.831833,0.535714



--------------------------------------------------------------------------------------------
Cases with frozen candidates          : 95
Universally available configurations  : 36
Outer leave-one-asset-out folds        : 36
Inner grouped folds per outer fold     : 5
Selected outer prediction cases        : 95
Official CARE criticality threshold    : 72
Candidate prediction freeze SHA-256    : f8382ed1a94f4a6a6d15cc16729445d4b2a64b9196aeff051158ad4ad69c0588
Cell 8 implementation SHA-256          : 8623bee69428a833ac3e7672af20812c093461fe1a2e0dee3e0d3ac44905d6e0
Cell 8 receipt SHA-256                 : 8d39da53929a8f81fdacd12b40d9fece044a20a4c9ef8ecf67c1feb0b005ee6d
Cell 8 state                           : new Cell 8 receipt frozen
Candidate predictions before outcomes : Yes
Outer labels used for selection       : No
Outer outcomes scored                 : No — reserved for Cell 9
Official CARE reference checks        : PASS
Artifact/prediction checks             : PASS
CELL 8 COMPLETED 

In [14]:
"""CELL 9 — single-use pooled outer evaluation.

Paste this complete file into the ninth UC-RCF-NBM notebook cell and run it
only after the corrected Cell 8 v1.0.1 has completed successfully.

This cell is the first and only stage that scores the frozen leave-one-asset-out
predictions on their held-out outcomes.  It verifies the complete Cell 8
selection receipt and every selected prediction artifact before opening the
outcome lockbox.  It then reports the pooled official CARE endpoint, the frozen
secondary endpoints, farm-stratified diagnostics, and 10,000-replicate
asset-cluster percentile bootstrap intervals.  No configuration is selected or
changed in this cell.
"""

from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print


# =============================================================================
# 0. Bind Cell 9 to the exact frozen Cell 8 handoff
# =============================================================================

EXPECTED_CONTRACT_SHA256_CELL9 = (
    "827641aecd8e807193ad193d64c274319756092faa53bdb5084f310f62041f49"
)
EXPECTED_DATASET_MANIFEST_SHA256_CELL9 = (
    "62484bab1219888aa1d0788965ecd77db2b85f0bbb9b476cd3240f4143026f1f"
)
EXPECTED_OUTCOME_LOCKBOX_SHA256_CELL9 = (
    "69c7c75ea8157e2e12cb61c596375168a90c73bef7ee283c618dab080a447e10"
)
EXPECTED_CELL8_IMPLEMENTATION_SHA256 = (
    "8623bee69428a833ac3e7672af20812c093461fe1a2e0dee3e0d3ac44905d6e0"
)
EXPECTED_CANDIDATE_PREDICTION_FREEZE_SHA256 = (
    "f8382ed1a94f4a6a6d15cc16729445d4b2a64b9196aeff051158ad4ad69c0588"
)
EXPECTED_CELL8_RECEIPT_SHA256 = (
    "8d39da53929a8f81fdacd12b40d9fece044a20a4c9ef8ecf67c1feb0b005ee6d"
)

_required_objects_cell9 = (
    "CONTRACT_SHA256",
    "DATASET_MANIFEST_SHA256",
    "OUTCOME_LOCKBOX_SHA256",
    "CELL8_IMPLEMENTATION_SHA256",
    "CANDIDATE_PREDICTION_FREEZE_SHA256",
    "CELL8_RECEIPT_SHA256",
    "CELL8_RECEIPT_PATH",
    "CANDIDATE_PREDICTION_FREEZE_PATH",
    "CELL8_SELECTED_ROOT",
    "CELL8_QUALITY_ROOT",
    "DATASET",
    "EVALUATION",
    "CARE",
    "REPRODUCIBILITY",
    "CASE_REGISTRY",
    "UNIVERSAL_CONFIGURATION_SHA256",
    "CANDIDATE_PREDICTION_REGISTRY",
    "OUTER_SELECTION_GRID",
    "INNER_FOLD_SCORE_AUDIT",
    "OUTER_SELECTED_CONFIGURATIONS",
    "LABEL_ACCESS_AUDIT",
    "SELECTED_OUTER_PREDICTION_REGISTRY",
    "SELECTED_CONFIGURATION_FREQUENCY",
    "cell8_load_outcome_lockbox",
    "cell8_case_configuration_metrics",
    "cell8_aggregate_care",
    "cell8_care_score",
    "file_sha256",
    "dataframe_sha256",
    "save_csv_atomic",
    "save_json",
    "sha256_json",
    "utc_now",
    "QUALITY_DIR",
)
_missing_objects_cell9 = [
    name for name in _required_objects_cell9 if name not in globals()
]
if _missing_objects_cell9:
    raise RuntimeError(
        "Run UC-RCF-NBM Cells 1–8 before Cell 9. Missing objects: "
        + ", ".join(_missing_objects_cell9)
    )

_observed_handoff_cell9 = {
    "contract": CONTRACT_SHA256,
    "dataset_manifest": DATASET_MANIFEST_SHA256,
    "outcome_lockbox": OUTCOME_LOCKBOX_SHA256,
    "cell8_implementation": CELL8_IMPLEMENTATION_SHA256,
    "candidate_prediction_freeze": CANDIDATE_PREDICTION_FREEZE_SHA256,
    "cell8_receipt": CELL8_RECEIPT_SHA256,
}
_expected_handoff_cell9 = {
    "contract": EXPECTED_CONTRACT_SHA256_CELL9,
    "dataset_manifest": EXPECTED_DATASET_MANIFEST_SHA256_CELL9,
    "outcome_lockbox": EXPECTED_OUTCOME_LOCKBOX_SHA256_CELL9,
    "cell8_implementation": EXPECTED_CELL8_IMPLEMENTATION_SHA256,
    "candidate_prediction_freeze": EXPECTED_CANDIDATE_PREDICTION_FREEZE_SHA256,
    "cell8_receipt": EXPECTED_CELL8_RECEIPT_SHA256,
}
if _observed_handoff_cell9 != _expected_handoff_cell9:
    raise RuntimeError(
        "Cell 9 is bound to the exact completed Cell 8 handoff. "
        f"Observed={_observed_handoff_cell9}, expected={_expected_handoff_cell9}."
    )

if EVALUATION.primary_endpoint != "pooled outer-fold official CARE":
    raise RuntimeError("The frozen primary endpoint has changed.")
if EVALUATION.bootstrap_unit != "asset":
    raise RuntimeError("Cell 9 requires asset-cluster bootstrap resampling.")
if int(EVALUATION.bootstrap_replicates) != 10_000:
    raise RuntimeError("Cell 9 is frozen to 10,000 bootstrap replicates.")
if int(EVALUATION.bootstrap_seed) != 42:
    raise RuntimeError("Cell 9 is frozen to bootstrap seed 42.")
if EVALUATION.outer_labels_available_during_selection:
    raise RuntimeError("Outer labels cannot have been available during selection.")
if not REPRODUCIBILITY.save_outer_predictions_before_label_scoring:
    raise RuntimeError("Outer predictions must be frozen before Cell 9 scoring.")
if not CARE.raw_timestamp_predictions_only:
    raise RuntimeError("Official CARE must use raw timestamp predictions.")


# =============================================================================
# 1. Verify the Cell 8 receipts, tables, and label-access audit before labels
# =============================================================================

_cell8_receipt_document_cell9 = json.loads(
    Path(CELL8_RECEIPT_PATH).read_text(encoding="utf-8")
)
_candidate_receipt_document_cell9 = json.loads(
    Path(CANDIDATE_PREDICTION_FREEZE_PATH).read_text(encoding="utf-8")
)
if _cell8_receipt_document_cell9.get("cell8_receipt_sha256") != CELL8_RECEIPT_SHA256:
    raise RuntimeError("The on-disk Cell 8 receipt does not match notebook state.")
if _candidate_receipt_document_cell9.get(
    "candidate_prediction_freeze_sha256"
) != CANDIDATE_PREDICTION_FREEZE_SHA256:
    raise RuntimeError("The on-disk candidate-freeze receipt does not match state.")
if _candidate_receipt_document_cell9.get("outcomes_accessed") is not False:
    raise RuntimeError("Candidate predictions were not frozen label-free.")
if _candidate_receipt_document_cell9.get("labels_used") is not False:
    raise RuntimeError("Candidate predictions used forbidden outcome labels.")
if _cell8_receipt_document_cell9.get("outer_labels_used_for_selection") is not False:
    raise RuntimeError("Cell 8 reports outer-label use during selection.")
if _cell8_receipt_document_cell9.get("outer_outcomes_scored") is not False:
    raise RuntimeError("Outer outcomes were scored before the reserved Cell 9 stage.")
if _cell8_receipt_document_cell9.get(
    "pooled_outer_endpoint_reserved_for_cell9"
) is not True:
    raise RuntimeError("The pooled endpoint was not reserved for Cell 9.")

_candidate_component_hashes_cell9 = {
    "universal_configuration_sha256": UNIVERSAL_CONFIGURATION_SHA256,
    "candidate_prediction_registry_sha256": dataframe_sha256(
        CANDIDATE_PREDICTION_REGISTRY,
        ("farm", "event_id"),
    ),
}
if _candidate_receipt_document_cell9.get("component_hashes") != (
    _candidate_component_hashes_cell9
):
    raise RuntimeError("Live candidate tables differ from their frozen receipt.")

_cell8_component_hashes_cell9 = {
    **_candidate_component_hashes_cell9,
    "outer_selection_grid_sha256": dataframe_sha256(
        OUTER_SELECTION_GRID,
        ("outer_fold", "configuration_id"),
    ),
    "inner_fold_score_audit_sha256": dataframe_sha256(
        INNER_FOLD_SCORE_AUDIT,
        ("outer_fold", "inner_fold", "configuration_id"),
    ),
    "outer_selected_configurations_sha256": dataframe_sha256(
        OUTER_SELECTED_CONFIGURATIONS,
        ("outer_fold",),
    ),
    "label_access_audit_sha256": dataframe_sha256(
        LABEL_ACCESS_AUDIT,
        ("outer_fold",),
    ),
    "selected_outer_prediction_registry_sha256": dataframe_sha256(
        SELECTED_OUTER_PREDICTION_REGISTRY,
        ("outer_fold", "farm", "event_id"),
    ),
    "selected_configuration_frequency_sha256": dataframe_sha256(
        SELECTED_CONFIGURATION_FREQUENCY,
        ("configuration_id",),
    ),
}
if _cell8_receipt_document_cell9.get("component_hashes") != (
    _cell8_component_hashes_cell9
):
    raise RuntimeError("Live Cell 8 tables differ from their frozen receipt.")
if OUTER_SELECTED_CONFIGURATIONS["outer_labels_used"].astype(bool).any():
    raise RuntimeError("An outer label entered configuration selection.")
if LABEL_ACCESS_AUDIT["outer_case_entered_objective"].astype(bool).any():
    raise RuntimeError("An outer case entered its own development objective.")
if len(OUTER_SELECTED_CONFIGURATIONS) != DATASET.expected_assets:
    raise RuntimeError("The selected-configuration table does not cover 36 assets.")
if len(SELECTED_OUTER_PREDICTION_REGISTRY) != DATASET.expected_total_cases:
    raise RuntimeError("The selected prediction registry does not cover 95 cases.")

CELL8_HANDOFF_VERIFIED_BEFORE_OUTCOME_ACCESS = True


# =============================================================================
# 2. Verify and load the immutable selected-prediction artifacts
# =============================================================================

_selected_prediction_arrays_cell9: dict[str, dict[str, np.ndarray]] = {}
_required_selected_arrays_cell9 = (
    "prediction_row_indices",
    "timestamp_ns",
    "segment_id",
    "normal_status",
    "configuration_id",
    "coverage",
    "minimum_r2",
    "smoothing_steps",
    "threshold_quantile",
    "threshold",
    "raw_health",
    "observed_weight_fraction",
    "smoothed_health",
    "raw_alarm",
    "normalized_interval_width",
    "interval_covered_fraction",
    "standardized_interval_score",
)

for _registry_row_cell9 in SELECTED_OUTER_PREDICTION_REGISTRY.sort_values(
    ["outer_fold", "farm", "event_id"], kind="stable"
).itertuples(index=False):
    _metadata_path_cell9 = (
        Path(CELL8_SELECTED_ROOT) / str(_registry_row_cell9.metadata_relative_path)
    )
    _artifact_path_cell9 = (
        Path(CELL8_SELECTED_ROOT) / str(_registry_row_cell9.artifact_relative_path)
    )
    if not _metadata_path_cell9.exists() or not _artifact_path_cell9.exists():
        raise FileNotFoundError(
            f"Selected prediction artifact is missing for {_registry_row_cell9.case_key}."
        )
    _artifact_hash_cell9 = file_sha256(_artifact_path_cell9)
    if _artifact_hash_cell9 != str(_registry_row_cell9.artifact_sha256):
        raise RuntimeError(
            f"Selected artifact hash failed for {_registry_row_cell9.case_key}."
        )
    _metadata_cell9 = json.loads(_metadata_path_cell9.read_text(encoding="utf-8"))
    if _metadata_cell9.get("artifact_sha256") != _artifact_hash_cell9:
        raise RuntimeError(
            f"Selected metadata hash failed for {_registry_row_cell9.case_key}."
        )
    if _metadata_cell9.get("outer_outcome_fields_used") is not False:
        raise RuntimeError("A selected prediction used outer outcome fields.")
    if _metadata_cell9.get("outer_performance_scored") is not False:
        raise RuntimeError("A selected prediction was scored before Cell 9.")
    with np.load(_artifact_path_cell9, allow_pickle=False) as _npz_cell9:
        _missing_arrays_cell9 = [
            name for name in _required_selected_arrays_cell9 if name not in _npz_cell9
        ]
        if _missing_arrays_cell9:
            raise RuntimeError(
                f"Selected artifact schema drift for {_registry_row_cell9.case_key}: "
                + ", ".join(_missing_arrays_cell9)
            )
        _arrays_cell9 = {
            name: np.asarray(_npz_cell9[name])
            for name in _required_selected_arrays_cell9
        }
    _rows_cell9 = len(_arrays_cell9["timestamp_ns"])
    for _vector_name_cell9 in (
        "prediction_row_indices",
        "segment_id",
        "normal_status",
        "raw_health",
        "observed_weight_fraction",
        "smoothed_health",
        "raw_alarm",
        "normalized_interval_width",
        "interval_covered_fraction",
        "standardized_interval_score",
    ):
        if _arrays_cell9[_vector_name_cell9].shape != (_rows_cell9,):
            raise RuntimeError(
                f"Selected vector shape drift for {_registry_row_cell9.case_key}."
            )
    if int(_arrays_cell9["configuration_id"].item()) != int(
        _registry_row_cell9.configuration_id
    ):
        raise RuntimeError("Selected artifact configuration drift was detected.")
    _reproduced_alarm_cell9 = np.isfinite(_arrays_cell9["smoothed_health"]) & (
        _arrays_cell9["smoothed_health"] > float(_arrays_cell9["threshold"].item())
    )
    if not np.array_equal(
        _arrays_cell9["raw_alarm"].astype(bool), _reproduced_alarm_cell9
    ):
        raise RuntimeError("Selected raw alarms cannot be reproduced exactly.")
    _selected_prediction_arrays_cell9[str(_registry_row_cell9.case_key)] = (
        _arrays_cell9
    )

if len(_selected_prediction_arrays_cell9) != DATASET.expected_total_cases:
    raise RuntimeError("Not all selected prediction artifacts were verified.")
SELECTED_ARTIFACTS_VERIFIED_BEFORE_OUTCOME_ACCESS = True


# =============================================================================
# 3. Open the lockbox after verification and score each held-out case exactly once
# =============================================================================

_OUTCOME_LOCKBOX_CELL9 = cell8_load_outcome_lockbox()
OUTER_OUTCOME_ACCESS_BEGAN_AFTER_CELL8_VERIFICATION = True


def cell9_selected_case_metrics(outcome_row: Any) -> dict[str, Any]:
    """Score one frozen selected artifact without changing its predictions."""
    case_key = str(outcome_row.case_key)
    arrays = _selected_prediction_arrays_cell9[case_key]
    registry_match = SELECTED_OUTER_PREDICTION_REGISTRY.loc[
        SELECTED_OUTER_PREDICTION_REGISTRY["case_key"].eq(case_key)
    ]
    if len(registry_match) != 1:
        raise RuntimeError(f"Selected registry mismatch for {case_key}.")
    registry_row = registry_match.iloc[0]
    configuration_id = int(arrays["configuration_id"].item())
    wrapped_artifact = {
        "configuration_ids": np.asarray([configuration_id], dtype=np.int32),
        "timestamp_ns": np.asarray(arrays["timestamp_ns"], dtype=np.int64),
        "segment_id": np.asarray(arrays["segment_id"], dtype=np.int32),
        "normal_status": np.asarray(arrays["normal_status"], dtype=bool),
        "raw_alarm": np.asarray(arrays["raw_alarm"], dtype=bool)[:, None],
        "normalized_interval_width": np.asarray(
            arrays["normalized_interval_width"], dtype=np.float64
        )[:, None],
        "interval_covered_fraction": np.asarray(
            arrays["interval_covered_fraction"], dtype=np.float64
        )[:, None],
        "standardized_interval_score": np.asarray(
            arrays["standardized_interval_score"], dtype=np.float64
        )[:, None],
    }
    result = cell8_case_configuration_metrics(
        outcome_row, wrapped_artifact, configuration_id
    )

    timestamps = np.asarray(arrays["timestamp_ns"], dtype=np.int64)
    timestamp_values = pd.to_datetime(timestamps, unit="ns")
    is_anomaly = bool(outcome_row.is_anomaly)
    if is_anomaly:
        ground_truth = np.asarray(
            (timestamp_values >= pd.Timestamp(outcome_row.event_start))
            & (timestamp_values <= pd.Timestamp(outcome_row.event_end)),
            dtype=bool,
        )
    else:
        ground_truth = np.zeros(len(timestamps), dtype=bool)
    actionable = (
        np.ones(len(timestamps), dtype=bool)
        if str(outcome_row.farm) == "Wind Farm A" and is_anomaly
        else np.asarray(arrays["normal_status"], dtype=bool)
    )
    accepted = actionable & np.isfinite(
        np.asarray(arrays["smoothed_health"], dtype=np.float64)
    )
    alarms = np.asarray(arrays["raw_alarm"], dtype=bool)
    selective_errors = int(np.sum(alarms[accepted] != ground_truth[accepted]))
    selective_observations = int(accepted.sum())
    nominal_coverage = float(arrays["coverage"].item())
    result.update(
        {
            "outer_fold": int(registry_row["outer_fold"]),
            "nominal_interval_coverage": nominal_coverage,
            "signed_interval_coverage_error": (
                float(result["mean_interval_covered_fraction"] - nominal_coverage)
                if np.isfinite(result["mean_interval_covered_fraction"])
                else np.nan
            ),
            "absolute_interval_coverage_error": (
                float(abs(result["mean_interval_covered_fraction"] - nominal_coverage))
                if np.isfinite(result["mean_interval_covered_fraction"])
                else np.nan
            ),
            "selective_observations": selective_observations,
            "selective_errors": selective_errors,
            "selective_prediction_coverage": float(
                selective_observations / int(actionable.sum())
            ),
            "selective_risk": (
                float(selective_errors / selective_observations)
                if selective_observations > 0
                else np.nan
            ),
            "mean_observed_weight_fraction": float(
                np.nanmean(
                    np.asarray(arrays["observed_weight_fraction"], dtype=np.float64)[
                        actionable
                    ]
                )
            ),
        }
    )
    return result


_outer_case_metric_records_cell9: list[dict[str, Any]] = []
for _outcome_row_cell9 in _OUTCOME_LOCKBOX_CELL9.sort_values(
    ["farm", "event_id"], kind="stable"
).itertuples(index=False):
    _outer_case_metric_records_cell9.append(
        cell9_selected_case_metrics(_outcome_row_cell9)
    )

OUTER_CASE_METRICS = pd.DataFrame(_outer_case_metric_records_cell9).sort_values(
    ["outer_fold", "farm", "event_id"], kind="stable"
).reset_index(drop=True)
if len(OUTER_CASE_METRICS) != DATASET.expected_total_cases:
    raise RuntimeError("Outer scoring did not produce 95 unique case metrics.")
if OUTER_CASE_METRICS["case_key"].duplicated().any():
    raise RuntimeError("A held-out case was scored more than once.")
if int(OUTER_CASE_METRICS["is_anomaly"].sum()) != DATASET.expected_anomaly_cases:
    raise RuntimeError("The scored anomaly-case count is not 45.")
if int((~OUTER_CASE_METRICS["is_anomaly"]).sum()) != DATASET.expected_normal_cases:
    raise RuntimeError("The scored normal-case count is not 50.")


# =============================================================================
# 4. Pooled official CARE and uncertainty-aware secondary endpoints
# =============================================================================


def cell9_complete_summary(case_metrics: pd.DataFrame) -> dict[str, Any]:
    base = cell8_aggregate_care(case_metrics)
    anomaly = case_metrics["is_anomaly"].astype(bool)
    normal = ~anomaly
    selective_observations = int(case_metrics["selective_observations"].sum())
    selective_errors = int(case_metrics["selective_errors"].sum())
    actionable = int(case_metrics["care_actionable_timestamps"].sum())
    normal_nominal = float(
        case_metrics.loc[normal, "nominal_interval_coverage"].mean()
    )
    base.update(
        {
            "cases": int(len(case_metrics)),
            "assets": int(case_metrics["asset_id"].nunique()),
            "anomaly_cases": int(anomaly.sum()),
            "normal_cases": int(normal.sum()),
            "event_sensitivity": float(
                base["event_true_positive"] / int(anomaly.sum())
            ),
            "anomaly_miss_count": int(base["event_false_negative"]),
            "total_event_alarm_count": int(case_metrics["event_alarm"].sum()),
            "nominal_normal_interval_coverage": normal_nominal,
            "signed_normal_coverage_error": float(
                base["normal_interval_coverage"] - normal_nominal
            ),
            "absolute_normal_coverage_error": float(
                abs(base["normal_interval_coverage"] - normal_nominal)
            ),
            "selective_prediction_coverage": float(
                selective_observations / actionable
            ),
            "selective_risk": (
                float(selective_errors / selective_observations)
                if selective_observations > 0
                else np.nan
            ),
            "selective_observations": selective_observations,
            "selective_errors": selective_errors,
        }
    )
    return base


POOLED_OUTER_CARE_SUMMARY = pd.DataFrame(
    [cell9_complete_summary(OUTER_CASE_METRICS)]
)
if not bool(POOLED_OUTER_CARE_SUMMARY.iloc[0]["care_defined"]):
    raise RuntimeError("The pooled official CARE endpoint is undefined.")
if not np.isfinite(float(POOLED_OUTER_CARE_SUMMARY.iloc[0]["care_score"])):
    raise RuntimeError("The pooled official CARE score is not finite.")

_farm_summary_records_cell9: list[dict[str, Any]] = []
for _farm_cell9, _farm_metrics_cell9 in OUTER_CASE_METRICS.groupby(
    "farm", sort=True
):
    _farm_record_cell9 = cell9_complete_summary(_farm_metrics_cell9)
    _farm_record_cell9["farm"] = str(_farm_cell9)
    _farm_summary_records_cell9.append(_farm_record_cell9)
OUTER_FARM_CARE_SUMMARY = pd.DataFrame(_farm_summary_records_cell9).sort_values(
    "farm", kind="stable"
).reset_index(drop=True)


# =============================================================================
# 5. Asset-level sufficient statistics and 10,000 cluster-bootstrap replicates
# =============================================================================


def cell9_asset_sufficient_statistics(case_metrics: pd.DataFrame) -> pd.DataFrame:
    records: list[dict[str, Any]] = []
    for asset_id, frame in case_metrics.groupby("asset_id", sort=True):
        anomaly = frame["is_anomaly"].astype(bool)
        normal = ~anomaly
        normal_coverage = frame.loc[normal, "mean_interval_covered_fraction"]
        normal_width = frame.loc[normal, "mean_normalized_interval_width"]
        normal_score = frame.loc[normal, "mean_standardized_interval_score"]
        normal_nominal = frame.loc[normal, "nominal_interval_coverage"]
        records.append(
            {
                "asset_id": str(asset_id),
                "farm": str(frame["farm"].iloc[0]),
                "cases": int(len(frame)),
                "anomaly_cases": int(anomaly.sum()),
                "normal_cases": int(normal.sum()),
                "coverage_fbeta_sum": float(
                    frame.loc[anomaly, "coverage_fbeta"].sum()
                ),
                "earliness_score_sum": float(
                    frame.loc[anomaly, "earliness_score"].sum()
                ),
                "normal_accuracy_sum": float(
                    frame.loc[normal, "point_accuracy"].sum()
                ),
                "event_true_positive": int(
                    frame.loc[anomaly, "event_alarm"].astype(bool).sum()
                ),
                "event_false_negative": int(
                    anomaly.sum()
                    - frame.loc[anomaly, "event_alarm"].astype(bool).sum()
                ),
                "event_false_positive": int(
                    frame.loc[normal, "event_alarm"].astype(bool).sum()
                ),
                "event_true_negative": int(
                    normal.sum() - frame.loc[normal, "event_alarm"].astype(bool).sum()
                ),
                "normal_interval_coverage_sum": float(normal_coverage.sum()),
                "normal_interval_coverage_count": int(normal_coverage.notna().sum()),
                "normal_interval_width_sum": float(normal_width.sum()),
                "normal_interval_width_count": int(normal_width.notna().sum()),
                "normal_interval_score_sum": float(normal_score.sum()),
                "normal_interval_score_count": int(normal_score.notna().sum()),
                "normal_nominal_coverage_sum": float(normal_nominal.sum()),
                "normal_nominal_coverage_count": int(normal_nominal.notna().sum()),
                "selective_observations": int(frame["selective_observations"].sum()),
                "selective_errors": int(frame["selective_errors"].sum()),
                "care_actionable_timestamps": int(
                    frame["care_actionable_timestamps"].sum()
                ),
            }
        )
    return pd.DataFrame(records).sort_values("asset_id", kind="stable").reset_index(
        drop=True
    )


OUTER_ASSET_SUFFICIENT_STATISTICS = cell9_asset_sufficient_statistics(
    OUTER_CASE_METRICS
)
if len(OUTER_ASSET_SUFFICIENT_STATISTICS) != DATASET.expected_assets:
    raise RuntimeError("Asset sufficient statistics do not cover all 36 assets.")


def cell9_asset_cluster_bootstrap(
    asset_statistics: pd.DataFrame,
    pooled_summary: pd.Series,
) -> pd.DataFrame:
    replicates = int(EVALUATION.bootstrap_replicates)
    asset_count = len(asset_statistics)
    rng = np.random.default_rng(int(EVALUATION.bootstrap_seed))
    draws = rng.integers(0, asset_count, size=(replicates, asset_count))
    weights = np.zeros((replicates, asset_count), dtype=np.int16)
    np.add.at(
        weights,
        (np.repeat(np.arange(replicates), asset_count), draws.reshape(-1)),
        1,
    )
    weights_float = weights.astype(np.float64)

    def weighted(column: str) -> np.ndarray:
        return weights_float @ asset_statistics[column].to_numpy(dtype=np.float64)

    def ratio(numerator: str, denominator: str) -> np.ndarray:
        num = weighted(numerator)
        den = weighted(denominator)
        return np.divide(
            num,
            den,
            out=np.full(replicates, np.nan, dtype=np.float64),
            where=den > 0,
        )

    anomaly_cases = weighted("anomaly_cases")
    normal_cases = weighted("normal_cases")
    coverage = np.divide(
        weighted("coverage_fbeta_sum"),
        anomaly_cases,
        out=np.full(replicates, np.nan),
        where=anomaly_cases > 0,
    )
    earliness = np.divide(
        weighted("earliness_score_sum"),
        anomaly_cases,
        out=np.full(replicates, np.nan),
        where=anomaly_cases > 0,
    )
    normal_accuracy = np.divide(
        weighted("normal_accuracy_sum"),
        normal_cases,
        out=np.full(replicates, np.nan),
        where=normal_cases > 0,
    )
    event_tp = weighted("event_true_positive")
    event_fp = weighted("event_false_positive")
    event_fn = weighted("event_false_negative")
    beta_squared = float(CARE.beta) ** 2
    reliability_numerator = (1.0 + beta_squared) * event_tp
    reliability_denominator = (
        reliability_numerator + beta_squared * event_fn + event_fp
    )
    reliability = np.divide(
        reliability_numerator,
        reliability_denominator,
        out=np.zeros(replicates, dtype=np.float64),
        where=reliability_denominator > 0,
    )
    event_alarm_count = event_tp + event_fp
    weighted_care = (
        coverage
        + earliness
        + reliability
        + 2.0 * normal_accuracy
    ) / 5.0
    care_score = weighted_care.copy()
    care_score[event_alarm_count == 0] = 0.0
    random_or_worse = (event_alarm_count > 0) & (normal_accuracy <= 0.5)
    care_score[random_or_worse] = normal_accuracy[random_or_worse]

    event_sensitivity = np.divide(
        event_tp,
        anomaly_cases,
        out=np.full(replicates, np.nan),
        where=anomaly_cases > 0,
    )
    normal_interval_coverage = ratio(
        "normal_interval_coverage_sum", "normal_interval_coverage_count"
    )
    normal_interval_width = ratio(
        "normal_interval_width_sum", "normal_interval_width_count"
    )
    normal_interval_score = ratio(
        "normal_interval_score_sum", "normal_interval_score_count"
    )
    nominal_coverage = ratio(
        "normal_nominal_coverage_sum", "normal_nominal_coverage_count"
    )
    absolute_coverage_error = np.abs(normal_interval_coverage - nominal_coverage)
    selective_risk = ratio("selective_errors", "selective_observations")
    selective_coverage = ratio(
        "selective_observations", "care_actionable_timestamps"
    )

    metric_values = {
        "care_score": care_score,
        "coverage_fbeta": coverage,
        "earliness_weighted_score": earliness,
        "event_reliability_fbeta": reliability,
        "normal_accuracy": normal_accuracy,
        "event_sensitivity": event_sensitivity,
        "event_false_alarm_count": event_fp,
        "normal_interval_coverage": normal_interval_coverage,
        "mean_normalized_normal_interval_width": normal_interval_width,
        "absolute_normal_coverage_error": absolute_coverage_error,
        "normal_interval_score": normal_interval_score,
        "selective_prediction_coverage": selective_coverage,
        "selective_risk": selective_risk,
    }
    direction = {
        "care_score": "higher_is_better",
        "coverage_fbeta": "higher_is_better",
        "earliness_weighted_score": "higher_is_better",
        "event_reliability_fbeta": "higher_is_better",
        "normal_accuracy": "higher_is_better",
        "event_sensitivity": "higher_is_better",
        "event_false_alarm_count": "lower_is_better",
        "normal_interval_coverage": "calibration_target",
        "mean_normalized_normal_interval_width": "lower_is_better",
        "absolute_normal_coverage_error": "lower_is_better",
        "normal_interval_score": "lower_is_better",
        "selective_prediction_coverage": "higher_is_better",
        "selective_risk": "lower_is_better",
    }
    records: list[dict[str, Any]] = []
    for metric, values in metric_values.items():
        finite = values[np.isfinite(values)]
        if len(finite) < int(0.99 * replicates):
            raise RuntimeError(
                f"Too few valid cluster-bootstrap replicates for {metric}: {len(finite)}."
            )
        records.append(
            {
                "metric": metric,
                "estimate": float(pooled_summary[metric]),
                "bootstrap_mean": float(np.mean(finite)),
                "ci_lower_2p5": float(np.quantile(finite, 0.025)),
                "ci_upper_97p5": float(np.quantile(finite, 0.975)),
                "valid_replicates": int(len(finite)),
                "bootstrap_replicates": replicates,
                "bootstrap_unit": "asset",
                "confidence_interval": "two-sided percentile 95%",
                "direction": direction[metric],
            }
        )
    return pd.DataFrame(records)


ASSET_BOOTSTRAP_CONFIDENCE_INTERVALS = cell9_asset_cluster_bootstrap(
    OUTER_ASSET_SUFFICIENT_STATISTICS,
    POOLED_OUTER_CARE_SUMMARY.iloc[0],
)


# =============================================================================
# 6. Freeze the single-use endpoint receipt and publication-ready tables
# =============================================================================

CELL9_VERSION = "1.0.0"
CELL9_NAMESPACE = "cell9_single_use_outer_evaluation_v1_0_0"
CELL9_QUALITY_ROOT = Path(QUALITY_DIR) / CELL9_NAMESPACE
CELL9_QUALITY_ROOT.mkdir(parents=True, exist_ok=True)

CELL9_IMPLEMENTATION = {
    "version": CELL9_VERSION,
    "primary_endpoint": EVALUATION.primary_endpoint,
    "prediction_source": "immutable Cell 8 selected outer artifacts",
    "configuration_selection_in_cell9": False,
    "cell8_receipt_verified_before_outcome_access": True,
    "selected_artifacts_verified_before_outcome_access": True,
    "outer_case_scoring": "each held-out case exactly once",
    "official_care": {
        "implementation": "Cell 8 frozen reference implementation",
        "beta": float(CARE.beta),
        "component_weights": tuple(float(value) for value in CARE.component_weights),
        "criticality_threshold": int(CARE.criticality_threshold),
        "post_alarm_latching": False,
        "raw_timestamp_predictions_only": True,
    },
    "secondary_uncertainty_endpoints": {
        "normal_interval_coverage": "case-average empirical target coverage",
        "normal_interval_width": "case-average normalized interval width",
        "coverage_error": (
            "absolute difference between pooled empirical normal coverage and "
            "the selected nominal coverage"
        ),
        "interval_score": "case-average standardized interval score",
        "selective_risk": (
            "timestamp misclassification risk conditional on a finite smoothed "
            "health score; pooled by accepted timestamp"
        ),
        "selective_prediction_coverage": (
            "fraction of CARE-actionable timestamps with a finite smoothed health score"
        ),
    },
    "bootstrap": {
        "unit": EVALUATION.bootstrap_unit,
        "replicates": int(EVALUATION.bootstrap_replicates),
        "seed": int(EVALUATION.bootstrap_seed),
        "interval": "two-sided percentile 95%",
        "asset_multiplicity": "all cases for a sampled asset resampled together",
    },
    "multiple_comparison_adjustment": {
        "method_reserved": EVALUATION.multiple_comparison_adjustment,
        "applied_in_cell9": False,
        "reason": "single proposed-method endpoint; no baseline family tested here",
    },
}
CELL9_IMPLEMENTATION_SHA256 = sha256_json(CELL9_IMPLEMENTATION)

_cell9_component_hashes = {
    "outer_case_metrics_sha256": dataframe_sha256(
        OUTER_CASE_METRICS, ("outer_fold", "farm", "event_id")
    ),
    "pooled_outer_care_summary_sha256": dataframe_sha256(
        POOLED_OUTER_CARE_SUMMARY, ("care_score",)
    ),
    "outer_farm_care_summary_sha256": dataframe_sha256(
        OUTER_FARM_CARE_SUMMARY, ("farm",)
    ),
    "outer_asset_sufficient_statistics_sha256": dataframe_sha256(
        OUTER_ASSET_SUFFICIENT_STATISTICS, ("asset_id",)
    ),
    "asset_bootstrap_confidence_intervals_sha256": dataframe_sha256(
        ASSET_BOOTSTRAP_CONFIDENCE_INTERVALS, ("metric",)
    ),
}
CELL9_RECEIPT_SHA256 = sha256_json(
    {
        "contract_sha256": CONTRACT_SHA256,
        "outcome_lockbox_sha256": OUTCOME_LOCKBOX_SHA256,
        "cell8_receipt_sha256": CELL8_RECEIPT_SHA256,
        "candidate_prediction_freeze_sha256": CANDIDATE_PREDICTION_FREEZE_SHA256,
        "cell9_implementation_sha256": CELL9_IMPLEMENTATION_SHA256,
        "component_hashes": _cell9_component_hashes,
        "outer_predictions_changed": False,
        "primary_endpoint_scored": True,
    }
)
CELL9_RECEIPT_PATH = CELL9_QUALITY_ROOT / "cell9_outer_evaluation_receipt.json"
if CELL9_RECEIPT_PATH.exists():
    _existing_receipt_cell9 = json.loads(
        CELL9_RECEIPT_PATH.read_text(encoding="utf-8")
    )
    if _existing_receipt_cell9.get("cell9_receipt_sha256") != CELL9_RECEIPT_SHA256:
        raise RuntimeError(
            "A different Cell 9 evaluation receipt exists. Do not overwrite the "
            "registered outer endpoint."
        )
    CELL9_STATE = "existing identical Cell 9 endpoint verified"
    _write_cell9_receipt = False
else:
    CELL9_STATE = "new single-use Cell 9 outer endpoint frozen"
    _write_cell9_receipt = True

save_csv_atomic(
    OUTER_CASE_METRICS,
    CELL9_QUALITY_ROOT / "outer_case_metrics.csv",
)
save_csv_atomic(
    POOLED_OUTER_CARE_SUMMARY,
    CELL9_QUALITY_ROOT / "pooled_outer_care_summary.csv",
)
save_csv_atomic(
    OUTER_FARM_CARE_SUMMARY,
    CELL9_QUALITY_ROOT / "outer_farm_care_summary.csv",
)
save_csv_atomic(
    OUTER_ASSET_SUFFICIENT_STATISTICS,
    CELL9_QUALITY_ROOT / "outer_asset_sufficient_statistics.csv",
)
save_csv_atomic(
    ASSET_BOOTSTRAP_CONFIDENCE_INTERVALS,
    CELL9_QUALITY_ROOT / "asset_bootstrap_confidence_intervals.csv",
)
save_json(
    CELL9_IMPLEMENTATION,
    CELL9_QUALITY_ROOT / "cell9_implementation.json",
)

if _write_cell9_receipt:
    save_json(
        {
            "cell9_version": CELL9_VERSION,
            "created_at_utc": utc_now(),
            "contract_sha256": CONTRACT_SHA256,
            "dataset_manifest_sha256": DATASET_MANIFEST_SHA256,
            "outcome_lockbox_sha256": OUTCOME_LOCKBOX_SHA256,
            "cell8_implementation_sha256": CELL8_IMPLEMENTATION_SHA256,
            "candidate_prediction_freeze_sha256": (
                CANDIDATE_PREDICTION_FREEZE_SHA256
            ),
            "cell8_receipt_sha256": CELL8_RECEIPT_SHA256,
            "cell9_implementation_sha256": CELL9_IMPLEMENTATION_SHA256,
            "component_hashes": _cell9_component_hashes,
            "cell9_receipt_sha256": CELL9_RECEIPT_SHA256,
            "cell8_handoff_verified_before_outcome_access": True,
            "selected_artifacts_verified_before_outcome_access": True,
            "outer_predictions_changed": False,
            "outer_cases_scored_once": True,
            "primary_endpoint_scored": True,
            "configuration_selected_in_cell9": False,
            "bootstrap_unit": EVALUATION.bootstrap_unit,
            "bootstrap_replicates": int(EVALUATION.bootstrap_replicates),
            "bootstrap_seed": int(EVALUATION.bootstrap_seed),
        },
        CELL9_RECEIPT_PATH,
    )


# =============================================================================
# 7. Concise notebook report
# =============================================================================

_headline_columns_cell9 = [
    "care_score",
    "coverage_fbeta",
    "earliness_weighted_score",
    "event_reliability_fbeta",
    "normal_accuracy",
    "event_sensitivity",
    "event_false_alarm_count",
    "detected_anomaly_cases",
    "anomaly_miss_count",
    "median_lead_hours",
]
_uncertainty_columns_cell9 = [
    "nominal_normal_interval_coverage",
    "normal_interval_coverage",
    "absolute_normal_coverage_error",
    "mean_normalized_normal_interval_width",
    "normal_interval_score",
    "selective_prediction_coverage",
    "selective_risk",
]

print("\n" + "=" * 92)
print("UC-RCF-NBM CELL 9 — SINGLE-USE POOLED OUTER EVALUATION")
print("=" * 92)
print("\nPRIMARY POOLED OUTER-FOLD OFFICIAL CARE ENDPOINT")
display(POOLED_OUTER_CARE_SUMMARY.loc[:, _headline_columns_cell9])
print("\nUNCERTAINTY-AWARE SECONDARY ENDPOINTS")
display(POOLED_OUTER_CARE_SUMMARY.loc[:, _uncertainty_columns_cell9])
print("\nASSET-CLUSTER BOOTSTRAP 95% CONFIDENCE INTERVALS")
display(
    ASSET_BOOTSTRAP_CONFIDENCE_INTERVALS.loc[
        :,
        [
            "metric",
            "estimate",
            "ci_lower_2p5",
            "ci_upper_97p5",
            "valid_replicates",
            "direction",
        ],
    ]
)
print("\nFARM-STRATIFIED OUTER PERFORMANCE (DESCRIPTIVE)")
display(
    OUTER_FARM_CARE_SUMMARY.loc[
        :,
        [
            "farm",
            "cases",
            "anomaly_cases",
            "normal_cases",
            "care_score",
            "coverage_fbeta",
            "earliness_weighted_score",
            "event_reliability_fbeta",
            "normal_accuracy",
            "event_sensitivity",
            "event_false_alarm_count",
            "median_lead_hours",
        ],
    ]
)

_primary_cell9 = POOLED_OUTER_CARE_SUMMARY.iloc[0]
_care_ci_cell9 = ASSET_BOOTSTRAP_CONFIDENCE_INTERVALS.loc[
    ASSET_BOOTSTRAP_CONFIDENCE_INTERVALS["metric"].eq("care_score")
].iloc[0]
print("\n" + "-" * 76)
print(f"Held-out cases scored                 : {len(OUTER_CASE_METRICS)}")
print(f"Held-out assets                       : {OUTER_CASE_METRICS['asset_id'].nunique()}")
print(
    "Anomaly / normal cases              : "
    f"{int(OUTER_CASE_METRICS['is_anomaly'].sum())} / "
    f"{int((~OUTER_CASE_METRICS['is_anomaly']).sum())}"
)
print(f"Pooled official CARE                 : {float(_primary_cell9['care_score']):.6f}")
print(
    "Asset-bootstrap CARE 95% CI          : "
    f"[{float(_care_ci_cell9['ci_lower_2p5']):.6f}, "
    f"{float(_care_ci_cell9['ci_upper_97p5']):.6f}]"
)
print(f"Cell 8 handoff verified pre-outcome  : {CELL8_HANDOFF_VERIFIED_BEFORE_OUTCOME_ACCESS}")
print(f"Selected artifacts verified          : {len(_selected_prediction_arrays_cell9)} / 95")
print("Configurations selected in Cell 9   : No")
print("Outer predictions modified           : No")
print("Holm adjustment applied              : No — no baseline family tested")
print(f"Cell 9 implementation SHA-256        : {CELL9_IMPLEMENTATION_SHA256}")
print(f"Cell 9 receipt SHA-256               : {CELL9_RECEIPT_SHA256}")
print(f"Cell 9 state                         : {CELL9_STATE}")
print("=" * 92)
print("CELL 9 COMPLETED SUCCESSFULLY — POOLED OUTER ENDPOINT LOCKED")

# Keep outcome-bearing publication tables, but remove the raw lockbox and all
# in-memory timestamp arrays after scoring.  Predictions remain immutable on disk.
del _OUTCOME_LOCKBOX_CELL9
del _selected_prediction_arrays_cell9




UC-RCF-NBM CELL 9 — SINGLE-USE POOLED OUTER EVALUATION

PRIMARY POOLED OUTER-FOLD OFFICIAL CARE ENDPOINT


,care_score,coverage_fbeta,earliness_weighted_score,event_reliability_fbeta,normal_accuracy,event_sensitivity,event_false_alarm_count,detected_anomaly_cases,anomaly_miss_count,median_lead_hours
0,0.581494,0.393868,0.312809,0.536481,0.832156,0.555556,22,25,20,402.0



UNCERTAINTY-AWARE SECONDARY ENDPOINTS


,nominal_normal_interval_coverage,normal_interval_coverage,absolute_normal_coverage_error,mean_normalized_normal_interval_width,normal_interval_score,selective_prediction_coverage,selective_risk
0,0.98,0.983334,0.003334,7.326621e+17,7.326621e+17,0.996347,0.365771



ASSET-CLUSTER BOOTSTRAP 95% CONFIDENCE INTERVALS


,metric,estimate,ci_lower_2p5,ci_upper_97p5,valid_replicates,direction
0,care_score,5.814939e-01,0.531633,6.328289e-01,10000,higher_is_better
1,coverage_fbeta,3.938680e-01,0.309121,4.855180e-01,10000,higher_is_better
2,earliness_weighted_score,3.128086e-01,0.213002,4.290904e-01,10000,higher_is_better
3,event_reliability_fbeta,5.364807e-01,0.452754,6.150870e-01,10000,higher_is_better
4,normal_accuracy,8.321561e-01,0.760682,8.968272e-01,10000,higher_is_better
5,event_sensitivity,5.555556e-01,0.404255,7.272727e-01,10000,higher_is_better
6,event_false_alarm_count,2.200000e+01,15.000000,2.900000e+01,10000,lower_is_better
7,normal_interval_coverage,9.833344e-01,0.977069,9.879519e-01,10000,calibration_target
8,mean_normalized_normal_interval_width,7.326621e+17,230.357404,2.289569e+18,10000,lower_is_better
9,absolute_normal_coverage_error,3.334402e-03,0.000213,7.978221e-03,10000,lower_is_better



FARM-STRATIFIED OUTER PERFORMANCE (DESCRIPTIVE)


,farm,cases,anomaly_cases,normal_cases,care_score,coverage_fbeta,earliness_weighted_score,event_reliability_fbeta,normal_accuracy,event_sensitivity,event_false_alarm_count,median_lead_hours
0,Wind Farm A,22,12,10,0.506876,0.169852,0.047530,0.416667,0.950165,0.166667,1,417.750000
1,Wind Farm B,15,6,9,0.617884,0.447123,0.216171,0.600000,0.913064,1.000000,5,671.666667
2,Wind Farm C,58,27,31,0.601914,0.481596,0.452185,0.534591,0.770599,0.629630,16,325.833333



----------------------------------------------------------------------------
Held-out cases scored                 : 95
Held-out assets                       : 36
Anomaly / normal cases              : 45 / 50
Pooled official CARE                 : 0.581494
Asset-bootstrap CARE 95% CI          : [0.531633, 0.632829]
Cell 8 handoff verified pre-outcome  : True
Selected artifacts verified          : 95 / 95
Configurations selected in Cell 9   : No
Outer predictions modified           : No
Holm adjustment applied              : No — no baseline family tested
Cell 9 implementation SHA-256        : e89c89d6ea299be75309c575077c1e486ebe581c4c8e7017db1ef8003491b86a
Cell 9 receipt SHA-256               : f804fdf42c5156d399504eec8026e2bf65318457f32d414c0e65f0360b7ab235
Cell 9 state                         : new single-use Cell 9 outer endpoint frozen
CELL 9 COMPLETED SUCCESSFULLY — POOLED OUTER ENDPOINT LOCKED


In [15]:
"""CELL 10 — post-lock diagnostic audit.

Paste this complete file into the tenth UC-RCF-NBM notebook cell and run it
only after Cell 9 has locked the pooled outer endpoint.

This cell is diagnostic only.  It verifies the frozen Cell 9 receipt before
reopening outcome metadata, never changes a prediction or configuration, and
does not redefine the primary endpoint.  It audits alarm timing, missed events,
false-alarm concentration, farm heterogeneity, and the numerical source of the
very large normalized conformal interval width/score.
"""

from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print


# =============================================================================
# 0. Bind the audit to the exact locked Cell 9 endpoint
# =============================================================================

EXPECTED_CONTRACT_SHA256_CELL10 = (
    "827641aecd8e807193ad193d64c274319756092faa53bdb5084f310f62041f49"
)
EXPECTED_OUTCOME_LOCKBOX_SHA256_CELL10 = (
    "69c7c75ea8157e2e12cb61c596375168a90c73bef7ee283c618dab080a447e10"
)
EXPECTED_CELL9_IMPLEMENTATION_SHA256 = (
    "e89c89d6ea299be75309c575077c1e486ebe581c4c8e7017db1ef8003491b86a"
)
EXPECTED_CELL9_RECEIPT_SHA256 = (
    "f804fdf42c5156d399504eec8026e2bf65318457f32d414c0e65f0360b7ab235"
)

_required_objects_cell10 = (
    "CONTRACT_SHA256",
    "DATASET_MANIFEST_SHA256",
    "OUTCOME_LOCKBOX_SHA256",
    "CELL9_IMPLEMENTATION_SHA256",
    "CELL9_RECEIPT_SHA256",
    "CELL9_RECEIPT_PATH",
    "CELL8_SELECTED_ROOT",
    "DATASET",
    "CARE",
    "EVALUATION",
    "OUTER_CASE_METRICS",
    "POOLED_OUTER_CARE_SUMMARY",
    "OUTER_FARM_CARE_SUMMARY",
    "OUTER_ASSET_SUFFICIENT_STATISTICS",
    "ASSET_BOOTSTRAP_CONFIDENCE_INTERVALS",
    "OUTER_SELECTED_CONFIGURATIONS",
    "SELECTED_OUTER_PREDICTION_REGISTRY",
    "cell8_load_outcome_lockbox",
    "cell8_criticality_trajectory",
    "load_case_cache",
    "load_case_mean_model",
    "load_case_health_model",
    "predict_case_conformal_interval",
    "exact_health_grid_index",
    "cell8_weighted_row_mean",
    "file_sha256",
    "dataframe_sha256",
    "save_csv_atomic",
    "save_json",
    "sha256_json",
    "utc_now",
    "QUALITY_DIR",
)
_missing_objects_cell10 = [
    name for name in _required_objects_cell10 if name not in globals()
]
if _missing_objects_cell10:
    raise RuntimeError(
        "Run UC-RCF-NBM Cells 1–9 before Cell 10. Missing objects: "
        + ", ".join(_missing_objects_cell10)
    )

_observed_handoff_cell10 = {
    "contract": CONTRACT_SHA256,
    "outcome_lockbox": OUTCOME_LOCKBOX_SHA256,
    "cell9_implementation": CELL9_IMPLEMENTATION_SHA256,
    "cell9_receipt": CELL9_RECEIPT_SHA256,
}
_expected_handoff_cell10 = {
    "contract": EXPECTED_CONTRACT_SHA256_CELL10,
    "outcome_lockbox": EXPECTED_OUTCOME_LOCKBOX_SHA256_CELL10,
    "cell9_implementation": EXPECTED_CELL9_IMPLEMENTATION_SHA256,
    "cell9_receipt": EXPECTED_CELL9_RECEIPT_SHA256,
}
if _observed_handoff_cell10 != _expected_handoff_cell10:
    raise RuntimeError(
        "Cell 10 is bound to the exact locked Cell 9 endpoint. "
        f"Observed={_observed_handoff_cell10}, expected={_expected_handoff_cell10}."
    )

_cell9_receipt_document_cell10 = json.loads(
    Path(CELL9_RECEIPT_PATH).read_text(encoding="utf-8")
)
if _cell9_receipt_document_cell10.get("cell9_receipt_sha256") != (
    CELL9_RECEIPT_SHA256
):
    raise RuntimeError("The on-disk Cell 9 receipt does not match notebook state.")
if _cell9_receipt_document_cell10.get("primary_endpoint_scored") is not True:
    raise RuntimeError("The Cell 9 primary endpoint was not frozen.")
if _cell9_receipt_document_cell10.get("outer_predictions_changed") is not False:
    raise RuntimeError("Cell 9 reports a change to frozen predictions.")
if _cell9_receipt_document_cell10.get("configuration_selected_in_cell9") is not False:
    raise RuntimeError("Cell 9 reports post-outcome configuration selection.")

_live_cell9_component_hashes_cell10 = {
    "outer_case_metrics_sha256": dataframe_sha256(
        OUTER_CASE_METRICS, ("outer_fold", "farm", "event_id")
    ),
    "pooled_outer_care_summary_sha256": dataframe_sha256(
        POOLED_OUTER_CARE_SUMMARY, ("care_score",)
    ),
    "outer_farm_care_summary_sha256": dataframe_sha256(
        OUTER_FARM_CARE_SUMMARY, ("farm",)
    ),
    "outer_asset_sufficient_statistics_sha256": dataframe_sha256(
        OUTER_ASSET_SUFFICIENT_STATISTICS, ("asset_id",)
    ),
    "asset_bootstrap_confidence_intervals_sha256": dataframe_sha256(
        ASSET_BOOTSTRAP_CONFIDENCE_INTERVALS, ("metric",)
    ),
}
if _cell9_receipt_document_cell10.get("component_hashes") != (
    _live_cell9_component_hashes_cell10
):
    raise RuntimeError("Live Cell 9 tables differ from their frozen receipt.")
CELL9_ENDPOINT_VERIFIED_BEFORE_DIAGNOSTICS = True


# =============================================================================
# 1. Frozen diagnostic definitions (not selection or endpoint rules)
# =============================================================================

CELL10_VERSION = "1.0.1"
CELL10_NAMESPACE = "cell10_post_lock_diagnostic_audit_v1_0_1"
CELL10_QUALITY_ROOT = Path(QUALITY_DIR) / CELL10_NAMESPACE
CELL10_QUALITY_ROOT.mkdir(parents=True, exist_ok=True)

EXPECTED_STEP_NS_CELL10 = int(
    pd.Timedelta(minutes=DATASET.sampling_minutes).value
)
INTERVAL_PATHOLOGY_WIDTH = 1.0e6
EXTREME_INTERVAL_WIDTH = 1.0e12
TINY_TARGET_SCALE = 1.0e-6
VERY_TINY_TARGET_SCALE = 1.0e-9
TARGET_DIAGNOSTIC_NORMAL_CASES = 12

CELL10_IMPLEMENTATION = {
    "version": CELL10_VERSION,
    "role": "post-lock descriptive diagnostic audit",
    "cell9_endpoint_verified_before_diagnostics": True,
    "prediction_or_configuration_changes": False,
    "primary_endpoint_recalculated_or_redefined": False,
    "diagnostics": (
        "case alarm timing relative to event onset and end",
        "miss and false-alarm concentration",
        "farm heterogeneity",
        "case-level interval width/score tail concentration",
        "active-target scale census",
        "target-level interval decomposition for the 12 widest normal cases",
    ),
    "descriptive_flags": {
        "interval_pathology_width": INTERVAL_PATHOLOGY_WIDTH,
        "extreme_interval_width": EXTREME_INTERVAL_WIDTH,
        "tiny_target_scale": TINY_TARGET_SCALE,
        "very_tiny_target_scale": VERY_TINY_TARGET_SCALE,
        "not_inferential_thresholds": True,
    },
    "interpretation_guardrails": {
        "cell9_care_remains_primary": True,
        "lead_time_reference": "hours from first criticality crossing to event end",
        "early_warning_reference": "hours from first criticality crossing to event onset",
        "diagnostics_cannot_feed_back_into_experiment_01": True,
        "method_changes_require_new_experiment_id": True,
    },
}
CELL10_IMPLEMENTATION_SHA256 = sha256_json(CELL10_IMPLEMENTATION)


# =============================================================================
# 2. Load outcome boundaries only after the Cell 9 receipt is verified
# =============================================================================

_OUTCOME_LOCKBOX_CELL10 = cell8_load_outcome_lockbox()
OUTCOMES_OPENED_AFTER_CELL9_VERIFICATION = True


def cell10_selected_arrays(case_key: str) -> dict[str, np.ndarray]:
    match = SELECTED_OUTER_PREDICTION_REGISTRY.loc[
        SELECTED_OUTER_PREDICTION_REGISTRY["case_key"].eq(case_key)
    ]
    if len(match) != 1:
        raise KeyError(f"Unknown or duplicate selected case: {case_key}")
    row = match.iloc[0]
    path = Path(CELL8_SELECTED_ROOT) / str(row["artifact_relative_path"])
    if file_sha256(path) != str(row["artifact_sha256"]):
        raise RuntimeError(f"Selected artifact hash failed for {case_key}.")
    with np.load(path, allow_pickle=False) as archive:
        return {name: np.asarray(archive[name]) for name in archive.files}


def cell10_finite_statistic(
    values: np.ndarray,
    statistic: str,
    quantile: float | None = None,
) -> float:
    finite = np.asarray(values, dtype=np.float64)
    finite = finite[np.isfinite(finite)]
    if len(finite) == 0:
        return np.nan
    if statistic == "mean":
        return float(np.mean(finite))
    if statistic == "median":
        return float(np.median(finite))
    if statistic == "max":
        return float(np.max(finite))
    if statistic == "quantile" and quantile is not None:
        return float(np.quantile(finite, quantile))
    raise ValueError(f"Unknown statistic: {statistic}")


def cell10_longest_alarm_run(
    alarm: np.ndarray,
    actionable: np.ndarray,
    timestamps: np.ndarray,
    segments: np.ndarray,
) -> int:
    longest = 0
    current = 0
    for index in range(len(alarm)):
        continuous = index == 0 or (
            segments[index] == segments[index - 1]
            and timestamps[index] - timestamps[index - 1] == EXPECTED_STEP_NS_CELL10
        )
        if not continuous:
            current = 0
        if bool(actionable[index]) and bool(alarm[index]):
            current += 1
            longest = max(longest, current)
        else:
            current = 0
    return int(longest)


# =============================================================================
# 3. Case-level alarm timing, misses, false alarms, and interval tails
# =============================================================================

_case_diagnostic_records_cell10: list[dict[str, Any]] = []
for _outcome_cell10 in _OUTCOME_LOCKBOX_CELL10.sort_values(
    ["farm", "event_id"], kind="stable"
).itertuples(index=False):
    _case_key_cell10 = str(_outcome_cell10.case_key)
    _arrays_cell10 = cell10_selected_arrays(_case_key_cell10)
    _timestamps_cell10 = np.asarray(_arrays_cell10["timestamp_ns"], dtype=np.int64)
    _segments_cell10 = np.asarray(_arrays_cell10["segment_id"], dtype=np.int32)
    _normal_status_cell10 = np.asarray(_arrays_cell10["normal_status"], dtype=bool)
    _alarms_cell10 = np.asarray(_arrays_cell10["raw_alarm"], dtype=bool)
    _smoothed_cell10 = np.asarray(_arrays_cell10["smoothed_health"], dtype=np.float64)
    _width_cell10 = np.asarray(
        _arrays_cell10["normalized_interval_width"], dtype=np.float64
    )
    _score_cell10 = np.asarray(
        _arrays_cell10["standardized_interval_score"], dtype=np.float64
    )
    _coverage_cell10 = np.asarray(
        _arrays_cell10["interval_covered_fraction"], dtype=np.float64
    )
    _is_anomaly_cell10 = bool(_outcome_cell10.is_anomaly)
    _actionable_cell10 = (
        np.ones(len(_timestamps_cell10), dtype=bool)
        if str(_outcome_cell10.farm) == "Wind Farm A" and _is_anomaly_cell10
        else _normal_status_cell10.copy()
    )
    if not _actionable_cell10.any():
        raise RuntimeError(f"No actionable rows for {_case_key_cell10}.")

    if _is_anomaly_cell10:
        _event_start_ns_cell10 = int(pd.Timestamp(_outcome_cell10.event_start).value)
        _event_end_ns_cell10 = int(pd.Timestamp(_outcome_cell10.event_end).value)
        _scope_cell10 = _timestamps_cell10 <= _event_end_ns_cell10
        _pre_onset_cell10 = _timestamps_cell10 < _event_start_ns_cell10
        _within_event_cell10 = (
            (_timestamps_cell10 >= _event_start_ns_cell10)
            & (_timestamps_cell10 <= _event_end_ns_cell10)
        )
        _post_event_cell10 = _timestamps_cell10 > _event_end_ns_cell10
    else:
        _event_start_ns_cell10 = None
        _event_end_ns_cell10 = None
        _scope_cell10 = np.ones(len(_timestamps_cell10), dtype=bool)
        _pre_onset_cell10 = np.zeros(len(_timestamps_cell10), dtype=bool)
        _within_event_cell10 = np.zeros(len(_timestamps_cell10), dtype=bool)
        _post_event_cell10 = np.zeros(len(_timestamps_cell10), dtype=bool)

    _criticality_cell10 = cell8_criticality_trajectory(
        _alarms_cell10[_scope_cell10],
        _actionable_cell10[_scope_cell10],
        _timestamps_cell10[_scope_cell10],
        _segments_cell10[_scope_cell10],
    )
    _crossing_local_cell10 = np.flatnonzero(
        _criticality_cell10 >= int(CARE.criticality_threshold)
    )
    _crossing_ns_cell10: int | None = None
    if len(_crossing_local_cell10):
        _crossing_ns_cell10 = int(
            _timestamps_cell10[_scope_cell10][int(_crossing_local_cell10[0])]
        )
    _event_alarm_cell10 = _crossing_ns_cell10 is not None

    if _is_anomaly_cell10 and _event_alarm_cell10:
        _hours_from_onset_cell10 = float(
            (_crossing_ns_cell10 - _event_start_ns_cell10) / 3.6e12
        )
        _hours_to_end_cell10 = float(
            (_event_end_ns_cell10 - _crossing_ns_cell10) / 3.6e12
        )
        _detection_phase_cell10 = (
            "pre_onset" if _crossing_ns_cell10 < _event_start_ns_cell10 else "within_event"
        )
    elif _is_anomaly_cell10:
        _hours_from_onset_cell10 = np.nan
        _hours_to_end_cell10 = np.nan
        _detection_phase_cell10 = "missed"
    else:
        _hours_from_onset_cell10 = np.nan
        _hours_to_end_cell10 = np.nan
        _detection_phase_cell10 = "false_alarm" if _event_alarm_cell10 else "correct_normal"

    _metric_match_cell10 = OUTER_CASE_METRICS.loc[
        OUTER_CASE_METRICS["case_key"].eq(_case_key_cell10)
    ]
    if len(_metric_match_cell10) != 1:
        raise RuntimeError(f"Cell 9 metric mismatch for {_case_key_cell10}.")
    _metric_cell10 = _metric_match_cell10.iloc[0]
    if bool(_metric_cell10["event_alarm"]) != _event_alarm_cell10:
        raise RuntimeError(f"Cell 9 event alarm cannot be reproduced for {_case_key_cell10}.")
    if int(_metric_cell10["maximum_criticality"]) != int(_criticality_cell10.max()):
        raise RuntimeError(f"Cell 9 criticality cannot be reproduced for {_case_key_cell10}.")

    _interval_mask_cell10 = _actionable_cell10 & np.isfinite(_width_cell10)
    _case_diagnostic_records_cell10.append(
        {
            "case_key": _case_key_cell10,
            "farm": str(_outcome_cell10.farm),
            "asset_id": str(_outcome_cell10.asset_id),
            "event_id": int(_outcome_cell10.event_id),
            "is_anomaly": _is_anomaly_cell10,
            "configuration_id": int(_arrays_cell10["configuration_id"].item()),
            "prediction_rows": int(len(_timestamps_cell10)),
            "actionable_rows": int(_actionable_cell10.sum()),
            "finite_health_rows": int(
                (_actionable_cell10 & np.isfinite(_smoothed_cell10)).sum()
            ),
            "raw_alarm_rows": int((_actionable_cell10 & _alarms_cell10).sum()),
            "raw_alarm_fraction": float(
                (_actionable_cell10 & _alarms_cell10).sum()
                / _actionable_cell10.sum()
            ),
            "longest_raw_alarm_run_steps": cell10_longest_alarm_run(
                _alarms_cell10,
                _actionable_cell10,
                _timestamps_cell10,
                _segments_cell10,
            ),
            "maximum_criticality": int(_criticality_cell10.max()),
            "criticality_margin_to_alarm": int(
                int(_criticality_cell10.max()) - int(CARE.criticality_threshold)
            ),
            "event_alarm": _event_alarm_cell10,
            "detection_phase": _detection_phase_cell10,
            "first_crossing_timestamp": (
                pd.Timestamp(_crossing_ns_cell10).isoformat()
                if _crossing_ns_cell10 is not None
                else ""
            ),
            "hours_from_event_onset": _hours_from_onset_cell10,
            "hours_to_event_end": _hours_to_end_cell10,
            "pre_onset_raw_alarm_rows": int(
                (_actionable_cell10 & _alarms_cell10 & _pre_onset_cell10).sum()
            ),
            "within_event_raw_alarm_rows": int(
                (_actionable_cell10 & _alarms_cell10 & _within_event_cell10).sum()
            ),
            "post_event_raw_alarm_rows": int(
                (_actionable_cell10 & _alarms_cell10 & _post_event_cell10).sum()
            ),
            "mean_normalized_interval_width": cell10_finite_statistic(
                _width_cell10[_interval_mask_cell10], "mean"
            ),
            "median_normalized_interval_width": cell10_finite_statistic(
                _width_cell10[_interval_mask_cell10], "median"
            ),
            "p95_normalized_interval_width": cell10_finite_statistic(
                _width_cell10[_interval_mask_cell10], "quantile", 0.95
            ),
            "p99_normalized_interval_width": cell10_finite_statistic(
                _width_cell10[_interval_mask_cell10], "quantile", 0.99
            ),
            "maximum_normalized_interval_width": cell10_finite_statistic(
                _width_cell10[_interval_mask_cell10], "max"
            ),
            "fraction_width_above_1e6": float(
                np.mean(_width_cell10[_interval_mask_cell10] > INTERVAL_PATHOLOGY_WIDTH)
            ),
            "fraction_width_above_1e12": float(
                np.mean(_width_cell10[_interval_mask_cell10] > EXTREME_INTERVAL_WIDTH)
            ),
            "mean_standardized_interval_score": cell10_finite_statistic(
                _score_cell10[_actionable_cell10], "mean"
            ),
            "empirical_interval_coverage": cell10_finite_statistic(
                _coverage_cell10[_actionable_cell10], "mean"
            ),
        }
    )

CASE_DIAGNOSTIC_AUDIT = pd.DataFrame(_case_diagnostic_records_cell10).sort_values(
    ["farm", "event_id"], kind="stable"
).reset_index(drop=True)
if len(CASE_DIAGNOSTIC_AUDIT) != DATASET.expected_total_cases:
    raise RuntimeError("The diagnostic audit does not cover all 95 cases.")


# =============================================================================
# 4. Detection phase, farm heterogeneity, and false-alarm concentration
# =============================================================================

_anomaly_diagnostics_cell10 = CASE_DIAGNOSTIC_AUDIT.loc[
    CASE_DIAGNOSTIC_AUDIT["is_anomaly"].astype(bool)
].copy()
_normal_diagnostics_cell10 = CASE_DIAGNOSTIC_AUDIT.loc[
    ~CASE_DIAGNOSTIC_AUDIT["is_anomaly"].astype(bool)
].copy()

_phase_order_cell10 = ("pre_onset", "within_event", "missed")
_phase_records_cell10: list[dict[str, Any]] = []
for _phase_cell10 in _phase_order_cell10:
    _phase_frame_cell10 = _anomaly_diagnostics_cell10.loc[
        _anomaly_diagnostics_cell10["detection_phase"].eq(_phase_cell10)
    ]
    _phase_records_cell10.append(
        {
            "detection_phase": _phase_cell10,
            "anomaly_cases": int(len(_phase_frame_cell10)),
            "fraction_of_anomaly_cases": float(
                len(_phase_frame_cell10) / DATASET.expected_anomaly_cases
            ),
            "median_hours_from_event_onset": cell10_finite_statistic(
                _phase_frame_cell10["hours_from_event_onset"].to_numpy(), "median"
            ),
            "median_hours_to_event_end": cell10_finite_statistic(
                _phase_frame_cell10["hours_to_event_end"].to_numpy(), "median"
            ),
        }
    )
DETECTION_PHASE_SUMMARY = pd.DataFrame(_phase_records_cell10)

_farm_diagnostic_records_cell10: list[dict[str, Any]] = []
for _farm_cell10, _farm_frame_cell10 in CASE_DIAGNOSTIC_AUDIT.groupby(
    "farm", sort=True
):
    _farm_anomaly_cell10 = _farm_frame_cell10.loc[
        _farm_frame_cell10["is_anomaly"].astype(bool)
    ]
    _farm_normal_cell10 = _farm_frame_cell10.loc[
        ~_farm_frame_cell10["is_anomaly"].astype(bool)
    ]
    _farm_care_cell10 = OUTER_FARM_CARE_SUMMARY.loc[
        OUTER_FARM_CARE_SUMMARY["farm"].eq(_farm_cell10)
    ].iloc[0]
    _farm_diagnostic_records_cell10.append(
        {
            "farm": str(_farm_cell10),
            "cases": int(len(_farm_frame_cell10)),
            "anomaly_cases": int(len(_farm_anomaly_cell10)),
            "normal_cases": int(len(_farm_normal_cell10)),
            "care_score": float(_farm_care_cell10["care_score"]),
            "event_sensitivity": float(_farm_care_cell10["event_sensitivity"]),
            "pre_onset_detections": int(
                _farm_anomaly_cell10["detection_phase"].eq("pre_onset").sum()
            ),
            "within_event_detections": int(
                _farm_anomaly_cell10["detection_phase"].eq("within_event").sum()
            ),
            "missed_anomalies": int(
                _farm_anomaly_cell10["detection_phase"].eq("missed").sum()
            ),
            "normal_event_false_alarms": int(_farm_normal_cell10["event_alarm"].sum()),
            "normal_case_false_alarm_rate": float(
                _farm_normal_cell10["event_alarm"].mean()
            ),
            "median_normal_raw_alarm_fraction": cell10_finite_statistic(
                _farm_normal_cell10["raw_alarm_fraction"].to_numpy(), "median"
            ),
            "median_anomaly_maximum_criticality": cell10_finite_statistic(
                _farm_anomaly_cell10["maximum_criticality"].to_numpy(), "median"
            ),
            "median_normal_maximum_criticality": cell10_finite_statistic(
                _farm_normal_cell10["maximum_criticality"].to_numpy(), "median"
            ),
        }
    )
FARM_DIAGNOSTIC_AUDIT = pd.DataFrame(_farm_diagnostic_records_cell10).sort_values(
    "farm", kind="stable"
).reset_index(drop=True)

FALSE_ALARM_CASE_AUDIT = _normal_diagnostics_cell10.sort_values(
    ["event_alarm", "maximum_criticality", "raw_alarm_fraction"],
    ascending=[False, False, False],
    kind="stable",
).reset_index(drop=True)
MISSED_ANOMALY_CASE_AUDIT = _anomaly_diagnostics_cell10.loc[
    _anomaly_diagnostics_cell10["detection_phase"].eq("missed")
].sort_values(
    ["maximum_criticality", "raw_alarm_fraction"],
    ascending=[False, False],
    kind="stable",
).reset_index(drop=True)


# =============================================================================
# 5. Normal-case interval tail concentration and active target-scale census
# =============================================================================

NORMAL_CASE_INTERVAL_DIAGNOSTICS = _normal_diagnostics_cell10.sort_values(
    "mean_normalized_interval_width", ascending=False, kind="stable"
).reset_index(drop=True)
_normal_width_means_cell10 = NORMAL_CASE_INTERVAL_DIAGNOSTICS[
    "mean_normalized_interval_width"
].to_numpy(dtype=np.float64)
_finite_normal_width_cell10 = _normal_width_means_cell10[
    np.isfinite(_normal_width_means_cell10)
]
_width_total_cell10 = float(np.sum(_finite_normal_width_cell10))
_sorted_width_cell10 = np.sort(_finite_normal_width_cell10)[::-1]
INTERVAL_CONCENTRATION_SUMMARY = pd.DataFrame(
    [
        {
            "normal_cases": int(len(NORMAL_CASE_INTERVAL_DIAGNOSTICS)),
            "cell9_mean_normalized_width": float(
                POOLED_OUTER_CARE_SUMMARY.iloc[0][
                    "mean_normalized_normal_interval_width"
                ]
            ),
            "reproduced_mean_of_case_means": float(
                np.mean(_finite_normal_width_cell10)
            ),
            "median_case_mean_width": float(np.median(_finite_normal_width_cell10)),
            "p95_case_mean_width": float(
                np.quantile(_finite_normal_width_cell10, 0.95)
            ),
            "maximum_case_mean_width": float(np.max(_finite_normal_width_cell10)),
            "top_1_case_share_of_total_width": float(
                _sorted_width_cell10[:1].sum() / _width_total_cell10
            ),
            "top_5_case_share_of_total_width": float(
                _sorted_width_cell10[:5].sum() / _width_total_cell10
            ),
            "cases_mean_width_above_1e6": int(
                (_finite_normal_width_cell10 > INTERVAL_PATHOLOGY_WIDTH).sum()
            ),
            "cases_mean_width_above_1e12": int(
                (_finite_normal_width_cell10 > EXTREME_INTERVAL_WIDTH).sum()
            ),
        }
    ]
)
if not np.isclose(
    float(INTERVAL_CONCENTRATION_SUMMARY.iloc[0]["cell9_mean_normalized_width"]),
    float(INTERVAL_CONCENTRATION_SUMMARY.iloc[0]["reproduced_mean_of_case_means"]),
    rtol=1.0e-6,
    atol=1.0e-6,
):
    raise RuntimeError("Cell 9 normalized interval width cannot be reproduced.")

_target_scale_records_cell10: list[dict[str, Any]] = []
for _outcome_cell10 in _OUTCOME_LOCKBOX_CELL10.sort_values(
    ["farm", "event_id"], kind="stable"
).itertuples(index=False):
    _case_key_cell10 = str(_outcome_cell10.case_key)
    _selection_cell10 = OUTER_SELECTED_CONFIGURATIONS.loc[
        OUTER_SELECTED_CONFIGURATIONS["outer_asset_id"].eq(
            str(_outcome_cell10.asset_id)
        )
    ]
    if len(_selection_cell10) != 1:
        raise RuntimeError(f"Missing selected configuration for {_case_key_cell10}.")
    _selection_cell10 = _selection_cell10.iloc[0]
    _mean_model_cell10 = load_case_mean_model(_case_key_cell10)
    _health_model_cell10 = load_case_health_model(_case_key_cell10)
    _coverage_index_cell10 = exact_health_grid_index(
        _health_model_cell10["health_coverages"],
        float(_selection_cell10["coverage"]),
        "coverage",
    )
    _r2_index_cell10 = exact_health_grid_index(
        _health_model_cell10["modelability_thresholds"],
        float(_selection_cell10["minimum_r2"]),
        "minimum_r2",
    )
    _head_index_cell10 = _health_model_cell10["metadata"]["evidence_heads"].index(
        str(_selection_cell10["evidence_head"])
    )
    _weights_cell10 = np.asarray(
        _health_model_cell10["sensor_weights"]
        [_coverage_index_cell10, _r2_index_cell10, _head_index_cell10],
        dtype=np.float64,
    )
    _target_scale_cell10 = np.asarray(
        _mean_model_cell10["target_scale"], dtype=np.float64
    )
    _active_cell10 = (
        (_weights_cell10 > 0.0)
        & np.isfinite(_target_scale_cell10)
        & (_target_scale_cell10 > 0.0)
    )
    if not _active_cell10.any():
        raise RuntimeError(f"No active targets for {_case_key_cell10}.")
    _target_scale_records_cell10.append(
        {
            "case_key": _case_key_cell10,
            "farm": str(_outcome_cell10.farm),
            "asset_id": str(_outcome_cell10.asset_id),
            "event_id": int(_outcome_cell10.event_id),
            "is_anomaly": bool(_outcome_cell10.is_anomaly),
            "active_targets": int(_active_cell10.sum()),
            "effective_targets": float(
                1.0 / np.square(_weights_cell10[_active_cell10]).sum()
            ),
            "minimum_active_target_scale": float(
                np.min(_target_scale_cell10[_active_cell10])
            ),
            "median_active_target_scale": float(
                np.median(_target_scale_cell10[_active_cell10])
            ),
            "active_targets_scale_below_1em6": int(
                (_target_scale_cell10[_active_cell10] < TINY_TARGET_SCALE).sum()
            ),
            "active_targets_scale_below_1em9": int(
                (_target_scale_cell10[_active_cell10] < VERY_TINY_TARGET_SCALE).sum()
            ),
            "weight_on_scale_below_1em6": float(
                _weights_cell10[
                    _active_cell10 & (_target_scale_cell10 < TINY_TARGET_SCALE)
                ].sum()
            ),
            "weight_on_scale_below_1em9": float(
                _weights_cell10[
                    _active_cell10 & (_target_scale_cell10 < VERY_TINY_TARGET_SCALE)
                ].sum()
            ),
            "maximum_single_target_weight": float(
                np.max(_weights_cell10[_active_cell10])
            ),
        }
    )
TARGET_SCALE_CASE_CENSUS = pd.DataFrame(_target_scale_records_cell10).sort_values(
    ["minimum_active_target_scale", "farm", "event_id"], kind="stable"
).reset_index(drop=True)


# =============================================================================
# 6. Target-level decomposition of the 12 widest normal cases
# =============================================================================

_diagnostic_case_keys_cell10 = NORMAL_CASE_INTERVAL_DIAGNOSTICS.head(
    TARGET_DIAGNOSTIC_NORMAL_CASES
)["case_key"].tolist()
_target_interval_records_cell10: list[dict[str, Any]] = []

for _case_rank_cell10, _case_key_cell10 in enumerate(
    _diagnostic_case_keys_cell10, start=1
):
    _outcome_cell10 = _OUTCOME_LOCKBOX_CELL10.loc[
        _OUTCOME_LOCKBOX_CELL10["case_key"].eq(_case_key_cell10)
    ].iloc[0]
    _selection_cell10 = OUTER_SELECTED_CONFIGURATIONS.loc[
        OUTER_SELECTED_CONFIGURATIONS["outer_asset_id"].eq(
            str(_outcome_cell10["asset_id"])
        )
    ].iloc[0]
    _selected_cell10 = cell10_selected_arrays(_case_key_cell10)
    _prediction_rows_cell10 = np.asarray(
        _selected_cell10["prediction_row_indices"], dtype=np.int32
    )
    _cache_cell10 = load_case_cache(_case_key_cell10)
    _mean_model_cell10 = load_case_mean_model(_case_key_cell10)
    _health_model_cell10 = load_case_health_model(_case_key_cell10)
    _interval_cell10 = predict_case_conformal_interval(
        _case_key_cell10,
        row_selector=_prediction_rows_cell10,
        coverage=float(_selection_cell10["coverage"]),
        minimum_r2=float(_selection_cell10["minimum_r2"]),
    )
    _coverage_index_cell10 = exact_health_grid_index(
        _health_model_cell10["health_coverages"],
        float(_selection_cell10["coverage"]),
        "coverage",
    )
    _r2_index_cell10 = exact_health_grid_index(
        _health_model_cell10["modelability_thresholds"],
        float(_selection_cell10["minimum_r2"]),
        "minimum_r2",
    )
    _head_index_cell10 = _health_model_cell10["metadata"]["evidence_heads"].index(
        str(_selection_cell10["evidence_head"])
    )
    _weights_cell10 = np.asarray(
        _health_model_cell10["sensor_weights"]
        [_coverage_index_cell10, _r2_index_cell10, _head_index_cell10],
        dtype=np.float64,
    )
    _target_scale_cell10 = np.asarray(
        _mean_model_cell10["target_scale"], dtype=np.float64
    )
    _target_r2_cell10 = np.asarray(
        _mean_model_cell10["target_crossfit_r2"], dtype=np.float64
    )
    _target_temperature_cell10 = np.asarray(
        _mean_model_cell10["target_temperature"], dtype=bool
    )
    _target_names_cell10 = list(_mean_model_cell10["metadata"]["target_names"])
    _targets_cell10 = np.asarray(_cache_cell10["targets"], dtype=np.float64)[
        _prediction_rows_cell10
    ]
    _mean_cell10 = np.asarray(_interval_cell10["mean"], dtype=np.float64)
    _half_width_cell10 = np.asarray(
        _interval_cell10["half_width"], dtype=np.float64
    )
    _total_scale_cell10 = np.asarray(
        _interval_cell10["total_predictive_scale"], dtype=np.float64
    )
    _residual_cell10 = np.abs(_targets_cell10 - _mean_cell10)
    _standardized_width_cell10 = np.full_like(
        _half_width_cell10, np.nan, dtype=np.float64
    )
    _normalized_residual_cell10 = np.full_like(
        _residual_cell10, np.nan, dtype=np.float64
    )
    _valid_scale_cell10 = np.isfinite(_target_scale_cell10) & (
        _target_scale_cell10 > 0.0
    )
    np.divide(
        2.0 * _half_width_cell10,
        _target_scale_cell10[None, :],
        out=_standardized_width_cell10,
        where=_valid_scale_cell10[None, :],
    )
    np.divide(
        _residual_cell10,
        _target_scale_cell10[None, :],
        out=_normalized_residual_cell10,
        where=_valid_scale_cell10[None, :],
    )
    _alpha_cell10 = 1.0 - float(_selection_cell10["coverage"])
    _target_score_cell10 = _standardized_width_cell10 + (
        2.0 / _alpha_cell10
    ) * np.maximum(
        _normalized_residual_cell10 - 0.5 * _standardized_width_cell10,
        0.0,
    )
    _reconstructed_width_cell10 = cell8_weighted_row_mean(
        _standardized_width_cell10, _weights_cell10
    )
    _reconstructed_score_cell10 = cell8_weighted_row_mean(
        _target_score_cell10, _weights_cell10
    )
    _stored_width_cell10 = np.asarray(
        _selected_cell10["normalized_interval_width"], dtype=np.float64
    )
    _stored_score_cell10 = np.asarray(
        _selected_cell10["standardized_interval_score"], dtype=np.float64
    )
    if not np.allclose(
        _reconstructed_width_cell10,
        _stored_width_cell10,
        rtol=1.0e-4,
        atol=1.0e-5,
        equal_nan=True,
    ):
        raise RuntimeError(
            f"Target-level width decomposition drift for {_case_key_cell10}."
        )
    if not np.allclose(
        _reconstructed_score_cell10,
        _stored_score_cell10,
        rtol=1.0e-4,
        atol=1.0e-5,
        equal_nan=True,
    ):
        raise RuntimeError(
            f"Target-level interval-score decomposition drift for {_case_key_cell10}."
        )

    _active_indices_cell10 = np.flatnonzero(
        (_weights_cell10 > 0.0) & _valid_scale_cell10
    )
    for _target_index_cell10 in _active_indices_cell10:
        _target_width_cell10 = _standardized_width_cell10[:, _target_index_cell10]
        _target_total_scale_cell10 = _total_scale_cell10[:, _target_index_cell10]
        _target_score_values_cell10 = _target_score_cell10[:, _target_index_cell10]
        _mean_width_cell10 = cell10_finite_statistic(
            _target_width_cell10, "mean"
        )
        _p99_width_cell10 = cell10_finite_statistic(
            _target_width_cell10, "quantile", 0.99
        )
        _p99_total_scale_cell10 = cell10_finite_statistic(
            _target_total_scale_cell10, "quantile", 0.99
        )
        _scale_value_cell10 = float(_target_scale_cell10[_target_index_cell10])
        if _mean_width_cell10 > INTERVAL_PATHOLOGY_WIDTH:
            if _scale_value_cell10 < TINY_TARGET_SCALE:
                _cause_cell10 = "near_zero_target_scale"
            elif (
                np.isfinite(_p99_total_scale_cell10)
                and _p99_total_scale_cell10 / _scale_value_cell10
                > 0.5 * INTERVAL_PATHOLOGY_WIDTH
            ):
                _cause_cell10 = "predictive_scale_explosion"
            else:
                _cause_cell10 = "combined_or_tail_effect"
        else:
            _cause_cell10 = "not_pathological"
        _target_interval_records_cell10.append(
            {
                "case_width_rank": int(_case_rank_cell10),
                "case_key": _case_key_cell10,
                "farm": str(_outcome_cell10["farm"]),
                "asset_id": str(_outcome_cell10["asset_id"]),
                "event_id": int(_outcome_cell10["event_id"]),
                "target_index": int(_target_index_cell10),
                "target_name": str(_target_names_cell10[_target_index_cell10]),
                "target_temperature": bool(
                    _target_temperature_cell10[_target_index_cell10]
                ),
                "target_weight": float(_weights_cell10[_target_index_cell10]),
                "target_crossfit_r2": float(_target_r2_cell10[_target_index_cell10]),
                "target_scale": _scale_value_cell10,
                "mean_standardized_width": _mean_width_cell10,
                "median_standardized_width": cell10_finite_statistic(
                    _target_width_cell10, "median"
                ),
                "p99_standardized_width": _p99_width_cell10,
                "maximum_standardized_width": cell10_finite_statistic(
                    _target_width_cell10, "max"
                ),
                "p99_total_predictive_scale": _p99_total_scale_cell10,
                "mean_standardized_interval_score": cell10_finite_statistic(
                    _target_score_values_cell10, "mean"
                ),
                "weighted_mean_width_contribution": float(
                    _weights_cell10[_target_index_cell10] * _mean_width_cell10
                ),
                "weighted_mean_score_contribution": float(
                    _weights_cell10[_target_index_cell10]
                    * cell10_finite_statistic(_target_score_values_cell10, "mean")
                ),
                "width_pathology_flag": bool(
                    _mean_width_cell10 > INTERVAL_PATHOLOGY_WIDTH
                ),
                "diagnostic_cause": _cause_cell10,
            }
        )
    del _cache_cell10, _mean_model_cell10, _health_model_cell10, _interval_cell10

TARGET_INTERVAL_PATHOLOGY_DIAGNOSTICS = pd.DataFrame(
    _target_interval_records_cell10
).sort_values(
    ["weighted_mean_width_contribution", "case_width_rank", "target_index"],
    ascending=[False, True, True],
    kind="stable",
).reset_index(drop=True)


# =============================================================================
# 7. Frozen diagnostic decision summary
# =============================================================================

_pooled_false_alarm_rate_cell10 = float(
    _normal_diagnostics_cell10["event_alarm"].mean()
)
_farm_sensitivity_range_cell10 = float(
    FARM_DIAGNOSTIC_AUDIT["event_sensitivity"].max()
    - FARM_DIAGNOSTIC_AUDIT["event_sensitivity"].min()
)
_interval_pathology_detected_cell10 = bool(
    INTERVAL_CONCENTRATION_SUMMARY.iloc[0]["maximum_case_mean_width"]
    > INTERVAL_PATHOLOGY_WIDTH
)
_target_pathology_detected_cell10 = bool(
    TARGET_INTERVAL_PATHOLOGY_DIAGNOSTICS["width_pathology_flag"].any()
)

DIAGNOSTIC_DECISION_SUMMARY = pd.DataFrame(
    [
        {
            "cell9_primary_care_remains_valid": True,
            "cell9_predictions_or_configuration_changed": False,
            "pooled_care_score": float(
                POOLED_OUTER_CARE_SUMMARY.iloc[0]["care_score"]
            ),
            "anomaly_event_sensitivity": float(
                POOLED_OUTER_CARE_SUMMARY.iloc[0]["event_sensitivity"]
            ),
            "normal_case_event_false_alarm_rate": _pooled_false_alarm_rate_cell10,
            "farm_sensitivity_range": _farm_sensitivity_range_cell10,
            "interval_pathology_detected": _interval_pathology_detected_cell10,
            "target_level_pathology_detected": _target_pathology_detected_cell10,
            "interval_coverage_publishable": bool(
                float(
                    POOLED_OUTER_CARE_SUMMARY.iloc[0][
                        "absolute_normal_coverage_error"
                    ]
                )
                <= 0.02
            ),
            "mean_interval_width_publishable_without_qualification": bool(
                not _interval_pathology_detected_cell10
            ),
            "uniform_cross_farm_robustness_supported": bool(
                _farm_sensitivity_range_cell10 <= 0.20
            ),
            "experiment_01_may_be_retuned": False,
            "new_experiment_id_required_for_method_changes": True,
            "new_experiment_recommended": bool(
                _interval_pathology_detected_cell10
                or _pooled_false_alarm_rate_cell10 > 0.20
                or _farm_sensitivity_range_cell10 > 0.20
            ),
        }
    ]
)


# =============================================================================
# 8. Save and lock the post-hoc diagnostic receipt
# =============================================================================

_cell10_component_hashes = {
    "case_diagnostic_audit_sha256": dataframe_sha256(
        CASE_DIAGNOSTIC_AUDIT, ("farm", "event_id")
    ),
    "detection_phase_summary_sha256": dataframe_sha256(
        DETECTION_PHASE_SUMMARY, ("detection_phase",)
    ),
    "farm_diagnostic_audit_sha256": dataframe_sha256(
        FARM_DIAGNOSTIC_AUDIT, ("farm",)
    ),
    "false_alarm_case_audit_sha256": dataframe_sha256(
        FALSE_ALARM_CASE_AUDIT, ("event_alarm", "farm", "event_id")
    ),
    "missed_anomaly_case_audit_sha256": dataframe_sha256(
        MISSED_ANOMALY_CASE_AUDIT, ("farm", "event_id")
    ),
    "normal_case_interval_diagnostics_sha256": dataframe_sha256(
        NORMAL_CASE_INTERVAL_DIAGNOSTICS,
        ("mean_normalized_interval_width", "farm", "event_id"),
    ),
    "interval_concentration_summary_sha256": dataframe_sha256(
        INTERVAL_CONCENTRATION_SUMMARY, ("normal_cases",)
    ),
    "target_scale_case_census_sha256": dataframe_sha256(
        TARGET_SCALE_CASE_CENSUS,
        ("minimum_active_target_scale", "farm", "event_id"),
    ),
    "target_interval_pathology_diagnostics_sha256": dataframe_sha256(
        TARGET_INTERVAL_PATHOLOGY_DIAGNOSTICS,
        ("weighted_mean_width_contribution", "case_width_rank", "target_index"),
    ),
    "diagnostic_decision_summary_sha256": dataframe_sha256(
        DIAGNOSTIC_DECISION_SUMMARY, ("pooled_care_score",)
    ),
}
CELL10_RECEIPT_SHA256 = sha256_json(
    {
        "contract_sha256": CONTRACT_SHA256,
        "outcome_lockbox_sha256": OUTCOME_LOCKBOX_SHA256,
        "cell9_receipt_sha256": CELL9_RECEIPT_SHA256,
        "cell10_implementation_sha256": CELL10_IMPLEMENTATION_SHA256,
        "component_hashes": _cell10_component_hashes,
        "predictions_or_configurations_changed": False,
        "primary_endpoint_changed": False,
        "post_lock_diagnostics": True,
    }
)
CELL10_RECEIPT_PATH = CELL10_QUALITY_ROOT / "cell10_diagnostic_audit_receipt.json"
if CELL10_RECEIPT_PATH.exists():
    _existing_receipt_cell10 = json.loads(
        CELL10_RECEIPT_PATH.read_text(encoding="utf-8")
    )
    if _existing_receipt_cell10.get("cell10_receipt_sha256") != (
        CELL10_RECEIPT_SHA256
    ):
        raise RuntimeError(
            "A different Cell 10 diagnostic receipt exists. Do not overwrite it."
        )
    CELL10_STATE = "existing identical Cell 10 diagnostic receipt verified"
    _write_cell10_receipt = False
else:
    CELL10_STATE = "new post-lock Cell 10 diagnostic receipt frozen"
    _write_cell10_receipt = True

_cell10_tables = {
    "case_diagnostic_audit.csv": CASE_DIAGNOSTIC_AUDIT,
    "detection_phase_summary.csv": DETECTION_PHASE_SUMMARY,
    "farm_diagnostic_audit.csv": FARM_DIAGNOSTIC_AUDIT,
    "false_alarm_case_audit.csv": FALSE_ALARM_CASE_AUDIT,
    "missed_anomaly_case_audit.csv": MISSED_ANOMALY_CASE_AUDIT,
    "normal_case_interval_diagnostics.csv": NORMAL_CASE_INTERVAL_DIAGNOSTICS,
    "interval_concentration_summary.csv": INTERVAL_CONCENTRATION_SUMMARY,
    "target_scale_case_census.csv": TARGET_SCALE_CASE_CENSUS,
    "target_interval_pathology_diagnostics.csv": (
        TARGET_INTERVAL_PATHOLOGY_DIAGNOSTICS
    ),
    "diagnostic_decision_summary.csv": DIAGNOSTIC_DECISION_SUMMARY,
}
for _filename_cell10, _table_cell10 in _cell10_tables.items():
    save_csv_atomic(_table_cell10, CELL10_QUALITY_ROOT / _filename_cell10)
save_json(
    CELL10_IMPLEMENTATION,
    CELL10_QUALITY_ROOT / "cell10_implementation.json",
)

if _write_cell10_receipt:
    save_json(
        {
            "cell10_version": CELL10_VERSION,
            "created_at_utc": utc_now(),
            "contract_sha256": CONTRACT_SHA256,
            "dataset_manifest_sha256": DATASET_MANIFEST_SHA256,
            "outcome_lockbox_sha256": OUTCOME_LOCKBOX_SHA256,
            "cell9_implementation_sha256": CELL9_IMPLEMENTATION_SHA256,
            "cell9_receipt_sha256": CELL9_RECEIPT_SHA256,
            "cell10_implementation_sha256": CELL10_IMPLEMENTATION_SHA256,
            "component_hashes": _cell10_component_hashes,
            "cell10_receipt_sha256": CELL10_RECEIPT_SHA256,
            "cell9_endpoint_verified_before_diagnostics": True,
            "outcomes_opened_after_cell9_verification": True,
            "predictions_or_configurations_changed": False,
            "primary_endpoint_changed": False,
            "post_lock_diagnostics": True,
        },
        CELL10_RECEIPT_PATH,
    )


# =============================================================================
# 9. Concise notebook report
# =============================================================================

print("\n" + "=" * 92)
print("UC-RCF-NBM CELL 10 — POST-LOCK DIAGNOSTIC AUDIT")
print("=" * 92)
print("\nDETECTION TIMING RELATIVE TO EVENT ONSET")
display(DETECTION_PHASE_SUMMARY)
print("\nFARM HETEROGENEITY AND EVENT-LEVEL FALSE ALARMS")
display(FARM_DIAGNOSTIC_AUDIT)
print("\nNORMAL-CASE INTERVAL WIDTH CONCENTRATION")
display(INTERVAL_CONCENTRATION_SUMMARY)
print("\nWIDEST NORMAL CASES")
display(
    NORMAL_CASE_INTERVAL_DIAGNOSTICS.head(15).loc[
        :,
        [
            "case_key",
            "farm",
            "asset_id",
            "event_id",
            "event_alarm",
            "mean_normalized_interval_width",
            "median_normalized_interval_width",
            "p99_normalized_interval_width",
            "maximum_normalized_interval_width",
            "mean_standardized_interval_score",
        ],
    ]
)
print("\nTOP TARGET-LEVEL CONTRIBUTORS TO INTERVAL PATHOLOGY")
display(
    TARGET_INTERVAL_PATHOLOGY_DIAGNOSTICS.head(30).loc[
        :,
        [
            "case_width_rank",
            "case_key",
            "target_name",
            "target_weight",
            "target_crossfit_r2",
            "target_scale",
            "mean_standardized_width",
            "p99_total_predictive_scale",
            "weighted_mean_width_contribution",
            "diagnostic_cause",
        ],
    ]
)
print("\nNORMAL FALSE-ALARM CASES WITH HIGHEST CRITICALITY")
display(
    FALSE_ALARM_CASE_AUDIT.head(20).loc[
        :,
        [
            "case_key",
            "farm",
            "asset_id",
            "event_id",
            "event_alarm",
            "maximum_criticality",
            "raw_alarm_fraction",
            "longest_raw_alarm_run_steps",
        ],
    ]
)
print("\nMISSED ANOMALIES NEAREST TO THE CRITICALITY THRESHOLD")
display(
    MISSED_ANOMALY_CASE_AUDIT.head(20).loc[
        :,
        [
            "case_key",
            "farm",
            "asset_id",
            "event_id",
            "maximum_criticality",
            "criticality_margin_to_alarm",
            "raw_alarm_fraction",
            "within_event_raw_alarm_rows",
        ],
    ]
)
print("\nPOST-LOCK SCIENTIFIC DECISION")
display(DIAGNOSTIC_DECISION_SUMMARY)

_decision_cell10 = DIAGNOSTIC_DECISION_SUMMARY.iloc[0]
print("\n" + "-" * 76)
print(f"Cell 9 endpoint verified first        : {CELL9_ENDPOINT_VERIFIED_BEFORE_DIAGNOSTICS}")
print(f"Cases audited                         : {len(CASE_DIAGNOSTIC_AUDIT)}")
print(
    "Pre-onset / within-event / missed   : "
    f"{int(DETECTION_PHASE_SUMMARY.iloc[0]['anomaly_cases'])} / "
    f"{int(DETECTION_PHASE_SUMMARY.iloc[1]['anomaly_cases'])} / "
    f"{int(DETECTION_PHASE_SUMMARY.iloc[2]['anomaly_cases'])}"
)
print(
    "Normal case event false-alarm rate  : "
    f"{float(_decision_cell10['normal_case_event_false_alarm_rate']):.4f}"
)
print(
    "Farm sensitivity range              : "
    f"{float(_decision_cell10['farm_sensitivity_range']):.4f}"
)
print(
    "Interval pathology detected         : "
    f"{bool(_decision_cell10['interval_pathology_detected'])}"
)
print(
    "New experiment recommended          : "
    f"{bool(_decision_cell10['new_experiment_recommended'])}"
)
print("Experiment 01 retuning allowed        : No")
print("Cell 9 primary endpoint changed       : No")
print(f"Cell 10 implementation SHA-256        : {CELL10_IMPLEMENTATION_SHA256}")
print(f"Cell 10 receipt SHA-256               : {CELL10_RECEIPT_SHA256}")
print(f"Cell 10 state                         : {CELL10_STATE}")
print("=" * 92)
print("CELL 10 COMPLETED SUCCESSFULLY — POST-LOCK DIAGNOSTIC AUDIT FROZEN")

del _OUTCOME_LOCKBOX_CELL10



UC-RCF-NBM CELL 10 — POST-LOCK DIAGNOSTIC AUDIT

DETECTION TIMING RELATIVE TO EVENT ONSET


,detection_phase,anomaly_cases,fraction_of_anomaly_cases,median_hours_from_event_onset,median_hours_to_event_end
0,pre_onset,3,0.066667,-35.833333,442.00
1,within_event,22,0.488889,71.416667,400.75
2,missed,20,0.444444,NaN,NaN



FARM HETEROGENEITY AND EVENT-LEVEL FALSE ALARMS


,farm,cases,anomaly_cases,normal_cases,care_score,event_sensitivity,pre_onset_detections,within_event_detections,missed_anomalies,normal_event_false_alarms,normal_case_false_alarm_rate,median_normal_raw_alarm_fraction,median_anomaly_maximum_criticality,median_normal_maximum_criticality
0,Wind Farm A,22,12,10,0.506876,0.166667,0,2,10,1,0.100000,0.024877,18.5,24.0
1,Wind Farm B,15,6,9,0.617884,1.000000,0,6,0,5,0.555556,0.073960,193.0,105.0
2,Wind Farm C,58,27,31,0.601914,0.629630,3,14,10,16,0.516129,0.085302,135.0,78.0



NORMAL-CASE INTERVAL WIDTH CONCENTRATION


...


MISSED ANOMALIES NEAREST TO THE CRITICALITY THRESHOLD


,case_key,farm,asset_id,event_id,maximum_criticality,criticality_margin_to_alarm,raw_alarm_fraction,within_event_raw_alarm_rows
0,Wind Farm C::event_12,Wind Farm C,Wind Farm C::2,12,65,-7,0.034952,108
1,Wind Farm A::event_73,Wind Farm A,Wind Farm A::0,73,59,-13,0.076345,84
2,Wind Farm C::event_76,Wind Farm C,Wind Farm C::53,76,57,-15,0.214072,114
3,Wind Farm C::event_49,Wind Farm C,Wind Farm C::33,49,55,-17,0.088022,97
4,Wind Farm C::event_78,Wind Farm C,Wind Farm C::15,78,50,-22,0.213849,57
5,Wind Farm A::event_42,Wind Farm A,Wind Farm A::10,42,39,-33,0.024637,39
6,Wind Farm A::event_0,Wind Farm A,Wind Farm A::0,0,32,-40,0.027132,45
7,Wind Farm C::event_4,Wind Farm C,Wind Farm C::34,4,29,-43,0.008213,29
8,Wind Farm A::event_22,Wind Farm A,Wind Farm A::21,22,24,-48,0.020906,24
9,Wind Farm C::event_16,Wind Farm C,Wind Farm C::53,16,14,-58,0.042697,19



POST-LOCK SCIENTIFIC DECISION


,cell9_primary_care_remains_valid,cell9_predictions_or_configuration_changed,pooled_care_score,anomaly_event_sensitivity,normal_case_event_false_alarm_rate,farm_sensitivity_range,interval_pathology_detected,target_level_pathology_detected,interval_coverage_publishable,mean_interval_width_publishable_without_qualification,uniform_cross_farm_robustness_supported,experiment_01_may_be_retuned,new_experiment_id_required_for_method_changes,new_experiment_recommended
0,True,False,0.581494,0.555556,0.44,0.833333,True,True,True,False,False,False,True,True



----------------------------------------------------------------------------
Cell 9 endpoint verified first        : True
Cases audited                         : 95
Pre-onset / within-event / missed   : 3 / 22 / 20
Normal case event false-alarm rate  : 0.4400
Farm sensitivity range              : 0.8333
Interval pathology detected         : True
New experiment recommended          : True
Experiment 01 retuning allowed        : No
Cell 9 primary endpoint changed       : No
Cell 10 implementation SHA-256        : 52a40f31b7ef4a85e69cd271a2664ac18e65f1773334034894ac80d54959a415
Cell 10 receipt SHA-256               : a5601127c4e101fc78d56e0b2c5527eb0628c4aa32b24afd51332f42c285f288
Cell 10 state                         : new post-lock Cell 10 diagnostic receipt frozen
CELL 10 COMPLETED SUCCESSFULLY — POST-LOCK DIAGNOSTIC AUDIT FROZEN
